In [ ]:
# Issue here is that data has only I50 ICD
# Contains DiD analysis

In [ ]:
# here

# Analysis of Patient Visit Patterns: Methodology

This document outlines the methodology for processing and analyzing patient encounter data to understand the relationship between Emergency and Primary Care visits.

## 1. Data Loading and Initial Cleaning

### 1.1. Load Dataset
First, we load the filtered and reordered dataset (`final_filtered_dataset_reordered.csv`). We explicitly define the `dtype` for each column during import. This is crucial for memory efficiency and to prevent incorrect type inference (e.g., ZIP codes being read as numbers). The `Date` column is parsed as a datetime object.

### 1.2. Filter "Other" ICD Codes
The dataset contains a "Other" category for `ICD_1char`. This category is non-specific and provides no analytical value. We remove all rows corresponding to this category to focus our analysis on specific, known diagnoses.

### 1.3. Initial Inspection
After loading and filtering, we will inspect:
1.  The data types (`.dtypes`) to confirm they loaded correctly.
2.  The new shape of the DataFrame (`.shape`) to see how many rows were removed.
3.  The top 10 most frequent ICD codes (`.value_counts()`) to verify "Other" is gone.
4.  The aggregated `EncounterCount` by ICD code to understand the most common diagnoses in our filtered data.

---

## 2. Filtering by Top Diagnoses

To focus our analysis on the most significant diagnoses and reduce noise from rare events, we will subset the data. Our strategy is to identify the most frequent ICD codes in both "Emergency" and "Regular" (non-Emergency) settings.

1.  We generate a list of the **top 50** most frequent `ICD_1char` (by summed `EncounterCount`) for `VisitType == "Emergency"`.
2.  We generate a separate list for the **top 50** codes for `VisitType != "Emergency"`.
3.  We create a combined `set` of these lists. Using a set automatically handles any duplicates (i.e., codes that are in the top 50 for both groups).
4.  We filter the main DataFrame to keep only the rows where `ICD_1char` is in this final combined set.

---

## 3. Data Sanitization

This is a critical data hygiene step. We need to ensure our categorical keys, especially ZIP codes, are standardized.

1.  **Enforce String Type**: We loop through all `object` columns and explicitly cast them to `str` to avoid mixed-type errors.
2.  **Clean ZIP Codes**: We identify all ZIP code-related columns. We then use a regular expression (`\.0$`) to find and remove any trailing `.0` suffixes. This is a common artifact from numeric-to-string conversion (e.g., `60601.0` becomes `60601`).
3.  **Final Check**: We perform one last check for any clearly invalid or suspicious ZIP code values (like "0" or "0.0") that might remain.

---

## 4. Aggregation and Final Cleaning

Our data is currently at the individual record level. To prepare it for panel regression, we must aggregate it to our chosen unit of analysis.

### 4.1. Define Key Columns and Drop NaNs
We define a list of `key_columns` that constitute a unique observation for our purposes (e.g., ZIP, Date, ICD, Department, etc.). We cannot use any row that is missing information in one of these keys, so we drop all rows with `NaN` values in this subset of columns.

### 4.2. Aggregate by Key
We group the data by these `key_columns` and aggregate the `EncounterCount` using `.sum()`. This creates our final analytical dataset, where each row represents the total number of encounters for that unique combination of characteristics.

### 4.3. Filter Invalid ZIP Codes
Finally, we remove any remaining records that have known invalid patient ZIP codes (e.g., "99999", "Other", "0", "00000"), as these do not represent real geographic areas.

### 4.4. Inspect Final DataFrame
We check the `.info()` of our `final_data` to ensure all dtypes are correct and there are no more null values.

---

## 5. Panel Data Preparation & Feature Engineering

With the data aggregated, we now engineer the specific features required for our panel regression models.

### 5.1. Set Panel Index
We create a copy of our `final_data` and set a `MultiIndex` of `(Filtered_Patient_ZipCode, Date)`. This format (Entity, Time) is required by the `linearmodels` library.

### 5.2. Create Dependent Variables
We "pivot" the `VisitType` column to create two separate, 0-filled columns:
* `Emergency_Visits`
* `Primary_Care_Visits` (from the "Regular" `VisitType`)

### 5.3. Create Lagged Independent Variables
The core of our model is to see how past visits affect current visits. We create lagged variables by grouping by `(Filtered_Patient_ZipCode, ICD_1char)` and using `.shift(1)`.
* `Lag_Emergency`: Emergency visits from the previous period (t-1).
* `Lag_Primary`: Primary care visits from the previous period (t-1).

### 5.4. Logarithmic Transformation
Visit count data is often highly skewed (many zeros, long tail). We apply a log-transform `np.log(x + 1)` to our dependent and key independent variables. This helps normalize the data and allows us to interpret the resulting coefficients as semi-elasticities.

### 5.5. Final Data for Model
Finally, we drop any rows containing `NaN` values, which are created by the `shift()` operation (i.e., the first observation for every group has no lag).

---

## 6. Panel Regression Analysis (Fixed Effects)

We will estimate two models to understand the relationship between Emergency and Primary Care visits, controlling for past behavior. We use a **Panel OLS with Fixed Effects** (`PanelOLS` with `entity_effects=True`). The "entity" is the `Filtered_Patient_ZipCode`, so this specification controls for all time-invariant, unobserved heterogeneity at the ZIP code level.

### 6.1. Define Control Variables
Before modeling, we must create dummy variables for our categorical controls:
* `FinancialClassNM` (Payer)
* `ICD_1char` (Diagnosis)

We also include `Year` and `Month` as time controls.

### 6.2. Model 1: Determinants of Emergency Visits
Our first model specifies `Log_Emergency_Visits` as the dependent variable.

**Specification:**
$Log\_Emergency_{it} = \beta_0 + \beta_1 Log\_Emergency\_Lag1_{it} + \beta_2 Log\_Primary\_Lag1_{it} + \gamma X_{it} + \alpha_i + \epsilon_{it}$
* $\alpha_i$ represents the ZIP code fixed effects.
* $X_{it}$ is our vector of controls (Time, Payer, ICD).
* We are most interested in $ \beta_2 $, which shows the cross-effect of past primary care visits on current emergency visits.

### 6.3. Model 2: Determinants of Primary Care Visits
Our second model is symmetric, specifying `Log_Primary_Care_Visits` as the dependent variable.

**Specification:**
$Log\_Primary_{it} = \beta_0 + \beta_1 Log\_Primary\_Lag1_{it} + \beta_2 Log\_Emergency\_Lag1_{it} + \gamma X_{it} + \alpha_i + \epsilon_{it}$
* Here, $\beta_2$ shows the effect of past emergency visits on current primary care utilization.

---

## 7. Interpretation of Results

* **Model 1 (`Log_Emergency_Visits`):**
    * `Log_Emergency_Lag1`: This shows persistence. A positive, significant coefficient suggests that ZIP codes with high emergency visits in one period tend to have high emergency visits in the next.
    * `Log_Primary_Lag1`: This is the key substitution/complementarity coefficient.
        * A **negative** coefficient would suggest **substitution**: more primary care visits in the past lead to *fewer* emergency visits now (e.g., better preventive care).
        * A **positive** coefficient would suggest **complementarity**: more primary care visits lead to *more* emergency visits (e.g., primary care acts as a referral system).

* **Model 2 (`Log_Primary_Care_Visits`):**
    * `Log_Primary_Lag1`: Shows persistence in primary care visits.
    * `Log_Emergency_Lag1`: The other key cross-effect.
        * A **positive** coefficient would suggest follow-up: an emergency visit leads to a scheduled primary care follow-up visit.
        * A **negative** coefficient is less likely but could imply that an emergency visit "solves" the issue, reducing the need for primary care.

* **Goodness-of-Fit:**
    * **R-squared (Within):** This is the most important R-squared for a Fixed Effects model. It tells us how much of the variation *within* each ZIP code over time is explained by our model.

---

## 8. Descriptive Statistics for Model Data

As a final check, we present descriptive statistics for the final dataset used in the regressions (`df_model_data`). This table shows the number of unique ZIP codes, unique dates, and total observations (rows) available for each `ICD_1char` *after* all filtering, lagging, and `dropna` steps. This helps us understand the data density for each diagnosis.

In [ ]:
import pandas as pd
import numpy as np
import warnings
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from IPython.display import display

# -------------------------------------------------------------------
# --- Step 1: Data Loading
# -------------------------------------------------------------------
# (Этот шаг остается почти без изменений, только убираем 'грязный' код)
print("Step 1: Loading and Initial Cleaning (v3 Logic)...")

filtered_data = pd.read_csv(
    "final_filtered_dataset_reordered.csv",
    parse_dates=["Date"],
    dtype={
        "Filtered_Patient_ZipCode": str, "Month": "int64", "Year": "int64",
        "Distance_km": "float64", 
        "ICD_1char": str,               # <--- НОВЫЙ
        "ICD_Group": str,               # <--- НОВЫЙ
        "LocationShortNM": str,
        "Department": str, "Department_ZipCode": str, "ICDDiagnosisCD": str,
        "ICDDiagnosisDSC": str, "FinancialClassNM": str, "PedsFLG": "int64",
        "EncounterCount": "int64", "VisitType": str, "EDPrimaryClinicalImpressionCD": str,
        "EDPrimaryClinicalImpressionDSC": str, "zcta5": str
    },
    low_memory=False
)
print(f"Initial shape: {filtered_data.shape}")

# -------------------------------------------------------------------
# --- Step 2: (НОВЫЙ) Creating Clinical Groups
# -------------------------------------------------------------------
#
# ❗️ ВАЖНО: Это заменяет "грязный" Step 2 (Top-50).
# Мы создаем клинически осмысленные группы.
#
# -------------------------------------------------------------------
print("\nStep 2: Creating Clinical Groups (v3 Logic)...")

# 1. Определяем условия для каждой группы
conditions = [
    (filtered_data['ICD_1char'] == 'R'),
    (filtered_data['ICD_1char'].isin(['I', 'E'])),
    (filtered_data['ICD_1char'].isin(['S', 'T'])),
    (filtered_data['ICD_1char'] == 'Z'),
    (filtered_data['ICD_1char'].isin(['0','1','2','3','4','5','6','7','8','9'])),
    (filtered_data['ICD_1char'] == 'Unknown'),
    (filtered_data['ICD_1char'].isin(['V', 'W', 'X', 'Y'])) # Внешние причины, часто идут с Травмами
]

# 2. Определяем названия групп
choices = [
    'Symptoms',        # R
    'Chronic',         # I, E
    'Trauma',          # S, T
    'Prevention',      # Z
    'Legacy_Numeric',  # 0-9 (старые коды или ошибки)
    'Unknown',         # 'Unknown'
    'External_Causes'  # V, W, X, Y
]

# 3. Создаем столбец. Все, что не попало, будет 'Other_Clinical'
# (Сюда войдут J, M, F, G, K, L, A, B, C и т.д.)
filtered_data['ICD_Group'] = np.select(conditions, choices, default='Other_Clinical')

print("\n✅ 'ICD_Group' создан. Распределение по группам:")
print(filtered_data['ICD_Group'].value_counts())


# -------------------------------------------------------------------
# --- Step 3: Data Sanitization (ZIP Codes)
# -------------------------------------------------------------------
# (Этот раздел остается без изменений)
print("\nStep 3: Sanitizing Data (ZIP Codes)...")
warnings.simplefilter(action='ignore', category=UserWarning)
pd.options.mode.chained_assignment = None
for col in filtered_data.select_dtypes(include="object").columns:
    if col != 'ICD_Group': # 'ICD_Group' уже правильный
        filtered_data[col] = filtered_data[col].astype(str)

zip_columns = ["Filtered_Patient_ZipCode", "Department_ZipCode", "zcta5"]
for col in zip_columns:
    filtered_data[col] = filtered_data[col].str.replace(r"\.0$", "", regex=True)

zip_series = filtered_data["Filtered_Patient_ZipCode"].astype(str)
suspicious_zips = zip_series[zip_series.str.fullmatch(r"0|0\.0|.*\.0")].unique()
print("🚨 Suspicious values found in Filtered_Patient_ZipCode (should be empty):")
print(suspicious_zips)


# -------------------------------------------------------------------
# --- Step 4: Final Aggregation and Cleaning
# -------------------------------------------------------------------
print("\nStep 4: Aggregating and Final Cleaning (v3 Logic)...")

# 🔹 ИЗМЕНЕНИЕ: Добавляем 'ICD_Group' в список ключей
key_columns = [
    "Filtered_Patient_ZipCode", "Year", "Month", "Date", 
    "ICD_1char",
    "ICD_Group",                # <--- НОВЫЙ КЛЮЧ
    "LocationShortNM", "Department", "Department_ZipCode", 
    "FinancialClassNM", "VisitType", "zcta5"
]

# (Эта часть остается без изменений)
filtered_data = filtered_data.dropna(subset=key_columns)
print(f"\n✅ Shape after dropping NaNs in key columns: {filtered_data.shape}")

# Агрегируем по НОВЫМ key_columns
aggregated_data = (
    filtered_data
    .groupby(key_columns, as_index=False)
    .agg({"EncounterCount": "sum"})
)
print(f"\n📊 Aggregated data shape: {aggregated_data.shape}")

# (Эта часть остается без изменений)
final_data = aggregated_data.copy()
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[
    (~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips))
]
print(f"\n✅ Shape after filtering invalid ZIPs: {final_data.shape}")
print("\n🧾 Info for final aggregated data:")
final_data.info()

print("\n🎉--- Скрипт успешно 'очищен' ---🎉")
print("Данные 'final_data' готовы для DiD-анализа.")
print("Теперь вы можете группировать или фильтровать по 'ICD_Group' (Symptoms, Chronic и т.д.)")


In [ ]:

# -------------------------------------------------------------------
# --- Step 5: Panel Data Preparation (Feature Engineering)
# -------------------------------------------------------------------
# (Этот раздел остается без изменений)
print("\nStep 5: Preparing Panel Data & Engineering Features...")
df_panel = final_data.copy()
df_panel = df_panel.set_index(["Filtered_Patient_ZipCode", "Date"])
df_panel["EncounterCount"] = pd.to_numeric(df_panel["EncounterCount"], errors="coerce")
df_panel["Emergency_Visits"] = np.where(df_panel["VisitType"] == "Emergency", df_panel["EncounterCount"], 0)
df_panel["Primary_Care_Visits"] = np.where(df_panel["VisitType"] == "Regular", df_panel["EncounterCount"], 0)
df_panel["Lag_Emergency"] = df_panel.groupby(["Filtered_Patient_ZipCode", "ICD_1char"])["Emergency_Visits"].shift(1)
df_panel["Lag_Primary"] = df_panel.groupby(["Filtered_Patient_ZipCode", "ICD_1char"])["Primary_Care_Visits"].shift(1)
df_panel["Log_Emergency_Visits"] = np.log(df_panel["Emergency_Visits"] + 1)
df_panel["Log_Primary_Care_Visits"] = np.log(df_panel["Primary_Care_Visits"] + 1)
df_panel["Log_Emergency_Lag1"] = np.log(df_panel["Lag_Emergency"] + 1)
df_panel["Log_Primary_Lag1"] = np.log(df_panel["Lag_Primary"] + 1)
df_model_data = df_panel.dropna(subset=["Log_Emergency_Lag1", "Log_Primary_Lag1"])
print(f"\n✅ Final model data shape after creating lags: {df_model_data.shape}")
print(df_model_data[['Log_Emergency_Visits', 'Log_Primary_Care_Visits', 'Log_Emergency_Lag1', 'Log_Primary_Lag1']].describe())


# -------------------------------------------------------------------
# --- Step 6: Panel Regression Models
# -------------------------------------------------------------------

print("\nStep 6: Running Panel Regression Models...")

# ❗ ИЗМЕНЕНИЕ 1: Сохраняем копию данных для Шага 7 *до* создания dummy-переменных
# Мы сбрасываем индекс, чтобы получить Filtered_Patient_ZipCode и Date как колонки
df_model_data_for_stats = df_model_data.reset_index()


# 1. Создаем dummy-переменные для регрессии
# ❗ ИЗМЕНЕНИЕ 2: Используем новое имя переменной для данных регрессии
df_model_data_regression = pd.get_dummies(df_model_data, columns=["FinancialClassNM", "ICD_1char"], drop_first=True)

# 2. Определяем списки ковариат
covariates = ["Year", "Month"]
# ❗ ИЗМЕНЕНИЕ 3: Убедимся, что ищем колонки в правильном DataFrame (df_model_data_regression)
payer_vars = [col for col in df_model_data_regression.columns if col.startswith("FinancialClassNM_")]
icd_vars = [col for col in df_model_data_regression.columns if col.startswith("ICD_1char_")]

# --- Model 1: Emergency Visits ---
print("\n--- Running Model 1: Log(Emergency Visits) ---")
y1 = df_model_data_regression["Log_Emergency_Visits"]
X1 = df_model_data_regression[["Log_Emergency_Lag1", "Log_Primary_Lag1"] + covariates + payer_vars + icd_vars]
X1 = sm.add_constant(X1, prepend=False)

model1 = PanelOLS(y1, X1, entity_effects=True, drop_absorbed=True)
results1 = model1.fit(cov_type='clustered', cluster_entity=True)

print(results1.summary)


# --- Model 2: Primary Care Visits ---
print("\n--- Running Model 2: Log(Primary Care Visits) ---")
y2 = df_model_data_regression["Log_Primary_Care_Visits"]
X2 = df_model_data_regression[["Log_Primary_Lag1", "Log_Emergency_Lag1"] + covariates + payer_vars + icd_vars]
X2 = sm.add_constant(X2, prepend=False)

model2 = PanelOLS(y2, X2, entity_effects=True, drop_absorbed=True)
results2 = model2.fit(cov_type='clustered', cluster_entity=True)

print(results2.summary)


# --- Save results to a file ---
print("\n💾 Saving regression results to 'panel_regression_results.txt'")
with open("panel_regression_results.txt", "w") as f:
    f.write("==================================================\n")
    f.write("          MODEL 1: Log(Emergency Visits)          \n")
    f.write("==================================================\n")
    f.write(results1.summary.as_text())
    f.write("\n\n\n")
    f.write("==================================================\n")
    f.write("        MODEL 2: Log(Primary Care Visits)         \n")
    f.write("==================================================\n")
    f.write(results2.summary.as_text())

print("\nRegression analysis complete.")

# -------------------------------------------------------------------
# --- Step 7: Descriptive Statistics of Model Data
# -------------------------------------------------------------------

print("\nStep 7: Generating Descriptive Statistics for Model Data...")

# ❗ ИЗМЕНЕНИЕ 4: Используем DataFrame, который мы сохранили ранее (df_model_data_for_stats)
# В нем все еще есть оригинальные колонки
df_stats_check = df_model_data_for_stats

# Group by ICD and get stats
icd_stats = df_stats_check.groupby("ICD_1char").agg(
    unique_zip_codes=("Filtered_Patient_ZipCode", "nunique"),
    unique_dates=("Date", "nunique"),
    total_observations=("ICD_1char", "count")
).reset_index().sort_values(by="total_observations", ascending=False)

# Display the table
print("\n📊 Descriptive Statistics of Final Model Data by ICD Code:")
# Note: In a pure .py script, `display` might not work.
# Use `print` if running outside Jupyter/IPython.
try:
    display(icd_stats)
except NameError:
    print(icd_stats)

print("\n--- Analysis Script Finished ---")

In [ ]:
import pandas as pd
import numpy as np

# --- 🚀 STARTING Step 0: Pre-processing Clinic File (MODIFIED) ---

# Define filenames
# This is your original file, as named in your script
original_clinic_file = "OSFResearch_Clinics_mastersheet_v1.csv" 
crosswalk_file = "Crosswalk_IL.csv"
new_clean_clinic_file = "OSFResearch_Clinics_mastersheet_v2.csv"

print(f"Loading '{original_clinic_file}' and '{crosswalk_file}'...")

try:
    # 1. Load files (with encoding fix for clinic file)
    df_clinics_info = pd.read_csv(original_clinic_file, encoding='windows-1252')
    df_crosswalk = pd.read_csv(crosswalk_file)
    print("✅ Original clinic and crosswalk files loaded.")

    # 2. Clean Clinic Column Names
    # --- THIS STEP IS NOW REMOVED ---
    # We are keeping the original column names (like 'Zip', 'New or Acquired')
    # so your other analysis scripts will work.
    print(f"✅ Step 2: Skipping column name cleaning to match your analysis scripts.")

    # 3. Prepare Crosswalk Coordinates
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    print(f"✅ Crosswalk prepared with {len(zip_coords)} unique ZIP coordinates.")

    # 4. Clean clinic 'zip' column
    # --- MODIFIED: Use the original 'Zip' column (capital Z) ---
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)

    # 5. Filter Clinics: Keep only those with coordinates
    original_count = len(df_clinics_info)
    
    # --- MODIFIED: Merge on the original 'Zip' column (capital Z) ---
    df_clinics_info_filtered = df_clinics_info.merge(
        zip_coords, 
        left_on='Zip',  # <-- Use original 'Zip'
        right_index=True, 
        how='inner'
    )
    # We drop the coordinates because your main scripts add them back in
    df_clinics_info_filtered = df_clinics_info_filtered.drop(columns=['IntPtLat', 'IntPtLon'])
    
    print(f"✅ Filtered clinics: Kept {len(df_clinics_info_filtered)} of {original_count} clinics.")

    # 6. Save the new, clean, filtered file
    df_clinics_info_filtered.to_csv(new_clean_clinic_file, index=False)
    
    print(f"\n✅🎉 Success! Saved new filtered file as '{new_clean_clinic_file}'.")
    print("This file now contains only clinics with coordinates and keeps original column names.")
    print("You can now run your main analysis code.")

except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}.")
    print(f"Make sure '{original_clinic_file}' and '{crosswalk_file}' are in the correct folder.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

print("--- END Step 0 ---")

In [ ]:
import pandas as pd
import numpy as np

# --- 0. Настройка вывода Pandas ---
pd.set_option('display.max_rows', None) 
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("--- 🚀 НАЧАЛО АУДИТА СВЯЗИ ДАННЫХ (СПРОС vs КЛИНИКИ) ---")

# --- Шаг 1: Анализ данных о визитах (Demand-Side Summary) ---
print("\n" + "="*50)
print("--- Шаг 1: Анализ данных о визитах (Demand-Side Summary) ---")
print("="*50)

if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден в памяти.")
else:
    print(f"✅ 'final_data' найден. {len(final_data)} строк в исходных данных.")
    
    try:
        visits_summary = final_data.groupby(
            ['LocationShortNM', 'Department', 'Department_ZipCode'], 
            as_index=False
        ).agg(
            Total_Encounters=('EncounterCount', 'sum'),
            Min_Date=('Date', 'min'),
            Max_Date=('Date', 'max'),
            Unique_Patient_ZIPs=('Filtered_Patient_ZipCode', 'nunique')
        )
        
        visits_summary = visits_summary.sort_values(by='Total_Encounters', ascending=False)
        
        print("\n📊 Сводка по визитам (visits_summary) - ВСЕ строки:")
        try:
            display(visits_summary)
        except NameError:
            print(visits_summary)
            
        # --- Сохранение 1 ---
        visits_summary_file = 'full_visits_summary.csv'
        visits_summary.to_csv(visits_summary_file, index=False, encoding='utf-8-sig')
        print(f"\n💾 ✅ Файл '{visits_summary_file}' успешно сохранен.")

    except KeyError as e:
        print(f"❌ ОШИБКА: Не найдена колонка {e}.")
    except Exception as e:
        print(f"❌ Произошла ошибка на Шаге 1: {e}")

    # --- Шаг 2: Подготовка данных о клиниках (Treatment-Side ZIPs) ---
    print("\n" + "="*50)
    print("--- Шаг 2: Подготовка данных о клиниках (Treatment-Side ZIPs) ---")
    print("="*50)
    
    clinic_file = 'OSFResearch_Clinics_mastersheet_v1.csv'
    try:
        df_clinics = pd.read_csv(clinic_file, encoding='windows-1252')
        print(f"✅ Файл '{clinic_file}' успешно загружен. {len(df_clinics)} строк.")
        
        if 'Zip' not in df_clinics.columns:
            print(f"❌ ОШИБКА: Колонка 'Zip' не найдена в файле '{clinic_file}'.")
            clinic_zips_set = set()
        else:
            df_clinics['Zip'] = df_clinics['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
            clinic_zips_set = set(df_clinics['Zip'].dropna().unique())
            print(f"📊 Найдено {len(clinic_zips_set)} уникальных ZIP-кодов в мастер-файле клиник.")

    except FileNotFoundError:
        print(f"❌ ОШИБКА: Файл '{clinic_file}' не найден.")
        clinic_zips_set = set()
    except Exception as e:
        print(f"❌ Произошла ошибка на Шаге 2: {e}")
        clinic_zips_set = set()

    # --- Шаг 3: Найти совпадения по ZIP-кодам (The "Matchmaking") ---
    print("\n" + "="*50)
    print("--- Шаг 3: Поиск совпадений по ZIP-кодам (\"Matchmaking\") ---")
    print("="*50)

    if 'visits_summary' in locals() and not clinic_zips_set == set():
        demand_zips_set = set(visits_summary['Department_ZipCode'].dropna().unique())
        print(f"📊 Найдено {len(demand_zips_set)} уникальных ZIP-кодов в данных о визитах (спрос).")

        # 1. Matched ZIPs (Совпадения)
        matched_zips = demand_zips_set.intersection(clinic_zips_set)
        print("\n--- 1. ✅ СОВПАДЕНИЯ (Matched ZIPs) ---")
        print(f"Найдено {len(matched_zips)} ZIP-кодов, которые есть и в визитах, и в клиниках:")
        print(sorted(list(matched_zips)))
        
        # ⬇️ *** НОВОЕ: Сохранение 2 *** ⬇️
        df_matched_zips = pd.DataFrame(sorted(list(matched_zips)), columns=['ZipCode'])
        df_matched_zips.to_csv('list_matched_zips.csv', index=False, encoding='utf-8-sig')
        print("💾 ✅ Файл 'list_matched_zips.csv' сохранен.")
        # ⬆️ *** КОНЕЦ НОВОГО БЛОКА *** ⬆️

        # 2. Clinics without Visits (Клиники БЕЗ визитов)
        clinics_only_zips = clinic_zips_set - demand_zips_set
        print("\n--- 2. ❗ КЛИНИКИ БЕЗ ВИЗИТОВ (Clinics without Visits) ---")
        print(f"Найдено {len(clinics_only_zips)} ZIP-кодов клиник, которых НЕТ в данных о визитах:")
        print(sorted(list(clinics_only_zips)))
        
        # ⬇️ *** НОВОЕ: Сохранение 3 *** ⬇️
        df_clinics_only = pd.DataFrame(sorted(list(clinics_only_zips)), columns=['ZipCode'])
        df_clinics_only.to_csv('list_clinics_without_visits.csv', index=False, encoding='utf-8-sig')
        print("💾 ✅ Файл 'list_clinics_without_visits.csv' сохранен.")
        # ⬆️ *** КОНЕЦ НОВОГО БЛОКА *** ⬆️

        # 3. Visits without Clinics (Визиты БЕЗ клиник)
        demand_only_zips = demand_zips_set - clinic_zips_set
        print("\n--- 3. ❓ ВИЗИТЫ БЕЗ КЛИНИК (Visits without Clinics) ---")
        print(f"Найдено {len(demand_only_zips)} ZIP-кодов из визитов, которых НЕТ в мастер-файле клиник:")
        print(sorted(list(demand_only_zips)))
        
        # ⬇️ *** НОВОЕ: Сохранение 4 *** ⬇️
        df_demand_only = pd.DataFrame(sorted(list(demand_only_zips)), columns=['ZipCode'])
        df_demand_only.to_csv('list_visits_without_clinics.csv', index=False, encoding='utf-8-sig')
        print("💾 ✅ Файл 'list_visits_without_clinics.csv' сохранен.")
        # ⬆️ *** КОНЕЦ НОВОГО БЛОКА *** ⬆️
        
    else:
        print("❌ Невозможно выполнить Шаг 3.")

    # --- Шаг 4: Углубленный анализ совпадений (Side-by-Side Comparison) ---
    print("\n" + "="*50)
    print("--- Шаг 4: Углубленный анализ совпадений (Side-by-Side) ---")
    print("="*50)

    if 'matched_zips' in locals() and len(matched_zips) > 0:
        # 1. Фильтруем visits_summary
        matched_visits_summary = visits_summary[
            visits_summary['Department_ZipCode'].isin(matched_zips)
        ].sort_values(by='Department_ZipCode') 
        
        print("\n--- ДАННЫЕ О ВИЗИТАХ (СПРОС) ДЛЯ СОВПАВШИХ ZIP-КОДОВ ---")
        try:
            display(matched_visits_summary)
        except NameError:
            print(matched_visits_summary)
            
        # --- Сохранение 5 ---
        matched_visits_file = 'matched_visits_summary.csv'
        matched_visits_summary.to_csv(matched_visits_file, index=False, encoding='utf-8-sig')
        print(f"\n💾 ✅ Файл '{matched_visits_file}' успешно сохранен.")

        # 2. Фильтруем df_clinics
        clinic_columns = [
            'Zip', 'Facility', 'Practice Name', 'Specialty', 
            'First Financial Date', 'New or Acquired'
        ]
        existing_clinic_cols = [col for col in clinic_columns if col in df_clinics.columns]
        
        matched_clinics = df_clinics[
            df_clinics['Zip'].isin(matched_zips)
        ][existing_clinic_cols].sort_values(by='Zip')

        print("\n--- ДАННЫЕ О КЛИНИКАХ (TREATMENT) ДЛЯ СОВПАВШИХ ZIP-КОДОВ ---")
        try:
            display(matched_clinics)
        except NameError:
            print(matched_clinics)
            
        # --- Сохранение 6 ---
        matched_clinics_file = 'matched_clinics.csv'
        matched_clinics.to_csv(matched_clinics_file, index=False, encoding='utf-8-sig')
        print(f"\n💾 ✅ Файл '{matched_clinics_file}' успешно сохранен.")
            
        print("\n✅ Анализ 'Side-by-Side' завершен.")

    elif 'matched_zips' in locals() and len(matched_zips) == 0:
        print("❌ Не найдено совпадений (matched_zips). Шаг 4 пропущен.")
    else:
        print("❌ Невозможно выполнить Шаг 4.")

print("\n--- ✅ АУДИТ СВЯЗИ ДАННЫХ ЗАВЕРШЕН (Все 6 файлов сохранены) ---")

# 9. Difference-in-Differences (DiD) Model Setup

In this section, we transition from panel regression to a Difference-in-Differences (DiD) estimation. The goal is to measure the causal impact of a clinic "event" (either a new clinic opening or an existing one being acquired) on patient visit volumes.

Our hypothesis is that the opening or acquisition of a clinic (the "treatment") will cause a change in patient visits (Emergency vs. Primary Care) in the surrounding area. Crucially, we hypothesize that this effect is heterogeneous and depends on the patient's distance from the clinic.

## 9.1. Data Preparation and Merging

1.  **Load Auxiliary Data**: We load two new files:
    * `OSFResearch_Clinics_mastersheet_v2.csv`: Contains information on each clinic, its type ('New' or 'Acquired'), and the `event_date` (the date of opening or acquisition).
    * `Crosswalk_IL.csv`: A geospatial crosswalk file containing the latitude and longitude for Illinois ZIP codes (ZCTAs).
2.  **Data Cleaning**: We perform a final cleaning pass on all DataFrames to ensure ZIP codes are standardized as strings and all required coordinates and dates are valid.

## 9.2. Defining Treatment: The "Treatment Map"

This is the most critical step in defining our treatment and control groups. We cannot simply use the `final_data`'s `Distance_km` column, as that measures distance to *any* department a patient visited.

Instead, we create a **"Treatment Map"**:
1.  We iterate through every unique `Filtered_Patient_ZipCode` from our `final_data`.
2.  For each patient ZIP, we calculate the geospatial distance (using `geopy.distance.geodesic`) to *every* clinic in our `df_clinics_info` file.
3.  We find the **single nearest clinic** to that patient ZIP code.
4.  We store this "match": `Patient_Zip` -> `Nearest_Clinic_Type`, `Nearest_Clinic_Event_Date`, and `Distance_to_Nearest_Clinic (km)`.
5.  This map definitively assigns each patient ZIP code to its closest "treatment" center.

## 9.3. Aggregation and DiD Variable Creation

1.  **Aggregate Data**: We aggregate our main `final_data` to the `(Filtered_Patient_ZipCode, Date)` level, summing the total `Emergency_Visits` and `Primary_Visits` for each ZIP-day.
2.  **Merge Treatment Map**: We merge the "Treatment Map" onto this aggregated DataFrame.
3.  **Create DiD Variables**:
    * `Treat`: A binary variable. We define the "treatment group" as any ZIP code **within a 50 km radius** of its nearest clinic (`Treat = 1` if `distance_km <= 50`, `0` otherwise).
    * `Post`: A binary variable that "switches on" after the treatment occurs. It is = 1 if the `Date` of the visit is on or after the associated clinic's `event_date` (`Post = 1` if `Date >= event_date`).
    * `distance_category`: To test our heterogeneous effects hypothesis, we bin the `distance_km` into categories: '0-5 km', '5-10 km', '10-20 km', and '20+ km'.

## 9.4. Model Specification (Triple Interaction)

We run a two-way fixed effects (TWFE) OLS model. The specification includes `C(Year)` and `C(Month)` (Time Fixed Effects) and `C(Zip)` (Entity Fixed Effects).

The key term is a **triple interaction**: `Treat * Post * C(distance_category)`

This specification allows the DiD effect (the `Treat:Post` interaction) to be *different* for each distance category. We set '0-5 km' as the reference (base) category.

$Y_{it} = \beta_0 + \beta_1(Treat_i \times Post_{it}) + \sum_{k} \delta_k(Treat_i \times Post_{it} \times DistanceCategory_{ik}) + \gamma_i + \lambda_t + \epsilon_{it}$

* $Y_{it}$ is the outcome (Primary or Emergency Visits) for ZIP $i$ at time $t$.
* $\beta_1$ is the DiD effect for the reference distance category ('0-5 km').
* $\delta_k$ is the *additional* effect for distance category $k$ (relative to the base).
* $\gamma_i$ are the ZIP Code Fixed Effects.
* $\lambda_t$ are the Time (Year + Month) Fixed Effects.

## 9.5. Analysis and Visualization

We run four separate models:
1.  **Model 1A**: `New` clinics, `Primary_Visits` outcome
2.  **Model 1B**: `New` clinics, `Emergency_Visits` outcome
3.  **Model 2A**: `Acquired` clinics, `Primary_Visits` outcome
4.  **Model 2B**: `Acquired` clinics, `Emergency_Visits` outcome

Finally, we define a function to extract the *absolute* DiD effect for each distance bin (e.g., `Treat:Post + Treat:Post:C(distance_category)[T.5-10 km]`) and plot these effects with their 95% confidence intervals. This visualization will show us how the treatment effect changes as we move further away from the clinic.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 START 'DiD Task Plan' (Correct Logic) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: 'final_data' DataFrame not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    print(f"✅ 'final_data' found. {len(final_data)} rows.")
    
    # --- 2. Load Auxiliary Files ---
    try:
        # df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v1.csv", encoding = 'utf-8')
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
        print("✅ 'OSFResearch_Clinics_mastersheet_v2.csv' and 'Crosswalk_IL.csv' loaded.")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    import pandas as pd

    # --- 2. Load Auxiliary Files ---
    try:
        # Use the new, clean filename
        clinic_file_name = "OSFResearch_Clinics_mastersheet_v1.csv" 
        
        # Use the 'windows-1252' encoding to fix the original error
        df_clinics_info = pd.read_csv(clinic_file_name, encoding='windows-1252')
        
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
        print("✅ Auxiliary files loaded successfully.")
    
    except FileNotFoundError:
        print(f"❌ ERROR: File not found. Make sure your file is named '{clinic_file_name}'")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    # --- 3. Data Cleaning ---
    
    # 3.1 Clean final_data
    # Remove ZIP '49829'
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    # Clean ZIP codes of '.0'
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    final_data['zcta5'] = final_data['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    # Remove unwanted ZIP codes (from your code)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    
    # 3.2 Clean Crosswalk (coordinates)
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # 3.3 Clean df_clinics_info (clinic info)
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    
    # Add coordinates to df_clinics_info
    df_clinics_info = df_clinics_info.merge(zip_coords, left_on='Zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['IntPtLat', 'IntPtLon', 'event_date', 'clinic_type'])
    print(f"✅ Cleaning complete. Found {len(df_clinics_info)} clinics with coordinates.")

    # --- 4. Create "Treatment Map" ---
    print("...Creating 'treatment map' (finding nearest clinic for each ZIP)...")
    
    # Unique patient ZIP codes
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords, left_on='Filtered_Patient_ZipCode', right_index=True)
    patient_zips_df = patient_zips_df.dropna() # Remove ZIPs without coordinates

    treatment_map = []
    
    # Loop over each patient ZIP code
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['IntPtLat'], patient_row['IntPtLon'])
        
        min_distance = np.inf
        nearest_clinic = {}
        
        # Find the nearest clinic
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['IntPtLat'], clinic_row['IntPtLon'])
            # Use kilometers to match Distance_km
            distance = geodesic(patient_coords, clinic_coords).kilometers
            
            if distance < min_distance:
                min_distance = distance
                nearest_clinic = {
                    'clinic_type': clinic_row['clinic_type'],
                    'event_date': clinic_row['event_date']
                }
        
        if nearest_clinic:
            treatment_map.append({
                'Filtered_Patient_ZipCode': patient_zip,
                'distance_km': min_distance,
                'clinic_type': nearest_clinic['clinic_type'],
                'event_date': nearest_clinic['event_date']
            })

    df_treatment_map = pd.DataFrame(treatment_map)
    print(f"✅ 'Treatment map' created for {len(df_treatment_map)} ZIP codes.")

    # --- 5. Aggregate Data and Create DiD Variables ---
    print("...Aggregating visits and creating DiD variables...")
    
    # Create Y_primary and Y_emergency
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    
    # Aggregate to ZIP-Date level
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({
        'Emergency_Visits': 'sum', 
        'Primary_Visits': 'sum'
    }).reset_index()

    # Join the "treatment map" to the aggregated data
    df_main = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left')
    df_main = df_main.dropna(subset=['distance_km', 'clinic_type', 'event_date'])
    
    # Create DiD variables
    TREATMENT_RADIUS_KM = 50 # Treatment zone - 50 km
    df_main['Treat'] = (df_main['distance_km'] <= TREATMENT_RADIUS_KM).astype(int)
    df_main['Post'] = (df_main['Date'] >= df_main['event_date']).astype(int)
    
    # Create distance categories (in KM)
    bins_km = [-np.inf, 5, 10, 20, np.inf]
    labels_km = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    df_main['distance_category'] = pd.cut(
        df_main['distance_km'],
        bins=bins_km,
        labels=labels_km,
        right=False
    )
    
    # Rename ZIP for the formula
    df_main = df_main.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    
    print(f"✅ Final DataFrame 'df_main' created. {len(df_main)} rows.")
    print(df_main['clinic_type'].value_counts())

    # --- 6. Define Formulas (Step 3 of Plan) ---
    Y_PRIMARY_NAME = 'Primary_Visits'
    Y_EMERGENCY_NAME = 'Emergency_Visits'
    CONTROLS_STR = 'C(Year) + C(Month) + C(Zip)'
    
    interaction_term = "Treat * Post * C(distance_category)"
    formula_primary = f"{Y_PRIMARY_NAME} ~ {interaction_term} + {CONTROLS_STR}"
    formula_emergency = f"{Y_EMERGENCY_NAME} ~ {interaction_term} + {CONTROLS_STR}"

    print(f"\n--- Formulas are ready ---")
    print(f"Formula (Primary): {formula_primary}")
    print(f"Formula (Emergency): {formula_emergency}")

    # --- 7. Run Models (Step 4 of Plan) ---
    print("\n--- Step 4: Running 4 models ---")
    
    # Set the base category
    df_main['distance_category'] = pd.Categorical(
        df_main['distance_category'],
        categories=labels_km,
        ordered=True
    )

    # Split the data
    df_new = df_main[df_main['clinic_type'] == 'New'].copy()
    df_acquired = df_main[df_main['clinic_type'] == 'Acquired'].copy()

    print(f"Records for 'New' clinics: {len(df_new)}")
    print(f"Records for 'Acquired' clinics: {len(df_acquired)}")
    
    models = {} # Dictionary to store results

    try:
        if not df_new.empty:
            print("...Running 1A: New, Primary")
            models['1A'] = smf.ols(formula_primary, data=df_new).fit(
                cov_type='cluster', cov_kwds={'groups': df_new['Zip']}
            )
            print("...Running 1B: New, Emergency")
            models['1B'] = smf.ols(formula_emergency, data=df_new).fit(
                cov_type='cluster', cov_kwds={'groups': df_new['Zip']}
            )
        else:
            print("⚠️ 'New' DataFrame is empty, skipping 1A and 1B.")

        if not df_acquired.empty:
            print("...Running 2A: Acquired, Primary")
            models['2A'] = smf.ols(formula_primary, data=df_acquired).fit(
                cov_type='cluster', cov_kwds={'groups': df_acquired['Zip']}
            )
            print("...Running 2B: Acquired, Emergency")
            models['2B'] = smf.ols(formula_emergency, data=df_acquired).fit(
                cov_type='cluster', cov_kwds={'groups': df_acquired['Zip']}
            )
        else:
            print("⚠️ 'Acquired' DataFrame is empty, skipping 2A and 2B.")
        
        print("✅ Model estimation complete.")
        
    except Exception as e:
        print(f"❌ ERROR DURING MODEL FITTING: {e}")
        raise

    # --- 8. Visualization (Step 5 of Plan) ---
    print("\n--- Step 5: Visualizing results ---")
    sns.set(style="whitegrid")

    # (Extraction and plotting functions - they are universal)
    def extract_absolute_did_effects(model_result, ref_category_label='0-5 km'):
        """
        Extracts the *absolute* DiD effect for each distance category
        by combining the base 'Treat:Post' term with the triple-interaction term.
        """
        params = model_result.params
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        
        # Base term (the reference category)
        if 'Treat:Post' not in params.index:
            print(f"Warning: 'Treat:Post' not found. Plot will be empty.")
            return pd.DataFrame(results)
            
        effect = params['Treat:Post']
        conf = model_result.conf_int().loc['Treat:Post']
        results['Distance'].append(ref_category_label)
        results['Effect'].append(effect)
        results['Conf_Low'].append(conf[0])
        results['Conf_High'].append(conf[1])

        # Other categories
        other_categories = [cat for cat in labels_km if cat != ref_category_label]
        
        for cat in other_categories:
            term_name = f"Treat:Post:C(distance_category)[T.{cat}]"
            if term_name in params.index:
                try:
                    # Use t_test to get the combined effect (Treat:Post + Interaction_Term)
                    # This is the *absolute* effect for this category
                    t_test_result = model_result.t_test(f"Treat:Post + {term_name}")
                    
                    effect = t_test_result.effect[0]
                    conf_low = t_test_result.conf_int()[0][0]
                    conf_high = t_test_result.conf_int()[0][1]
                    
                    results['Distance'].append(cat)
                    results['Effect'].append(effect)
                    results['Conf_Low'].append(conf_low)
                    results['Conf_High'].append(conf_high)
                except Exception as e:
                    print(f"Error during t_test for {term_name}: {e}")
        return pd.DataFrame(results)

    def plot_did_effects_by_distance(model_result, title, category_order):
        """
        Plots the extracted DiD effects with 95% CIs.
        """
        plot_data = extract_absolute_did_effects(model_result, ref_category_label=category_order[0])
        if plot_data.empty:
            print(f"No data to plot for: {title}")
            return

        plot_data['Distance'] = pd.Categorical(plot_data['Distance'], categories=category_order, ordered=True)
        plot_data = plot_data.sort_values('Distance')
        
        # Calculate error bar lengths
        plot_data['err_low'] = plot_data['Effect'] - plot_data['Conf_Low']
        plot_data['err_high'] = plot_data['Conf_High'] - plot_data['Effect']
        errors = [plot_data['err_low'], plot_data['err_high']]

        plt.figure(figsize=(10, 6))
        plt.errorbar(
            x=plot_data['Distance'], y=plot_data['Effect'], yerr=errors,
            fmt='o', capsize=5, linestyle='-', markersize=8, label='DiD Effect'
        )
        plt.axhline(y=0, color='red', linestyle='--', linewidth=1, label='No Effect (Y=0)')
        plt.title(title, fontsize=16, pad=20)
        plt.xlabel("Distance to Nearest Clinic (km)", fontsize=12)
        plt.ylabel("DiD Effect Size (Change in Visits)", fontsize=12)
        plt.legend()
        plt.tight_layout()
        plt.show()

    # --- Plot the 4 graphs ---
    if '1A' in models:
        plot_did_effects_by_distance(models['1A'], f"Model 1A: New Clinics - {Y_PRIMARY_NAME}", labels_km)
    if '1B' in models:
        plot_did_effects_by_distance(models['1B'], f"Model 1B: New Clinics - {Y_EMERGENCY_NAME}", labels_km)
    if '2A' in models:
        plot_did_effects_by_distance(models['2A'], f"Model 2A: Acquired Clinics - {Y_PRIMARY_NAME}", labels_km)
    if '2B' in models:
        plot_did_effects_by_distance(models['2B'], f"Model 2B: Acquired Clinics - {Y_EMERGENCY_NAME}", labels_km)

    print("\n🎉 --- DiD 'Task Plan' fully executed! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from IPython.display import display

# --- 0. Настройка вывода Pandas ---
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

print("--- 🚀 НАЧАЛО АНАЛИЗА: DiD 2.0 (Эффект Замещения ED) ---")

# --- Шаг 1: Определить "Treated" ZIP-коды и их детали ---
print("\n" + "="*50)
print("--- Шаг 1: Определение 'Treated' ZIP-кодов и событий ---")
print("="*50)

try:
    # 1.1. Загрузить "Treated" ZIP-коды
    treated_zips_file = "list_clinics_without_visits.csv"
    df_treated_list = pd.read_csv(treated_zips_file)
    # Очищаем (на всякий случай) и преобразуем в set для быстрого поиска
    standalone_zips_set = set(
        df_treated_list['ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    )
    print(f"✅ Загружен '{treated_zips_file}'. Найдено {len(standalone_zips_set)} 'standalone' (Treated) ZIP-кодов.")

    # 1.2. Загрузить мастер-файл клиник
    master_file = "OSFResearch_Clinics_mastersheet_v1.csv"
    df_clinics_master = pd.read_csv(master_file, encoding='windows-1252')
    
    # 1.3. Очистить данные
    df_clinics_master['Zip'] = df_clinics_master['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_master['First Financial Date'] = pd.to_datetime(
        df_clinics_master['First Financial Date'], errors='coerce'
    )
    
    # 1.4. Отфильтровать мастер-файл, оставив только наши "standalone" ZIP-коды
    df_standalone_details = df_clinics_master[
        df_clinics_master['Zip'].isin(standalone_zips_set)
    ].dropna(subset=['First Financial Date']) # Исключаем клиники без даты
    
    print(f"...Найдено {len(df_standalone_details)} записей о клиниках в 'standalone' ZIP-кодах.")

    # 1.5. Найти ПЕРВОЕ событие для каждого ZIP-кода
    # Сортируем по дате, чтобы idxmin() работал корректно
    df_standalone_details = df_standalone_details.sort_values(by='First Financial Date')
    
    # Находим индекс (loc) самой ранней записи для каждой группы 'Zip'
    first_event_indices = df_standalone_details.groupby('Zip')['First Financial Date'].idxmin()
    
    # Выбираем только эти строки
    df_treatment_map_raw = df_standalone_details.loc[first_event_indices]
    
    # 1.6. Создать чистую 'df_treatment_map'
    df_treatment_map = df_treatment_map_raw[[
        'Zip', 'First Financial Date', 'New or Acquired', 'Specialty'
    ]].rename(columns={
        'First Financial Date': 'event_date',
        'New or Acquired': 'clinic_type',
        'Specialty': 'specialty'
    })

    print(f"✅ Создана 'df_treatment_map' для {len(df_treatment_map)} уникальных ZIP-кодов с датой события.")
    print("--- Карта событий (df_treatment_map): ---")
    display(df_treatment_map.head())
    
except FileNotFoundError as e:
    print(f"❌ ОШИБКА: Файл не найден: {e.filename}. Шаги 1-6 не могут быть выполнены.")
except Exception as e:
    print(f"❌ Произошла ошибка на Шаге 1: {e}")


# --- Шаг 2: Подготовить данные по Y (Outcome Variable) — Визиты в ED ---
print("\n" + "="*50)
print("--- Шаг 2: Подготовка данных по визитам в ED (Y) ---")
print("="*50)

if 'final_data' not in locals():
    print("❌ ОШИБКА: 'final_data' не найден в памяти. Пожалуйста, запустите предыдущие скрипты.")
elif 'df_treatment_map' not in locals():
    print("❌ ОШИБКА: 'df_treatment_map' не был создан. Шаги 2-6 пропущены.")
else:
    print(f"...Обработка 'final_data' ({len(final_data)} строк)...")
    
    # 2.1. Отфильтровать только визиты в ED
    df_ed_visits = final_data[final_data['VisitType'] == 'Emergency'].copy()
    print(f"...Найдено {len(df_ed_visits)} записей о визитах в ED.")
    
    # 2.2. Агрегировать до уровня ZIP-Дата
    df_ed_visits_panel = df_ed_visits.groupby(
        ['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month'], 
        as_index=False
    ).agg(
        EncounterCount=('EncounterCount', 'sum')
    )
    print(f"✅ Данные по ED агрегированы. {len(df_ed_visits_panel)} строк (ZIP-Дата).")


    # --- Шаг 3: Создать финальный DiD DataFrame ---
    print("\n" + "="*50)
    print("--- Шаг 3: Создание финального DiD DataFrame ---")
    print("="*50)
    
    # 3.1. Соединить панель визитов (левая) с картой событий (правая)
    df_did_panel = df_ed_visits_panel.merge(
        df_treatment_map,
        left_on='Filtered_Patient_ZipCode',
        right_on='Zip',
        how='left'
    )
    
    # 3.2. Создать переменные DiD
    # Treated = 1, если для этого ZIP-кода есть дата события (т.е. он из 'standalone' списка)
    df_did_panel['Treated'] = np.where(df_did_panel['event_date'].notna(), 1, 0)
    
    # Post = 1, если дата визита >= даты события И это treated ZIP
    df_did_panel['Post'] = np.where(
        (df_did_panel['Date'] >= df_did_panel['event_date']) & (df_did_panel['Treated'] == 1), 
        1, 
        0
    )
    
    # DiD_Interaction = Treated * Post
    df_did_panel['DiD_Interaction'] = df_did_panel['Treated'] * df_did_panel['Post']
    
    # 3.3. Создать переменную Y
    df_did_panel['Log_ED_Visits'] = np.log(df_did_panel['EncounterCount'] + 1)
    
    # Очистка (удаляем дублирующую колонку Zip)
    df_did_panel = df_did_panel.drop(columns=['Zip'])
    
    print("✅ Финальный DiD DataFrame 'df_did_panel' создан.")
    print(f"...Всего 'Treated' наблюдений (строк): {len(df_did_panel[df_did_panel['Treated'] == 1])}")
    print(f"...Всего 'Post' наблюдений (строк): {len(df_did_panel[df_did_panel['Post'] == 1])}")


    # --- Шаг 4: Валидация (Проверка данных До/После) ---
    print("\n" + "="*50)
    print("--- Шаг 4: Валидация данных (До/После) ---")
    print("="*50)
    
    df_treated_only = df_did_panel[df_did_panel['Treated'] == 1].copy()
    
    if df_treated_only.empty:
        print("⚠️ Не найдено 'Treated' наблюдений. Валидация невозможна.")
    else:
        # Группируем по каждому treated ZIP-коду
        validation_summary = df_treated_only.groupby('Filtered_Patient_ZipCode').agg(
            # Получаем статические детали (они одинаковы для каждого ZIP)
            clinic_type=('clinic_type', 'first'),
            specialty=('specialty', 'first'),
            event_date=('event_date', 'first'),
            
            # Считаем месяцы (уникальные даты) до и после
            pre_treatment_months=('Post', lambda s: (s == 0).sum()),
            post_treatment_months=('Post', lambda s: (s == 1).sum())
        ).reset_index().rename(columns={'Filtered_Patient_ZipCode': 'ZipCode'})
        
        print("📊 Сводка по 'Treated' ZIP-кодам (сколько месяцев 'До' и 'После'):")
        display(validation_summary.sort_values(by=['pre_treatment_months', 'post_treatment_months']))


    # --- Шаг 5: Запустить Модель 1 (Общий эффект DiD) ---
    print("\n" + "="*50)
    print("--- Шаг 5: Модель 1 (Общий эффект DiD) ---")
    print("="*50)
    
    try:
        # 5.1. Подготовить данные для PanelOLS
        df_model_1 = df_did_panel.set_index(['Filtered_Patient_ZipCode', 'Date'])
        
        # 5.2. Определить Y и X
        Y1 = df_model_1['Log_ED_Visits']
        X1 = sm.add_constant(df_model_1['DiD_Interaction'])
        
        # 5.3. Запустить модель с Entity (ZIP) и Time (Date) Fixed Effects
        print("...Запуск PanelOLS (EntityEffects=True, TimeEffects=True)...")
        model1 = PanelOLS(Y1, X1, entity_effects=True, time_effects=True, drop_absorbed=True)
        
        # 5.4. Рассчитать модель с кластеризованными ошибками на уровне Entity (ZIP)
        results1 = model1.fit(cov_type='clustered', cluster_entity=True)
        
        # 5.5. Вывести результат
        print(results1.summary)
        
    except Exception as e:
        print(f"❌ ОШИБКА при запуске Модели 1: {e}")


    # --- Шаг 6: Запустить Модель 2 (Раздельный эффект New vs. Acquired) ---
    print("\n" + "="*50)
    print("--- Шаг 6: Модель 2 (Эффекты 'New' vs 'Acquired') ---")
    print("="*50)
    
    try:
        # 6.1. Создать раздельные переменные
        df_model_2_data = df_did_panel.copy()
        
        # DiD_New = 1, если Post=1 И clinic_type='New'
        df_model_2_data['DiD_New'] = np.where(
            (df_model_2_data['clinic_type'] == 'New') & (df_model_2_data['Post'] == 1), 1, 0
        )
        
        # DiD_Acquired = 1, если Post=1 И clinic_type='Acquired'
        df_model_2_data['DiD_Acquired'] = np.where(
            (df_model_2_data['clinic_type'] == 'Acquired') & (df_model_2_data['Post'] == 1), 1, 0
        )
        
        # 6.2. Подготовить данные
        df_model_2 = df_model_2_data.set_index(['Filtered_Patient_ZipCode', 'Date'])
        
        # 6.3. Определить Y и X
        Y2 = df_model_2['Log_ED_Visits']
        X2 = sm.add_constant(df_model_2[['DiD_New', 'DiD_Acquired']])
        
        # 6.4. Запустить модель
        print("...Запуск PanelOLS (Раздельные эффекты)...")
        model2 = PanelOLS(Y2, X2, entity_effects=True, time_effects=True, drop_absorbed=True)
        results2 = model2.fit(cov_type='clustered', cluster_entity=True)
        
        # 6.5. Вывести результат
        print(results2.summary)

    except Exception as e:
        print(f"❌ ОШИБКА при запуске Модели 2: {e}")


print("\n--- ✅ АНАЛИЗ DiD 2.0 ЗАВЕРШЕН ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from IPython.display import display

# --- 0. Настройка вывода Pandas ---
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

print("--- 🚀 НАЧАЛО АНАЛИЗА: DiD 3.0 ('Золотой список') ---")

# --- Шаг 1: Определить "Золотой список" (Hardcoded) ---
print("\n" + "="*50)
print("--- Шаг 1: 'Золотой список' (DiD 3.0) создан ---")
print("="*50)

# "Золотой список" из 9 клиник
gold_list_data = [
    {'Zip': '61568', 'event_date': '2016-04-01', 'clinic_type': 'Acquired', 'specialty': 'Family Practice'},
    {'Zip': '61517', 'event_date': '2018-01-31', 'clinic_type': 'Acquired', 'specialty': 'Family Practice'},
    {'Zip': '61434', 'event_date': '2018-10-01', 'clinic_type': 'Acquired', 'specialty': 'Family Practice'},
    {'Zip': '61361', 'event_date': '2018-10-01', 'clinic_type': 'Acquired', 'specialty': 'Family Practice'},
    {'Zip': '60450', 'event_date': '2018-10-17', 'clinic_type': 'New', 'specialty': 'Urgent Care_Urgo'},
    {'Zip': '62035', 'event_date': '2020-02-01', 'clinic_type': 'New', 'specialty': 'Family Practice/Prompt Care'},
    {'Zip': '61473', 'event_date': '2020-10-01', 'clinic_type': 'Acquired', 'specialty': 'Family Practice'},
    {'Zip': '61866', 'event_date': '2021-06-01', 'clinic_type': 'New', 'specialty': 'Urgent Care_Urgo'},
    {'Zip': '61376', 'event_date': '2021-07-01', 'clinic_type': 'Acquired', 'specialty': 'Family Practice'}
]

# Создаем DataFrame и преобразуем даты
df_treatment_map = pd.DataFrame(gold_list_data)
df_treatment_map['event_date'] = pd.to_datetime(df_treatment_map['event_date'])

print(f"✅ 'df_treatment_map' создан для {len(df_treatment_map)} 'Treated' ZIP-кодов.")
display(df_treatment_map)


# --- Шаг 2: Подготовить данные по Y (Outcome Variable) — Визиты в ED ---
print("\n" + "="*50)
print("--- Шаг 2: Подготовка данных по визитам в ED (Y) ---")
print("="*50)

if 'final_data' not in locals():
    print("❌ ОШИБКА: 'final_data' не найден в памяти. Пожалуйста, запустите предыдущие скрипты.")
else:
    # Агрегируем данные по ED
    df_ed_visits_panel = (
        final_data[final_data['VisitType'] == 'Emergency']
        .groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month'], as_index=False)
        .agg(EncounterCount=('EncounterCount', 'sum'))
    )
    print(f"✅ Данные по ED агрегированы. {len(df_ed_visits_panel)} строк (ZIP-Дата).")


    # --- Шаг 3: Создать финальный DiD DataFrame (на "золотом списке") ---
    print("\n" + "="*50)
    print("--- Шаг 3: Создание финального DiD DataFrame ---")
    print("="*50)

    # Соединяем ВСЕ визиты в ED с "золотым списком"
    df_did_panel = df_ed_visits_panel.merge(
        df_treatment_map,
        left_on='Filtered_Patient_ZipCode',
        right_on='Zip',
        how='left'
    )

    # Treated = 1, если ZIP есть в "золотом списке" (т.е. event_date не пустая)
    df_did_panel['Treated'] = (~df_did_panel['event_date'].isna()).astype(int)
    
    # Post = 1, если дата визита >= даты события И это treated ZIP
    df_did_panel['Post'] = (
        (df_did_panel['Date'] >= df_did_panel['event_date']) & (df_did_panel['Treated'] == 1)
    ).astype(int)
    
    # Outcome variable
    df_did_panel['Log_ED_Visits'] = np.log(df_did_panel['EncounterCount'] + 1)

    # DiD переменные для Модели 2
    # df_did_panel['clinic_type'] == 'New' вернет True/False/NaN. 
    # Умножение на Post (0 или 1) и .astype(int) корректно обработает NaN (превратив их в 0)
    df_did_panel['DiD_New'] = (
        df_did_panel['Post'] * (df_did_panel['clinic_type'] == 'New')
    ).astype(int)
    
    df_did_panel['DiD_Acquired'] = (
        df_did_panel['Post'] * (df_did_panel['clinic_type'] == 'Acquired')
    ).astype(int)

    print("✅ DiD переменные ('Treated', 'Post', 'Log_ED_Visits', 'DiD_New', 'DiD_Acquired') созданы.")
    print(f"...Всего 'Treated' наблюдений (строк): {df_did_panel['Treated'].sum()}")
    print(f"...Всего 'Post' наблюдений (строк): {df_did_panel['Post'].sum()}")


    # --- Шаг 4: Валидация (Критически важный шаг!) ---
    print("\n" + "="*50)
    print("--- Шаг 4: Валидация данных (До/После) ---")
    print("="*50)

    # Берем только наблюдения из "Treated" ZIP-кодов
    df_treated_obs = df_did_panel[df_did_panel['Treated'] == 1]

    if df_treated_obs.empty:
        print("❌ ОШИБКА: Не найдено ни одного наблюдения, соответствующего 'Treated' ZIP-кодам.")
    else:
        # Группируем и считаем месяцы "До" (Post=0) и "После" (Post=1)
        validation_summary = df_treated_obs.groupby(
            ['Filtered_Patient_ZipCode', 'clinic_type', 'specialty', 'event_date']
        ).agg(
            pre_treatment_months=('Post', lambda x: (x == 0).sum()),
            post_treatment_months=('Post', lambda x: (x == 1).sum())
        ).reset_index()

        print("📊 Сводка по 'Treated' ZIP-кодам (сколько месяцев 'До' и 'После'):")
        display(validation_summary)

        # Проверка на клиники без данных "после"
        clinics_with_no_post_data = validation_summary[validation_summary['post_treatment_months'] == 0]
        
        if not clinics_with_no_post_data.empty:
            print("\n" + "!"*60)
            print("🚨 ВНИМАНИЕ: Для этих клиник нет данных 'После'.")
            print("Они будут автоматически исключены из регрессии PanelOLS, т.к. для них нет вариации в 'Post'.")
            print("!"*60)
            display(clinics_with_no_post_data)
        else:
            print("\n✅ Отлично! У всех 'Treated' клиник в 'золотом списке' есть данные 'До' и 'После'.")


    # --- Шаг 5: Запустить Модель 2 (Раздельный эффект New vs. Acquired) ---
    print("\n" + "="*50)
    print("--- Шаг 5: Модель (Эффекты 'New' vs 'Acquired') ---")
    print("="*50)

    try:
        # Установка индекса
        df_model = df_did_panel.set_index(['Filtered_Patient_ZipCode', 'Date'])

        # Y (Outcome)
        Y = df_model['Log_ED_Visits']
        
        # X (Regressors)
        X = df_model[['DiD_New', 'DiD_Acquired']]
        X = sm.add_constant(X) # Добавляем константу

        print("...Запуск PanelOLS (EntityEffects=True, TimeEffects=True)...")
        
        # Модель с фиксированными эффектами для ZIP (entity) и Даты (time)
        model = PanelOLS(Y, X, entity_effects=True, time_effects=True, drop_absorbed=True)
        
        # Кластеризация ошибок на уровне ZIP (entity)
        results = model.fit(cov_type='clustered', cluster_entity=True)

        print("\n--- РЕЗУЛЬТАТЫ МОДЕЛИ DiD 3.0 ---")
        print(results.summary)

    except Exception as e:
        print(f"❌ ОШИБКА при запуске Модели: {e}")

print("\n--- ✅ АНАЛИЗ DiD 3.0 ЗАВЕРШЕН ---")

In [ ]:
import pandas as pd

# --- Отладочный скрипт ---
# Мы просто хотим загрузить файл и посмотреть на его колонки

clinics_file = "matched_clinics.csv"

try:
    df_test_load = pd.read_csv(clinics_file, encoding='windows-1252')
    
    print(f"✅ Файл '{clinics_file}' успешно загружен.")
    print("\n" + "="*30)
    print("СПИСОК КОЛОНОК В ФАЙЛЕ:")
    print(df_test_load.columns.to_list())
    print("="*30)
    
    print("\nПервые 5 строк файла (для визуальной проверки):")
    display(df_test_load.head())

except FileNotFoundError:
    print(f"❌ ОШИБКА: Файл '{clinics_file}' не найден.")
except Exception as e:
    print(f"❌ Произошла ошибка при загрузке: {e}")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from IPython.display import display

# --- 0. Настройка вывода Pandas ---
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

print("--- 🚀 НАЧАЛО АНАЛИЗА: 'Эффект Хаба' (Метод 2: ФИНАЛЬНОЕ ИСПРАВЛЕНИЕ) ---")

# --- Шаг 1: Подготовить данные по Y (Спрос в Хабах) ---
print("\n" + "="*50)
print("--- Шаг 1: Подготовка данных по Y (Спрос в Хабах) ---")
print("="*50)

if 'final_data' not in locals():
    print("❌ ОШИБКА: 'final_data' не найден в памяти. Пожалуйста, запустите предыдущие скрипты.")
else:
    print(f"...Обработка 'final_data' ({len(final_data)} строк)...")
    
    # Создаем временные колонки для условного суммирования
    final_data['Primary_Visits_Temp'] = np.where(
        final_data['VisitType'] != 'Emergency', final_data['EncounterCount'], 0
    )
    final_data['ED_Visits_Temp'] = np.where(
        final_data['VisitType'] == 'Emergency', final_data['EncounterCount'], 0
    )
    
    # Группируем по Хабу (Department_ZipCode) и Дате
    df_demand_panel = final_data.groupby(
        ['Department_ZipCode', 'Date', 'Year', 'Month'], 
        as_index=False
    ).agg(
        Total_Encounters=('EncounterCount', 'sum'),
        Primary_Visits=('Primary_Visits_Temp', 'sum'),
        ED_Visits=('ED_Visits_Temp', 'sum')
    )
    
    # Добавляем логарифмированную переменную Y
    df_demand_panel['Log_Total_Encounters'] = np.log(df_demand_panel['Total_Encounters'] + 1)
    
    # Удаляем временные колонки
    final_data = final_data.drop(columns=['Primary_Visits_Temp', 'ED_Visits_Temp'])
    
    print(f"✅ 'df_demand_panel' (Y) создан. {len(df_demand_panel)} строк (Хаб-Дата).")
    
    # --- Верификация Шага 1 ---
    print("\n--- Верификация (Шаг 1): df_demand_panel.info() ---")
    df_demand_panel.info()
    print("\n--- Верификация (Шаг 1): df_demand_panel.head() ---")
    display(df_demand_panel.head())


    # --- Шаг 2: Подготовить данные по X (Интенсивность "Лечения") ---
    print("\n" + "="*50)
    print("--- Шаг 2: Подготовка данных по X (Интенсивность 'Лечения') ---")
    print("="*50)
    
    clinics_file = "matched_clinics.csv"
    try:
        # !!! КЛЮЧЕВОЕ ИСПРАВЛЕНИЕ: Меняем encoding на 'utf-8-sig' !!!
        # Эта кодировка правильно обработает 'ï»¿' (BOM)
        df_clinics_events = pd.read_csv(clinics_file, encoding='utf-8-sig') 
        
        # 2.1. Очистка данных о событиях
        # Теперь pandas должен видеть колонку 'Zip' (с большой 'Z')
        
        if 'Zip' not in df_clinics_events.columns:
             print(f"❌ ОШИБКА: Кодировка 'utf-8-sig' не помогла. Колонки: {df_clinics_events.columns.to_list()}")
             raise KeyError("Колонка 'Zip' все еще не найдена.")

        df_clinics_events['Zip'] = df_clinics_events['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
        df_clinics_events['First Financial Date'] = pd.to_datetime(
            df_clinics_events['First Financial Date'], errors='coerce'
        )
        
        # Переименуем 'Zip' в 'Hub_ZipCode' для ясности
        df_clinics_events = df_clinics_events.rename(columns={
            'Zip': 'Hub_ZipCode', 
            'First Financial Date': 'Event_Date',
            'New or Acquired': 'clinic_type'
        }).dropna(subset=['Event_Date']) # Убираем события без даты
        
        print(f"✅ Загружен и очищен '{clinics_file}'. {len(df_clinics_events)} событий.")

        # 2.2. Создание "каркаса" (scaffold)
        hub_zips = df_demand_panel['Department_ZipCode'].unique()
        all_dates = df_demand_panel['Date'].unique()
        
        panel_index = pd.MultiIndex.from_product(
            [hub_zips, all_dates], 
            names=['Hub_ZipCode', 'Date']
        )
        df_panel_scaffold = pd.DataFrame(index=panel_index).reset_index()
        print(f"...Создан 'каркас' (scaffold) на {len(df_panel_scaffold)} строк.")

        # 2.3. Генерируем переменные X (эффективный метод)
        df_cross = pd.merge(
            df_panel_scaffold, 
            df_clinics_events[['Hub_ZipCode', 'Event_Date', 'clinic_type', 'Specialty']], 
            on='Hub_ZipCode', 
            how='left'
        )
        
        df_cumulative = df_cross[df_cross['Event_Date'] <= df_cross['Date']]
        
        df_cumulative['is_new'] = (df_cumulative['clinic_type'] == 'New').astype(int)
        df_cumulative['is_acquired'] = (df_cumulative['clinic_type'] == 'Acquired').astype(int)

        # 2.4. Группируем для получения кумулятивных итогов
        print("...Расчет кумулятивных переменных X...")
        df_grouped_stats = df_cumulative.groupby(['Hub_ZipCode', 'Date']).agg(
            Num_Total_Clinics_ht=('Event_Date', 'count'), 
            Num_New_Clinics_ht=('is_new', 'sum'),
            Num_Acquired_Clinics_ht=('is_acquired', 'sum'),
            Num_Unique_Specialties_ht=('Specialty', 'nunique')
        )
        
        # 2.5. Восстанавливаем полный "каркас"
        df_treatment_panel = df_grouped_stats.reindex(panel_index, fill_value=0)
        
        print(f"✅ 'df_treatment_panel' (X) создан. {len(df_treatment_panel)} строк.")
        
        # --- Верификация Шага 2 ---
        print("\n--- Верификация (Шаг 2): df_treatment_panel.info() ---")
        df_treatment_panel.info()
        print("\n--- Верификация (Шаг 2): df_treatment_panel.head() ---")
        display(df_treatment_panel.head())
        
    except FileNotFoundError:
        print(f"❌ ОШИБКА: Файл '{clinics_file}' не найден. Шаги 2-4 пропущены.")
    except KeyError as e:
        print(f"❌ ОШИБКА KeyError на Шаге 2: {e}.")
    except Exception as e:
        print(f"❌ Произошла ошибка на Шаге 2: {e}")


    # --- Шаг 3: Собрать финальную панель для модели ---
    print("\n" + "="*50)
    print("--- Шаг 3: Сборка финальной панели для модели ---")
    print("="*50)

    if 'df_demand_panel' in locals() and 'df_treatment_panel' in locals():
        df_hub_panel = df_demand_panel.merge(
            df_treatment_panel,
            left_on=['Department_ZipCode', 'Date'],
            right_index=True, # right_index=True использует ['Hub_ZipCode', 'Date']
            how='left'
        )
        
        x_vars = ['Num_Total_Clinics_ht', 'Num_New_Clinics_ht', 
                  'Num_Acquired_Clinics_ht', 'Num_Unique_Specialties_ht']
        
        for col in x_vars:
            if col not in df_hub_panel.columns:
                 df_hub_panel[col] = 0
            else:
                 df_hub_panel[col] = df_hub_panel[col].fillna(0)
        
        print(f"✅ Финальная 'df_hub_panel' собрана. {len(df_hub_panel)} строк.")
        
        # --- Верификация Шага 3 ---
        print("\n--- Верификация (Шаг 3): df_hub_panel.info() ---")
        df_hub_panel.info()
        print("\n--- Верификация (Шаг 3): df_hub_panel.head() ---")
        display(df_hub_panel.head())
    else:
        print("❌ Невозможно собрать панель: 'df_demand_panel' или 'df_treatment_panel' отсутствуют.")


    # --- Шаг 4: Запустить Модель Fixed Effects (PanelOLS) ---
    print("\n" + "="*50)
    print("--- Шаг 4: Запуск Модели Fixed Effects (PanelOLS) ---")
    print("="*50)

    if 'df_hub_panel' in locals():
        try:
            # 4.1. Подготовка данных для PanelOLS
            df_model = df_hub_panel.set_index(['Department_ZipCode', 'Date'])
            
            # 4.2. Определение Y и X
            Y = df_model['Log_Total_Encounters']
            X = sm.add_constant(df_model[['Num_New_Clinics_ht', 'Num_Acquired_Clinics_ht']])
            
            print("...Запуск PanelOLS (EntityEffects=True, TimeEffects=True)...")
            
            # 4.3. Запуск модели
            model = PanelOLS(Y, X, entity_effects=True, time_effects=True, drop_absorbed=True)
            
            # 4.4. Расчет
            results = model.fit(cov_type='clustered', cluster_entity=True)
            
            # 4.5. Вывод
            print("\n--- РЕЗУЛЬТАТЫ МОДЕЛИ 'ЭФФЕКТА ХАБА' (ИСПРАВЛЕННАЯ) ---")
            print(results.summary)

        except Exception as e:
            print(f"❌ ОШИБКА при запуске Модели: {e}")
            
    else:
        print("❌ Невозможно запустить модель: 'df_hub_panel' отсутствует.")


print("\n--- ✅ АНАЛИЗ 'ЭФФЕКТА ХАБА' (ИСПРАВЛЕННАЯ) ЗАВЕРШЕН ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from IPython.display import display

print("--- 🚀 НАЧАЛО: Дополнительный анализ 'Эффекта Хаба' (Primary vs. ED) ---")

# --- Шаг 1: Проверить наличие df_hub_panel ---
if 'df_hub_panel' not in locals():
    print("❌ ОШИБКА: DataFrame 'df_hub_panel' не найден в памяти.")
    print("Пожалуйста, сначала полностью запустите предыдущий скрипт ('Анализ Эффекта Хаба')")
else:
    print("✅ 'df_hub_panel' найден в памяти.")
    
    try:
        # --- Шаг 2: Добавить новые Log-переменные ---
        df_hub_panel['Log_Primary_Visits'] = np.log(df_hub_panel['Primary_Visits'] + 1)
        df_hub_panel['Log_ED_Visits'] = np.log(df_hub_panel['ED_Visits'] + 1)
        print("✅ Созданы 'Log_Primary_Visits' и 'Log_ED_Visits'.")

        # --- Шаг 3: Подготовить данные для PanelOLS ---
        df_model_multi = df_hub_panel.set_index(['Department_ZipCode', 'Date'])
        
        # Определяем X (он одинаковый для обеих моделей)
        X = sm.add_constant(df_model_multi[['Num_New_Clinics_ht', 'Num_Acquired_Clinics_ht']])

        # --- Шаг 4: Запустить Модель 1 (Primary Visits) ---
        print("\n" + "="*50)
        print("--- РЕЗУЛЬТАТЫ (ХАБ): Y = Primary Visits ---")
        print("="*50)
        
        Y_1 = df_model_multi['Log_Primary_Visits']
        
        model1 = PanelOLS(Y_1, X, entity_effects=True, time_effects=True, drop_absorbed=True)
        results1 = model1.fit(cov_type='clustered', cluster_entity=True)
        
        print(results1.summary)

        # --- Шаг 5: Запустить Модель 2 (ED Visits) ---
        print("\n" + "="*50)
        print("--- РЕЗУЛЬТАТЫ (ХАБ): Y = ED Visits ---")
        print("="*50)
        
        Y_2 = df_model_multi['Log_ED_Visits']
        
        model2 = PanelOLS(Y_2, X, entity_effects=True, time_effects=True, drop_absorbed=True)
        results2 = model2.fit(cov_type='clustered', cluster_entity=True)
        
        print(results2.summary)

    except KeyError as e:
        print(f"❌ ОШИBKA KeyError: {e}. Возможно, в 'df_hub_panel' отсутствуют 'Primary_Visits' или 'ED_Visits'.")
    except Exception as e:
        print(f"❌ Произошла ошибка при запуске моделей: {e}")

print("\n--- ✅ Дополнительный анализ 'Эффекта Хаба' ЗАВЕРШЕН ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from IPython.display import display

print("--- 🚀 НАЧАЛО: Финальный анализ 'Эффекта Хаба' (Y = Доля ED визитов) ---")

# --- Шаг 1: Проверить наличие df_hub_panel ---
if 'df_hub_panel' not in locals():
    print("❌ ОШИБКА: DataFrame 'df_hub_panel' не найден в памяти.")
    print("Пожалуйста, сначала полностью запустите скрипт ('Анализ Эффекта Хаба (Метод 2: ФИНАЛЬНОЕ ИСПРАВЛЕНИЕ)')")
else:
    print("✅ 'df_hub_panel' найден в памяти.")
    
    try:
        # --- Шаг 2: Создать новую Y-переменную (Share_ED) ---
        print("...Создание Y-переменной 'Share_ED'...")
        
        # 2.1. Создаем общий знаменатель
        df_hub_panel['Total_Visits_for_Ratio'] = (
            df_hub_panel['Primary_Visits'] + df_hub_panel['ED_Visits']
        )
        
        # 2.2. Создаем Share_ED с обработкой деления на ноль
        # Мы используем np.nan, если Total_Visits_for_Ratio == 0
        # PanelOLS автоматически отбросит эти строки при расчете.
        df_hub_panel['Share_ED'] = np.where(
            df_hub_panel['Total_Visits_for_Ratio'] == 0, 
            np.nan,  # Ставим NaN, если в этот день не было визитов
            df_hub_panel['ED_Visits'] / df_hub_panel['Total_Visits_for_Ratio']
        )
        
        # Проверим, сколько NaN мы создали
        nan_count = df_hub_panel['Share_ED'].isna().sum()
        print(f"✅ 'Share_ED' создана. {nan_count} наблюдений (дней без визитов) будут исключены.")


        # --- Шаг 3: Подготовить и запустить модель PanelOLS ---
        
        # 3.1. Установить мульти-индекс
        df_model_share = df_hub_panel.set_index(['Department_ZipCode', 'Date'])
        
        # 3.2. Определить Y и X
        Y_share = df_model_share['Share_ED']
        X_share = sm.add_constant(df_model_share[['Num_New_Clinics_ht', 'Num_Acquired_Clinics_ht']])

        print("\n" + "="*50)
        print("--- РЕЗУЛЬТАТЫ (ХАБ): Y = Доля ED визитов (Share_ED) ---")
        print("="*50)
        
        # 3.3. Запустить модель
        print("...Запуск PanelOLS (EntityEffects=True, TimeEffects=True)...")
        model_share = PanelOLS(Y_share, X_share, entity_effects=True, time_effects=True, drop_absorbed=True)
        results_share = model_share.fit(cov_type='clustered', cluster_entity=True)
        
        # 3.4. Вывести результат
        print(results_share.summary)

    except KeyError as e:
        print(f"❌ ОШИBKA KeyError: {e}. Убедитесь, что в 'df_hub_panel' есть колонки 'Primary_Visits' и 'ED_Visits'.")
    except Exception as e:
        print(f"❌ Произошла ошибка при запуске модели: {e}")

print("\n--- ✅ Финальный анализ 'Эффекта Хаба' ЗАВЕРШЕН ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from IPython.display import display

print("--- 🚀 НАЧАЛО: Анализ 'Эффекта Замещения' (Y = Regular Visits) ---")

# --- Шаг 1: Проверить наличие 'df_treatment_map' (Золотой список) ---
if 'df_treatment_map' not in locals():
    print("❌ ОШИБКА: DataFrame 'df_treatment_map' ('Золотой список') не найден в памяти.")
    print("Пожалуйста, сначала запустите скрипт 'Анализ DiD 3.0', чтобы создать этот DataFrame.")
elif 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден в памяти.")
else:
    print("✅ 'df_treatment_map' ('Золотой список') и 'final_data' найдены.")

    # --- Шаг 2: Подготовить данные по Y (Regular Visits) ---
    print("\n" + "="*50)
    print("--- Шаг 2: Подготовка данных по Y (Regular Visits) ---")
    print("="*50)

    try:
        # 2.1. Отфильтровать final_data, оставив только 'Regular' визиты
        df_regular_visits = final_data[final_data['VisitType'] != 'Emergency'].copy()
        
        # 2.2. Агрегировать по ZIP и Дате
        df_regular_visits_panel = (
            df_regular_visits
            .groupby(['Filtered_Patient_ZipCode', 'Date'], as_index=False)
            .agg(Regular_Visits=('EncounterCount', 'sum'))
        )
        print(f"✅ 'df_regular_visits_panel' (Y) создан. {len(df_regular_visits_panel)} строк (ZIP-Дата).")

        # --- Шаг 3: Создать финальный DiD DataFrame ---
        print("\n" + "="*50)
        print("--- Шаг 3: Создание финального DiD DataFrame ---")
        print("="*50)

        # 3.1. Соединить панель Regular_Visits (левая) с картой событий (правая)
        df_did_panel_reg = df_regular_visits_panel.merge(
            df_treatment_map,
            left_on='Filtered_Patient_ZipCode',
            right_on='Zip',
            how='left'
        )

        # 3.2. Создать переменные DiD
        # Treated = 1, если ZIP есть в "золотом списке"
        df_did_panel_reg['Treated'] = (~df_did_panel_reg['event_date'].isna()).astype(int)
        
        # Post = 1, если дата визита >= даты события И это treated ZIP
        df_did_panel_reg['Post'] = (
            (df_did_panel_reg['Date'] >= df_did_panel_reg['event_date']) & 
            (df_did_panel_reg['Treated'] == 1)
        ).astype(int)
        
        # 3.3. Создать Y-переменную (Log)
        df_did_panel_reg['Log_Regular_Visits'] = np.log(df_did_panel_reg['Regular_Visits'] + 1)

        # 3.4. Создать раздельные DiD-переменные
        df_did_panel_reg['DiD_New'] = (
            df_did_panel_reg['Post'] * (df_did_panel_reg['clinic_type'] == 'New')
        ).astype(int)
        
        df_did_panel_reg['DiD_Acquired'] = (
            df_did_panel_reg['Post'] * (df_did_panel_reg['clinic_type'] == 'Acquired')
        ).astype(int)

        print("✅ DiD переменные для 'Regular Visits' созданы.")
        print(f"...Всего 'Treated' наблюдений (строк): {df_did_panel_reg['Treated'].sum()}")
        print(f"...Всего 'Post' наблюдений (строк): {df_did_panel_reg['Post'].sum()}")


        # --- Шаг 4: Запустить Модель DiD (PanelOLS) ---
        print("\n" + "="*50)
        print("--- РЕЗУЛЬТАТЫ (STANDALONE): Y = Regular Visits ---")
        print("="*50)
    
        # 4.1. Установить мульти-индекс
        df_model_reg = df_did_panel_reg.set_index(['Filtered_Patient_ZipCode', 'Date'])

        # 4.2. Определить Y и X
        Y_reg = df_model_reg['Log_Regular_Visits']
        X_reg = sm.add_constant(df_model_reg[['DiD_New', 'DiD_Acquired']])

        # 4.3. Запустить модель
        print("...Запуск PanelOLS (EntityEffects=True, TimeEffects=True)...")
        model_reg = PanelOLS(Y_reg, X_reg, entity_effects=True, time_effects=True, drop_absorbed=True)
        results_reg = model_reg.fit(cov_type='clustered', cluster_entity=True)
        
        # 4.4. Вывести результат
        print(results_reg.summary)

    except Exception as e:
        print(f"❌ Произошла ошибка при выполнении скрипта: {e}")

print("\n--- ✅ Анализ 'Эффекта Замещения' (Regular Visits) ЗАВЕРШЕН ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from IPython.display import display

print("--- 🚀 НАЧАЛО: 'Эффект Замещения' (Y = Доля ED визитов) ---")

# --- Шаг 1: Проверить наличие df_final_data и df_treatment_map ---
if 'final_data' not in locals() or 'df_treatment_map' not in locals():
    print("❌ ОШИБКА: 'final_data' и/или 'df_treatment_map' ('Золотой список') не найдены в памяти.")
    print("Пожалуйста, запустите 'DiD 3.0' и убедитесь, что 'final_data' загружена.")
else:
    print("✅ 'final_data' и 'df_treatment_map' ('Золотой список') найдены.")
    
    try:
        # --- Шаг 2: Подготовить данные по Y (Оба типа визитов) ---
        print("\n" + "="*50)
        print("--- Шаг 2: Подготовка данных по Y (Regular vs. ED) ---")
        print("="*50)
        
        # 2.1. Создаем временные колонки для условного суммирования
        final_data['Regular_Visits_Temp'] = np.where(
            final_data['VisitType'] != 'Emergency', final_data['EncounterCount'], 0
        )
        final_data['ED_Visits_Temp'] = np.where(
            final_data['VisitType'] == 'Emergency', final_data['EncounterCount'], 0
        )
        
        # 2.2. Агрегируем по ZIP и Дате
        df_ratio_panel = final_data.groupby(
            ['Filtered_Patient_ZipCode', 'Date'], 
            as_index=False
        ).agg(
            Regular_Visits=('Regular_Visits_Temp', 'sum'),
            ED_Visits=('ED_Visits_Temp', 'sum')
        )
        
        # 2.3. Удаляем временные колонки из final_data
        final_data = final_data.drop(columns=['Regular_Visits_Temp', 'ED_Visits_Temp'])
        
        print(f"✅ 'df_ratio_panel' (Y) создан. {len(df_ratio_panel)} строк (ZIP-Дата).")
        
        
        # --- Шаг 3: Создать новую Y-переменную (Share_ED) ---
        print("\n" + "="*50)
        print("--- Шаг 3: Создание Y-переменной (Share_ED) ---")
        print("="*50)
        
        # 3.1. Создаем общий знаменатель
        df_ratio_panel['Total_Visits_for_Ratio'] = (
            df_ratio_panel['Regular_Visits'] + df_ratio_panel['ED_Visits']
        )
        
        # 3.2. Создаем Share_ED с обработкой деления на ноль
        # Используем np.nan, чтобы PanelOLS автоматически исключил эти наблюдения
        df_ratio_panel['Share_ED'] = np.where(
            df_ratio_panel['Total_Visits_for_Ratio'] == 0,
            np.nan, # Ставим NaN, если в этот день не было визитов
            df_ratio_panel['ED_Visits'] / df_ratio_panel['Total_Visits_for_Ratio']
        )
        
        nan_count = df_ratio_panel['Share_ED'].isna().sum()
        print(f"✅ 'Share_ED' создана. {nan_count} наблюдений (дней без визитов) будут исключены.")


        # --- Шаг 4: Создать финальный DiD DataFrame ---
        print("\n" + "="*50)
        print("--- Шаг 4: Создание финального DiD DataFrame ---")
        print("="*50)
        
        # 4.1. Соединяем панель 'долей' (левая) с картой событий (правая)
        df_did_panel_share = df_ratio_panel.merge(
            df_treatment_map,
            left_on='Filtered_Patient_ZipCode',
            right_on='Zip',
            how='left'
        )
        
        # 4.2. Создаем переменные DiD
        df_did_panel_share['Treated'] = (~df_did_panel_share['event_date'].isna()).astype(int)
        
        df_did_panel_share['Post'] = (
            (df_did_panel_share['Date'] >= df_did_panel_share['event_date']) & 
            (df_did_panel_share['Treated'] == 1)
        ).astype(int)
        
        df_did_panel_share['DiD_New'] = (
            df_did_panel_share['Post'] * (df_did_panel_share['clinic_type'] == 'New')
        ).astype(int)
        
        df_did_panel_share['DiD_Acquired'] = (
            df_did_panel_share['Post'] * (df_did_panel_share['clinic_type'] == 'Acquired')
        ).astype(int)

        print("✅ DiD переменные для 'Share_ED' созданы.")


        # --- Шаг 5: Запустить Модель DiD (PanelOLS) ---
        print("\n" + "="*50)
        print("--- РЕЗУЛЬТАТЫ (STANDALONE): Y = Доля ED визитов (Share_ED) ---")
        print("="*50)
        
        # 5.1. Установить мульти-индекс
        df_model_share = df_did_panel_share.set_index(['Filtered_Patient_ZipCode', 'Date'])

        # 5.2. Определить Y и X
        Y_share = df_model_share['Share_ED']
        X_share = sm.add_constant(df_model_share[['DiD_New', 'DiD_Acquired']])

        # 5.3. Запустить модель
        print("...Запуск PanelOLS (EntityEffects=True, TimeEffects=True)...")
        model_share = PanelOLS(Y_share, X_share, entity_effects=True, time_effects=True, drop_absorbed=True)
        results_share = model_share.fit(cov_type='clustered', cluster_entity=True)
        
        # 5.4. Вывести результат
        print(results_share.summary)

    except Exception as e:
        print(f"❌ Произошла ошибка при выполнении скрипта: {e}")

print("\n--- ✅ Анализ 'Эффекта Замещения' (Доля ED) ЗАВЕРШЕН ---")

In [ ]:
import pandas as pd
import os # Для проверки, что файл на месте

# Имя файла, который нужно проверить
filename = "OSFResearch_standalone_clinics_final_excel.xlsx"

print(f"--- 🚀 Попытка загрузить файл: {filename} ---")

try:
    # Проверяем, существует ли файл
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Файл '{filename}' не найден в текущей директории.")
        
    # Загружаем Excel-файл (без загрузки данных листов,
    # это самый быстрый способ получить имена)
    xls = pd.ExcelFile(filename)
    
    # Получаем список имен всех листов (sheets)
    sheet_names = xls.sheet_names
    
    print(f"\n✅ Файл '{filename}' успешно загружен.")
    print(f"Он содержит {len(sheet_names)} листов (страниц):")
    
    # Выводим все имена листов
    print("-" * 30)
    for name in sheet_names:
        print(name)
    print("-" * 30)

except FileNotFoundError as e:
    print(f"\n❌ ОШИБКА: {e}")
    print("Пожалуйста, убедитесь, что файл находится в той же папке, где запущен скрипт.")
except Exception as e:
    print(f"\n❌ Неожиданная ОШИБКА при чтении файла: {e}")
    print("Возможно, файл поврежден или это не Excel-файл.")

print("\n🎉 --- Проверка листов завершена ---")

In [ ]:
import pandas as pd
import numpy as np
import os # Для проверки существования файла

print("--- 🚀 НАЧАЛО: Расчет Индекса Конкуренции (Адаптированный под Excel) ---")

# 1. "Золотой список" (за вычетом 62035)
golden_list_zips = ['61568', '61517', '61434', '61361', '60450', '61473', '61866', '61376']
print(f"Обрабатываем {len(golden_list_zips)} ZIP-кодов из 'Золотого списка'...")

# 2. Имя ОДНОГО Excel-файла
excel_filename = "OSFResearch_standalone_clinics_final_excel.xlsx"

# 3. Подготовка цикла
competition_results = []

# 4. Проверяем, что ОСНОВНОЙ файл существует *перед* циклом
try:
    if not os.path.exists(excel_filename):
        raise FileNotFoundError(f"Главный файл не найден: {excel_filename}")
        
    # Загружаем файл один раз, чтобы получить список листов
    xls = pd.ExcelFile(excel_filename)
    available_sheets = xls.sheet_names
    print(f"✅ Главный файл '{excel_filename}' загружен. Доступные листы: {len(available_sheets)}")

    # 5. Обработка каждого листа из "золотого списка"
    for zip_code in golden_list_zips:
        
        # Проверяем, есть ли этот ZIP в списке листов
        if zip_code not in available_sheets:
            print(f"❌ ОШИБКА: Лист (sheet) с именем '{zip_code}' не найден в {excel_filename}. Записываю NaN.")
            competition_results.append({'ZipCode': zip_code, 'Num_Competitors': np.nan})
            continue # Переход к следующему zip_code

        # --- АДАПТАЦИЯ ---
        # Читаем "грязный" лист (sheet) из Excel, а не отдельный CSV
        # Убираем 'on_bad_lines', так как это параметр для read_csv
        try:
            df_comp = pd.read_excel(excel_filename, sheet_name=zip_code)
        except Exception as e_read:
            print(f"❌ ОШИБКА при чтении листа '{zip_code}': {e_read}. Записываю NaN.")
            competition_results.append({'ZipCode': zip_code, 'Num_Competitors': np.nan})
            continue

        # Проверяем, что колонка 'Clinic Name' существует
        if 'Clinic Name' not in df_comp.columns:
            print(f"⚠️ Предупреждение: Колонка 'Clinic Name' не найдена на листе '{zip_code}'. Пропуск.")
            competition_results.append({'ZipCode': zip_code, 'Num_Competitors': np.nan})
            continue 

        # 6. Очистка и подсчет (эта логика остается той же)
        
        # Убедимся, что 'Clinic Name' является строкой
        df_comp['Clinic Name'] = df_comp['Clinic Name'].astype(str)
        
        # Создаем фильтры
        is_not_osf = ~df_comp['Clinic Name'].str.contains("OSF", case=False, na=False)
        is_not_header = ~df_comp['Clinic Name'].str.contains("Clinic Name", case=False, na=False)
        is_not_empty = df_comp['Clinic Name'].str.len() > 1
        
        # Применяем фильтры
        df_competitors = df_comp[is_not_osf & is_not_header & is_not_empty]
        
        # Считаем
        count = len(df_competitors)
        
        # Записываем результат
        competition_results.append({'ZipCode': zip_code, 'Num_Competitors': count})
        print(f"  ✅ Лист '{zip_code}': Найдено {count} конкурентов.")

except FileNotFoundError as e:
    print(f"❌ КРИТИЧЕСКАЯ ОШИБКА: {e}")
    print("Работа остановлена. Убедитесь, что файл Excel находится в той же папке.")
except Exception as e:
    print(f"❌ Неожиданная ОШИБКА при чтении {excel_filename}: {e}")

# 7. Создание и Показ Результата
df_competition_index = pd.DataFrame(competition_results)

print("\n--- Итоговый Индекс Конкуренции ---")
print(df_competition_index)

# 8. Сохранение в памяти для следующего запроса
globals()['df_competition_index'] = df_competition_index
print("\n✅ DataFrame 'df_competition_index' сохранен в глобальной области.")
print("🎉 --- Задача выполнена! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
import warnings

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО: Модель с Взаимодействием (Конкуренция) ---")

# --- 1. Проверка наличия DataFrame ---
try:
    if 'final_data' not in locals() and 'final_data' not in globals():
        raise NameError("❌ ОШИБКА: DataFrame 'final_data' не найден.")
    if 'df_treatment_map' not in locals() and 'df_treatment_map' not in globals():
        raise NameError("❌ ОШИБКА: DataFrame 'df_treatment_map' не найден.")
    if 'df_competition_index' not in locals() and 'df_competition_index' not in globals():
        raise NameError("❌ ОШИБКА: DataFrame 'df_competition_index' не найден (Запрос 1).")
        
    print("✅ Все 3 DataFrame (final_data, df_treatment_map, df_competition_index) найдены.")

    # --- 2. Заново собрать Панель DiD ---
    print("... Шаг 2: Сборка панели DiD ...")
    
    # 2.1 Агрегация (без изменений)
    df_panel = final_data.copy()
    df_panel['ED_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Regular_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    
    df_all_visits_panel = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date']).agg(
        ED_Visits = pd.NamedAgg(column='ED_Visits', aggfunc='sum'),
        Regular_Visits = pd.NamedAgg(column='Regular_Visits', aggfunc='sum')
    ).reset_index()

    # 2.2 Создание Y-переменных (без изменений)
    df_all_visits_panel['Total_Visits_for_Ratio'] = df_all_visits_panel['ED_Visits'] + df_all_visits_panel['Regular_Visits']
    df_all_visits_panel['Share_ED'] = np.where(
        df_all_visits_panel['Total_Visits_for_Ratio'] > 0,
        df_all_visits_panel['ED_Visits'] / df_all_visits_panel['Total_Visits_for_Ratio'],
        0
    )
    df_all_visits_panel['Log_ED_Visits'] = np.log1p(df_all_visits_panel['ED_Visits'])
    df_all_visits_panel['Log_Regular_Visits'] = np.log1p(df_all_visits_panel['Regular_Visits'])

    # -----------------------------------------------------------------
    # --- 2.3 Мердж с картой лечения (ИСПРАВЛЕНИЕ ЗДЕСЬ) ---
    # Мы используем left_on и right_on, как в твоем рабочем скрипте
    # -----------------------------------------------------------------
    df_did_panel = pd.merge(
        df_all_visits_panel,
        df_treatment_map, 
        left_on='Filtered_Patient_ZipCode', # Key в df_all_visits_panel
        right_on='Zip',                 # Key в df_treatment_map
        how='left'
    )
    # -----------------------------------------------------------------

    # 2.4 Создание DiD переменных
    df_did_panel['event_date'] = pd.to_datetime(df_did_panel['event_date'])
    df_did_panel['Date'] = pd.to_datetime(df_did_panel['Date'])
    
    df_did_panel['Treated'] = df_did_panel['event_date'].notna().astype(int)
    df_did_panel['Post'] = (df_did_panel['Date'] >= df_did_panel['event_date']).astype(int)
    
    df_did_panel['DiD_New'] = ((df_did_panel['clinic_type'] == 'New') & (df_did_panel['Post'] == 1)).astype(int)
    df_did_panel['DiD_Acquired'] = ((df_did_panel['clinic_type'] == 'Acquired') & (df_did_panel['Post'] == 1)).astype(int)
    print("✅ Панель DiD создана.")

    # --- 3. Добавить Конкуренцию (НОВЫЙ ШАГ) ---
    print("... Шаг 3: Добавление Индекса Конкуренции ...")
    
    # Этот merge также использует left_on / right_on
    df_did_panel = pd.merge(
        df_did_panel,
        df_competition_index, # DataFrame из 8 ZIP
        left_on='Filtered_Patient_ZipCode',
        right_on='ZipCode',
        how='left'
    )
    
    # Очистка: удаляем дублирующую колонку 'ZipCode' (у нас их 2)
    # pd.merge создает 'ZipCode_x' (из df_treatment_map) и 'ZipCode_y' (из df_competition_index)
    if 'ZipCode_x' in df_did_panel.columns:
        df_did_panel = df_did_panel.drop(columns='ZipCode_x')
    if 'ZipCode_y' in df_did_panel.columns:
        df_did_panel = df_did_panel.drop(columns='ZipCode_y')
    # Также удалим колонку 'Zip' из df_treatment_map
    if 'Zip' in df_did_panel.columns:
        df_did_panel = df_did_panel.drop(columns='Zip')


    # Заполняем NaN нулями
    df_did_panel['Num_Competitors'] = df_did_panel['Num_Competitors'].fillna(0)
    
    # Создаем Переменные Взаимодействия
    df_did_panel['DiD_New_x_Comp'] = df_did_panel['DiD_New'] * df_did_panel['Num_Competitors']
    df_did_panel['DiD_Acq_x_Comp'] = df_did_panel['DiD_Acquired'] * df_did_panel['Num_Competitors']
    print("✅ Переменные взаимодействия (x_Comp) созданы.")
    
    # --- 4. Подготовить к запуску ---
    print("... Шаг 4: Подготовка данных к PanelOLS ...")
    
    X_vars = ['DiD_New', 'DiD_Acquired', 'DiD_New_x_Comp', 'DiD_Acq_x_Comp']
    Y_vars = ['Log_ED_Visits', 'Log_Regular_Visits', 'Share_ED']
    
    # Устанавливаем мульти-индекс
    df_model_ready = df_did_panel.set_index(['Filtered_Patient_ZipCode', 'Date'])
    
    initial_rows = len(df_model_ready)
    df_model_ready = df_model_ready.dropna(subset=Y_vars + X_vars)
    final_rows = len(df_model_ready)
    
    print(f"Изначально строк в панели: {initial_rows}")
    print(f"Строк для модели (после dropna): {final_rows} ({(initial_rows - final_rows)} удалено)")
    
    # Определяем X
    X = sm.add_constant(df_model_ready[X_vars])

    # --- 5. Запустить 3 Модели ---
    print("\n--- Шаг 5: Запуск 3 моделей PanelOLS ---")

    models_to_run = [
        ('Log_ED_Visits', "--- РЕЗУЛЬТАТЫ (ВЗАИМОДЕЙСТВИЕ): Y = Log_ED_Visits ---"),
        ('Log_Regular_Visits', "--- РЕЗУЛЬТАТЫ (ВЗАИМОДЕЙСТВИЕ): Y = Log_Regular_Visits ---"),
        ('Share_ED', "--- РЕЗУЛЬТАТЫ (ВЗАИМОДЕЙСТВИЕ): Y = Share_ED ---")
    ]

    for y_name, title in models_to_run:
        Y = df_model_ready[y_name]
        
        # Модель с FE для ZIP и Даты
        model = PanelOLS(Y, X, entity_effects=True, time_effects=True)
        
        # Кластеризуем ошибки на уровне ZIP
        results = model.fit(cov_type='clustered', cluster_entity=True)
        
        # Вывод
        print("\n" + "="*70)
        print(title)
        print("="*70)
        print(results.summary)

except NameError as e:
    print(f"\n❌ КРИТИЧЕСКАЯ ОШИБКА: {e}")
    print("Скрипт остановлен. Пожалуйста, убедитесь, что все необходимые DataFrame есть в памяти.")
except Exception as e:
    print(f"\n❌ Неожиданная ошибка: {e}")
    raise

print("\n🎉 --- Модель с Взаимодействием завершена! ---")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
from tqdm import tqdm
import os
import requests  # Для скачивания
import zipfile   # Для распаковки
import io        # Для работы с zip в памяти

# --- НОВЫЕ ИМПОРТЫ ---
try:
    import geopandas as gpd
    from shapely.geometry import Point
except ImportError:
    print("="*50)
    print("❌ ОШИБКА: Библиотека 'geopandas' не найдена.")
    print("Пожалуйста, установите ее, используя: conda install geopandas")
    print("="*50)
    raise

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid") 
plt.rcParams['figure.figsize'] = (10, 12) 

print("--- 🚀 НАЧАЛО СОЗДАНИЯ КАРТЫ ---")

# --- АВТО-ЗАГРУЗКА ФАЙЛА КАРТЫ ---
# Эта часть решает твою проблему.
# -----------------------------------
SHAPEFILE_DIR = 'tl_2023_us_state'
SHAPEFILE_NAME = 'tl_2023_us_state.shp'
SHAPEFILE_PATH = os.path.join(SHAPEFILE_DIR, SHAPEFILE_NAME)
SHAPEFILE_URL = 'https://www2.census.gov/geo/tiger/TIGER2023/STATE/tl_2023_us_state.zip'

try:
    # Проверяем, существует ли уже файл .shp
    if not os.path.exists(SHAPEFILE_PATH):
        print(f"Файл карты '{SHAPEFILE_PATH}' не найден.")
        print(f"... 🔽 Скачиваю файл карты с {SHAPEFILE_URL} ...")
        
        # Скачиваем zip-архив
        r = requests.get(SHAPEFILE_URL)
        r.raise_for_status() # Проверяем, что скачалось
        
        # Создаем папку
        os.makedirs(SHAPEFILE_DIR, exist_ok=True)
        
        # Распаковываем
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extractall(SHAPEFILE_DIR)
        
        print(f"✅ Файл карты скачан и распакован в папку '{SHAPEFILE_DIR}'.")
    else:
        print(f"✅ Файл карты '{SHAPEFILE_PATH}' уже существует.")

except requests.exceptions.RequestException as e:
    print(f"❌ ОШИБКА: Не удалось скачать файл карты: {e}")
    print("Проверьте интернет-соединение или URL.")
    raise
except Exception as e_zip:
    print(f"❌ ОШИБКА: Не удалось распаковать файл: {e_zip}")
    raise
# -----------------------------------
# --- КОНЕЦ АВТО-ЗАГРУЗКИ ---


# --- 1. Загрузка и Очистка Данных ---
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()

df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
print("✅ Данные клиник загружены и очищены.")

# --- 2. Идентификация Групп Клиник ---
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)

df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_pc_urgent, axis=1)].copy()

df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_urgent, axis=1))].copy()

def is_hospital(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return 'hospital' in facility_name or 'medical center' in facility_name

df_hospitals = df_clinics_info_full[df_clinics_info_full.apply(is_hospital, axis=1)].copy()
df_hospitals = df_hospitals.drop_duplicates(subset=['clinic_lat', 'clinic_lon'])


print(f"Идентифицировано: {len(df_hospitals)} Госпиталей")
print(f"Идентифицировано: {len(df_acquired_pc_clinics)} Acquired PC Клиник")
print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent Клиник")

# --- 3. Преобразование в GeoDataFrames ---
def create_gdf(df, lon_col='clinic_lon', lat_col='clinic_lat'):
    """Преобразует pandas.DataFrame в geopandas.GeoDataFrame"""
    if df.empty:
        return gpd.GeoDataFrame(geometry=[])
    geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    return gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

gdf_hospitals = create_gdf(df_hospitals)
gdf_acquired_pc = create_gdf(df_acquired_pc_clinics)
gdf_new_pc = create_gdf(df_new_PC_clinics)
print("✅ Данные преобразованы в GeoDataFrames.")

# --- 4. Отрисовка Карты ---
print(f"... Загрузка Shapefile из {SHAPEFILE_PATH} ...")
try:
    # Загружаем карту всех штатов
    # Теперь SHAPEFILE_PATH - это правильный путь к скачанному файлу
    usa_map = gpd.read_file(SHAPEFILE_PATH) 
    
    # Выбираем только Иллинойс (по имени)
    illinois_map = usa_map[usa_map['NAME'] == 'Illinois'] 
    
    if illinois_map.empty:
        print("❌ ОШИБКА: Не удалось найти 'Illinois' в Shapefile. Проверьте колонку 'NAME' или 'STUSPS'.")
        print("Доступные колонки:", usa_map.columns)
    else:
        print("✅ Shapefile Иллинойса загружен.")

        # Создаем полотно для карты
        fig, ax = plt.subplots(figsize=(10, 12))
        
        # 1. Рисуем контур Иллинойса
        illinois_map.plot(
            ax=ax, 
            color='white', 
            edgecolor='black', 
            linewidth=2 
        )
        
        # 2. Наносим Госпитали (Зеленые звезды)
        if not gdf_hospitals.empty:
            gdf_hospitals.plot(
                ax=ax,
                marker='*', 
                color='darkgreen', 
                markersize=250, 
                label=f'OSF Hospitals ({len(gdf_hospitals)})'
            )
            
        # 3. Наносим Acquired PC (Красные квадраты)
        if not gdf_acquired_pc.empty:
            gdf_acquired_pc.plot(
                ax=ax,
                marker='s', 
                color='red',
                markersize=80,
                label=f'Acquired PC Clinics ({len(gdf_acquired_pc)})'
            )

        # 4. Наносим New PC/Urgent (Синие круги)
        if not gdf_new_pc.empty:
            gdf_new_pc.plot(
                ax=ax,
                marker='o', 
                color='blue',
                markersize=80,
                label=f'New PC/Urgent Clinics ({len(gdf_new_pc)})'
            )

        # Настройка карты
        ax.set_title('Geographical Distribution of OSF HealthCare Facilities in Illinois', fontsize=16)
        
        # Добавляем легенду
        ax.legend(title='Facility Type', fontsize=12, title_fontsize=13, loc='upper right')
        
        # Убираем оси с широтой и долготой
        ax.set_axis_off() 
        
        plt.tight_layout()
        plt.show()
        print("✅ Карта создана!")

except FileNotFoundError:
    # Эта ошибка теперь не должна случиться, но оставим на всякий случай
    print(f"❌ ОШИБКА: Shapefile не найден по пути: {SHAPEFILE_PATH}")
except Exception as e:
    print(f"❌ Произошла ошибка при создании карты: {e}")

print("\n🎉 --- Создание Карты Завершено ---")

# 9. Difference-in-Differences (DiD) Analysis: Clinic Events

In this section, we shift our methodology to a Difference-in-Differences (DiD) estimation. The primary goal is to measure the causal impact of a clinic "event" — either the opening of a **New** clinic or the integration of an **Acquired** one — on patient visit volumes (Primary vs. Emergency).

Our central hypothesis is that this "treatment" (the clinic event) will cause a change in patient visit patterns in the surrounding geographic area. Crucially, we hypothesize that this effect is **heterogeneous** and will vary depending on a patient's proximity (distance) to the clinic.

## 9.1. Data Preparation and Merging

1.  **Load Auxiliary Data**: We begin by loading two essential helper files:
    * `OSFResearch_Clinics_mastersheet_v2.csv`: This file contains the list of clinics, their `clinic_type` ('New' or 'Acquired'), their `Zip` code, and the specific `event_date` (the date of opening or acquisition).
    * `Crosswalk_IL.csv`: A geospatial crosswalk file containing the precise latitude (`IntPtLat`) and longitude (`IntPtLon`) for Illinois ZIP codes (ZCTAs).
2.  **Data Cleaning**: We perform a final standardization pass on all DataFrames. This ensures ZIP codes are treated as strings (e.g., removing trailing `.0`), all clinic and patient ZIP codes have valid coordinates, and all `event_date` entries are in the correct datetime format.

## 9.2. Defining Treatment: The "Treatment Map"

This is the most critical step for defining our treatment and control groups. We cannot use the `Distance_km` from our original `final_data`, as that measures the distance to whichever department a patient happened to visit.

Instead, we must definitively link each patient to a *single* potential treatment. We do this by creating a **"Treatment Map"**:

1.  We compile a list of all unique `Filtered_Patient_ZipCode` values from our `final_data`.
2.  For each unique patient ZIP, we iterate through *every* clinic in our `df_clinics_info` file and calculate the precise geospatial distance (using `geopy.distance.geodesic`) to it.
3.  We find the **single nearest clinic** to that patient ZIP code.
4.  We then create a map that links each `Filtered_Patient_ZipCode` to its closest clinic's attributes: `clinic_type`, `event_date`, and `distance_km`.

This map assigns every patient ZIP code to its most relevant "treatment center," which is essential for the DiD setup.

## 9.3. Aggregation and DiD Variable Creation

1.  **Aggregate Data**: With the Treatment Map ready, we aggregate our main `final_data`. We group by `(Filtered_Patient_ZipCode, Date)` to get the total `Emergency_Visits` and `Primary_Visits` for each ZIP code on each day.
2.  **Merge Treatment Map**: We merge the Treatment Map onto this aggregated data. Now, each ZIP-day observation is tagged with its nearest clinic's info.
3.  **Create DiD Variables**: We then create our key analytic variables:
    * `Treat`: A binary variable defining the treatment group. We set a **50 km radius** as our "treatment area." `Treat = 1` if `distance_km <= 50`, and `0` otherwise.
    * `Post`: A binary variable that "switches on" after the treatment occurs. `Post = 1` if the observation's `Date` is on or *after* the clinic's `event_date`, and `0` before.
    * `distance_category`: To test our heterogeneous effects hypothesis, we bin the `distance_km` into four categories: '0-5 km', '5-10 km', '10-20 km', and '20+ km'.

## 9.4. Model Specification (Triple Interaction)

We estimate our model using OLS with two-way fixed effects (TWFE) to control for unobserved confounders.

* **Entity Fixed Effects**: `C(Zip)` controls for all time-invariant characteristics of a ZIP code (e.g., demographics, baseline health, travel infrastructure).
* **Time Fixed Effects**: `C(Year) + C(Month)` controls for common shocks or seasonality that affect all ZIP codes simultaneously.

The core of our specification is a **triple interaction term**: `Treat * Post * C(distance_category)`

This allows the main DiD effect (the `Treat:Post` interaction) to be *different* for each of our distance categories. We set '0-5 km' as the reference category. The full specification is:

$Y_{it} = \beta_0 + \beta_1(Treat_i \times Post_{it}) + \sum_{k} \delta_k(Treat_i \times Post_{it} \times DistanceCategory_{ik}) + \gamma_i + \lambda_t + \epsilon_{it}$

* $Y_{it}$ is the outcome (Primary or Emergency Visits) for ZIP $i$ at time $t$.
* $\gamma_i$ are the ZIP Code Fixed Effects.
* $\lambda_t$ are the Time (Year + Month) Fixed Effects.
* $\beta_1$ is the DiD effect for the reference distance category ('0-5 km').
* $\delta_k$ is the *additional* effect for distance category $k$ (e.g., '5-10 km') relative to the base category.

## 9.5. Analysis and Visualization

We run four separate estimations to isolate the effects:
1.  **Model 1A**: `New` clinics, `Primary_Visits` outcome
2.  **Model 1B**: `New` clinics, `Emergency_Visits` outcome
3.  **Model 2A**: `Acquired` clinics, `Primary_Visits` outcome
4.  **Model 2B**: `Acquired` clinics, `Emergency_Visits` outcome

Finally, we define a plotting function `plot_did_effects_by_distance`. This function is designed to extract the *absolute* DiD effect for each distance bin. It uses `model_result.t_test()` to calculate the linear combination of coefficients (e.g., $\beta_1 + \delta_k$) and its 95% confidence interval. This allows us to visualize how the total treatment effect changes as we move further away from the clinic.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 STARTING 'Task Plan' DiD (Correct Logic) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    print(f"✅ 'final_data' found. {len(final_data)} rows.")
    
    # --- 2. Load helper files ---
    try:
        # This file was created by our pre-processing script
        df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
        print("✅ 'OSFResearch_Clinics_mastersheet_v2.csv' and 'Crosswalk_IL.csv' loaded.")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        print("    Did you run the 'Step 0: Pre-processing' code block first?")
        raise
        
    # --- 3. Clean Data (as requested) ---
    
    # 3.1 Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    final_data['zcta5'] = final_data['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    
    # 3.2 Clean Crosswalk (coordinates)
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # 3.3 Clean df_clinics_info (clinic info)
    
    # ⬇️ *** ИСПРАВЛЕНИЕ 1 ***
    # Используем 'Zip' (с большой буквы) из файла
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    
    # ⬇️ *** ИСПРАВЛЕНИЕ 2 ***
    # Используем оригинальные имена колонок
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    
    # ⬇️ *** ИСПРАВЛЕНИЕ 3 ***
    # Объединяем по 'Zip' (с большой буквы)
    df_clinics_info = df_clinics_info.merge(zip_coords, left_on='Zip', right_index=True)
    
    # This dropna is good
    df_clinics_info = df_clinics_info.dropna(subset=['IntPtLat', 'IntPtLon', 'event_date', 'clinic_type'])
    print(f"✅ Cleaning complete. Found {len(df_clinics_info)} clinics with coordinates.")

    # --- 4. Create "Treatment Map" ---
    print("...Creating 'treatment map' (finding nearest clinic for each ZIP)...")
    
    # Get unique patient ZIPs from the main data
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords, left_on='Filtered_Patient_ZipCode', right_index=True)
    patient_zips_df = patient_zips_df.dropna() # Remove ZIPs without coordinates

    treatment_map = []
    
    # Loop over each patient ZIP
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['IntPtLat'], patient_row['IntPtLon'])
        
        min_distance = np.inf
        nearest_clinic = {}
        
        # Find the single closest clinic
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['IntPtLat'], clinic_row['IntPtLon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            
            if distance < min_distance:
                min_distance = distance
                nearest_clinic = {
                    'clinic_type': clinic_row['clinic_type'],
                    'event_date': clinic_row['event_date']
                }
        
        if nearest_clinic:
            treatment_map.append({
                'Filtered_Patient_ZipCode': patient_zip,
                'distance_km': min_distance,
                'clinic_type': nearest_clinic['clinic_type'],
                'event_date': nearest_clinic['event_date']
            })

    df_treatment_map = pd.DataFrame(treatment_map)
    print(f"✅ 'Treatment map' created for {len(df_treatment_map)} ZIP codes.")

    # --- 5. Aggregate Data and Create DiD Variables ---
    print("...Aggregating visits and creating DiD variables...")
    
    # Create Y_primary and Y_emergency
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    
    # Aggregate to the ZIP-Date level
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({
        'Emergency_Visits': 'sum', 
        'Primary_Visits': 'sum'
    }).reset_index()

    # Merge the treatment map to the aggregated data
    df_main = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left')
    df_main = df_main.dropna(subset=['distance_km', 'clinic_type', 'event_date'])
    
    # Create DiD variables
    TREATMENT_RADIUS_KM = 50 # Treatment area is 50 km
    df_main['Treat'] = (df_main['distance_km'] <= TREATMENT_RADIUS_KM).astype(int)
    df_main['Post'] = (df_main['Date'] >= df_main['event_date']).astype(int)
    
    # Create distance categories (in KM)
    bins_km = [-np.inf, 5, 10, 20, np.inf]
    labels_km = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    df_main['distance_category'] = pd.cut(
        df_main['distance_km'],
        bins=bins_km,
        labels=labels_km,
        right=False
    )
    
    # Rename ZIP for the formula
    df_main = df_main.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    
    print(f"✅ Final DataFrame 'df_main' created. {len(df_main)} rows.")
    print(df_main['clinic_type'].value_counts())

    # --- 6. Define Formulas (Step 3 from Plan) ---
    Y_PRIMARY_NAME = 'Primary_Visits'
    Y_EMERGENCY_NAME = 'Emergency_Visits'
    CONTROLS_STR = 'C(Year) + C(Month) + C(Zip)'
    
    interaction_term = "Treat * Post * C(distance_category)"
    formula_primary = f"{Y_PRIMARY_NAME} ~ {interaction_term} + {CONTROLS_STR}"
    formula_emergency = f"{Y_EMERGENCY_NAME} ~ {interaction_term} + {CONTROLS_STR}"

    print(f"\n--- Formulas are ready ---")
    print(f"Formula (Primary): {formula_primary}")
    print(f"Formula (Emergency): {formula_emergency}")

    # --- 7. Run Models (Step 4 from Plan) ---
    print("\n--- Step 4: Running 4 models ---")
    
    # Set base category
    df_main['distance_category'] = pd.Categorical(
        df_main['distance_category'],
        categories=labels_km,
        ordered=True
    )

    # Split data
    df_new = df_main[df_main['clinic_type'] == 'New'].copy()
    df_acquired = df_main[df_main['clinic_type'] == 'Acquired'].copy()

    print(f"Records for 'New' clinics: {len(df_new)}")
    print(f"Records for 'Acquired' clinics: {len(df_acquired)}")
    
    models = {} # Dictionary to store models

    try:
        if not df_new.empty:
            print("...Running 1A: New, Primary")
            models['1A'] = smf.ols(formula_primary, data=df_new).fit(
                cov_type='cluster', cov_kwds={'groups': df_new['Zip']}
            )
            print("...Running 1B: New, Emergency")
            models['1B'] = smf.ols(formula_emergency, data=df_new).fit(
                cov_type='cluster', cov_kwds={'groups': df_new['Zip']}
            )
        else:
            print("⚠️ 'New' DataFrame is empty, skipping 1A and 1B.")

        if not df_acquired.empty:
            print("...Running 2A: Acquired, Primary")
            models['2A'] = smf.ols(formula_primary, data=df_acquired).fit(
                cov_type='cluster', cov_kwds={'groups': df_acquired['Zip']}
            )
            print("...Running 2B: Acquired, Emergency")
            models['2B'] = smf.ols(formula_emergency, data=df_acquired).fit(
                cov_type='cluster', cov_kwds={'groups': df_acquired['Zip']}
            )
        else:
            print("⚠️ 'Acquired' DataFrame is empty, skipping 2A and 2B.")
        
        print("✅ Model estimation complete.")
        
    except Exception as e:
        print(f"❌ ERROR DURING MODEL FITTING: {e}")
        raise

    # --- 8. Visualization (Step 5 from Plan) ---
    print("\n--- Step 5: Visualizing Results ---")
    sns.set(style="whitegrid")

    # Function to extract absolute DiD effects (calculates sums of coefficients)
    def extract_absolute_did_effects(model_result, ref_category_label='0-5 km'):
        params = model_result.params
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        if 'Treat:Post' not in params.index:
            print(f"Warning: 'Treat:Post' not found. Plot cannot be generated.")
            return pd.DataFrame(results)
        
        # 1. Effect for the reference group
        effect = params['Treat:Post']
        conf = model_result.conf_int().loc['Treat:Post']
        results['Distance'].append(ref_category_label)
        results['Effect'].append(effect)
        results['Conf_Low'].append(conf[0])
        results['Conf_High'].append(conf[1])

        # 2. Effects for all other groups
        other_categories = [cat for cat in labels_km if cat != ref_category_label]
        
        for cat in other_categories:
            term_name = f"Treat:Post:C(distance_category)[T.{cat}]"
            if term_name in params.index:
                try:
                    # Test the hypothesis H_0: (Treat:Post + Interaction_Term) = 0
                    t_test_result = model_result.t_test(f"Treat:Post + {term_name}")
                    effect = t_test_result.effect[0]
                    conf_low = t_test_result.conf_int()[0][0]
                    conf_high = t_test_result.conf_int()[0][1]
                    results['Distance'].append(cat)
                    results['Effect'].append(effect)
                    results['Conf_Low'].append(conf_low)
                    results['Conf_High'].append(conf_high)
                except Exception as e:
                    print(f"t_test error for {term_name}: {e}")
        return pd.DataFrame(results)

    # Function to plot the results
    def plot_did_effects_by_distance(model_result, title, category_order):
        plot_data = extract_absolute_did_effects(model_result, ref_category_label=category_order[0])
        if plot_data.empty:
            print(f"No data to plot for: {title}")
            return

        plot_data['Distance'] = pd.Categorical(plot_data['Distance'], categories=category_order, ordered=True)
        plot_data = plot_data.sort_values('Distance')
        plot_data['err_low'] = plot_data['Effect'] - plot_data['Conf_Low']
        plot_data['err_high'] = plot_data['Conf_High'] - plot_data['Effect']
        errors = [plot_data['err_low'], plot_data['err_high']]

        plt.figure(figsize=(10, 6))
        plt.errorbar(
            x=plot_data['Distance'], y=plot_data['Effect'], yerr=errors,
            fmt='o', capsize=5, linestyle='-', markersize=8, label='DiD Effect'
        )
        plt.axhline(y=0, color='red', linestyle='--', linewidth=1, label='No Effect (Y=0)')
        plt.title(title, fontsize=16, pad=20)
        plt.xlabel("Distance Category to Nearest Clinic (km)", fontsize=12)
        plt.ylabel("DiD Effect Size (Change in Visits)", fontsize=12)
        plt.legend()
        plt.tight_layout()
        plt.show()

    # --- Plot the 4 graphs ---
    if '1A' in models:
        plot_did_effects_by_distance(models['1A'], f"Model 1A: New Clinics - {Y_PRIMARY_NAME}", labels_km)
    if '1B' in models:
        plot_did_effects_by_distance(models['1B'], f"Model 1B: New Clinics - {Y_EMERGENCY_NAME}", labels_km)
    if '2A' in models:
        plot_did_effects_by_distance(models['2A'], f"Model 2A: Acquired Clinics - {Y_PRIMARY_NAME}", labels_km)
    if '2B' in models:
        plot_did_effects_by_distance(models['2B'], f"Model 2B: Acquired Clinics - {Y_EMERGENCY_NAME}", labels_km)

    print("\n🎉 --- 'Task Plan' DiD Complete! ---")

# 10. DiD Analysis (Method 2): Visit Rates within a 15km Radius

This section presents an alternative specification to test the robustness of our findings. This method differs from the previous one in two fundamental ways:

1.  **Dependent Variable**: We use **visit rates per 1,000 people** instead of absolute visit counts.
2.  **Sample Definition**: We focus *only* on the "treatment zone," filtering our sample to include *only* ZIP codes within a 15km radius of their nearest clinic.

## 10.1. Data Preparation: Population & Visit Rates

To properly compare visit volumes across ZIP codes of different sizes, we must normalize our dependent variable.

1.  **Load Population Data**: We load `TotPopACS` (Total Population) from the `Crosswalk_IL.csv` file for each ZIP code.
2.  **Calculate Visit Rates**: We create two new dependent variables:
    * `Primary_Visits_Rate = (Primary_Visits / TotPopACS) * 1000`
    * `Emergency_Visits_Rate = (Emergency_Visits / TotPopACS) * 1000`
    
This transforms our outcome to "daily visits per 1,000 residents."

## 10.2. Sample Definition: 15km "Dive-In"

Instead of comparing a broad "treatment" group (e.g., < 50km) to a "control" group (> 50km), this method analyzes the treatment effect *within* a focused 15km radius.

1.  **Find Nearest Clinic**: We use the same "Treatment Map" logic as before to find the nearest clinic (and its `event_date` and `clinic_type`) for every patient ZIP code.
2.  **Filter Sample**: We create a new DataFrame, `df_dive`, by filtering the main dataset to keep *only* observations where `distance_patient_to_nearest_clinic_km <= 15`.
3.  **Define Bins**: We create more granular distance categories for this 15km radius: `0-2 km`, `2-6 km`, `6-10 km`, and `10-15 km`.

## 10.3. Model Specification (Simplified)

Since all ZIP codes in this `df_dive` sample are "treated" (i.e., within the 15km radius), our model specification changes.

We use a **Simplified Pooled OLS model** with time fixed effects. The model tests how the pre-post change in visit rates (`Post`) differs across the granular distance categories.

**Note**: This specification includes Time Fixed Effects (`C(Year) + C(Month)`) but **does not** include ZIP Code Fixed Effects (`C(Zip)`). This is a faster, simpler model but rests on the stronger assumption that there are no unobserved, time-invariant differences between the ZIP codes in our 15km sample.

The formula is:
$Y\_Rate_{it} = \beta_0 + \beta_1(Post_{it}) + \sum_{k} \delta_k(Post_{it} \times DistanceCategory_{ik}) + \lambda_t + \epsilon_{it}$

* $Y\_Rate_{it}$ is the visit rate for ZIP $i$ at time $t$.
* $\beta_1$ is the DiD effect for the reference distance category ('0-2 km').
* $\delta_k$ is the *additional* effect for distance category $k$ (relative to the '0-2 km' group).
* $\lambda_t$ are the Time (Year + Month) Fixed Effects.

## 10.4. Analysis & Visualization

We run the same four model variations on this new `df_dive` dataset. The final visualization plots the "New" and "Acquired" clinic effects on the same two graphs (one for Primary, one for Emergency) for direct comparison.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 START ANALYSIS: Method 2 (Radius around Clinic ZIP) with RATES ---")

# --- 1. Check final_data ---
if 'final_data' not in locals():
    print("❌ ERROR: 'final_data' DataFrame not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    # --- Load and Prepare Data ---
    df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")

    # Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # Load and clean population data
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    print(f"✅ Population data loaded and cleaned for {len(zip_population)} ZIP codes.")

    # ZIP Coordinates
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # Clinic Info (add clinic coordinates)
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'}) # Rename clinic's Zip
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    df_clinics_info = df_clinics_info.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])

    # --- Create Treatment Map (Method 2 - Radius around Clinic) ---
    print("...Creating 'treatment map' (finding nearest clinic for each patient ZIP)...")
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True).dropna()

    treatment_map = []
    # Find the nearest clinic for each patient (like in Method 1, to get event_date and type)
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        min_distance_to_any_clinic = np.inf
        nearest_clinic_info = {}
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_distance_to_any_clinic:
                min_distance_to_any_clinic = distance
                # Save info about the nearest clinic
                nearest_clinic_info = {
                    'distance_patient_to_nearest_clinic_km': distance, # This distance is used for bins
                    'clinic_type': clinic_row['clinic_type'],
                    'event_date': clinic_row['event_date']
                }
        if nearest_clinic_info:
            treatment_map.append({
                'Filtered_Patient_ZipCode': patient_zip,
                **nearest_clinic_info # Add info about the nearest clinic
            })
    df_treatment_map = pd.DataFrame(treatment_map)
    print(f"✅ 'Treatment map' (nearest clinic) created for {len(df_treatment_map)} ZIP codes.")


    # Aggregate Visits
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()

    # Merge Treatment Map and Population
    df_main = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left')
    df_main = df_main.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_main = df_main.dropna(subset=['distance_patient_to_nearest_clinic_km', 'clinic_type', 'event_date', 'TotPopACS'])

    # --- Define DiD Variables (Focus on 15km radius) ---
    ANALYSIS_RADIUS_KM = 15
    df_dive = df_main[df_main['distance_patient_to_nearest_clinic_km'] <= ANALYSIS_RADIUS_KM].copy()
    print(f"\n--- Focusing on {len(df_dive)} observations within a {ANALYSIS_RADIUS_KM} km radius of the nearest clinic ---")

    df_dive['Post'] = (df_dive['Date'] >= df_dive['event_date']).astype(int)

    # Granular bins based on distance to nearest clinic
    granular_bins_km = [-np.inf, 2, 6, 10, 15]
    granular_labels_km = ['0-2 km', '2-6 km', '6-10 km', '10-15 km']
    df_dive['distance_category_granular'] = pd.cut(
        df_dive['distance_patient_to_nearest_clinic_km'], # Use Patient-to-Clinic distance
        bins=granular_bins_km,
        labels=granular_labels_km,
        right=False
    )
    df_dive = df_dive.dropna(subset=['distance_category_granular'])
    print("\nCreated granular distance categories:")
    print(df_dive['distance_category_granular'].value_counts().sort_index())

    # Calculate Rates per 1000 people
    df_dive['Primary_Visits_Rate'] = (df_dive['Primary_Visits'] / df_dive['TotPopACS']) * 1000
    df_dive['Emergency_Visits_Rate'] = (df_dive['Emergency_Visits'] / df_dive['TotPopACS']) * 1000
    print("✅ Calculated visit rates per 1000 people.")

    # --- Define and Run SIMPLIFIED Models ---
    Y_PRIMARY_RATE_NAME = 'Primary_Visits_Rate'
    Y_EMERGENCY_RATE_NAME = 'Emergency_Visits_Rate'
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    # In this model, Treat=1 for all, as we filtered by radius
    interaction_term = "Post * C(distance_category_granular)"
    formula_primary_rate = f"{Y_PRIMARY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_rate = f"{Y_EMERGENCY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"

    print(f"\nFormula (Primary, Rate): {formula_primary_rate}")
    print(f"Formula (Emergency, Rate): {formula_emergency_rate}")

    print("\n--- Running models with rates ---")
    df_dive['distance_category_granular'] = pd.Categorical(df_dive['distance_category_granular'], categories=granular_labels_km, ordered=True)
    df_new_dive = df_dive[df_dive['clinic_type'] == 'New'].copy()
    df_acquired_dive = df_dive[df_dive['clinic_type'] == 'Acquired'].copy()
    models_dive_rate = {}
    try:
        if not df_new_dive.empty:
            print("...Running 1A (Rate): New, Primary")
            models_dive_rate['1A'] = smf.ols(formula_primary_rate, data=df_new_dive).fit()
            print("...Running 1B (Rate): New, Emergency")
            models_dive_rate['1B'] = smf.ols(formula_emergency_rate, data=df_new_dive).fit()
        if not df_acquired_dive.empty:
            print("...Running 2A (Rate): Acquired, Primary")
            models_dive_rate['2A'] = smf.ols(formula_primary_rate, data=df_acquired_dive).fit()
            print("...Running 2B (Rate): Acquired, Emergency")
            models_dive_rate['2B'] = smf.ols(formula_emergency_rate, data=df_acquired_dive).fit()
        print("✅ Models with rates have been estimated.")
    except Exception as e:
        print(f"❌ ERROR DURING MODEL FITTING (RATES): {e}")
        raise

    # --- Visualization ---
    print("\n--- Visualizing results with rates ---")
    sns.set(style="whitegrid")

    # (Functions extract_effects_simple and plot_comparison_graphs_simple remain the same)
    def extract_effects_simple(model_result, category_order):
        """
        Extracts the absolute 'Post' effect for each distance category.
        """
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        
        # Base effect (reference category)
        if 'Post' not in params.index:
            print("Warning: 'Post' coefficient not found. Cannot plot.")
            return pd.DataFrame(results)
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Post'])
        results['Conf_Low'].append(conf.loc['Post', 0])
        results['Conf_High'].append(conf.loc['Post', 1])
        
        # Interaction effects
        for cat in category_order[1:]:
            term_name = f"Post:C(distance_category_granular)[T.{cat}]"
            if term_name in params.index:
                # Get the absolute effect (Base + Interaction)
                t_test = model_result.t_test(f"Post + {term_name}")
                results['Distance'].append(cat)
                results['Effect'].append(t_test.effect[0])
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
                
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    def plot_comparison_graphs_simple(models_dict, category_order, y_primary, y_emergency):
        """
        Plots New vs. Acquired on two subplots (Primary and Emergency).
        """
        fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 16), sharex=True)

        # --- Primary Visits Plot ---
        ax1.set_title(f"DiD Effect on {y_primary} (per 1000 ppl)", fontsize=16, pad=20)
        plot_made = False
        if '1A' in models_dict:
            data = extract_effects_simple(models_dict['1A'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax1.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='-o', capsize=5, label='New Clinics')
                plot_made = True
        if '2A' in models_dict:
            data = extract_effects_simple(models_dict['2A'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax1.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='--s', capsize=5, label='Acquired Clinics')
                plot_made = True
        if plot_made:
            ax1.axhline(0, color='black', linestyle='--')
            ax1.set_ylabel("DiD Effect Size (Change in Daily Visits per 1000 People)")
            ax1.legend()
        else:
            ax1.text(0.5, 0.5, "No data to plot", horizontalalignment='center', verticalalignment='center', transform=ax1.transAxes)

        # --- Emergency Visits Plot ---
        ax2.set_title(f"DiD Effect on {y_emergency} (per 1000 ppl)", fontsize=16, pad=20)
        plot_made = False
        if '1B' in models_dict:
            data = extract_effects_simple(models_dict['1B'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax2.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='-o', capsize=5, label='New Clinics')
                plot_made = True
        if '2B' in models_dict:
            data = extract_effects_simple(models_dict['2B'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax2.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='--s', capsize=5, label='Acquired Clinics')
                plot_made = True
        if plot_made:
            ax2.axhline(0, color='black', linestyle='--')
            ax2.set_xlabel("Distance Category to Nearest Clinic (km)")
            ax2.set_ylabel("DiD Effect Size (Change in Daily Visits per 1000 People)")
            ax2.legend()
        else:
            ax2.text(0.5, 0.5, "No data to plot", horizontalalignment='center', verticalalignment='center', transform=ax2.transAxes)

        plt.tight_layout(pad=2.0)
        plt.show()

    # --- Run the plotting function for the rate-based models ---
    plot_comparison_graphs_simple(models_dive_rate, granular_labels_km, Y_PRIMARY_RATE_NAME, Y_EMERGENCY_RATE_NAME)

    print("\n🎉 --- Analysis of Method 2 (Radius around Clinic ZIP) with RATES is complete! ---")

# 9. DiD Visualization: Combined Comparative Plots

This is the final step for our first DiD analysis. Instead of plotting four separate graphs (one for each model), we will create two **comparative plots**. This is a much stronger way to present the findings, as it allows for a direct visual comparison between the "New" and "Acquired" clinic effects.

1.  **Plot 1: Primary Visits**: The first graph will display the DiD effect on `Primary_Visits`. It will plot the results from **Model 1A (New clinics)** and **Model 2A (Acquired clinics)** on the same axes. This directly answers: "Is the effect of opening a new clinic different from acquiring an existing one?"

2.  **Plot 2: Emergency Visits**: The second graph will do the same for `Emergency_Visits`, comparing **Model 1B (New clinics)** and **Model 2B (Acquired clinics)**.

We will use the same `extract_absolute_did_effects` helper function as before, which correctly calculates the full effect (and 95% confidence interval) for each distance category by combining the base `Treat:Post` term with its corresponding triple-interaction term.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("\n--- Step 5 (Revised): Visualizing Combined Results ---")
sns.set(style="whitegrid")

# (Helper function to extract data from a model result)
# We need to make sure this function is defined in this cell
def extract_absolute_did_effects(model_result, ref_category_label='0-5 km'):
    # This list must match the labels used in Step 5 of the main script
    all_categories = ['0-5 km', '5-10 km', '10-20 km', '20+ km']

    params = model_result.params
    results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    if 'Treat:Post' not in params.index:
        print(f"Warning: 'Treat:Post' not found. Plot cannot be generated.")
        return pd.DataFrame(results)

    # 1. Effect for the reference group (0-5 km)
    effect = params['Treat:Post']
    conf = model_result.conf_int().loc['Treat:Post']
    results['Distance'].append(ref_category_label)
    results['Effect'].append(effect)
    results['Conf_Low'].append(conf[0])
    results['Conf_High'].append(conf[1])

    # 2. Effects for all other groups
    other_categories = [cat for cat in all_categories if cat != ref_category_label]

    for cat in other_categories:
        term_name = f"Treat:Post:C(distance_category)[T.{cat}]"
        if term_name in params.index:
            try:
                t_test_result = model_result.t_test(f"Treat:Post + {term_name}")
                effect = t_test_result.effect[0]
                conf_low = t_test_result.conf_int()[0][0]
                conf_high = t_test_result.conf_int()[0][1]
                results['Distance'].append(cat)
                results['Effect'].append(effect)
                results['Conf_Low'].append(conf_low)
                results['Conf_High'].append(conf_high)
            except Exception as e:
                print(f"t_test error for {term_name}: {e}")

    # Ensure data is returned in the correct order
    df = pd.DataFrame(results)
    df['Distance'] = pd.Categorical(df['Distance'], categories=all_categories, ordered=True)
    df = df.sort_values('Distance')
    return df

# --- New Plotting Function ---
def plot_comparison_graphs(models_dict, category_order, y_primary, y_emergency):
    """
    Plots two graphs:
    1. Primary Visits (New vs. Acquired)
    2. Emergency Visits (New vs. Acquired)
    """

    # Create a 2-row, 1-column figure
    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 16), sharex=True)

    # --- Plot 1: Primary Visits ---
    ax1.set_title(f"DiD Effect on {y_primary} (New vs. Acquired)", fontsize=16, pad=20)

    # Check for and plot Model 1A (New, Primary)
    if '1A' in models_dict:
        data_1A = extract_absolute_did_effects(models_dict['1A'], ref_category_label=category_order[0])
        if not data_1A.empty:
            errors_1A = [data_1A['Effect'] - data_1A['Conf_Low'], data_1A['Conf_High'] - data_1A['Effect']]
            ax1.errorbar(x=data_1A['Distance'], y=data_1A['Effect'], yerr=errors_1A,
                         fmt='-o', capsize=5, markersize=8, label='New Clinics', color='blue')

    # Check for and plot Model 2A (Acquired, Primary)
    if '2A' in models_dict:
        data_2A = extract_absolute_did_effects(models_dict['2A'], ref_category_label=category_order[0])
        if not data_2A.empty:
            errors_2A = [data_2A['Effect'] - data_2A['Conf_Low'], data_2A['Conf_High'] - data_2A['Effect']]
            ax1.errorbar(x=data_2A['Distance'], y=data_2A['Effect'], yerr=errors_2A,
                         fmt='--s', capsize=5, markersize=8, label='Acquired Clinics', color='red')

    ax1.axhline(y=0, color='black', linestyle='--', linewidth=1, label='No Effect (Y=0)')
    ax1.set_ylabel("DiD Effect Size (Change in Visits)", fontsize=12)
    ax1.legend(fontsize=12)
    ax1.grid(True)

    # --- Plot 2: Emergency Visits ---
    ax2.set_title(f"DiD Effect on {y_emergency} (New vs. Acquired)", fontsize=16, pad=20)

    # Check for and plot Model 1B (New, Emergency)
    if '1B' in models_dict:
        data_1B = extract_absolute_did_effects(models_dict['1B'], ref_category_label=category_order[0])
        if not data_1B.empty:
            # CORRECTED LINE BELOW:
            errors_1B = [data_1B['Effect'] - data_1B['Conf_Low'], data_1B['Conf_High'] - data_1B['Effect']]
            ax2.errorbar(x=data_1B['Distance'], y=data_1B['Effect'], yerr=errors_1B,
                         fmt='-o', capsize=5, markersize=8, label='New Clinics', color='blue')

    # Check for and plot Model 2B (Acquired, Emergency)
    if '2B' in models_dict:
        data_2B = extract_absolute_did_effects(models_dict['2B'], ref_category_label=category_order[0])
        if not data_2B.empty:
            errors_2B = [data_2B['Effect'] - data_2B['Conf_Low'], data_2B['Conf_High'] - data_2B['Effect']]
            ax2.errorbar(x=data_2B['Distance'], y=data_2B['Effect'], yerr=errors_2B,
                         fmt='--s', capsize=5, markersize=8, label='Acquired Clinics', color='red')

    ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, label='No Effect (Y=0)')
    ax2.set_xlabel("Distance Category to Nearest Clinic (km)", fontsize=12)
    ax2.set_ylabel("DiD Effect Size (Change in Visits)", fontsize=12)
    ax2.legend(fontsize=12)
    ax2.grid(True)

    plt.tight_layout(pad=2.0)
    plt.show()

# --- Run the new plotting function ---
if 'models' in locals() and models:
    # These must match the variables from the previous cell
    category_labels_km = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    y_primary_name = 'Primary_Visits'
    y_emergency_name = 'Emergency_Visits'

    plot_comparison_graphs(models, category_labels_km, y_primary_name, y_emergency_name)
else:
    print("❌ ERROR: 'models' dictionary not found. Please re-run the analysis cell (Step 7) first.")

print("\n🎉 --- 'Task Plan' Visualization Complete! ---")

# 11. DiD Analysis (Method 3): Two-Way Fixed Effects (TWFE) with Heterogeneous Effects

This section implements our primary Difference-in-Differences (DiD) model. This approach uses a Two-Way Fixed Effects (TWFE) OLS regression to measure the causal impact of clinic openings/acquisitions, allowing the effect to vary by distance.

## 11.1. Treatment & Control Group Definition

This method relies on a clear-cut "treatment" and "control" group.

1.  **Treatment Group (`Treat = 1`)**: We define a "treatment" radius of 50km. We iterate through every patient ZIP code and find its single *nearest* clinic. If this nearest clinic is **within 50km**, that patient ZIP code is assigned to the **treatment group** (`Treat = 1`).
2.  **Control Group (`Treat = 0`)**: Any patient ZIP code whose nearest clinic is *further* than 50km away is assigned to the **control group** (`Treat = 0`).
3.  **Data Split**: The analysis is run twice.
    * **"New" Model**: Compares patient ZIPs treated by a 'New' clinic against the *entire* control group.
    * **"Acquired" Model**: Compares patient ZIPs treated by an 'Acquired' clinic against the *entire* control group.

## 11.2. DiD Variable Creation

* **`Treat`**: Binary. 1 if the patient ZIP's nearest clinic is <= 50km, 0 otherwise.
* **`Post`**: Binary. For the treatment group (`Treat=1`), this "switches on" to 1 if `Date >= event_date`. For the control group (`Treat=0`), `Post` is always 0.
* **`distance_category`**: For the treatment group, we bin their distance to the nearest clinic ('0-5 km', '5-10 km', '10-20 km', '20+ km'). For the control group, we assign a separate category: 'Control (>50km)'.

## 11.3. Model Specification (TWFE)

We use a full Two-Way Fixed Effects (TWFE) model, which includes fixed effects for both entity (`C(Zip)`) and time (`C(Year) + C(Month)`). This is a very robust specification that controls for:
1.  All time-invariant characteristics of a ZIP code (e.g., demographics, geography, baseline health).
2.  All common time shocks that affect all ZIP codes (e.g., seasonality, holidays).

The formula uses a **triple interaction**:
$Y_{it} = \beta_0 + \sum_{k} \delta_k(Treat_i \times Post_{it} \times DistanceCategory_{ik}) + \gamma_i + \lambda_t + \epsilon_{it}$

* $Y_{it}$ is the outcome (Primary or Emergency Visits) for ZIP $i$ at time $t$.
* $\gamma_i$ are the ZIP Code Fixed Effects (`C(Zip)`).
* $\lambda_t$ are the Time Fixed Effects (`C(Year) + C(Month)`).
* The `Treat * Post * C(distance_category)` term measures the DiD effect, allowing it to be different for each distance bin $k$.

## 11.4. Visualization & Interpretation

The regression output will give us the effect for the reference bin (e.g., '0-5 km') via the `Treat:Post` coefficient, and *relative* effects for other bins (e.g., `Treat:Post:C(distance_category)[T.5-10 km]`).

Our plotting function uses `model.t_test()` to automatically combine these coefficients. This allows us to plot the **absolute DiD effect** (and its 95% confidence interval) for each distance bin, providing a clear visualization of how the treatment impact changes as distance from the clinic increases.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 STARTING 'Task Plan' DiD (Clinic ZIP Radius Logic) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("   Please run the cell that creates 'final_data' first.")
else:
    print(f"✅ 'final_data' found. {len(final_data)} rows.")

    # --- 2. Load helper files ---
    try:
        df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
        print("✅ 'OSFResearch_Clinics_mastersheet_v2.csv' and 'Crosswalk_IL.csv' loaded.")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 3. Clean Data ---
    # 3.1 Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    final_data['zcta5'] = final_data['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # 3.2 Clean Crosswalk (coordinates)
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # 3.3 Clean df_clinics_info (clinic info)
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    df_clinics_info = df_clinics_info.merge(zip_coords.rename(columns={'IntPtLat':'Clinic_Lat', 'IntPtLon':'Clinic_Lon'}), left_on='Zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['Clinic_Lat', 'Clinic_Lon', 'event_date', 'clinic_type'])
    print(f"✅ Cleaning complete. Found {len(df_clinics_info)} clinics with coordinates.")

    # --- 4. NEW: Create "Treatment Map" based on distance to CLINIC's ZIP ---
    print("...Creating 'treatment map' (finding nearest clinic within 50km based on CLINIC ZIP distance)...")

    # Get unique patient ZIPs with coordinates
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords.rename(columns={'IntPtLat':'Patient_Lat', 'IntPtLon':'Patient_Lon'}),
                                            left_on='Filtered_Patient_ZipCode', right_index=True)
    patient_zips_df = patient_zips_df.dropna()

    treatment_map = []
    TREATMENT_RADIUS_KM = 50 # Max distance to consider a patient ZIP 'treated'

    # Loop over each patient ZIP
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['Patient_Lat'], patient_row['Patient_Lon'])

        min_distance_to_clinic_zip = np.inf
        relevant_clinic = {}

        # Find the single closest clinic *within the radius*
        for _, clinic_row in df_clinics_info.iterrows():
            # Calculate distance between PATIENT ZIP and CLINIC ZIP
            clinic_coords = (clinic_row['Clinic_Lat'], clinic_row['Clinic_Lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers

            # Is this clinic the closest one *found so far* AND *within the treatment radius*?
            if distance <= TREATMENT_RADIUS_KM and distance < min_distance_to_clinic_zip:
                min_distance_to_clinic_zip = distance
                relevant_clinic = {
                    'clinic_zip': clinic_row['Zip'], # Store the clinic's zip
                    'clinic_type': clinic_row['clinic_type'],
                    'event_date': clinic_row['event_date']
                }

        # Only add to map if we found a relevant clinic within the radius
        if relevant_clinic:
            treatment_map.append({
                'Filtered_Patient_ZipCode': patient_zip,
                'distance_patient_zip_to_clinic_zip_km': min_distance_to_clinic_zip, # This is the key distance now
                'clinic_type': relevant_clinic['clinic_type'],
                'event_date': relevant_clinic['event_date']
            })

    df_treatment_map = pd.DataFrame(treatment_map)
    print(f"✅ 'Treatment map' created for {len(df_treatment_map)} ZIP codes within {TREATMENT_RADIUS_KM}km of a clinic.")

    # --- 5. Aggregate Data and Create DiD Variables ---
    print("...Aggregating visits and creating DiD variables...")

    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)

    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({
        'Emergency_Visits': 'sum',
        'Primary_Visits': 'sum'
    }).reset_index()

    # Merge treatment map
    df_main = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left')

    # Define Treat: 1 if the patient_zip is in the map (has a clinic within 50km), 0 otherwise
    df_main['Treat'] = df_main['event_date'].notna().astype(int)

    # Define Post relative to the event_date *for treated observations*
    df_main['Post'] = 0
    df_main.loc[df_main['Treat'] == 1, 'Post'] = (df_main['Date'] >= df_main['event_date']).astype(int)

    # Define distance_category based on the distance to the clinic's ZIP
    # Only meaningful for treated observations, but apply to all for formula simplicity
    bins_km = [-np.inf, 5, 10, 20, np.inf]
    labels_km = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    df_main['distance_category'] = pd.cut(
        df_main['distance_patient_zip_to_clinic_zip_km'], # Use the new distance
        bins=bins_km,
        labels=labels_km,
        right=False
    )
    # Fill NA distance category for control group (distance > 50km or no clinic found)
    # We can assign them to the furthest category '20+ km' as they are effectively controls
    # Or create a separate 'Control' category if preferred. Let's use '20+ km'.
    if df_main['distance_category'].isnull().any():
       df_main['distance_category'] = df_main['distance_category'].cat.add_categories(['Control (>50km)'])
       df_main['distance_category'].fillna('Control (>50km)', inplace=True)
       labels_km.append('Control (>50km)') # Add to our list for plotting order


    # Rename ZIP for the formula
    df_main = df_main.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

    # Drop rows where essential DiD info might be missing (shouldn't happen with Treat=0 logic, but safe)
    df_main = df_main.dropna(subset=['clinic_type', 'distance_category'], how='any')


    print(f"✅ Final DataFrame 'df_main' created. {len(df_main)} rows.")
    print("Clinic type distribution in final data:")
    print(df_main[df_main['Treat']==1]['clinic_type'].value_counts()) # Show types only for treated


    # --- 6. Define Formulas ---
    Y_PRIMARY_NAME = 'Primary_Visits'
    Y_EMERGENCY_NAME = 'Emergency_Visits'
    CONTROLS_STR = 'C(Year) + C(Month) + C(Zip)' # Zip fixed effects control for baseline differences

    interaction_term = "Treat * Post * C(distance_category)"
    formula_primary = f"{Y_PRIMARY_NAME} ~ {interaction_term} + {CONTROLS_STR}"
    formula_emergency = f"{Y_EMERGENCY_NAME} ~ {interaction_term} + {CONTROLS_STR}"

    print(f"\n--- Formulas are ready ---")
    print(f"Formula (Primary): {formula_primary}")
    print(f"Formula (Emergency): {formula_emergency}")

    # --- 7. Run Models ---
    print("\n--- Step 7: Running 4 models ---")

    # Ensure distance_category has the correct order, including the potential Control category
    df_main['distance_category'] = pd.Categorical(
        df_main['distance_category'],
        categories=labels_km, # Use the potentially updated list
        ordered=True
    )

    # Split data based on clinic type *for the treated group*
    # Control group observations (Treat=0) are needed in both regressions
    df_new = df_main[(df_main['clinic_type'] == 'New') | (df_main['Treat'] == 0)].copy()
    df_acquired = df_main[(df_main['clinic_type'] == 'Acquired') | (df_main['Treat'] == 0)].copy()

    print(f"Records for 'New' regression (incl. controls): {len(df_new)}")
    print(f"Records for 'Acquired' regression (incl. controls): {len(df_acquired)}")

    models = {}

    try:
        if not df_new[df_new['Treat']==1].empty: # Check if there are treated 'New' observations
            print("...Running 1A: New, Primary")
            # Ensure the reference level for distance category is set correctly
            # Note: Patsy might choose a different reference if '0-5 km' has no data. Check summary.
            models['1A'] = smf.ols(formula_primary, data=df_new).fit(
                cov_type='cluster', cov_kwds={'groups': df_new['Zip']}
            )
            print("...Running 1B: New, Emergency")
            models['1B'] = smf.ols(formula_emergency, data=df_new).fit(
                cov_type='cluster', cov_kwds={'groups': df_new['Zip']}
            )
        else:
            print("⚠️ No treated 'New' observations found, skipping 1A and 1B.")

        if not df_acquired[df_acquired['Treat']==1].empty: # Check if there are treated 'Acquired' observations
            print("...Running 2A: Acquired, Primary")
            models['2A'] = smf.ols(formula_primary, data=df_acquired).fit(
                cov_type='cluster', cov_kwds={'groups': df_acquired['Zip']}
            )
            print("...Running 2B: Acquired, Emergency")
            models['2B'] = smf.ols(formula_emergency, data=df_acquired).fit(
                cov_type='cluster', cov_kwds={'groups': df_acquired['Zip']}
            )
        else:
            print("⚠️ No treated 'Acquired' observations found, skipping 2A and 2B.")

        print("✅ Model estimation complete.")

    except Exception as e:
        print(f"❌ ERROR DURING MODEL FITTING: {e}")
        # Add more specific error checks if needed, e.g., for singular matrix
        if "Singular matrix" in str(e):
             print("   This might be due to perfect multicollinearity. Check fixed effects and interactions.")
        raise

    # --- 8. Visualization ---
    print("\n--- Step 8: Visualizing Combined Results ---")
    sns.set(style="whitegrid")

    # (Helper function to extract data - updated categories)
    def extract_absolute_did_effects(model_result, category_order):
        ref_category_label = category_order[0] # Usually '0-5 km'
        params = model_result.params
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}

        # Check if the main Treat:Post interaction exists
        if 'Treat:Post' not in params.index:
            print(f"Warning: Base 'Treat:Post' coefficient not found. Cannot calculate effects.")
            # Attempt to infer reference category from distance interactions if possible
            base_effect_found = False
            for cat in category_order[1:]: # Check interactions if base is missing
                 term_name_base_ref = f"Treat:Post:C(distance_category)[T.{cat}]"
                 if term_name_base_ref in params.index:
                      # If interactions exist but base doesn't, implies base effect is absorbed or zero
                      print(f"  Base Treat:Post missing, assuming effect for {ref_category_label} is 0.")
                      results['Distance'].append(ref_category_label)
                      results['Effect'].append(0)
                      results['Conf_Low'].append(0)
                      results['Conf_High'].append(0)
                      base_effect_found = True
                      break # Only need to add base effect once
            if not base_effect_found:
                 print("  Cannot determine base effect. Plotting skipped.")
                 return pd.DataFrame(results) # Return empty if base effect cannot be determined

        else: # Normal case: Treat:Post exists
             effect = params['Treat:Post']
             conf = model_result.conf_int().loc['Treat:Post']
             results['Distance'].append(ref_category_label)
             results['Effect'].append(effect)
             results['Conf_Low'].append(conf[0])
             results['Conf_High'].append(conf[1])

        # Calculate effects for other categories relative to base
        other_categories = [cat for cat in category_order if cat != ref_category_label and cat != 'Control (>50km)']

        for cat in other_categories:
            term_name = f"Treat:Post:C(distance_category)[T.{cat}]"
            if term_name in params.index:
                try:
                    # Calculate total effect: Base + Interaction
                    hypothesis = f"Treat:Post + {term_name}" if 'Treat:Post' in params.index else term_name
                    t_test_result = model_result.t_test(hypothesis)
                    effect = t_test_result.effect[0]
                    conf_low = t_test_result.conf_int()[0][0]
                    conf_high = t_test_result.conf_int()[0][1]
                    results['Distance'].append(cat)
                    results['Effect'].append(effect)
                    results['Conf_Low'].append(conf_low)
                    results['Conf_High'].append(conf_high)
                except Exception as e:
                    print(f"t_test error for {term_name}: {e}")
            # Handle cases where interaction term might be missing (e.g., absorbed)
            elif 'Treat:Post' in params.index :
                 print(f"Warning: Interaction term {term_name} not found. Assuming effect for {cat} is same as base.")
                 base_effect_row = next((item for item in results if item["Distance"] == ref_category_label), None)
                 if base_effect_row:
                      results['Distance'].append(cat)
                      results['Effect'].append(base_effect_row['Effect'])
                      results['Conf_Low'].append(base_effect_row['Conf_Low'])
                      results['Conf_High'].append(base_effect_row['Conf_High'])


        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        df = df.sort_values('Distance')
        return df

    # (Plotting function - largely the same, adjusted category label)
    def plot_comparison_graphs(models_dict, category_order, y_primary, y_emergency):
        fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 16), sharex=True)

        # Plot 1: Primary Visits
        ax1.set_title(f"DiD Effect on {y_primary} (New vs. Acquired)", fontsize=16, pad=20)
        plot_successful = False
        if '1A' in models_dict:
            data_1A = extract_absolute_did_effects(models_dict['1A'], category_order)
            if not data_1A.empty:
                errors_1A = [data_1A['Effect'] - data_1A['Conf_Low'], data_1A['Conf_High'] - data_1A['Effect']]
                ax1.errorbar(x=data_1A['Distance'].astype(str), y=data_1A['Effect'], yerr=errors_1A,
                             fmt='-o', capsize=5, markersize=8, label='New Clinics', color='blue')
                plot_successful = True
        if '2A' in models_dict:
            data_2A = extract_absolute_did_effects(models_dict['2A'], category_order)
            if not data_2A.empty:
                errors_2A = [data_2A['Effect'] - data_2A['Conf_Low'], data_2A['Conf_High'] - data_2A['Effect']]
                ax1.errorbar(x=data_2A['Distance'].astype(str), y=data_2A['Effect'], yerr=errors_2A,
                             fmt='--s', capsize=5, markersize=8, label='Acquired Clinics', color='red')
                plot_successful = True

        if plot_successful:
            ax1.axhline(y=0, color='black', linestyle='--', linewidth=1, label='No Effect (Y=0)')
            ax1.set_ylabel("DiD Effect Size (Change in Visits)", fontsize=12)
            ax1.legend(fontsize=12)
            ax1.grid(True)
        else:
             ax1.text(0.5, 0.5, "No data to plot for Primary Visits", ha='center', va='center', fontsize=14)


        # Plot 2: Emergency Visits
        ax2.set_title(f"DiD Effect on {y_emergency} (New vs. Acquired)", fontsize=16, pad=20)
        plot_successful = False
        if '1B' in models_dict:
            data_1B = extract_absolute_did_effects(models_dict['1B'], category_order)
            if not data_1B.empty:
                errors_1B = [data_1B['Effect'] - data_1B['Conf_Low'], data_1B['Conf_High'] - data_1B['Effect']]
                ax2.errorbar(x=data_1B['Distance'].astype(str), y=data_1B['Effect'], yerr=errors_1B,
                             fmt='-o', capsize=5, markersize=8, label='New Clinics', color='blue')
                plot_successful = True
        if '2B' in models_dict:
            data_2B = extract_absolute_did_effects(models_dict['2B'], category_order)
            if not data_2B.empty:
                errors_2B = [data_2B['Effect'] - data_2B['Conf_Low'], data_2B['Conf_High'] - data_2B['Effect']]
                ax2.errorbar(x=data_2B['Distance'].astype(str), y=data_2B['Effect'], yerr=errors_2B,
                             fmt='--s', capsize=5, markersize=8, label='Acquired Clinics', color='red')
                plot_successful = True

        if plot_successful:
             ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, label='No Effect (Y=0)')
             # Use only relevant categories for x-axis labels if 'Control' category exists
             plot_categories = [cat for cat in category_order if cat != 'Control (>50km)']
             ax2.set_xticks(range(len(plot_categories))) # Set ticks numerically
             ax2.set_xticklabels(plot_categories) # Set labels as strings
             ax2.set_xlabel("Distance Category (Patient ZIP to Clinic ZIP, km)", fontsize=12)
             ax2.set_ylabel("DiD Effect Size (Change in Visits)", fontsize=12)
             ax2.legend(fontsize=12)
             ax2.grid(True)
        else:
             ax2.text(0.5, 0.5, "No data to plot for Emergency Visits", ha='center', va='center', fontsize=14)


        plt.tight_layout(pad=2.0)
        plt.show()

    # --- Run the plotting function ---
    if 'models' in locals() and models:
        # Define category labels including potential control group
        category_labels_km = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
        if 'Control (>50km)' in df_main['distance_category'].cat.categories:
             category_labels_km.append('Control (>50km)')

        y_primary_name = 'Primary_Visits'
        y_emergency_name = 'Emergency_Visits'

        plot_comparison_graphs(models, category_labels_km, y_primary_name, y_emergency_name)
    else:
        print("❌ ERROR: 'models' dictionary not found or empty. Cannot plot results.")

    print("\n🎉 --- 'Task Plan' DiD Complete! ---")

# 12. DiD Analysis (Method 4): Pooled OLS "Deep Dive" (0-15km)

This section performs a "deep dive" robustness check. We "zoom in" on the 15km radius, which our previous models identified as the primary area of action.

This analysis differs from the main TWFE model (Method 3) in two key ways:

1.  **Sample**: The analysis is filtered *only* to patient ZIPs whose nearest clinic is **within 15km**. We are no longer using the >50km "control group" in this regression.
2.  **Model**: We use a simpler **Pooled OLS** model that **removes ZIP Code Fixed Effects (`C(Zip)`)**. This is a much faster model that relies on the assumption that all ZIPs within the 15km radius are relatively comparable.
3.  **Bins**: We use more **granular distance bins** (e.g., `0-2 km`, `2-4 km`, `4-6 km`, etc.) to get a finer-grained look at how the effect decays with distance.

## 12.1. Model Specification

Since all ZIP codes in this sample are "treated," the `Treat` variable is always 1 and is omitted. The model estimates the effect of the clinic event (`Post`) and how that effect differs across our new, granular distance categories.

The formula is a Pooled OLS with Time Fixed Effects:

$Y_{it} = \beta_0 + \beta_1(Post_{it}) + \sum_{k} \delta_k(Post_{it} \times DistanceCategory_{ik}) + \lambda_t + \epsilon_{it}$

* $Y_{it}$ is the outcome (Primary or Emergency Visits) for ZIP $i$ at time $t$.
* $\beta_1$ is the DiD effect for the reference distance category (e.g., '0-2 km').
* $\delta_k$ is the *additional* effect for distance category $k$ (relative to the base).
* $\lambda_t$ are the Time (Year + Month) Fixed Effects.

The visualization will plot the absolute effect ($\beta_1 + \delta_k$) for each distance bin.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 STARTING DEEP DIVE ANALYSIS (0-15 km) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    # --- The data preparation steps are the same as before ---
    # (These are unchanged)
    df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    df_clinics_info = df_clinics_info.merge(zip_coords, left_on='Zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['IntPtLat', 'IntPtLon', 'event_date', 'clinic_type'])
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords, left_on='Filtered_Patient_ZipCode', right_index=True)
    patient_zips_df = patient_zips_df.dropna()
    treatment_map = []
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['IntPtLat'], patient_row['IntPtLon'])
        min_distance = np.inf
        nearest_clinic = {}
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['IntPtLat'], clinic_row['IntPtLon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_distance:
                min_distance = distance
                nearest_clinic = {'clinic_type': clinic_row['clinic_type'], 'event_date': clinic_row['event_date']}
        if nearest_clinic:
            treatment_map.append({'Filtered_Patient_ZipCode': patient_zip, 'distance_km': min_distance, 'clinic_type': nearest_clinic['clinic_type'], 'event_date': nearest_clinic['event_date']})
    df_treatment_map = pd.DataFrame(treatment_map)
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()
    df_main_full = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left').dropna(subset=['distance_km', 'clinic_type', 'event_date'])

    # --- NEW: Filter data for the 0-15 km deep dive ---
    ANALYSIS_RADIUS_KM = 15
    df_dive = df_main_full[df_main_full['distance_km'] <= ANALYSIS_RADIUS_KM].copy()
    print(f"\n--- Focusing analysis on {len(df_dive)} observations within {ANALYSIS_RADIUS_KM} km of a clinic ---")

    # Create DiD variables on this focused dataset
    df_dive['Treat'] = 1 
    df_dive['Post'] = (df_dive['Date'] >= df_dive['event_date']).astype(int)

    # --- NEW: Create more granular distance bins ---
    granular_bins_km = [-np.inf, 2, 4, 6, 10, 15]
    # We keep this original list *only* for the `pd.cut` labels
    granular_labels_km_original = ['0-2 km', '2-4 km', '4-6 km', '6-10 km', '10-15 km'] 
    df_dive['distance_category_granular'] = pd.cut(
        df_dive['distance_km'],
        bins=granular_bins_km,
        labels=granular_labels_km_original, # Use original list here
        right=False
    )
    df_dive = df_dive.dropna(subset=['distance_category_granular'])
    
    # --- FIX 1: Dynamically get the *actual* categories that have data ---
    # This avoids using the empty '2-4 km' bin
    granular_labels_km_actual = df_dive['distance_category_granular'].cat.categories.tolist()
    print(f"✅ Original bins defined: {granular_labels_km_original}")
    print(f"✅ Actual non-empty bins being used: {granular_labels_km_actual}")

    # --- NEW: Define SIMPLIFIED formula (NO C(Zip)) ---
    Y_PRIMARY_NAME = 'Primary_Visits'
    Y_EMERGENCY_NAME = 'Emergency_Visits'
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    
    interaction_term = "Post * C(distance_category_granular)" 
    formula_primary_simple = f"{Y_PRIMARY_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_simple = f"{Y_EMERGENCY_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    
    print(f"\nSimplified Formula (Primary): {formula_primary_simple}")

    # --- Run Models with Simplified Formula ---
    print("\n--- Running models with simplified formula ---")
    
    # --- FIX 2: Use the *actual* (non-empty) category list to order the data ---
    df_dive['distance_category_granular'] = pd.Categorical(
        df_dive['distance_category_granular'], 
        categories=granular_labels_km_actual, # Use the dynamic, actual list
        ordered=True
    )
    
    df_new_dive = df_dive[df_dive['clinic_type'] == 'New'].copy()
    df_acquired_dive = df_dive[df_dive['clinic_type'] == 'Acquired'].copy()

    models_dive = {}
    try:
        if not df_new_dive.empty:
            print("...Running Deep Dive 1A: New, Primary")
            models_dive['1A'] = smf.ols(formula_primary_simple, data=df_new_dive).fit() 
            print("...Running Deep Dive 1B: New, Emergency")
            models_dive['1B'] = smf.ols(formula_emergency_simple, data=df_new_dive).fit()
        else:
            print("⚠️ No 'New' clinic data within 15km.")

        if not df_acquired_dive.empty:
            print("...Running Deep Dive 2A: Acquired, Primary")
            models_dive['2A'] = smf.ols(formula_primary_simple, data=df_acquired_dive).fit()
            print("...Running Deep Dive 2B: Acquired, Emergency")
            models_dive['2B'] = smf.ols(formula_emergency_simple, data=df_acquired_dive).fit()
        else:
            print("⚠️ No 'Acquired' clinic data within 15km.")
        print("✅ Deep dive model estimation complete.")
    except Exception as e:
        print(f"❌ ERROR DURING DEEP DIVE MODEL FITTING: {e}")
        raise

    # --- Visualization for Deep Dive ---
    print("\n--- Visualizing Deep Dive Results ---")
    
    # This function should now work, as the model and category list match
    def extract_effects_simple(model_result, category_order):
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        
        if 'Post' not in params.index:
            # This warning should no longer appear, but we leave it as a safeguard
            print(f"Warning: Base 'Post' coefficient (for ref category {ref_label}) not found. Plot may be incomplete.")
            return pd.DataFrame(results) # Return empty
            
        # Effect for reference group is just 'Post' coefficient
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Post'])
        results['Conf_Low'].append(conf.loc['Post', 0])
        results['Conf_High'].append(conf.loc['Post', 1])

        # Effects for other groups
        for cat in category_order[1:]:
            term_name = f"Post:C(distance_category_granular)[T.{cat}]"
            if term_name in params.index:
                # Total effect is Post + Interaction
                total_effect = params['Post'] + params[term_name]
                # Use t_test to get correct confidence interval for the sum
                t_test = model_result.t_test(f"Post + {term_name}")
                
                results['Distance'].append(cat)
                results['Effect'].append(total_effect)
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
            # If term_name is not in params, it's because it was an empty bin 
            # (e.g., '2-4 km') and was correctly excluded. We just skip it.
        
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    # (Assuming 'plot_comparison_graphs' is defined in a previous cell)
    # --- FIX 3: Pass the *actual* (non-empty) category list to the plotting function ---
    print(f"Plotting with categories: {granular_labels_km_actual}")
    
    # We must define the plotting function here if it's not in memory
    # This is the function from your PREVIOUS script (Method 2)
    def plot_comparison_graphs(models_dict, category_order, y_primary, y_emergency):
        fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 16), sharex=True)

        # --- Primary Visits Plot ---
        ax1.set_title(f"DiD Effect on {y_primary}", fontsize=16, pad=20)
        plot_made = False
        if '1A' in models_dict:
            data = extract_effects_simple(models_dict['1A'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax1.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='-o', capsize=5, label='New Clinics')
                plot_made = True
        if '2A' in models_dict:
            data = extract_effects_simple(models_dict['2A'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax1.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='--s', capsize=5, label='Acquired Clinics')
                plot_made = True
        if plot_made:
            ax1.axhline(0, color='black', linestyle='--')
            ax1.set_ylabel("DiD Effect Size (Change in Daily Visits)")
            ax1.legend()
        else:
            ax1.text(0.5, 0.5, "No data to plot", horizontalalignment='center', verticalalignment='center', transform=ax1.transAxes)

        # --- Emergency Visits Plot ---
        ax2.set_title(f"DiD Effect on {y_emergency}", fontsize=16, pad=20)
        plot_made = False
        if '1B' in models_dict:
            data = extract_effects_simple(models_dict['1B'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax2.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='-o', capsize=5, label='New Clinics')
                plot_made = True
        if '2B' in models_dict:
            data = extract_effects_simple(models_dict['2B'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax2.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='--s', capsize=5, label='Acquired Clinics')
                plot_made = True
        if plot_made:
            ax2.axhline(0, color='black', linestyle='--')
            ax2.set_xlabel("Distance Category to Nearest Clinic (km)")
            ax2.set_ylabel("DiD Effect Size (Change in Daily Visits)")
            ax2.legend()
        else:
            ax2.text(0.5, 0.5, "No data to plot", horizontalalignment='center', verticalalignment='center', transform=ax2.transAxes)

        plt.tight_layout(pad=2.0)
        plt.show()
    
    # Now this call uses the clean, non-empty list of categories
    plot_comparison_graphs(models_dive, granular_labels_km_actual, Y_PRIMARY_NAME, Y_EMERGENCY_NAME)

# 13. DiD Analysis (Method 5): "Deep Dive" with Visit Rates (0-15km)

This is our final and most robust specification. It combines the strengths of the previous two methods:

1.  **"Deep Dive" Sample**: It focuses *only* on the 0-15km radius where the treatment effect is expected to be strongest.
2.  **"Visit Rates" Dependent Variable**: It normalizes the outcome variable by population, using **visits per 1,000 people**.

This allows us to get a granular view of the distance decay (using bins like `0-2 km`, `2-6 km`, etc.) while properly controlling for the fact that different ZIP codes have different population sizes.

## 13.1. Data Preparation: Population & Visit Rates

1.  **Load Population Data**: We load `TotPopACS` (Total Population) from the `Crosswalk_IL.csv` file. We use `pd.to_numeric` with `errors='coerce'` to robustly handle any non-numeric entries.
2.  **Merge & Filter**: We merge the population data onto our main aggregated dataset. Any ZIP-day observation without corresponding population data is dropped.
3.  **Filter for "Deep Dive"**: We create the `df_dive` DataFrame by keeping *only* observations where `distance_km <= 15`.
4.  **Calculate Visit Rates**: We create our two final dependent variables:
    * `Primary_Visits_Rate = (Primary_Visits / TotPopACS) * 1000`
    * `Emergency_Visits_Rate = (Emergency_Visits / TotPopACS) * 1000`

## 13.2. Model Specification (Pooled OLS with Rates)

As in the previous "Deep Dive," we use a Pooled OLS model with Time Fixed Effects (`C(Year) + C(Month)`) and no ZIP Code Fixed Effects. The model estimates the pre-post change in *visit rates* and how this change differs across our granular distance bins.

The formula is:
$Y\_Rate_{it} = \beta_0 + \beta_1(Post_{it}) + \sum_{k} \delta_k(Post_{it} \times DistanceCategory_{ik}) + \lambda_t + \epsilon_{it}$

* $Y\_Rate_{it}$ is the visit rate per 1,000 people.
* $\beta_1$ is the DiD effect for the reference distance category ('0-2 km').
* $\delta_k$ is the *additional* effect for distance category $k$.
* $\lambda_t$ are the Time (Year + Month) Fixed Effects.

The final visualization plots the absolute DiD effect ($\beta_1 + \delta_k$) for each distance bin, showing the change in **daily visits per 1,000 people**.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 STARTING 'Deep Dive with Rates' ANALYSIS (CORRECTED) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("   Please run the cell that creates 'final_data' first.")
else:
    # --- Load and Prepare Data (as before) ---
    df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")

    # Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # --- CORRECTED SECTION: Load and clean population data robustly ---
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    
    # FIX IS HERE: Use pd.to_numeric with errors='coerce'
    zip_population['TotPopACS'] = pd.to_numeric(
        zip_population['TotPopACS'].str.replace(',', '', regex=False),
        errors='coerce'  # This turns bad text into NaN instead of crashing
    )
    
    # Now, drop any rows that became NaN
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    print(f"✅ Population data loaded and cleaned for {len(zip_population)} ZIP codes.")

    # ZIP Coordinates
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # Clinic Info
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    df_clinics_info = df_clinics_info.merge(zip_coords, left_on='Zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['IntPtLat', 'IntPtLon', 'event_date', 'clinic_type'])

    # Treatment Map
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords, left_on='Filtered_Patient_ZipCode', right_index=True).dropna()
    treatment_map = []
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['IntPtLat'], patient_row['IntPtLon'])
        min_distance = np.inf
        nearest_clinic = {}
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['IntPtLat'], clinic_row['IntPtLon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_distance:
                min_distance = distance
                nearest_clinic = {'clinic_type': clinic_row['clinic_type'], 'event_date': clinic_row['event_date']}
        if nearest_clinic:
            treatment_map.append({'Filtered_Patient_ZipCode': patient_zip, 'distance_km': min_distance, 'clinic_type': nearest_clinic['clinic_type'], 'event_date': nearest_clinic['event_date']})
    df_treatment_map = pd.DataFrame(treatment_map)

    # Aggregate Visits
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()
    df_main_full = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left').dropna(subset=['distance_km', 'clinic_type', 'event_date'])

    # Merge population
    df_main_full = df_main_full.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    initial_rows = len(df_main_full)
    df_main_full = df_main_full.dropna(subset=['TotPopACS'])
    print(f"ℹ️ Dropped {initial_rows - len(df_main_full)} rows with missing population data.")

    # Filter for deep dive (0-15 km)
    ANALYSIS_RADIUS_KM = 15
    df_dive = df_main_full[df_main_full['distance_km'] <= ANALYSIS_RADIUS_KM].copy()
    print(f"\n--- Focusing analysis on {len(df_dive)} observations within {ANALYSIS_RADIUS_KM} km ---")

    # Create DiD variables
    df_dive['Post'] = (df_dive['Date'] >= df_dive['event_date']).astype(int)
    granular_bins_km = [-np.inf, 2, 6, 10, 15]
    granular_labels_km = ['0-2 km', '2-6 km', '6-10 km', '10-15 km']
    df_dive['distance_category_granular'] = pd.cut(df_dive['distance_km'], bins=granular_bins_km, labels=granular_labels_km, right=False)
    df_dive = df_dive.dropna(subset=['distance_category_granular'])
    print("\nCreated granular distance categories:")
    print(df_dive['distance_category_granular'].value_counts().sort_index())

    # Calculate rates per 1000 people
    df_dive['Primary_Visits_Rate'] = (df_dive['Primary_Visits'] / df_dive['TotPopACS']) * 1000
    df_dive['Emergency_Visits_Rate'] = (df_dive['Emergency_Visits'] / df_dive['TotPopACS']) * 1000
    print("✅ Calculated visit rates per 1000 people.")

    # Define formulas with RATES
    Y_PRIMARY_RATE_NAME = 'Primary_Visits_Rate'
    Y_EMERGENCY_RATE_NAME = 'Emergency_Visits_Rate'
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    interaction_term = "Post * C(distance_category_granular)"
    formula_primary_rate = f"{Y_PRIMARY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_rate = f"{Y_EMERGENCY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    print(f"\nFormula (Primary, Rate): {formula_primary_rate}")
    print(f"Formula (Emergency, Rate): {formula_emergency_rate}")

    # Run Models
    print("\n--- Running models with rates ---")
    df_dive['distance_category_granular'] = pd.Categorical(df_dive['distance_category_granular'], categories=granular_labels_km, ordered=True)
    df_new_dive = df_dive[df_dive['clinic_type'] == 'New'].copy()
    df_acquired_dive = df_dive[df_dive['clinic_type'] == 'Acquired'].copy()
    models_dive_rate = {}
    try:
        if not df_new_dive.empty:
            print("...Running 1A (Rate): New, Primary")
            models_dive_rate['1A'] = smf.ols(formula_primary_rate, data=df_new_dive).fit()
            print("...Running 1B (Rate): New, Emergency")
            models_dive_rate['1B'] = smf.ols(formula_emergency_rate, data=df_new_dive).fit()
        if not df_acquired_dive.empty:
            print("...Running 2A (Rate): Acquired, Primary")
            models_dive_rate['2A'] = smf.ols(formula_primary_rate, data=df_acquired_dive).fit()
            print("...Running 2B (Rate): Acquired, Emergency")
            models_dive_rate['2B'] = smf.ols(formula_emergency_rate, data=df_acquired_dive).fit()
        print("✅ Models with rates are estimated.")
    except Exception as e:
        print(f"❌ ERROR FITTING MODELS WITH RATES: {e}")
        raise

    # --- Visualization (using the corrected extraction function) ---
    print("\n--- Visualizing results with rates ---")
    sns.set(style="whitegrid")

    def extract_effects_simple(model_result, category_order):
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        if 'Post' not in params.index:
            print("Warning: 'Post' coefficient not found. Cannot plot.")
            return pd.DataFrame(results)
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Post'])
        results['Conf_Low'].append(conf.loc['Post', 0])
        results['Conf_High'].append(conf.loc['Post', 1])
        for cat in category_order[1:]:
            term_name = f"Post:C(distance_category_granular)[T.{cat}]"
            if term_name in params.index:
                t_test = model_result.t_test(f"Post + {term_name}")
                results['Distance'].append(cat)
                results['Effect'].append(t_test.effect[0])
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    def plot_comparison_graphs_simple(models_dict, category_order, y_primary, y_emergency):
        fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 16), sharex=True)
        
        ax1.set_title(f"DiD Effect on {y_primary} (per 1000 ppl)", fontsize=16, pad=20)
        if '1A' in models_dict:
            data = extract_effects_simple(models_dict['1A'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax1.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='-o', capsize=5, label='New Clinics')
        if '2A' in models_dict:
            data = extract_effects_simple(models_dict['2A'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax1.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='--s', capsize=5, label='Acquired Clinics')
        ax1.axhline(0, color='black', linestyle='--')
        ax1.set_ylabel("DiD Effect Size (Change in Daily Visits per 1000 People)")
        ax1.legend()

        ax2.set_title(f"DiD Effect on {y_emergency} (per 1000 ppl)", fontsize=16, pad=20)
        if '1B' in models_dict:
            data = extract_effects_simple(models_dict['1B'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax2.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='-o', capsize=5, label='New Clinics')
        if '2B' in models_dict:
            data = extract_effects_simple(models_dict['2B'], category_order)
            if not data.empty:
                errors = [data['Effect'] - data['Conf_Low'], data['Conf_High'] - data['Effect']]
                ax2.errorbar(x=data['Distance'], y=data['Effect'], yerr=errors, fmt='--s', capsize=5, label='Acquired Clinics')
        ax2.axhline(0, color='black', linestyle='--')
        ax2.set_xlabel("Distance Category (km)")
        ax2.set_ylabel("DiD Effect Size (Change in Daily Visits per 1000 People)")
        ax2.legend()
        
        plt.tight_layout(pad=2.0)
        plt.show()

    # Run the plotting function for the rate-based models
    plot_comparison_graphs_simple(models_dive_rate, granular_labels_km, Y_PRIMARY_RATE_NAME, Y_EMERGENCY_RATE_NAME)

    print("\n🎉 --- 'Deep Dive with Rates' analysis complete! ---")

# 13. DiD Analysis (Method 5): "Deep Dive" with Visit Rates (0-15km)

This is our final and most robust specification. It combines the strengths of the previous two methods:

1.  **"Deep Dive" Sample**: It focuses *only* on the 0-15km radius where the treatment effect is expected to be strongest.
2.  **"Visit Rates" Dependent Variable**: It normalizes the outcome variable by population, using **visits per 1,000 people**.

This allows us to get a granular view of the distance decay (using bins like `0-2 km`, `2-6 km`, etc.) while properly controlling for the fact that different ZIP codes have different population sizes.

## 13.1. Data Preparation: Population & Visit Rates

1.  **Load Population Data**: We load `TotPopACS` (Total Population) from the `Crosswalk_IL.csv` file. We use `pd.to_numeric` with `errors='coerce'` to robustly handle any non-numeric entries.
2.  **Merge & Filter**: We merge the population data onto our main aggregated dataset. Any ZIP-day observation without corresponding population data is dropped.
3.  **Filter for "Deep Dive"**: We create the `df_dive` DataFrame by keeping *only* observations where `distance_km <= 15`.
4.  **Calculate Visit Rates**: We create our two final dependent variables:
    * `Primary_Visits_Rate = (Primary_Visits / TotPopACS) * 1000`
    * `Emergency_Visits_Rate = (Emergency_Visits / TotPopACS) * 1000`

## 13.2. Model Specification (Pooled OLS with Rates)

As in the previous "Deep Dive," we use a Pooled OLS model with Time Fixed Effects (`C(Year) + C(Month)`) and no ZIP Code Fixed Effects. The model estimates the pre-post change in *visit rates* and how this change differs across our granular distance bins.

The formula is:
$Y\_Rate_{it} = \beta_0 + \beta_1(Post_{it}) + \sum_{k} \delta_k(Post_{it} \times DistanceCategory_{ik}) + \lambda_t + \epsilon_{it}$

* $Y\_Rate_{it}$ is the visit rate per 1,000 people.
* $\beta_1$ is the DiD effect for the reference distance category ('0-2 km').
* $\delta_k$ is the *additional* effect for distance category $k$.
* $\lambda_t$ are the Time (Year + Month) Fixed Effects.

## 13.3. Visualization: Absolute and Relative Effects

The final visualization is a 2x2 grid of four plots, one for each model (New/Primary, New/Emergency, Acquired/Primary, Acquired/Emergency).

Each plot shows the DiD effect in two ways:
* **Left Y-Axis (Absolute)**: Shows the "Change in Daily Visits per 1000 People".
* **Right Y-Axis (Relative)**: Shows the "Estimate (As % of Mean)". This is calculated by dividing the absolute effect by the **pre-treatment mean** of the dependent variable for that specific group. This mirrors the academic example plot and shows economic significance.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
from matplotlib.ticker import PercentFormatter # Import the formatter

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 STARTING 'Deep Dive with Rates' ANALYSIS (CORRECTED) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    # --- Load and Prepare Data (as before) ---
    df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")

    # Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # --- CORRECTED SECTION: Load and clean population data robustly ---
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    
    zip_population['TotPopACS'] = pd.to_numeric(
        zip_population['TotPopACS'].str.replace(',', '', regex=False),
        errors='coerce'  # This turns bad text into NaN instead of crashing
    )
    
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    print(f"✅ Population data loaded and cleaned for {len(zip_population)} ZIP codes.")

    # ZIP Coordinates
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # Clinic Info
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    df_clinics_info = df_clinics_info.merge(zip_coords, left_on='Zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['IntPtLat', 'IntPtLon', 'event_date', 'clinic_type'])

    # Treatment Map
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords, left_on='Filtered_Patient_ZipCode', right_index=True).dropna()
    treatment_map = []
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['IntPtLat'], patient_row['IntPtLon'])
        min_distance = np.inf
        nearest_clinic = {}
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['IntPtLat'], clinic_row['IntPtLon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_distance:
                min_distance = distance
                nearest_clinic = {'clinic_type': clinic_row['clinic_type'], 'event_date': clinic_row['event_date']}
        if nearest_clinic:
            treatment_map.append({'Filtered_Patient_ZipCode': patient_zip, 'distance_km': min_distance, 'clinic_type': nearest_clinic['clinic_type'], 'event_date': nearest_clinic['event_date']})
    df_treatment_map = pd.DataFrame(treatment_map)

    # Aggregate Visits
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()
    df_main_full = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left').dropna(subset=['distance_km', 'clinic_type', 'event_date'])

    # Merge population
    df_main_full = df_main_full.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    initial_rows = len(df_main_full)
    df_main_full = df_main_full.dropna(subset=['TotPopACS'])
    print(f"ℹ️ Dropped {initial_rows - len(df_main_full)} rows with missing population data.")

    # Filter for deep dive (0-15 km)
    ANALYSIS_RADIUS_KM = 15
    df_dive = df_main_full[df_main_full['distance_km'] <= ANALYSIS_RADIUS_KM].copy()
    print(f"\n--- Focusing analysis on {len(df_dive)} observations within {ANALYSIS_RADIUS_KM} km ---")

    # Create DiD variables
    df_dive['Post'] = (df_dive['Date'] >= df_dive['event_date']).astype(int)
    granular_bins_km = [-np.inf, 2, 6, 10, 15]
    granular_labels_km = ['0-2 km', '2-6 km', '6-10 km', '10-15 km']
    df_dive['distance_category_granular'] = pd.cut(df_dive['distance_km'], bins=granular_bins_km, labels=granular_labels_km, right=False)
    
    # --- FIX: Dynamically get non-empty bins ---
    df_dive = df_dive.dropna(subset=['distance_category_granular'])
    granular_labels_km_actual = df_dive['distance_category_granular'].cat.categories.tolist()
    print("\nCreated granular distance categories:")
    print(df_dive['distance_category_granular'].value_counts().sort_index())
    if len(granular_labels_km_actual) != len(granular_labels_km):
        print(f"Warning: Using only non-empty categories: {granular_labels_km_actual}")


    # Calculate rates per 1000 people
    df_dive['Primary_Visits_Rate'] = (df_dive['Primary_Visits'] / df_dive['TotPopACS']) * 1000
    df_dive['Emergency_Visits_Rate'] = (df_dive['Emergency_Visits'] / df_dive['TotPopACS']) * 1000
    print("✅ Calculated visit rates per 1000 people.")

    # Define formulas with RATES
    Y_PRIMARY_RATE_NAME = 'Primary_Visits_Rate'
    Y_EMERGENCY_RATE_NAME = 'Emergency_Visits_Rate'
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    interaction_term = "Post * C(distance_category_granular)"
    formula_primary_rate = f"{Y_PRIMARY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_rate = f"{Y_EMERGENCY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    print(f"\nFormula (Primary, Rate): {formula_primary_rate}")
    print(f"Formula (Emergency, Rate): {formula_emergency_rate}")

    # Run Models
    print("\n--- Running models with rates ---")
    df_dive['distance_category_granular'] = pd.Categorical(df_dive['distance_category_granular'], categories=granular_labels_km_actual, ordered=True)
    df_new_dive = df_dive[df_dive['clinic_type'] == 'New'].copy()
    df_acquired_dive = df_dive[df_dive['clinic_type'] == 'Acquired'].copy()
    
    models_dive_rate = {}
    try:
        if not df_new_dive.empty:
            print("...Running 1A (Rate): New, Primary")
            models_dive_rate['1A'] = smf.ols(formula_primary_rate, data=df_new_dive).fit()
            print("...Running 1B (Rate): New, Emergency")
            models_dive_rate['1B'] = smf.ols(formula_emergency_rate, data=df_new_dive).fit()
        if not df_acquired_dive.empty:
            print("...Running 2A (Rate): Acquired, Primary")
            models_dive_rate['2A'] = smf.ols(formula_primary_rate, data=df_acquired_dive).fit()
            print("...Running 2B (Rate): Acquired, Emergency")
            models_dive_rate['2B'] = smf.ols(formula_emergency_rate, data=df_acquired_dive).fit()
        print("✅ Models with rates are estimated.")
    except Exception as e:
        print(f"❌ ERROR FITTING MODELS WITH RATES: {e}")
        raise

    # --- NEW: Calculate pre-treatment means for relative axis ---
    
    def get_pre_mean(df, outcome):
        """Helper function to safely calculate the pre-treatment mean."""
        if df.empty or 'Post' not in df.columns or outcome not in df.columns:
            return 0
        pre_data = df[df['Post'] == 0][outcome]
        if pre_data.empty:
            return 0
        return pre_data.mean()

    # Calculate the four means
    mean_new_primary = get_pre_mean(df_new_dive, Y_PRIMARY_RATE_NAME)
    mean_new_emergency = get_pre_mean(df_new_dive, Y_EMERGENCY_RATE_NAME)
    mean_acq_primary = get_pre_mean(df_acquired_dive, Y_PRIMARY_RATE_NAME)
    mean_acq_emergency = get_pre_mean(df_acquired_dive, Y_EMERGENCY_RATE_NAME)
    
    # Store in a dictionary for easy access
    means_dict = {
        '1A': mean_new_primary,
        '1B': mean_new_emergency,
        '2A': mean_acq_primary,
        '2B': mean_acq_emergency
    }
    
    print("\nCalculated Pre-Treatment Means (Visits per 1000 ppl):")
    print(f"  New, Primary:    {mean_new_primary:.4f}")
    print(f"  New, Emergency:  {mean_new_emergency:.4f}")
    print(f"  Acq, Primary:    {mean_acq_primary:.4f}")
    print(f"  Acq, Emergency:  {mean_acq_emergency:.4f}")

    # --- NEW: Visualization with Dual Axis ---
    print("\n--- Visualizing results with absolute and relative axes ---")
    sns.set(style="whitegrid")

    # This extraction function is the same as before
    def extract_effects_simple(model_result, category_order):
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        if 'Post' not in params.index:
            print(f"Warning: 'Post' coefficient (for {ref_label}) not found. Cannot plot.")
            return pd.DataFrame(results)
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Post'])
        results['Conf_Low'].append(conf.loc['Post', 0])
        results['Conf_High'].append(conf.loc['Post', 1])
        for cat in category_order[1:]:
            term_name = f"Post:C(distance_category_granular)[T.{cat}]"
            if term_name in params.index:
                t_test = model_result.t_test(f"Post + {term_name}")
                results['Distance'].append(cat)
                results['Effect'].append(t_test.effect[0])
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    # --- NEW: Plotting function that creates a *single* plot with two axes ---
    def plot_single_did_with_relative_axis(model_result, mean_value, category_order, title, y_label_abs, ax, color):
        """
        Plots a single DiD result on a given axis (ax) with a twin relative axis.
        """
        # 1. Extract data
        plot_data = extract_effects_simple(model_result, category_order)
        if plot_data.empty:
            ax.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title, fontsize=14, pad=10)
            return

        # 2. Plot the absolute effect (left axis)
        errors = [plot_data['Effect'] - plot_data['Conf_Low'], plot_data['Conf_High'] - plot_data['Effect']]
        ax.errorbar(x=plot_data['Distance'], y=plot_data['Effect'], yerr=errors, fmt='-o', capsize=5, color=color, label='Absolute Effect')
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
        
        # Set labels and title
        ax.set_title(title, fontsize=14, pad=10)
        ax.set_ylabel(y_label_abs, fontsize=12, color=color)
        ax.tick_params(axis='y', labelcolor=color)
        ax.set_xlabel("Distance Category (km)", fontsize=12)

        # 3. Create the relative axis (right axis)
        if mean_value is None or mean_value == 0: # Handle division by zero or missing mean
            print(f"Warning: Mean value is {mean_value} for '{title}'. Cannot plot relative axis.")
            return 

        ax_twin = ax.twinx()
        
        # Get limits from left axis
        y_min, y_max = ax.get_ylim()
        
        # Calculate corresponding limits for right axis
        rel_min = (y_min / mean_value) * 100
        rel_max = (y_max / mean_value) * 100
        
        # Set limits and format
        ax_twin.set_ylim(rel_min, rel_max)
        ax_twin.set_ylabel("Estimate (As % of Mean)", fontsize=12)
        # Format as percentage
        ax_twin.yaxis.set_major_formatter(PercentFormatter(decimals=0))

    # --- NEW: Create a 2x2 plot grid ---
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 16))
    fig.suptitle("DiD 'Deep Dive' Analysis (0-15 km) with Visit Rates", fontsize=20, y=1.03)

    y_label_abs = "Change in Daily Visits per 1000 Ppl"

    # Plot 1: New, Primary (Top-Left)
    if '1A' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['1A'], means_dict['1A'], granular_labels_km_actual,
                                           "New Clinics - Primary Visits Rate", y_label_abs, axes[0, 0], color='C0') # C0 is blue
    else:
        axes[0, 0].set_title("New Clinics - Primary Visits Rate", fontsize=14, pad=10)
        axes[0, 0].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    # Plot 2: New, Emergency (Top-Right)
    if '1B' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['1B'], means_dict['1B'], granular_labels_km_actual,
                                           "New Clinics - Emergency Visits Rate", y_label_abs, axes[0, 1], color='C3') # C3 is red
    else:
        axes[0, 1].set_title("New Clinics - Emergency Visits Rate", fontsize=14, pad=10)
        axes[0, 1].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    # Plot 3: Acquired, Primary (Bottom-Left)
    if '2A' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['2A'], means_dict['2A'], granular_labels_km_actual,
                                           "Acquired Clinics - Primary Visits Rate", y_label_abs, axes[1, 0], color='C2') # C2 is green
    else:
        axes[1, 0].set_title("Acquired Clinics - Primary Visits Rate", fontsize=14, pad=10)
        axes[1, 0].text(0.5, 0.5, "Model not estimated", ha='center', va='center')
        
    # Plot 4: Acquired, Emergency (Bottom-Right)
    if '2B' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['2B'], means_dict['2B'], granular_labels_km_actual,
                                           "Acquired Clinics - Emergency Visits Rate", y_label_abs, axes[1, 1], color='C1') # C1 is orange
    else:
        axes[1, 1].set_title("Acquired Clinics - Emergency Visits Rate", fontsize=14, pad=10)
        axes[1, 1].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()

    print("\n🎉 --- 'Deep Dive with Rates' analysis complete! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
from matplotlib.ticker import PercentFormatter # Import the formatter

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 STARTING 'Deep Dive with Rates' ANALYSIS (CORRECTED) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    # --- Load and Prepare Data (as before) ---
    df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")

    # Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # --- CORRECTED SECTION: Load and clean population data robustly ---
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    
    zip_population['TotPopACS'] = pd.to_numeric(
        zip_population['TotPopACS'].str.replace(',', '', regex=False),
        errors='coerce'  # This turns bad text into NaN instead of crashing
    )
    
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    print(f"✅ Population data loaded and cleaned for {len(zip_population)} ZIP codes.")

    # ZIP Coordinates
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # Clinic Info
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    df_clinics_info = df_clinics_info.merge(zip_coords, left_on='Zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['IntPtLat', 'IntPtLon', 'event_date', 'clinic_type'])

    # Treatment Map
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords, left_on='Filtered_Patient_ZipCode', right_index=True).dropna()
    treatment_map = []
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['IntPtLat'], patient_row['IntPtLon'])
        min_distance = np.inf
        nearest_clinic = {}
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['IntPtLat'], clinic_row['IntPtLon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_distance:
                min_distance = distance
                nearest_clinic = {'clinic_type': clinic_row['clinic_type'], 'event_date': clinic_row['event_date']}
        if nearest_clinic:
            treatment_map.append({'Filtered_Patient_ZipCode': patient_zip, 'distance_km': min_distance, 'clinic_type': nearest_clinic['clinic_type'], 'event_date': nearest_clinic['event_date']})
    df_treatment_map = pd.DataFrame(treatment_map)

    # Aggregate Visits
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()
    df_main_full = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left').dropna(subset=['distance_km', 'clinic_type', 'event_date'])

    # Merge population
    df_main_full = df_main_full.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    initial_rows = len(df_main_full)
    df_main_full = df_main_full.dropna(subset=['TotPopACS'])
    print(f"ℹ️ Dropped {initial_rows - len(df_main_full)} rows with missing population data.")

    # Filter for deep dive (0-15 km)
    ANALYSIS_RADIUS_KM = 15
    df_dive = df_main_full[df_main_full['distance_km'] <= ANALYSIS_RADIUS_KM].copy()
    print(f"\n--- Focusing analysis on {len(df_dive)} observations within {ANALYSIS_RADIUS_KM} km ---")

    # Create DiD variables
    df_dive['Post'] = (df_dive['Date'] >= df_dive['event_date']).astype(int)
    granular_bins_km = [-np.inf, 2, 6, 10, 15]
    granular_labels_km = ['0-2 km', '2-6 km', '6-10 km', '10-15 km']
    df_dive['distance_category_granular'] = pd.cut(df_dive['distance_km'], bins=granular_bins_km, labels=granular_labels_km, right=False)
    
    # --- FIX: Dynamically get non-empty bins ---
    df_dive = df_dive.dropna(subset=['distance_category_granular'])
    granular_labels_km_actual = df_dive['distance_category_granular'].cat.categories.tolist()
    print("\nCreated granular distance categories:")
    print(df_dive['distance_category_granular'].value_counts().sort_index())
    if len(granular_labels_km_actual) != len(granular_labels_km):
        print(f"Warning: Using only non-empty categories: {granular_labels_km_actual}")


    # Calculate rates per 1000 people
    df_dive['Primary_Visits_Rate'] = (df_dive['Primary_Visits'] / df_dive['TotPopACS']) * 1000
    df_dive['Emergency_Visits_Rate'] = (df_dive['Emergency_Visits'] / df_dive['TotPopACS']) * 1000
    print("✅ Calculated visit rates per 1000 people.")

    # Define formulas with RATES
    Y_PRIMARY_RATE_NAME = 'Primary_Visits_Rate'
    Y_EMERGENCY_RATE_NAME = 'Emergency_Visits_Rate'
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    interaction_term = "Post * C(distance_category_granular)"
    formula_primary_rate = f"{Y_PRIMARY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_rate = f"{Y_EMERGENCY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    print(f"\nFormula (Primary, Rate): {formula_primary_rate}")
    print(f"Formula (Emergency, Rate): {formula_emergency_rate}")

    # Run Models
    print("\n--- Running models with rates ---")
    df_dive['distance_category_granular'] = pd.Categorical(df_dive['distance_category_granular'], categories=granular_labels_km_actual, ordered=True)
    df_new_dive = df_dive[df_dive['clinic_type'] == 'New'].copy()
    df_acquired_dive = df_dive[df_dive['clinic_type'] == 'Acquired'].copy()
    
    models_dive_rate = {}
    try:
        if not df_new_dive.empty:
            print("...Running 1A (Rate): New, Primary")
            models_dive_rate['1A'] = smf.ols(formula_primary_rate, data=df_new_dive).fit()
            print("...Running 1B (Rate): New, Emergency")
            models_dive_rate['1B'] = smf.ols(formula_emergency_rate, data=df_new_dive).fit()
        if not df_acquired_dive.empty:
            print("...Running 2A (Rate): Acquired, Primary")
            models_dive_rate['2A'] = smf.ols(formula_primary_rate, data=df_acquired_dive).fit()
            print("...Running 2B (Rate): Acquired, Emergency")
            models_dive_rate['2B'] = smf.ols(formula_emergency_rate, data=df_acquired_dive).fit()
        print("✅ Models with rates are estimated.")
    except Exception as e:
        print(f"❌ ERROR FITTING MODELS WITH RATES: {e}")
        raise

    # --- Calculate pre-treatment means for relative axis ---
    
    def get_pre_mean(df, outcome):
        if df.empty or 'Post' not in df.columns or outcome not in df.columns:
            return 0
        pre_data = df[df['Post'] == 0][outcome]
        if pre_data.empty:
            return 0
        return pre_data.mean()

    means_dict = {
        '1A': get_pre_mean(df_new_dive, Y_PRIMARY_RATE_NAME),
        '1B': get_pre_mean(df_new_dive, Y_EMERGENCY_RATE_NAME),
        '2A': get_pre_mean(df_acquired_dive, Y_PRIMARY_RATE_NAME),
        '2B': get_pre_mean(df_acquired_dive, Y_EMERGENCY_RATE_NAME)
    }
    
    print("\nCalculated Pre-Treatment Means (Visits per 1000 ppl):")
    print(f"  New, Primary:    {means_dict['1A']:.4f}")
    print(f"  New, Emergency:  {means_dict['1B']:.4f}")
    print(f"  Acq, Primary:    {means_dict['2A']:.4f}")
    print(f"  Acq, Emergency:  {means_dict['2B']:.4f}")

    # --- Visualization with Dual Axis ---
    print("\n--- Visualizing results with absolute and relative axes ---")
    sns.set(style="whitegrid")

    # This extraction function is the same as before
    def extract_effects_simple(model_result, category_order):
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        if 'Post' not in params.index:
            print(f"Warning: 'Post' coefficient (for {ref_label}) not found. Cannot plot.")
            return pd.DataFrame(results)
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Post'])
        results['Conf_Low'].append(conf.loc['Post', 0])
        results['Conf_High'].append(conf.loc['Post', 1])
        for cat in category_order[1:]:
            term_name = f"Post:C(distance_category_granular)[T.{cat}]"
            if term_name in params.index:
                t_test = model_result.t_test(f"Post + {term_name}")
                results['Distance'].append(cat)
                results['Effect'].append(t_test.effect[0])
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    # --- MODIFICATION: Updated plotting function ---
    def plot_single_did_with_relative_axis(model_result, mean_value, category_order, title, y_label_abs, ax, color):
        """
        Plots a single DiD result on a given axis (ax) with a twin relative axis
        using a shaded confidence band.
        """
        # 1. Extract data
        plot_data = extract_effects_simple(model_result, category_order)
        if plot_data.empty:
            ax.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title, fontsize=14, pad=10)
            return

        # --- START OF MODIFICATION ---
        
        # 2. Plot the 95% confidence interval band (shaded area)
        ax.fill_between(
            plot_data['Distance'],
            plot_data['Conf_Low'],
            plot_data['Conf_High'],
            color=color,
            alpha=0.2,  # This creates the semi-transparent fill
            label='95% Confidence Interval'
        )

        # 3. Plot the point estimate line on top
        ax.plot(
            plot_data['Distance'],
            plot_data['Effect'],
            color=color,
            marker='o',
            linestyle='-',
            label='Point Estimate'
        )
        
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
        
        # --- END OF MODIFICATION ---

        # Set labels and title
        ax.set_title(title, fontsize=14, pad=10)
        ax.set_ylabel(y_label_abs, fontsize=12, color=color)
        ax.tick_params(axis='y', labelcolor=color)
        ax.set_xlabel("Distance Category (km)", fontsize=12)
        ax.legend(loc='best') # Add a legend

        # 4. Create the relative axis (right axis)
        if mean_value is None or mean_value == 0: 
            print(f"Warning: Mean value is {mean_value} for '{title}'. Cannot plot relative axis.")
            return 

        ax_twin = ax.twinx()
        
        # Get limits from left axis
        y_min, y_max = ax.get_ylim()
        
        # Calculate corresponding limits for right axis
        # Use 1.0 for PercentFormatter(1.0) -> 0.2 = 20%
        rel_min = y_min / mean_value 
        rel_max = y_max / mean_value
        
        # Set limits and format
        ax_twin.set_ylim(rel_min, rel_max)
        ax_twin.set_ylabel("Estimate (As % of Mean)", fontsize=12)
        # Format as percentage, e.g., 0.2 -> 20%
        ax_twin.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)) 
        # --- End of plotting function ---


    # --- Create a 2x2 plot grid (No change here) ---
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 16))
    fig.suptitle("DiD 'Deep Dive' Analysis (0-15 km) with Visit Rates", fontsize=20, y=1.03)

    y_label_abs = "Change in Daily Visits per 1000 Ppl"

    # Plot 1: New, Primary (Top-Left)
    if '1A' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['1A'], means_dict['1A'], granular_labels_km_actual,
                                           "New Clinics - Primary Visits Rate", y_label_abs, axes[0, 0], color='C0') # C0 is blue
    else:
        axes[0, 0].set_title("New Clinics - Primary Visits Rate", fontsize=14, pad=10)
        axes[0, 0].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    # Plot 2: New, Emergency (Top-Right)
    if '1B' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['1B'], means_dict['1B'], granular_labels_km_actual,
                                           "New Clinics - Emergency Visits Rate", y_label_abs, axes[0, 1], color='C3') # C3 is red
    else:
        axes[0, 1].set_title("New Clinics - Emergency Visits Rate", fontsize=14, pad=10)
        axes[0, 1].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    # Plot 3: Acquired, Primary (Bottom-Left)
    if '2A' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['2A'], means_dict['2A'], granular_labels_km_actual,
                                           "Acquired Clinics - Primary Visits Rate", y_label_abs, axes[1, 0], color='C2') # C2 is green
    else:
        axes[1, 0].set_title("Acquired Clinics - Primary Visits Rate", fontsize=14, pad=10)
        axes[1, 0].text(0.5, 0.5, "Model not estimated", ha='center', va='center')
        
    # Plot 4: Acquired, Emergency (Bottom-Right)
    if '2B' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['2B'], means_dict['2B'], granular_labels_km_actual,
                                           "Acquired Clinics - Emergency Visits Rate", y_label_abs, axes[1, 1], color='C1') # C1 is orange
    else:
        axes[1, 1].set_title("Acquired Clinics - Emergency Visits Rate", fontsize=14, pad=10)
        axes[1, 1].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()

    # --- FIX: Added the missing closing quote ---
    print("\n🎉 --- 'Deep Dive with Rates' analysis complete! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
from matplotlib.ticker import PercentFormatter # Import the formatter

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 STARTING 'Deep Dive with Rates' ANALYSIS (CORRECTED) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    # --- Load and Prepare Data (as before) ---
    df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")

    # Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # --- CORRECTED SECTION: Load and clean population data robustly ---
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    
    zip_population['TotPopACS'] = pd.to_numeric(
        zip_population['TotPopACS'].str.replace(',', '', regex=False),
        errors='coerce'  # This turns bad text into NaN instead of crashing
    )
    
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    print(f"✅ Population data loaded and cleaned for {len(zip_population)} ZIP codes.")

    # ZIP Coordinates
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # Clinic Info
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    df_clinics_info = df_clinics_info.merge(zip_coords, left_on='Zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['IntPtLat', 'IntPtLon', 'event_date', 'clinic_type'])

    # Treatment Map
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords, left_on='Filtered_Patient_ZipCode', right_index=True).dropna()
    treatment_map = []
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['IntPtLat'], patient_row['IntPtLon'])
        min_distance = np.inf
        nearest_clinic = {}
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['IntPtLat'], clinic_row['IntPtLon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_distance:
                min_distance = distance
                nearest_clinic = {'clinic_type': clinic_row['clinic_type'], 'event_date': clinic_row['event_date']}
        if nearest_clinic:
            treatment_map.append({'Filtered_Patient_ZipCode': patient_zip, 'distance_km': min_distance, 'clinic_type': nearest_clinic['clinic_type'], 'event_date': nearest_clinic['event_date']})
    df_treatment_map = pd.DataFrame(treatment_map)

    # Aggregate Visits
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()
    df_main_full = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left').dropna(subset=['distance_km', 'clinic_type', 'event_date'])

    # Merge population
    df_main_full = df_main_full.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    initial_rows = len(df_main_full)
    df_main_full = df_main_full.dropna(subset=['TotPopACS'])
    print(f"ℹ️ Dropped {initial_rows - len(df_main_full)} rows with missing population data.")

    # Filter for deep dive (0-15 km)
    ANALYSIS_RADIUS_KM = 15
    df_dive = df_main_full[df_main_full['distance_km'] <= ANALYSIS_RADIUS_KM].copy()
    print(f"\n--- Focusing analysis on {len(df_dive)} observations within {ANALYSIS_RADIUS_KM} km ---")

    # Create DiD variables
    df_dive['Post'] = (df_dive['Date'] >= df_dive['event_date']).astype(int)
    granular_bins_km = [-np.inf, 2, 6, 10, 15]
    granular_labels_km = ['0-2 km', '2-6 km', '6-10 km', '10-15 km']
    df_dive['distance_category_granular'] = pd.cut(df_dive['distance_km'], bins=granular_bins_km, labels=granular_labels_km, right=False)
    
    # --- FIX: Dynamically get non-empty bins ---
    df_dive = df_dive.dropna(subset=['distance_category_granular'])
    granular_labels_km_actual = df_dive['distance_category_granular'].cat.categories.tolist()
    print("\nCreated granular distance categories:")
    print(df_dive['distance_category_granular'].value_counts().sort_index())
    if len(granular_labels_km_actual) != len(granular_labels_km):
        print(f"Warning: Using only non-empty categories: {granular_labels_km_actual}")


    # Calculate rates per 1000 people
    df_dive['Primary_Visits_Rate'] = (df_dive['Primary_Visits'] / df_dive['TotPopACS']) * 1000
    df_dive['Emergency_Visits_Rate'] = (df_dive['Emergency_Visits'] / df_dive['TotPopACS']) * 1000
    print("✅ Calculated visit rates per 1000 people.")

    # Define formulas with RATES
    Y_PRIMARY_RATE_NAME = 'Primary_Visits_Rate'
    Y_EMERGENCY_RATE_NAME = 'Emergency_Visits_Rate'
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    interaction_term = "Post * C(distance_category_granular)"
    formula_primary_rate = f"{Y_PRIMARY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_rate = f"{Y_EMERGENCY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    print(f"\nFormula (Primary, Rate): {formula_primary_rate}")
    print(f"Formula (Emergency, Rate): {formula_emergency_rate}")

    # Run Models
    print("\n--- Running models with rates ---")
    df_dive['distance_category_granular'] = pd.Categorical(df_dive['distance_category_granular'], categories=granular_labels_km_actual, ordered=True)
    df_new_dive = df_dive[df_dive['clinic_type'] == 'New'].copy()
    df_acquired_dive = df_dive[df_dive['clinic_type'] == 'Acquired'].copy()
    
    models_dive_rate = {}
    try:
        if not df_new_dive.empty:
            print("...Running 1A (Rate): New, Primary")
            models_dive_rate['1A'] = smf.ols(formula_primary_rate, data=df_new_dive).fit()
            print("...Running 1B (Rate): New, Emergency")
            models_dive_rate['1B'] = smf.ols(formula_emergency_rate, data=df_new_dive).fit()
        if not df_acquired_dive.empty:
            print("...Running 2A (Rate): Acquired, Primary")
            models_dive_rate['2A'] = smf.ols(formula_primary_rate, data=df_acquired_dive).fit()
            print("...Running 2B (Rate): Acquired, Emergency")
            models_dive_rate['2B'] = smf.ols(formula_emergency_rate, data=df_acquired_dive).fit()
        print("✅ Models with rates are estimated.")
    except Exception as e:
        print(f"❌ ERROR FITTING MODELS WITH RATES: {e}")
        raise

    # --- NEW: Calculate pre-treatment means for relative axis ---
    
    def get_pre_mean(df, outcome):
        """Helper function to safely calculate the pre-treatment mean."""
        if df.empty or 'Post' not in df.columns or outcome not in df.columns:
            return 0
        pre_data = df[df['Post'] == 0][outcome]
        if pre_data.empty:
            return 0
        return pre_data.mean()

    # Calculate the four means
    mean_new_primary = get_pre_mean(df_new_dive, Y_PRIMARY_RATE_NAME)
    mean_new_emergency = get_pre_mean(df_new_dive, Y_EMERGENCY_RATE_NAME)
    mean_acq_primary = get_pre_mean(df_acquired_dive, Y_PRIMARY_RATE_NAME)
    mean_acq_emergency = get_pre_mean(df_acquired_dive, Y_EMERGENCY_RATE_NAME)
    
    # Store in a dictionary for easy access
    means_dict = {
        '1A': mean_new_primary,
        '1B': mean_new_emergency,
        '2A': mean_acq_primary,
        '2B': mean_acq_emergency
    }
    
    print("\nCalculated Pre-Treatment Means (Visits per 1000 ppl):")
    print(f"  New, Primary:    {mean_new_primary:.4f}")
    print(f"  New, Emergency:  {mean_new_emergency:.4f}")
    print(f"  Acq, Primary:    {mean_acq_primary:.4f}")
    print(f"  Acq, Emergency:  {mean_acq_emergency:.4f}")

    # --- NEW: Visualization with Dual Axis ---
    print("\n--- Visualizing results with absolute and relative axes ---")
    sns.set(style="whitegrid")

    # This extraction function is the same as before
    def extract_effects_simple(model_result, category_order):
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        if 'Post' not in params.index:
            print(f"Warning: 'Post' coefficient (for {ref_label}) not found. Cannot plot.")
            return pd.DataFrame(results)
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Post'])
        results['Conf_Low'].append(conf.loc['Post', 0])
        results['Conf_High'].append(conf.loc['Post', 1])
        for cat in category_order[1:]:
            term_name = f"Post:C(distance_category_granular)[T.{cat}]"
            if term_name in params.index:
                t_test = model_result.t_test(f"Post + {term_name}")
                results['Distance'].append(cat)
                results['Effect'].append(t_test.effect[0])
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    # --- NEW: Plotting function that creates a *single* plot with two axes ---
    def plot_single_did_with_relative_axis(model_result, mean_value, category_order, title, y_label_abs, ax, color):
        """
        Plots a single DiD result on a given axis (ax) with a twin relative axis.
        """
        # 1. Extract data
        plot_data = extract_effects_simple(model_result, category_order)
        if plot_data.empty:
            ax.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title, fontsize=14, pad=10)
            return

        # 2. Plot the absolute effect (left axis)
        errors = [plot_data['Effect'] - plot_data['Conf_Low'], plot_data['Conf_High'] - plot_data['Effect']]
        ax.errorbar(x=plot_data['Distance'], y=plot_data['Effect'], yerr=errors, fmt='-o', capsize=5, color=color, label='Absolute Effect')
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
        
        # Set labels and title
        ax.set_title(title, fontsize=14, pad=10)
        ax.set_ylabel(y_label_abs, fontsize=12, color=color)
        ax.tick_params(axis='y', labelcolor=color)
        ax.set_xlabel("Distance Category (km)", fontsize=12)

        # 3. Create the relative axis (right axis)
        if mean_value is None or mean_value == 0: # Handle division by zero or missing mean
            print(f"Warning: Mean value is {mean_value} for '{title}'. Cannot plot relative axis.")
            return 

        ax_twin = ax.twinx()
        
        # Get limits from left axis
        y_min, y_max = ax.get_ylim()
        
        # Calculate corresponding limits for right axis
        rel_min = (y_min / mean_value) * 100
        rel_max = (y_max / mean_value) * 100
        
        # Set limits and format
        ax_twin.set_ylim(rel_min, rel_max)
        ax_twin.set_ylabel("Estimate (As % of Mean)", fontsize=12)
        # Format as percentage
        ax_twin.yaxis.set_major_formatter(PercentFormatter(decimals=0))

    # --- NEW: Create a 2x2 plot grid ---
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 16))
    fig.suptitle("DiD 'Deep Dive' Analysis (0-15 km) with Visit Rates", fontsize=20, y=1.03)

    y_label_abs = "Change in Daily Visits per 1000 Ppl"

    # Plot 1: New, Primary (Top-Left)
    if '1A' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['1A'], means_dict['1A'], granular_labels_km_actual,
                                           "New Clinics - Primary Visits Rate", y_label_abs, axes[0, 0], color='C0') # C0 is blue
    else:
        axes[0, 0].set_title("New Clinics - Primary Visits Rate", fontsize=14, pad=10)
        axes[0, 0].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    # Plot 2: New, Emergency (Top-Right)
    if '1B' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['1B'], means_dict['1B'], granular_labels_km_actual,
                                           "New Clinics - Emergency Visits Rate", y_label_abs, axes[0, 1], color='C3') # C3 is red
    else:
        axes[0, 1].set_title("New Clinics - Emergency Visits Rate", fontsize=14, pad=10)
        axes[0, 1].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    # Plot 3: Acquired, Primary (Bottom-Left)
    if '2A' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['2A'], means_dict['2A'], granular_labels_km_actual,
                                           "Acquired Clinics - Primary Visits Rate", y_label_abs, axes[1, 0], color='C2') # C2 is green
    else:
        axes[1, 0].set_title("Acquired Clinics - Primary Visits Rate", fontsize=14, pad=10)
        axes[1, 0].text(0.5, 0.5, "Model not estimated", ha='center', va='center')
        
    # Plot 4: Acquired, Emergency (Bottom-Right)
    if '2B' in models_dive_rate:
        plot_single_did_with_relative_axis(models_dive_rate['2B'], means_dict['2B'], granular_labels_km_actual,
                                           "Acquired Clinics - Emergency Visits Rate", y_label_abs, axes[1, 1], color='C1') # C1 is orange
    else:
        axes[1, 1].set_title("Acquired Clinics - Emergency Visits Rate", fontsize=14, pad=10)
        axes[1, 1].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()

    print("\n🎉 --- 'Deep Dive with Rates' analysis complete! ---")

# 14. DiD Analysis (Method 1): Pooled OLS with Patient-to-Nearest-Clinic Distance

This section implements a Pooled OLS Difference-in-Differences model. The core logic is to assign every patient ZIP code to its **single nearest clinic** and then define "treatment" based on that distance.

## 14.1. Treatment Map & Group Definition

This is the key logic for this method:

1.  **Find Nearest Clinic**: We iterate through *every* patient ZIP code and find its single closest clinic, regardless of distance.
2.  **Store Assignment**: We create a "treatment map" that links each `Patient_Zip` to its `nearest_clinic_type`, `event_date`, and `distance_to_nearest_clinic_km`.
3.  **Define Treatment Group**: A ZIP code is in the "treatment group" (`Treat = 1`) if its single nearest clinic is **within 50km**.
4.  **Define Control Group**: A ZIP code is in the "control group" (`Treat = 0`) if its single nearest clinic is **further than 50km away**.

## 14.2. Dependent Variable: Visit Rates

To control for differences in population, our dependent variable is **visit rates per 1,000 people**.

* `Primary_Visits_Rate = (Primary_Visits / TotPopACS) * 1000`
* `Emergency_Visits_Rate = (Emergency_Visits / TotPopACS) * 1000`

## 14.3. Model Specification (Pooled OLS)

We use a Pooled OLS model (which does **not** include ZIP Code Fixed Effects) and control for Time Fixed Effects. This is a simpler, faster alternative to the TWFE model.

The formula tests how the DiD effect (`Treat * Post`) varies by the distance categories:

$Y\_Rate_{it} = \beta_0 + \sum_{k} \delta_k(Treat_i \times Post_{it} \times DistanceCategory_{ik}) + \text{Main Effects} + \lambda_t + \epsilon_{it}$

* $Y\_Rate_{it}$ is the visit rate per 1,000 people.
* The `Treat * Post * C(distance_category)` term estimates the DiD effect for each distance bin $k$.
* $\lambda_t$ are the Time (Year + Month) Fixed Effects.
* The model also includes all lower-order main effects (e.g., `Treat`, `Post`, `C(distance_category)`).

The visualization plots the absolute DiD effect for each distance bin by combining the base `Treat:Post` term with the triple-interaction term, allowing us to see the distance decay of the treatment effect.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
from matplotlib.ticker import PercentFormatter # Import the formatter

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 STARTING ANALYSIS: Method 1 (Patient-to-Nearest-Clinic Distance) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    # --- Load and Prepare Data ---
    df_clinics_info = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")

    # Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # Load and clean population data
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    print(f"✅ Population data loaded and cleaned for {len(zip_population)} ZIP codes.")

    # ZIP Coordinates
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # Clinic Info
    df_clinics_info['Zip'] = df_clinics_info['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info = df_clinics_info.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date'})
    df_clinics_info['event_date'] = pd.to_datetime(df_clinics_info['event_date'])
    df_clinics_info = df_clinics_info.merge(zip_coords, left_on='Zip', right_index=True)
    df_clinics_info = df_clinics_info.dropna(subset=['IntPtLat', 'IntPtLon', 'event_date', 'clinic_type'])

    # --- THIS IS THE KEY LOGIC FOR METHOD 1 ---
    print("...Creating 'treatment map' by finding the single nearest clinic for each patient ZIP...")
    patient_zips_df = pd.DataFrame(final_data['Filtered_Patient_ZipCode'].unique(), columns=['Filtered_Patient_ZipCode'])
    patient_zips_df = patient_zips_df.merge(zip_coords, left_on='Filtered_Patient_ZipCode', right_index=True).dropna()
    
    treatment_map = []
    for _, patient_row in patient_zips_df.iterrows():
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['IntPtLat'], patient_row['IntPtLon'])
        min_distance = np.inf
        nearest_clinic = {}
        # Find the single closest clinic, regardless of distance
        for _, clinic_row in df_clinics_info.iterrows():
            clinic_coords = (clinic_row['IntPtLat'], clinic_row['IntPtLon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_distance:
                min_distance = distance
                nearest_clinic = {'clinic_type': clinic_row['clinic_type'], 'event_date': clinic_row['event_date']}
        if nearest_clinic:
            treatment_map.append({
                'Filtered_Patient_ZipCode': patient_zip, 
                'distance_to_nearest_clinic_km': min_distance,  # This is our key distance
                'clinic_type': nearest_clinic['clinic_type'], 
                'event_date': nearest_clinic['event_date']
            })
    df_treatment_map = pd.DataFrame(treatment_map)
    print(f"✅ 'Treatment map' created for {len(df_treatment_map)} ZIP codes.")

    # Aggregate Visits & Merge Data
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()
    df_main = df_agg.merge(df_treatment_map, on='Filtered_Patient_ZipCode', how='left')
    df_main = df_main.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_main = df_main.dropna(subset=['distance_to_nearest_clinic_km', 'clinic_type', 'event_date', 'TotPopACS'])

    # --- Define DiD Variables using this distance ---
    TREATMENT_RADIUS_KM = 50 # Define a broad "treatment" area
    df_main['Treat'] = (df_main['distance_to_nearest_clinic_km'] <= TREATMENT_RADIUS_KM).astype(int)
    df_main['Post'] = (df_main['Date'] >= df_main['event_date']).astype(int)

    # Use standard distance bins
    bins_km = [-np.inf, 5, 10, 20, np.inf]
    labels_km = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    df_main['distance_category'] = pd.cut(
        df_main['distance_to_nearest_clinic_km'],
        bins=bins_km,
        labels=labels_km,
        right=False
    )
    df_main = df_main.dropna(subset=['distance_category'])
    
    # Calculate Rates per 1000 people
    df_main['Primary_Visits_Rate'] = (df_main['Primary_Visits'] / df_main['TotPopACS']) * 1000
    df_main['Emergency_Visits_Rate'] = (df_main['Emergency_Visits'] / df_main['TotPopACS']) * 1000
    print(f"✅ Final DataFrame created with {len(df_main)} rows for analysis.")

    # --- Define and Run SIMPLIFIED Models ---
    Y_PRIMARY_RATE_NAME = 'Primary_Visits_Rate'
    Y_EMERGENCY_RATE_NAME = 'Emergency_Visits_Rate'
    # NOTE: NO C(Zip)
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    interaction_term = "Treat * Post * C(distance_category)"
    formula_primary_rate = f"{Y_PRIMARY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_rate = f"{Y_EMERGENCY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    
    print("\n--- Running Simplified Models on 'Patient-to-Nearest-Clinic' Distance ---")
    df_main['distance_category'] = pd.Categorical(df_main['distance_category'], categories=labels_km, ordered=True)
    df_new = df_main[df_main['clinic_type'] == 'New'].copy()
    df_acquired = df_main[df_main['clinic_type'] == 'Acquired'].copy()
    
    models_method1 = {}
    try:
        if not df_new.empty:
            models_method1['1A'] = smf.ols(formula_primary_rate, data=df_new).fit()
            models_method1['1B'] = smf.ols(formula_emergency_rate, data=df_new).fit()
        if not df_acquired.empty:
            models_method1['2A'] = smf.ols(formula_primary_rate, data=df_acquired).fit()
            models_method1['2B'] = smf.ols(formula_emergency_rate, data=df_acquired).fit()
        print("✅ Models estimated successfully.")
    except Exception as e:
        print(f"❌ ERROR FITTING MODELS: {e}")
        raise

    # --- NEW: Calculate pre-treatment means for relative axis ---
    def get_pre_treatment_mean(df, outcome):
        """Calculates the pre-treatment mean for the *treated* group."""
        if df.empty:
            return 0
        # Filter for the treated group (<=50km) in the pre-period
        pre_data = df[(df['Treat'] == 1) & (df['Post'] == 0)][outcome]
        if pre_data.empty:
            return 0
        return pre_data.mean()

    means_dict = {
        '1A': get_pre_treatment_mean(df_new, Y_PRIMARY_RATE_NAME),
        '1B': get_pre_treatment_mean(df_new, Y_EMERGENCY_RATE_NAME),
        '2A': get_pre_treatment_mean(df_acquired, Y_PRIMARY_RATE_NAME),
        '2B': get_pre_treatment_mean(df_acquired, Y_EMERGENCY_RATE_NAME)
    }

    print("\nCalculated Pre-Treatment Means (for Treated Group, Visits per 1000 ppl):")
    print(f"  New, Primary:    {means_dict['1A']:.4f}")
    print(f"  New, Emergency:  {means_dict['1B']:.4f}")
    print(f"  Acq, Primary:    {means_dict['2A']:.4f}")
    print(f"  Acq, Emergency:  {means_dict['2B']:.4f}")

    # --- NEW: Visualization setup with shaded CIs and dual axis ---
    print("\n--- Visualizing Results for Method 1 ---")
    sns.set(style="whitegrid")
    
    def extract_effects_method1(model_result, category_order):
        # This function correctly interprets the Treat*Post*Distance interaction
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        
        if 'Treat:Post' not in params.index:
            print("Warning: 'Treat:Post' not found. Cannot plot.")
            return pd.DataFrame(results)

        # Effect for reference group
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Treat:Post'])
        results['Conf_Low'].append(conf.loc['Treat:Post', 0])
        results['Conf_High'].append(conf.loc['Treat:Post', 1])

        # Effects for other groups
        for cat in category_order[1:]:
            term_name = f"Treat:Post:C(distance_category)[T.{cat}]"
            if term_name in params.index:
                t_test = model_result.t_test(f"Treat:Post + {term_name}")
                results['Distance'].append(cat)
                results['Effect'].append(t_test.effect[0])
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
        
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    def plot_single_did_with_relative_axis(model_result, mean_value, category_order, title, y_label_abs, ax, color):
        """
        Plots a single DiD result on a given axis (ax) with a twin relative axis
        using a shaded confidence band.
        """
        # 1. Extract data
        plot_data = extract_effects_method1(model_result, category_order)
        if plot_data.empty:
            ax.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title, fontsize=14, pad=10)
            return

        # 2. Plot the 95% confidence interval band (shaded area)
        ax.fill_between(
            plot_data['Distance'],
            plot_data['Conf_Low'],
            plot_data['Conf_High'],
            color=color,
            alpha=0.2,  # Semi-transparent fill
            label='95% Confidence Interval'
        )

        # 3. Plot the point estimate line on top
        ax.plot(
            plot_data['Distance'],
            plot_data['Effect'],
            color=color,
            marker='o',
            linestyle='-',
            label='Point Estimate'
        )
        
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

        # Set labels and title
        ax.set_title(title, fontsize=14, pad=10)
        ax.set_ylabel(y_label_abs, fontsize=12, color=color)
        ax.tick_params(axis='y', labelcolor=color)
        ax.set_xlabel("Distance to Nearest Clinic (km)", fontsize=12)
        ax.legend(loc='best')

        # 4. Create the relative axis (right axis)
        if mean_value is None or mean_value == 0: 
            print(f"Warning: Mean value is {mean_value} for '{title}'. Cannot plot relative axis.")
            return 

        ax_twin = ax.twinx()
        
        # Get limits from left axis
        y_min, y_max = ax.get_ylim()
        
        # Calculate corresponding limits for right axis
        rel_min = y_min / mean_value 
        rel_max = y_max / mean_value
        
        # Set limits and format
        ax_twin.set_ylim(rel_min, rel_max)
        ax_twin.set_ylabel("Estimate (As % of Mean)", fontsize=12)
        # Format as percentage
        ax_twin.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)) 
    
    # --- Create a 2x2 plot grid ---
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 16))
    fig.suptitle("DiD Pooled OLS Analysis (50km Treatment Zone) with Visit Rates", fontsize=20, y=1.03)

    y_label_abs = "Change in Daily Visits per 1000 Ppl"

    # Plot 1: New, Primary (Top-Left)
    if '1A' in models_method1:
        plot_single_did_with_relative_axis(models_method1['1A'], means_dict['1A'], labels_km,
                                           "New Clinics - Primary Visits Rate", y_label_abs, axes[0, 0], color='C0')
    else:
        axes[0, 0].set_title("New Clinics - Primary Visits Rate", fontsize=14, pad=10)
        axes[0, 0].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    # Plot 2: New, Emergency (Top-Right)
    if '1B' in models_method1:
        plot_single_did_with_relative_axis(models_method1['1B'], means_dict['1B'], labels_km,
                                           "New Clinics - Emergency Visits Rate", y_label_abs, axes[0, 1], color='C3')
    else:
        axes[0, 1].set_title("New Clinics - Emergency Visits Rate", fontsize=14, pad=10)
        axes[0, 1].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    # Plot 3: Acquired, Primary (Bottom-Left)
    if '2A' in models_method1:
        plot_single_did_with_relative_axis(models_method1['2A'], means_dict['2A'], labels_km,
                                           "Acquired Clinics - Primary Visits Rate", y_label_abs, axes[1, 0], color='C2')
    else:
        axes[1, 0].set_title("Acquired Clinics - Primary Visits Rate", fontsize=14, pad=10)
        axes[1, 0].text(0.5, 0.5, "Model not estimated", ha='center', va='center')
        
    # Plot 4: Acquired, Emergency (Bottom-Right)
    if '2B' in models_method1:
        plot_single_did_with_relative_axis(models_method1['2B'], means_dict['2B'], labels_km,
                                           "Acquired Clinics - Emergency Visits Rate", y_label_abs, axes[1, 1], color='C1')
    else:
        axes[1, 1].set_title("Acquired Clinics - Emergency Visits Rate", fontsize=14, pad=10)
        axes[1, 1].text(0.5, 0.5, "Model not estimated", ha='center', va='center')

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()

    print("\n🎉 --- 'Method 1' analysis complete! ---")

# 14. DiD Analysis (Method 6): Clinic-by-Clinic Heterogeneity Analysis

This section moves from a *pooled* effect to a *disaggregated* one. Our previous models (Methods 4 & 5) grouped all "New" clinics together, assuming their impact was uniform. This is a strong assumption.

This analysis tests for **effect heterogeneity** by running a separate, individual DiD model *for each 'New' clinic*. This allows us to see which specific clinics are driving the results and whether the effect is consistent across different locations.

## 14.1. Methodology: A Loop of Case Studies

Instead of one large regression, this script performs a series of "case studies" by looping through every individual clinic identified as 'New'.

**For each 'New' clinic in the list:**

1.  **Define Sample**: A temporary, clinic-specific dataset is created. It consists of all patient-day observations from patient ZIP codes that are **within a 15km radius of that specific clinic**.
2.  **Define Treatment**: The `Post` variable is created based on *that specific clinic's* unique `event_date`.
3.  **Define Bins**: Granular distance bins (`0-2 km`, `2-6 km`, etc.) are created based on the distance from the patient ZIP *to that specific clinic*.
4.  **Calculate Rates**: The `Primary_Visits_Rate` and `Emergency_Visits_Rate` (per 1,000 people) are calculated for this clinic-specific sample.
5.  **Run Model**: The same Pooled OLS "Deep Dive" model (from Method 5) is run *only* on this small, clinic-specific dataset.
    $Y\_Rate_{it} = \beta_0 + \beta_1(Post_{it}) + \sum_{k} \delta_k(Post_{it} \times DistanceCategory_{ik}) + \lambda_t + \epsilon_{it}$
6.  **Generate Output**: A two-panel plot (Primary vs. Emergency) is generated for *this specific clinic* and saved as a `.png` file in the `clinic_specific_did_plots/` folder.

This process repeats for every 'New' clinic, resulting in a folder full of individual outcome plots.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import os # For creating folders
import re # <-- IMPORTED for fixing filenames
from matplotlib.ticker import PercentFormatter # <-- IMPORTED for plotting

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None
pd.set_option('display.max_rows', 100)

print("--- 🚀 STARTING ANALYSIS FOR EACH 'NEW' CLINIC (with RATES) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    # --- Load and Prepare Data (as before) ---
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # Load and clean population
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')

    # ZIP Coordinates
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # Clinic Info
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])

    # --- Select ONLY 'New' clinics for the loop ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy().reset_index()
    print(f"✅ Found {len(df_new_clinics)} 'New' clinics for individual analysis:")
    print(df_new_clinics[['clinic_zip', 'Facility', 'event_date']].to_string())


    # --- Prepare the main DataFrame with visits and population ---
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()
    df_agg = df_agg.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_agg = df_agg.dropna(subset=['TotPopACS'])
    df_agg = df_agg.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_agg = df_agg.dropna(subset=['patient_lat', 'patient_lon'])


    # --- Analysis Settings ---
    ANALYSIS_RADIUS_KM = 15 # Radius around the clinic to analyze
    granular_bins_km = [-np.inf, 2, 6, 10, 15]
    granular_labels_km = ['0-2 km', '2-6 km', '6-10 km', '10-15 km']
    Y_PRIMARY_RATE_NAME = 'Primary_Visits_Rate'
    Y_EMERGENCY_RATE_NAME = 'Emergency_Visits_Rate'
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    interaction_term = "Post * C(distance_category_granular)"
    formula_primary_rate = f"{Y_PRIMARY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_rate = f"{Y_EMERGENCY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"

    # --- Folder for saving plots ---
    output_folder = "clinic_specific_did_plots"
    os.makedirs(output_folder, exist_ok=True)
    print(f"\n📂 Plots will be saved to folder: {output_folder}")

    # --- HELPER FUNCTIONS (defined outside the loop) ---
    def get_pre_mean(df, outcome):
        """Safely calculates the pre-treatment mean."""
        if df.empty or 'Post' not in df.columns or outcome not in df.columns:
            return 0
        pre_data = df[df['Post'] == 0][outcome]
        if pre_data.empty:
            # Fallback to all data if pre-period is empty
            all_data_mean = df[outcome].mean()
            return all_data_mean if not pd.isna(all_data_mean) and all_data_mean != 0 else 1.0 # Avoid zero
        pre_mean = pre_data.mean()
        return pre_mean if not pd.isna(pre_mean) and pre_mean != 0 else 1.0 # Avoid zero

    def extract_effects_simple(model_result, category_order):
        """Extracts DiD effects for the simplified (Post * Category) model."""
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        if 'Post' not in params.index: 
            print(f"Warning: 'Post' (for {ref_label}) not found.")
            return pd.DataFrame(results) # Check
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Post'])
        results['Conf_Low'].append(conf.loc['Post', 0])
        results['Conf_High'].append(conf.loc['Post', 1])
        for cat in category_order[1:]:
            term_name = f"Post:C(distance_category_granular)[T.{cat}]"
            if term_name in params.index:
                t_test = model_result.t_test(f"Post + {term_name}")
                results['Distance'].append(cat)
                results['Effect'].append(t_test.effect[0])
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    def plot_single_did_with_relative_axis(model_result, mean_value, category_order, title, y_label_abs, ax, color):
        """
        Plots a single DiD result with a shaded confidence band and twin axis.
        """
        plot_data = extract_effects_simple(model_result, category_order)
        if plot_data.empty:
            ax.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title, fontsize=12, pad=10)
            return

        # Plot the 95% confidence interval band (shaded area)
        ax.fill_between(
            plot_data['Distance'], plot_data['Conf_Low'], plot_data['Conf_High'],
            color=color, alpha=0.2, label='95% Confidence Interval'
        )
        # Plot the point estimate line on top
        ax.plot(
            plot_data['Distance'], plot_data['Effect'],
            color=color, marker='o', linestyle='-', label='Point Estimate'
        )
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

        # Set labels and title
        ax.set_title(title, fontsize=12, pad=10)
        ax.set_ylabel(y_label_abs, fontsize=10, color=color)
        ax.tick_params(axis='y', labelcolor=color)
        ax.set_xlabel(f"Distance from Clinic (km)", fontsize=10)
        ax.legend(loc='best')

        # Create the relative axis (right axis)
        if mean_value is None or mean_value == 0: 
            print(f"Warning: Mean value is {mean_value} for '{title}'. Cannot plot relative axis.")
            return 
        ax_twin = ax.twinx()
        y_min, y_max = ax.get_ylim()
        rel_min = y_min / mean_value 
        rel_max = y_max / mean_value
        ax_twin.set_ylim(rel_min, rel_max)
        ax_twin.set_ylabel("Estimate (As % of Mean)", fontsize=10)
        ax_twin.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)) 
    # --- END OF HELPER FUNCTIONS ---


    # --- Main LOOP through each 'New' clinic ---
    all_clinic_models = {} 

    for index, clinic_row in df_new_clinics.iterrows():
        clinic_zip = clinic_row['clinic_zip']
        clinic_name = clinic_row.get('Facility', f'Clinic_{clinic_zip}') 
        clinic_event_date = clinic_row['event_date']
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])

        print(f"\n--- Analyzing Clinic: {clinic_name} (ZIP: {clinic_zip}) ---")

        # 1. Select patient ZIP codes within the radius of THIS clinic
        df_clinic_zone = df_agg.copy()
        df_clinic_zone['distance_to_this_clinic_km'] = df_clinic_zone.apply(
            lambda row: geodesic((row['patient_lat'], row['patient_lon']), clinic_coords).kilometers,
            axis=1
        )
        df_clinic_zone = df_clinic_zone[df_clinic_zone['distance_to_this_clinic_km'] <= ANALYSIS_RADIUS_KM]

        if df_clinic_zone.empty:
            print(f"⚠️ Skipped: No patient data found within the {ANALYSIS_RADIUS_KM} km radius.")
            continue

        # 2. Create DiD variables for THIS clinic
        df_clinic_zone['Post'] = (df_clinic_zone['Date'] >= clinic_event_date).astype(int)
        
        # --- DYNAMIC BINNING CHECK ---
        df_clinic_zone['distance_category_granular'] = pd.cut(
            df_clinic_zone['distance_to_this_clinic_km'],
            bins=granular_bins_km,
            labels=granular_labels_km,
            right=False
        )
        df_clinic_zone = df_clinic_zone.dropna(subset=['distance_category_granular'])
        
        # Get the actual non-empty categories *for this clinic*
        clinic_actual_labels = df_clinic_zone['distance_category_granular'].cat.categories.tolist()
        df_clinic_zone['distance_category_granular'] = pd.Categorical(
             df_clinic_zone['distance_category_granular'], 
             categories=clinic_actual_labels, 
             ordered=True
        )

        if df_clinic_zone.empty or df_clinic_zone['Post'].nunique() < 2 or len(clinic_actual_labels) < 2 :
             print(f"⚠️ Skipped: Insufficient variation in data (Post or distance) to analyze this clinic.")
             continue

        # 3. Calculate rates and pre-treatment means
        df_clinic_zone['Primary_Visits_Rate'] = (df_clinic_zone['Primary_Visits'] / df_clinic_zone['TotPopACS']) * 1000
        df_clinic_zone['Emergency_Visits_Rate'] = (df_clinic_zone['Emergency_Visits'] / df_clinic_zone['TotPopACS']) * 1000
        
        primary_pre_mean = get_pre_mean(df_clinic_zone, Y_PRIMARY_RATE_NAME)
        emergency_pre_mean = get_pre_mean(df_clinic_zone, Y_EMERGENCY_RATE_NAME)

        # 4. Run models for THIS clinic
        models_this_clinic = {}
        try:
            print(f"...Running Primary Rate model (Observations: {len(df_clinic_zone)})")
            models_this_clinic['Primary'] = smf.ols(formula_primary_rate, data=df_clinic_zone).fit()
            print(f"...Running Emergency Rate model (Observations: {len(df_clinic_zone)})")
            models_this_clinic['Emergency'] = smf.ols(formula_emergency_rate, data=df_clinic_zone).fit()
            all_clinic_models[clinic_zip] = models_this_clinic
        except Exception as e:
            print(f"❌ ERROR fitting model for {clinic_name}: {e}")
            continue 

        # --- 5. Visualization for THIS clinic ---
        print("...Visualizing results...")
        fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10, 14), sharex=True)
        fig.suptitle(f"DiD Effect for Clinic: {clinic_name} (ZIP: {clinic_zip})", fontsize=16, y=1.02)

        # Plot Primary
        if 'Primary' in models_this_clinic:
            plot_single_did_with_relative_axis(
                models_this_clinic['Primary'], primary_pre_mean, clinic_actual_labels,
                f"Effect on {Y_PRIMARY_RATE_NAME} (per 1000 ppl)", "DiD Effect Size",
                ax1, color='C0'
            )
        else:
             ax1.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax1.transAxes)
             ax1.set_title(f"Effect on {Y_PRIMARY_RATE_NAME} (per 1000 ppl)", fontsize=12)

        # Plot Emergency
        if 'Emergency' in models_this_clinic:
            plot_single_did_with_relative_axis(
                models_this_clinic['Emergency'], emergency_pre_mean, clinic_actual_labels,
                f"Effect on {Y_EMERGENCY_RATE_NAME} (per 1000 ppl)", "DiD Effect Size",
                ax2, color='C3'
            )
        else:
             ax2.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax2.transAxes)
             ax2.set_title(f"Effect on {Y_EMERGENCY_RATE_NAME} (per 1000 ppl)", fontsize=12)

        plt.tight_layout(rect=[0, 0.03, 1, 0.98]) 

        # --- FIX: Save the plot with a sanitized filename ---
        # This removes ALL illegal characters: | / \ : * ? " < >
        safe_clinic_name = re.sub(r'[\\/*?:"<>|]', '_', clinic_name)
        safe_clinic_name = safe_clinic_name.replace(' ', '_') # Also replace spaces
        
        plot_filename = os.path.join(output_folder, f"did_effect_{safe_clinic_name}_{clinic_zip}.png")
        try:
            plt.savefig(plot_filename)
            print(f"✅ Plot saved: {plot_filename}")
        except Exception as e:
            print(f"❌ ERROR saving plot {plot_filename}: {e}")
        plt.close(fig) 

    print("\n🎉 --- Analysis for all 'New' clinics complete! ---")

# 15. DiD Analysis (Method 7): Clinic-by-Clinic Heterogeneity (Acquired Clinics)

This analysis mirrors the previous step (Method 6), but instead of looping through 'New' clinics, we now loop through every **'Acquired' clinic**.

The goal remains the same: to test for effect heterogeneity and identify which specific clinic acquisitions (if any) are driving the pooled results found in our earlier models.

## 15.1. Methodology: A Loop of Case Studies

The script performs a series of "case studies" by looping through every individual clinic identified as **'Acquired'**.

**For each 'Acquired' clinic in the list:**

1.  **Define Sample**: A temporary, clinic-specific dataset is created. It consists of all patient-day observations from patient ZIP codes that are **within a 15km radius of that specific clinic**.
2.  **Define Treatment**: The `Post` variable is created based on *that specific clinic's* unique `event_date`.
3.  **Define Bins**: Granular distance bins (`0-2 km`, `2-6 km`, etc.) are created based on the distance from the patient ZIP *to that specific clinic*.
4.  **Calculate Rates**: The `Primary_Visits_Rate` and `Emergency_Visits_Rate` (per 1,000 people) are calculated for this clinic-specific sample.
5.  **Run Model**: The same Pooled OLS "Deep Dive" model (from Method 5) is run *only* on this small, clinic-specific dataset.
    $Y\_Rate_{it} = \beta_0 + \beta_1(Post_{it}) + \sum_{k} \delta_k(Post_{it} \times DistanceCategory_{ik}) + \lambda_t + \epsilon_{it}$
6.  **Generate Output**: A two-panel plot (Primary vs. Emergency) is generated for *this specific clinic* and saved as a `.png` file in the `acquired_clinic_specific_did_plots/` folder.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import os # For creating folders
import re # <-- IMPORTED for fixing filenames
from matplotlib.ticker import PercentFormatter # <-- IMPORTED for plotting

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None
pd.set_option('display.max_rows', 100)

print("--- 🚀 STARTING ANALYSIS FOR EACH 'ACQUIRED' CLINIC (with RATES) ---")

# --- 1. Check if final_data exists ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    # --- Load and Prepare Data (as before) ---
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # Clean final_data
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    # Load and clean population
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')

    # ZIP Coordinates
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    # Clinic Info
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])

    # --- CHANGE: Select ONLY 'Acquired' clinics for the loop ---
    df_acquired_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'Acquired'].copy().reset_index()
    print(f"✅ Found {len(df_acquired_clinics)} 'Acquired' clinics for individual analysis:")
    print(df_acquired_clinics[['clinic_zip', 'Facility', 'event_date']].to_string())


    # --- Prepare main DataFrame with visits and population (no change) ---
    df_panel = final_data.copy()
    df_panel['Emergency_Visits'] = np.where(df_panel['VisitType'] == 'Emergency', df_panel['EncounterCount'], 0)
    df_panel['Primary_Visits'] = np.where(df_panel['VisitType'] != 'Emergency', df_panel['EncounterCount'], 0)
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month']).agg({'Emergency_Visits': 'sum', 'Primary_Visits': 'sum'}).reset_index()
    df_agg = df_agg.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_agg = df_agg.dropna(subset=['TotPopACS'])
    df_agg = df_agg.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_agg = df_agg.dropna(subset=['patient_lat', 'patient_lon'])


    # --- Analysis Settings (no change) ---
    ANALYSIS_RADIUS_KM = 15
    granular_bins_km = [-np.inf, 2, 6, 10, 15]
    granular_labels_km = ['0-2 km', '2-6 km', '6-10 km', '10-15 km']
    Y_PRIMARY_RATE_NAME = 'Primary_Visits_Rate'
    Y_EMERGENCY_RATE_NAME = 'Emergency_Visits_Rate'
    SIMPLIFIED_CONTROLS = 'C(Year) + C(Month)'
    interaction_term = "Post * C(distance_category_granular)"
    formula_primary_rate = f"{Y_PRIMARY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"
    formula_emergency_rate = f"{Y_EMERGENCY_RATE_NAME} ~ {interaction_term} + {SIMPLIFIED_CONTROLS}"

    # --- CHANGE: Folder for saving plots ---
    output_folder = "acquired_clinic_specific_did_plots" # New folder name
    os.makedirs(output_folder, exist_ok=True)
    print(f"\n📂 Plots will be saved to folder: {output_folder}")

    # --- HELPER FUNCTIONS (defined outside the loop) ---
    def get_pre_mean(df, outcome):
        """Safely calculates the pre-treatment mean."""
        if df.empty or 'Post' not in df.columns or outcome not in df.columns:
            return 0
        pre_data = df[df['Post'] == 0][outcome]
        if pre_data.empty:
            all_data_mean = df[outcome].mean()
            return all_data_mean if not pd.isna(all_data_mean) and all_data_mean != 0 else 1.0
        pre_mean = pre_data.mean()
        return pre_mean if not pd.isna(pre_mean) and pre_mean != 0 else 1.0

    def extract_effects_simple(model_result, category_order):
        """Extracts DiD effects for the simplified (Post * Category) model."""
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        if 'Post' not in params.index: 
            print(f"Warning: 'Post' (for {ref_label}) not found.")
            return pd.DataFrame(results)
        results['Distance'].append(ref_label)
        results['Effect'].append(params['Post'])
        results['Conf_Low'].append(conf.loc['Post', 0])
        results['Conf_High'].append(conf.loc['Post', 1])
        for cat in category_order[1:]:
            term_name = f"Post:C(distance_category_granular)[T.{cat}]"
            if term_name in params.index:
                t_test = model_result.t_test(f"Post + {term_name}")
                results['Distance'].append(cat)
                results['Effect'].append(t_test.effect[0])
                results['Conf_Low'].append(t_test.conf_int()[0][0])
                results['Conf_High'].append(t_test.conf_int()[0][1])
        df = pd.DataFrame(results)
        df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
        return df.sort_values('Distance')

    def plot_single_did_with_relative_axis(model_result, mean_value, category_order, title, y_label_abs, ax, color):
        """
        Plots a single DiD result with a shaded confidence band and twin axis.
        """
        plot_data = extract_effects_simple(model_result, category_order)
        if plot_data.empty:
            ax.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title, fontsize=12, pad=10)
            return

        # Plot the 95% confidence interval band (shaded area)
        ax.fill_between(
            plot_data['Distance'], plot_data['Conf_Low'], plot_data['Conf_High'],
            color=color, alpha=0.2, label='95% Confidence Interval'
        )
        # Plot the point estimate line on top
        ax.plot(
            plot_data['Distance'], plot_data['Effect'],
            color=color, marker='o', linestyle='-', label='Point Estimate'
        )
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

        # Set labels and title
        ax.set_title(title, fontsize=12, pad=10)
        ax.set_ylabel(y_label_abs, fontsize=10, color=color)
        ax.tick_params(axis='y', labelcolor=color)
        ax.set_xlabel(f"Distance from Clinic (km)", fontsize=10)
        ax.legend(loc='best')

        # Create the relative axis (right axis)
        if mean_value is None or mean_value == 0: 
            print(f"Warning: Mean value is {mean_value} for '{title}'. Cannot plot relative axis.")
            return 
        ax_twin = ax.twinx()
        y_min, y_max = ax.get_ylim()
        rel_min = y_min / mean_value 
        rel_max = y_max / mean_value
        ax_twin.set_ylim(rel_min, rel_max)
        ax_twin.set_ylabel("Estimate (As % of Mean)", fontsize=10)
        ax_twin.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)) 
    # --- END OF HELPER FUNCTIONS ---


    # --- Main LOOP through each 'Acquired' clinic ---
    all_acquired_clinic_models = {} 

    for index, clinic_row in df_acquired_clinics.iterrows():
        clinic_zip = clinic_row['clinic_zip']
        clinic_name = clinic_row.get('Facility', f'Acquired_Clinic_{clinic_zip}')
        clinic_event_date = clinic_row['event_date']
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])

        print(f"\n--- Analyzing Clinic: {clinic_name} (ZIP: {clinic_zip}) ---")

        # 1. Select patient ZIP codes in the radius around THIS clinic
        df_clinic_zone = df_agg.copy()
        df_clinic_zone['distance_to_this_clinic_km'] = df_clinic_zone.apply(
            lambda row: geodesic((row['patient_lat'], row['patient_lon']), clinic_coords).kilometers,
            axis=1
        )
        df_clinic_zone = df_clinic_zone[df_clinic_zone['distance_to_this_clinic_km'] <= ANALYSIS_RADIUS_KM]

        if df_clinic_zone.empty:
            print(f"⚠️ Skipped: No patient data found within the {ANALYSIS_RADIUS_KM} km radius.")
            continue

        # 2. Create DiD variables for THIS clinic
        df_clinic_zone['Post'] = (df_clinic_zone['Date'] >= clinic_event_date).astype(int)
        
        # --- DYNAMIC BINNING CHECK ---
        df_clinic_zone['distance_category_granular'] = pd.cut(
            df_clinic_zone['distance_to_this_clinic_km'],
            bins=granular_bins_km,
            labels=granular_labels_km,
            right=False
        )
        df_clinic_zone = df_clinic_zone.dropna(subset=['distance_category_granular'])
        
        clinic_actual_labels = df_clinic_zone['distance_category_granular'].cat.categories.tolist()
        df_clinic_zone['distance_category_granular'] = pd.Categorical(
             df_clinic_zone['distance_category_granular'], 
             categories=clinic_actual_labels, 
             ordered=True
        )

        if df_clinic_zone.empty or df_clinic_zone['Post'].nunique() < 2 or len(clinic_actual_labels) < 2 :
             print(f"⚠️ Skipped: Insufficient variation in data (Post or distance) to analyze this clinic.")
             continue

        # 3. Calculate rates and pre-treatment means
        df_clinic_zone['Primary_Visits_Rate'] = (df_clinic_zone['Primary_Visits'] / df_clinic_zone['TotPopACS']) * 1000
        df_clinic_zone['Emergency_Visits_Rate'] = (df_clinic_zone['Emergency_Visits'] / df_clinic_zone['TotPopACS']) * 1000
        
        primary_pre_mean = get_pre_mean(df_clinic_zone, Y_PRIMARY_RATE_NAME)
        emergency_pre_mean = get_pre_mean(df_clinic_zone, Y_EMERGENCY_RATE_NAME)

        # 4. Run models for THIS clinic
        models_this_clinic = {}
        try:
            print(f"...Running Primary Rate model (Observations: {len(df_clinic_zone)})")
            if df_clinic_zone['distance_category_granular'].nunique() > 1 and df_clinic_zone['Post'].nunique() > 1:
                 models_this_clinic['Primary'] = smf.ols(formula_primary_rate, data=df_clinic_zone).fit()
            else:
                 print("    Skipping Primary: insufficient variation.")

            print(f"...Running Emergency Rate model (Observations: {len(df_clinic_zone)})")
            if df_clinic_zone['distance_category_granular'].nunique() > 1 and df_clinic_zone['Post'].nunique() > 1:
                models_this_clinic['Emergency'] = smf.ols(formula_emergency_rate, data=df_clinic_zone).fit()
            else:
                 print("    Skipping Emergency: insufficient variation.")

            if models_this_clinic:
                all_acquired_clinic_models[clinic_zip] = models_this_clinic
        except Exception as e:
            print(f"❌ ERROR fitting model for {clinic_name}: {e}")
            continue 

        # --- 5. Visualization for THIS clinic ---
        if not models_this_clinic: 
             print("...Visualization skipped (models were not estimated).")
             continue

        print("...Visualizing results...")
        fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10, 14), sharex=True)
        fig.suptitle(f"DiD Effect for Clinic: {clinic_name} (ZIP: {clinic_zip}) - Acquired", fontsize=16, y=1.02)

        # Plot Primary
        if 'Primary' in models_this_clinic:
            plot_single_did_with_relative_axis(
                models_this_clinic['Primary'], primary_pre_mean, clinic_actual_labels,
                f"Effect on {Y_PRIMARY_RATE_NAME} (per 1000 ppl)", "DiD Effect Size",
                ax1, color='C2' # Green
            )
        else:
             ax1.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax1.transAxes)
             ax1.set_title(f"Effect on {Y_PRIMARY_RATE_NAME} (per 1000 ppl)", fontsize=12)

        # Plot Emergency
        if 'Emergency' in models_this_clinic:
            plot_single_did_with_relative_axis(
                models_this_clinic['Emergency'], emergency_pre_mean, clinic_actual_labels,
                f"Effect on {Y_EMERGENCY_RATE_NAME} (per 1000 ppl)", "DiD Effect Size",
                ax2, color='C1' # Orange
            )
        else:
             ax2.text(0.5, 0.5, "No results", ha='center', va='center', transform=ax2.transAxes)
             ax2.set_title(f"Effect on {Y_EMERGENCY_RATE_NAME} (per 1000 ppl)", fontsize=12)

        plt.tight_layout(rect=[0, 0.03, 1, 0.98]) 

        # --- FIX: Save the plot with a sanitized filename ---
        # This removes ALL illegal characters: | / \ : * ? " < >
        safe_clinic_name = re.sub(r'[\\/*?:"<>|]', '_', clinic_name)
        safe_clinic_name = safe_clinic_name.replace(' ', '_') 
        
        plot_filename = os.path.join(output_folder, f"did_effect_{safe_clinic_name}_{clinic_zip}_acquired.png")
        try:
            plt.savefig(plot_filename)
            print(f"✅ Plot saved: {plot_filename}")
        except Exception as e:
            print(f"❌ ERROR saving plot {plot_filename}: {e}")
        
        # --- NEW: Show the plot in the notebook ---
        plt.show()
        # We remove plt.close(fig) to allow the plot to display

    print("\n🎉 --- Analysis for all 'Acquired' clinics complete! ---")

# 16. DiD Analysis: Testing Acquisition Mechanisms (Plans A & B)

This analysis moves beyond a simple "Acquired" group to test two specific hypotheses about the *mechanisms* of an acquisition. We first categorize all 'Acquired' clinics into 'PC/General' and 'Specialty' groups.

## 16.1. Data Preparation: Specialty-Specific Outcomes

To test these hypotheses, we must first create new, specialized dependent variables:
1.  **Clinic Categorization**: We categorize all 'Acquired' clinics into groups like 'PC/General', 'Specialty - Cardiology', etc., based on keywords in their 'Facility' or 'Specialty' names.
2.  **New Outcome Variables**: We aggregate patient visits to the ZIP-Day level, but this time we create separate *rates* based on ICD codes:
    * `Total_ER_Rate`: All emergency visits.
    * `Cardiology_ER_Rate`: Emergency visits with a Cardiology-related ICD code (e.g., "Ixx").
    * `Ortho_ER_Rate`: Emergency visits with an Orthopedics-related ICD code (e.g., "Mxx").

## 16.2. Plan A: "Reputation/System Effect" Hypothesis

This plan tests if acquiring a *Primary Care* (PC) clinic creates a system-wide "brand" or "reputation" effect that changes overall ER usage.

* **Hypothesis**: Acquiring PC clinics changes the `Total_ER_Rate` for nearby patients.
* **Model**: A standard Difference-in-Differences model with triple interaction:
    $Y_{it} = \beta_0 + \delta(Treat_i \times Post_{it} \times C(Distance_k)) + \gamma_i + \lambda_t + \epsilon_{it}$
* **Treatment Group (`Treat=1`)**: Patient ZIPs whose *nearest PC clinic* (from the 'Acquired' list) is within a 50km radius.
* **Outcome (`Y`
    )**: `Total_ER_Rate`.
* **Fixed Effects**: Pooled OLS with Time (`C(Year) + C(Month)`) fixed effects. (Note: This is a faster model that does not include `C(Zip)` fixed effects).

## 16.3. Plan B: "Specialty-Specific Leakage" Hypothesis (Event Study)

This plan tests a more specific "leakage" or "referral capture" hypothesis. It uses a powerful **Event Study** design.

* **Hypothesis (Test 1)**: Acquiring a *Cardiology* clinic should *specifically* reduce `Cardiology_ER_Rate` for nearby patients, as those emergencies are now captured by the in-system specialist.
* **Hypothesis (Test 2 - Placebo)**: Acquiring a *Cardiology* clinic should have **no effect** on `Ortho_ER_Rate`. This validates that we are measuring a specific specialty effect, not just a general trend.
* **Model**: An Event Study model with full Two-Way Fixed Effects (TWFE). This plots the effect for each month relative to the event, which is crucial for checking pre-trends.
    $Y_{it} = \beta_0 + \sum_{k \neq -1} \delta_k D_{it}^k + \gamma_i + \lambda_t + \epsilon_{it}$
    * $Y_{it}$ is the outcome (e.g., `Cardiology_ER_Rate`).
    * $D_{it}^k$ is a dummy for month $k$ relative to the event (where $k=-1$ is the omitted base period).
    * $\gamma_i$ are the ZIP Code Fixed Effects (`C(Zip)`).
    * $\lambda_t$ are the Time Fixed Effects (`C(Year) + C(Month)`).

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re # For cleaning file names
import os

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
pd.set_option('display.max_rows', 200) # Show more rows for the clinic list

print("--- 🚀 STARTING 'ACQUIRED' CLINIC ANALYSIS (Plans A & B) ---")

# --- 1. Check and Load Data ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
    print("    Please run the cell that creates 'final_data' first.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Clean Data ---
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Data loaded and cleaned.")

    # --- Step 1: Categorize 'Acquired' Clinics ---
    df_acquired_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'Acquired'].copy()

    # Define keywords (can be expanded)
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    specialty_keywords = ['Cardiology', 'CV', 'Gynecology', 'Ortho', 'Hospitalist', 'Audiology', 'OB/GYN', 'Lab']
    cardiology_keywords = ['Cardiology', 'CV']

    # Function to categorize
    def categorize_clinic(row):
        facility_name = str(row.get('Facility', '')).lower()
        specialty_name = str(row.get('Specialty', '')).lower() # Also check Specialty column
        
        # First check for Specialty keywords
        for keyword in specialty_keywords:
            if keyword.lower() in facility_name or keyword.lower() in specialty_name:
                # Determine the specific specialty
                if any(ck.lower() in facility_name or ck.lower() in specialty_name for ck in cardiology_keywords):
                    return 'Specialty - Cardiology'
                # Can add 'if' for Ortho, etc., if needed
                return 'Specialty - Other'
        # Then check for PC keywords
        for keyword in pc_keywords:
             if keyword.lower() in facility_name or keyword.lower() in specialty_name:
                 return 'PC/General'
        # If nothing matched
        return 'Unknown/Other'

    df_acquired_clinics_all['Category'] = df_acquired_clinics_all.apply(categorize_clinic, axis=1)

    # Create clinic lists for Plans A and B
    acquired_pc_clinics = df_acquired_clinics_all[df_acquired_clinics_all['Category'] == 'PC/General']
    acquired_specialty_clinics = df_acquired_clinics_all[df_acquired_clinics_all['Category'].str.startswith('Specialty')]
    cardiology_clinics = df_acquired_clinics_all[df_acquired_clinics_all['Category'] == 'Specialty - Cardiology']

    print("\n--- 'Acquired' Clinic Categorization ---")
    print(f"Total Acquired: {len(df_acquired_clinics_all)}")
    print(f"  PC/General: {len(acquired_pc_clinics)}")
    print(f"  Specialty: {len(acquired_specialty_clinics)}")
    print(f"    Including Cardiology: {len(cardiology_clinics)}")

    # --- Prepare Base DataFrame with Visits ---
    df_panel = final_data.copy()
    # Add patient coordinates
    df_panel = df_panel.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    # Add population
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['patient_lat', 'patient_lon', 'TotPopACS'])

    # Create variables for ER visits
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Is_Cardiology_ICD'] = df_panel['ICD_1char'].str.startswith('I', na=False).astype(int)
    df_panel['Is_Ortho_ICD'] = df_panel['ICD_1char'].str.startswith('M', na=False).astype(int)

    # Aggregate to PatientZIP-Date level
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month', 'patient_lat', 'patient_lon', 'TotPopACS']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Cardiology_ER_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[(df_panel.loc[x.index, 'Is_Emergency'] == 1) & (df_panel.loc[x.index, 'Is_Cardiology_ICD'] == 1)].sum()),
        Ortho_ER_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[(df_panel.loc[x.index, 'Is_Emergency'] == 1) & (df_panel.loc[x.index, 'Is_Ortho_ICD'] == 1)].sum())
    ).reset_index()

    # Calculate Rates per 1000 people
    df_agg['Total_ER_Rate'] = (df_agg['Total_Emergency_Visits'] / df_agg['TotPopACS']) * 1000
    df_agg['Cardiology_ER_Rate'] = (df_agg['Cardiology_ER_Visits'] / df_agg['TotPopACS']) * 1000
    df_agg['Ortho_ER_Rate'] = (df_agg['Ortho_ER_Visits'] / df_agg['TotPopACS']) * 1000
    print("✅ Aggregated ER visit rates calculated.")


    # --- Function to find nearest clinic from a given group ---
    def find_nearest_clinic_in_group(patient_coords, clinic_group_df):
        min_dist = np.inf
        nearest_info = {}
        if clinic_group_df.empty:
            return nearest_info # Return empty if group is empty

        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist:
                min_dist = distance
                nearest_info = {
                    'distance_km': min_dist,
                    'event_date': clinic_row['event_date']
                }
        return nearest_info

    # --- PLAN A: Reputation Effect (PC Clinics, Total ER Rate) ---
    print("\n--- PLAN A: Testing Reputation Effect ---")

    # 1. Create treatment map for PC clinics
    treatment_map_A = []
    patient_zips_coords = df_agg[['Filtered_Patient_ZipCode', 'patient_lat', 'patient_lon']].drop_duplicates().dropna()
    for _, patient_row in patient_zips_coords.iterrows():
         patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
         nearest_pc_info = find_nearest_clinic_in_group(patient_coords, acquired_pc_clinics)
         if nearest_pc_info:
             treatment_map_A.append({
                 'Filtered_Patient_ZipCode': patient_row['Filtered_Patient_ZipCode'],
                 'distance_to_nearest_pc_km': nearest_pc_info['distance_km'],
                 'pc_event_date': nearest_pc_info['event_date']
             })
    df_treatment_map_A = pd.DataFrame(treatment_map_A)

    # 2. Prepare data for Plan A regression
    df_plan_A = df_agg.merge(df_treatment_map_A, on='Filtered_Patient_ZipCode', how='left')
    df_plan_A = df_plan_A.dropna(subset=['distance_to_nearest_pc_km', 'pc_event_date']) # Analyze only ZIPs with a nearest PC clinic

    TREATMENT_RADIUS_KM_A = 50 # Define analysis zone
    df_plan_A['Treat'] = (df_plan_A['distance_to_nearest_pc_km'] <= TREATMENT_RADIUS_KM_A).astype(int)
    df_plan_A['Post'] = (df_plan_A['Date'] >= df_plan_A['pc_event_date']).astype(int)

    bins_A = [-np.inf, 5, 10, 20, np.inf]
    labels_A = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    df_plan_A['distance_category'] = pd.cut(df_plan_A['distance_to_nearest_pc_km'], bins=bins_A, labels=labels_A, right=False)
    df_plan_A = df_plan_A.dropna(subset=['distance_category'])

    print(f"Prepared {len(df_plan_A)} observations for Plan A.")
    print("Distribution by distance category:")
    print(df_plan_A['distance_category'].value_counts())


    # 3. Run Plan A regression
    formula_A = "Total_ER_Rate ~ Treat * Post * C(distance_category) + C(Year) + C(Month)"
    model_A_result = None
    try:
        print("...Running Plan A model...")
        # Set base category
        df_plan_A['distance_category'] = pd.Categorical(df_plan_A['distance_category'], categories=labels_A, ordered=True)
        model_A = smf.ols(formula_A, data=df_plan_A)
        model_A_result = model_A.fit()
        print("✅ Plan A model estimated.")
        # print(model_A_result.summary()) # Uncomment to see table
    except Exception as e:
        print(f"❌ ERROR fitting Plan A model: {e}")


    # 4. Visualization for Plan A
    def extract_effects_method1(model_result, category_order):
        # (This is the extraction function you provided)
        ref_label = category_order[0]
        params = model_result.params
        conf = model_result.conf_int()
        results = {'Distance': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        base_term = 'Treat:Post' if 'Treat:Post' in params.index else ('Post' if 'Post' in params.index else None)
        if base_term is None :
             print("Warning: Neither 'Treat:Post' nor 'Post' found. Cannot plot.")
             return pd.DataFrame(results)

        results['Distance'].append(ref_label)
        results['Effect'].append(params[base_term])
        results['Conf_Low'].append(conf.loc[base_term, 0])
        results['Conf_High'].append(conf.loc[base_term, 1])

        for cat in category_order[1:]:
             term_name_interact = f"Treat:Post:C(distance_category)[T.{cat}]"
             if term_name_interact in params.index and base_term == 'Treat:Post':
                 t_test = model_result.t_test(f"{base_term} + {term_name_interact}")
                 results['Distance'].append(cat)
                 results['Effect'].append(t_test.effect[0])
                 results['Conf_Low'].append(t_test.conf_int()[0][0])
                 results['Conf_High'].append(t_test.conf_int()[0][1])
             else:
                 term_name_simple = f"Post:C(distance_category)[T.{cat}]"
                 if term_name_simple in params.index and base_term == 'Post':
                     t_test = model_result.t_test(f"{base_term} + {term_name_simple}")
                     results['Distance'].append(cat)
                     results['Effect'].append(t_test.effect[0])
                     results['Conf_Low'].append(t_test.conf_int()[0][0])
                     results['Conf_High'].append(t_test.conf_int()[0][1])

        df = pd.DataFrame(results)
        if not df.empty:
             df['Distance'] = pd.Categorical(df['Distance'], categories=category_order, ordered=True)
             df = df.sort_values('Distance')
        return df


    print("\n--- Visualizing Plan A ---")
    if model_A_result:
        plot_data_A = extract_effects_method1(model_A_result, labels_A)
        if not plot_data_A.empty:
            fig_A, ax_A = plt.subplots(figsize=(10, 6))
            errors_A = [plot_data_A['Effect'] - plot_data_A['Conf_Low'], plot_data_A['Conf_High'] - plot_data_A['Effect']]
            ax_A.errorbar(x=plot_data_A['Distance'], y=plot_data_A['Effect'], yerr=errors_A, fmt='-s', capsize=5, label='Acquired PC Clinics Effect')
            ax_A.axhline(0, color='black', linestyle='--')
            ax_A.set_title("Plan A: Effect of Acquiring PC Clinics on Total ER Rate", fontsize=14)
            ax_A.set_xlabel("Distance to Nearest Acquired PC Clinic (km)")
            ax_A.set_ylabel("DiD Effect (Change in Daily ER Visits per 1000 Ppl)")
            ax_A.legend()
            plt.tight_layout()
            plt.show()
        else:
            print("Could not extract data for Plan A plot.")
    else:
        print("Plan A model was not estimated, skipping plot.")


    # --- PLAN B: Specificity Effect (Cardio Clinics, Cardio/Ortho ER Rates) ---
    print("\n\n--- PLAN B: Testing Specificity Effect ---")

    # 1. Create treatment map for Cardio clinics
    treatment_map_B = []
    # Use the same patient_zips_coords from Plan A
    for _, patient_row in patient_zips_coords.iterrows():
         patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
         # Find nearest cardio clinic
         nearest_cardio_info = find_nearest_clinic_in_group(patient_coords, cardiology_clinics)
         if nearest_cardio_info:
             treatment_map_B.append({
                 'Filtered_Patient_ZipCode': patient_row['Filtered_Patient_ZipCode'],
                 'distance_to_nearest_cardio_km': nearest_cardio_info['distance_km'],
                 'cardio_event_date': nearest_cardio_info['event_date']
             })
    df_treatment_map_B = pd.DataFrame(treatment_map_B)

    # 2. Prepare data for Plan B regression
    df_plan_B = df_agg.merge(df_treatment_map_B, on='Filtered_Patient_ZipCode', how='inner') # INNER join!
    df_plan_B = df_plan_B.dropna(subset=['cardio_event_date'])

    # Create Post relative to cardio event
    df_plan_B['Post'] = (df_plan_B['Date'] >= df_plan_B['cardio_event_date']).astype(int)

    # Create relative_month for Event Study
    df_plan_B['event_month'] = pd.to_datetime(df_plan_B['cardio_event_date']).dt.to_period('M')
    df_plan_B['current_month'] = pd.to_datetime(df_plan_B['Date']).dt.to_period('M')
    df_plan_B['relative_month'] = (df_plan_B['current_month'] - df_plan_B['event_month']).apply(lambda x: x.n)

    # Limit Event Study window
    EVENT_WINDOW = 12
    df_plan_B = df_plan_B[df_plan_B['relative_month'].between(-EVENT_WINDOW, EVENT_WINDOW)]

    # --- FIX IS HERE: Use Treatment(reference=...) in formula ---
    # Base period for Event Study (usually -1)
    BASE_PERIOD = -1
    # Ensure all relative_month values are int
    df_plan_B['relative_month'] = df_plan_B['relative_month'].astype(int)

    print(f"Prepared {len(df_plan_B)} observations for Plan B (Event Study).")


    # 3. Run Plan B regressions (Event Study)
    models_B = {}
    # --- FIX IS HERE: Use Treatment(reference=...) in formula ---
    formula_B_event_study = "{outcome} ~ C(relative_month, Treatment(reference={ref})) + C(Year) + C(Month)"

    # Test 1: Cardiology on Cardio ER
    outcome_B1 = "Cardiology_ER_Rate"
    try:
        print(f"...Running Plan B model (Test 1): {outcome_B1}")
        df_plan_B_test1 = df_plan_B.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
        # Add C(Zip)
        formula_B1 = formula_B_event_study.format(outcome=outcome_B1, ref=BASE_PERIOD) + " + C(Zip)"
        model_B1 = smf.ols(formula_B1, data=df_plan_B_test1)
        models_B['Test1_Cardio'] = model_B1.fit(cov_type='cluster', cov_kwds={'groups': df_plan_B_test1['Zip']})
        print("  ✅ Test 1 model estimated.")
    except Exception as e:
        print(f"  ❌ ERROR fitting Test 1 model: {e}")

    # Test 2: Cardiology on Ortho ER (Placebo)
    outcome_B2 = "Ortho_ER_Rate"
    try:
        print(f"...Running Plan B model (Test 2): {outcome_B2} (Placebo)")
        df_plan_B_test2 = df_plan_B.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
        formula_B2 = formula_B_event_study.format(outcome=outcome_B2, ref=BASE_PERIOD) + " + C(Zip)"
        model_B2 = smf.ols(formula_B2, data=df_plan_B_test2)
        models_B['Test2_Placebo'] = model_B2.fit(cov_type='cluster', cov_kwds={'groups': df_plan_B_test2['Zip']})
        print("  ✅ Test 2 (Placebo) model estimated.")
    except Exception as e:
        print(f"  ❌ ERROR fitting Test 2 model: {e}")

    # 4. Visualization for Plan B (Event Study for Test 1)
    print("\n--- Visualizing Plan B (Event Study Test 1) ---")

    def extract_event_study_effects(model_result, base_period):
        # --- FIX IS HERE: Updated coefficient parsing ---
        params = model_result.params.filter(like="C(relative_month") # Find coefficients
        conf = model_result.conf_int().filter(like="C(relative_month", axis=0)
        
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        
        # Add base period with 0 effect
        results['Relative_Month'].append(base_period)
        results['Effect'].append(0)
        results['Conf_Low'].append(0)
        results['Conf_High'].append(0)
        
        # Extract other periods
        for idx in params.index:
            try:
                # Updated regex for names like C(relative_month...)[T.month]
                match = re.search(r'\[T\.(-?\d+\.?\d*)\]', idx)
                if match:
                    month = int(float(match.group(1))) # Convert to float then int
                    # Skip base period if it somehow gets in (shouldn't)
                    if month == base_period:
                        continue
                    results['Relative_Month'].append(month)
                    results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0])
                    results['Conf_High'].append(conf.loc[idx, 1])
                # else: # Debugging output if name not recognized
                #   print(f"  Did not recognize coefficient name: {idx}")

            except (AttributeError, ValueError, IndexError, TypeError) as e:
                 print(f"  Warning: Could not parse month from '{idx}': {e}")

        df = pd.DataFrame(results)
        # Drop duplicate base periods if they appeared
        df = df.drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)

    if 'Test1_Cardio' in models_B:
        event_study_data_B1 = extract_event_study_effects(models_B['Test1_Cardio'], BASE_PERIOD)
        if not event_study_data_B1.empty:
            fig_B, ax_B = plt.subplots(figsize=(12, 7))
            errors_B = [event_study_data_B1['Effect'] - event_study_data_B1['Conf_Low'],
                         event_study_data_B1['Conf_High'] - event_study_data_B1['Effect']]
            # Check for NaNs in errors (can happen with model issues)
            valid_error = ~np.isnan(errors_B[0]) & ~np.isnan(errors_B[1])
            if valid_error.all():
                 ax_B.errorbar(x=event_study_data_B1['Relative_Month'], y=event_study_data_B1['Effect'],
                               yerr=errors_B, fmt='-o', capsize=5, label='Effect on Cardio ER Rate')
            else:
                # Plot without error bars if issues
                 print("  Warning: Could not calculate CIs for some points.")
                 ax_B.plot(event_study_data_B1['Relative_Month'], event_study_data_B1['Effect'],
                           marker='o', linestyle='-', label='Effect on Cardio ER Rate (No CI)')

            ax_B.axhline(0, color='black', linestyle='--')
            ax_B.axvline(x=-0.5, color='grey', linestyle=':', label=f'Acquisition Event (Month {BASE_PERIOD+1})') # Event line
            ax_B.set_title("Plan B (Test 1): Event Study - Effect of Acquiring Cardio Clinic on Cardiology ER Rate", fontsize=14)
            ax_B.set_xlabel("Months Relative to Clinic Acquisition")
            ax_B.set_ylabel("Change in Daily Cardiology ER Visits per 1000 Ppl")
            ax_B.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2)) # Ticks every 2 months
            ax_B.legend()
            ax_B.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else:
            print("Could not extract data for Plan B Event Study plot.")
    else:
        print("Plan B Test 1 model was not estimated, skipping plot.")


    # Display Placebo Test (Test 2) results
    if 'Test2_Placebo' in models_B:
         print("\n--- Placebo Test Results (Test 2: Cardio clinics on Ortho visits) ---")
         placebo_coeffs = models_B['Test2_Placebo'].params.filter(like="C(relative_month")
         post_event_coeffs = {}
         for idx in placebo_coeffs.index:
             try:
                 match = re.search(r'\[T\.(-?\d+\.?\d*)\]', idx)
                 if match:
                     month = int(float(match.group(1)))
                     if month >= 0:
                          post_event_coeffs[idx] = placebo_coeffs[idx]
             except:
                 continue

         if post_event_coeffs:
             print("Coefficients for POST-event months:")
             for name, val in post_event_coeffs.items():
                 p_val = models_B['Test2_Placebo'].pvalues.get(name, float('nan'))
                 print(f"  {name}: {val:.4f} (p={p_val:.3f})")
         else:
              print("No post-event coefficients found in placebo test.")

    print("\n🎉 --- 'Acquired' Clinic Analysis (Plans A & B) complete! ---")

# 17. DiD Analysis (Method 8): Parallel Trends Test for Plan A

This is a crucial diagnostic step. The central identifying assumption of any Difference-in-Differences (DiD) model is **parallel trends**. This assumption states that, *in the absence of the treatment*, the treatment group and control group would have followed the same trend.

We cannot prove this counterfactual, but we can test for it visually by plotting an **event study**.

## 17.1. The Parallel Trends Test

The logic is as follows:
1.  We run an event study regression that estimates a separate effect for each month *before* and *after* the acquisition event.
2.  We plot these monthly coefficients with their 95% confidence intervals.
3.  **If the parallel trends assumption holds**, we should see that all the coefficients for the *pre-treatment* periods (e.g., months -12 to -2) are statistically indistinguishable from zero (i.e., their confidence intervals overlap the zero line).
4.  If we see a clear upward or downward trend *before* the event, the assumption is violated, and our DiD results may be biased.

## 17.2. Methodology for this Test

This script specifically tests the assumption for **Plan A (Reputation Effect)**:
* **Outcome**: `Total_ER_Rate` (per 1,000 people).
* **Event**: The acquisition of the nearest Primary Care (PC) clinic.

To create a "treatment" and "control" group for this test, we split the data into two distinct samples:
1.  **"Near" Group (Treatment)**: All patient ZIPs within a **10km radius** of their nearest acquired PC clinic.
2.  **"Far" Group (Control)**: All patient ZIPs more than **20km away** from their nearest acquired PC clinic. (The 10-20km "donut" ring is dropped to create a clearer separation).

We then run *two separate* Two-Way Fixed Effects (TWFE) event study models—one for the "Near" group and one for the "Far" group. Both models include `C(Zip)` and Time (`C(Year) + C(Month)`) fixed effects.

$Y\_Rate_{it} = \beta_0 + \sum_{k \neq -1} \delta_k D_{it}^k + \gamma_i + \lambda_t + \epsilon_{it}$

The final plot overlays the coefficients ($\delta_k$) from both models. We hope to see both lines hovering around zero before the event at t=-0.5.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re # For cleaning file names
import os
# --- NO PROBLEMATIC IMPORTS ---

# --- 0. Setup ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
# --- NO PROBLEMATIC WARNING FILTERS ---
pd.options.mode.chained_assignment = None
pd.set_option('display.max_rows', 200) # Show more rows for clinic list

print("--- 🚀 STARTING PARALLEL TRENDS CHECK (Plan A: Acquired PC, Total ER Rate) ---")

# --- 1. Check and Load Data ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' not found.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Clean Data ---
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Data loaded and cleaned.")

    # --- 3. Categorize 'Acquired' and select PC clinics ---
    df_acquired_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'Acquired'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower()
        specialty_name = str(row.get('Specialty', '')).lower()
        for keyword in pc_keywords:
             if keyword.lower() in facility_name or keyword.lower() in specialty_name:
                 return True
        return False
        
    acquired_pc_clinics = df_acquired_clinics_all[df_acquired_clinics_all.apply(is_pc_clinic, axis=1)].copy()
    print(f"\n✅ Found {len(acquired_pc_clinics)} Acquired PC/General clinics for analysis.")

    # --- 4. Prepare base DataFrame with visits ---
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['patient_lat', 'patient_lon', 'TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    
    # Aggregate **only** total ER visits
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month', 'patient_lat', 'patient_lon', 'TotPopACS']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    df_agg['Total_ER_Rate'] = (df_agg['Total_Emergency_Visits'] / df_agg['TotPopACS']) * 1000
    print("✅ Aggregated total ER visit rates calculated.")

    # --- 5. Create treatment map for PC clinics ---
    def find_nearest_clinic_in_group(patient_coords, clinic_group_df):
        # (function copied from previous code)
        min_dist = np.inf
        nearest_info = {}
        if clinic_group_df.empty: return nearest_info
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist:
                min_dist = distance
                nearest_info = {'distance_km': min_dist, 'event_date': clinic_row['event_date']}
        return nearest_info

    treatment_map_A = []
    patient_zips_coords = df_agg[['Filtered_Patient_ZipCode', 'patient_lat', 'patient_lon']].drop_duplicates().dropna()
    for _, patient_row in patient_zips_coords.iterrows():
         patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
         nearest_pc_info = find_nearest_clinic_in_group(patient_coords, acquired_pc_clinics)
         if nearest_pc_info:
             treatment_map_A.append({
                 'Filtered_Patient_ZipCode': patient_row['Filtered_Patient_ZipCode'],
                 'distance_to_nearest_pc_km': nearest_pc_info['distance_km'],
                 'pc_event_date': nearest_pc_info['event_date']
             })
    df_treatment_map_A = pd.DataFrame(treatment_map_A)

    # --- 6. Prepare data for Event Study ---
    df_event_A = df_agg.merge(df_treatment_map_A, on='Filtered_Patient_ZipCode', how='left')
    df_event_A = df_event_A.dropna(subset=['distance_to_nearest_pc_km', 'pc_event_date'])

    # Create relative_month
    df_event_A['event_month'] = pd.to_datetime(df_event_A['pc_event_date']).dt.to_period('M')
    df_event_A['current_month'] = pd.to_datetime(df_event_A['Date']).dt.to_period('M')
    df_event_A['relative_month'] = (df_event_A['current_month'] - df_event_A['event_month']).apply(lambda x: x.n)

    # Limit window and define base period
    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    df_event_A = df_event_A[df_event_A['relative_month'].between(-EVENT_WINDOW, EVENT_WINDOW)]
    df_event_A['relative_month'] = df_event_A['relative_month'].astype(int)

    # --- 7. Split into groups by distance ---
    # "Near" group: 0-10 km
    df_event_A_near = df_event_A[df_event_A['distance_to_nearest_pc_km'] <= 10].copy()
    # "Far" group (Control): > 20 km
    df_event_A_far = df_event_A[df_event_A['distance_to_nearest_pc_km'] > 20].copy()

    print(f"\nPrepared data for Event Study:")
    print(f"  Near (<=10 km): {len(df_event_A_near)} observations")
    print(f"  Far (>20 km): {len(df_event_A_far)} observations")


    # --- 8. Run SEPARATE Event Study regressions ---
    models_event_A = {}
    OUTCOME = "Total_ER_Rate"
    # Formula with C(Zip) for controlling baseline differences
    FORMULA_EVENT = f"{OUTCOME} ~ C(relative_month, Treatment(reference={BASE_PERIOD})) + C(Year) + C(Month) + C(Zip)"

    # Model for "Near" group
    if not df_event_A_near.empty:
        try:
            print("...Running Event Study for 'Near' group (0-10 km)...")
            df_event_A_near = df_event_A_near.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
            model_near = smf.ols(FORMULA_EVENT, data=df_event_A_near)
            models_event_A['Near'] = model_near.fit(cov_type='cluster', cov_kwds={'groups': df_event_A_near['Zip']})
            print("  ✅ 'Near' model estimated.")
        except Exception as e:
            print(f"  ❌ ERROR fitting 'Near' model: {e}")
            if "requires non-empty data" in str(e) or "PerfectSeparationError" in str(e):
                 print("      -> Likely insufficient data or variation in this group.")

    # Model for "Far" group
    if not df_event_A_far.empty:
         try:
            print("...Running Event Study for 'Far' group (>20 km)...")
            df_event_A_far = df_event_A_far.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
            model_far = smf.ols(FORMULA_EVENT, data=df_event_A_far)
            models_event_A['Far'] = model_far.fit(cov_type='cluster', cov_kwds={'groups': df_event_A_far['Zip']})
            print("  ✅ 'Far' model estimated.")
         except Exception as e:
            print(f"  ❌ ERROR fitting 'Far' model: {e}")
            if "requires non-empty data" in str(e) or "PerfectSeparationError" in str(e):
                 print("      -> Likely insufficient data or variation in this group.")

    # --- 9. Extract and Visualize Results ---
    print("\n--- Visualizing Parallel Trends (Plan A) ---")

    # (Event study extraction function - same as before)
    def extract_event_study_effects(model_result, base_period):
        # (Copied from previous code)
        params = model_result.params.filter(like="C(relative_month")
        conf = model_result.conf_int().filter(like="C(relative_month", axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        results['Relative_Month'].append(base_period)
        results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
        for idx in params.index:
            try:
                match = re.search(r'\[T\.(-?\d+\.?\d*)\]', idx)
                if match:
                    month = int(float(match.group(1)))
                    if month == base_period: continue
                    results['Relative_Month'].append(month)
                    results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0])
                    results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)


    fig_trends, ax_trends = plt.subplots(figsize=(12, 7))
    plot_made = False

    # Line for "Near"
    if 'Near' in models_event_A:
        data_near = extract_event_study_effects(models_event_A['Near'], BASE_PERIOD)
        if not data_near.empty:
            errors_near = [data_near['Effect'] - data_near['Conf_Low'], data_near['Conf_High'] - data_near['Effect']]
            valid_error = ~np.isnan(errors_near[0]) & ~np.isnan(errors_near[1])
            if valid_error.all():
                 ax_trends.errorbar(x=data_near['Relative_Month'], y=data_near['Effect'], yerr=errors_near,
                                      fmt='-o', capsize=3, label='Near (0-10 km)', color='blue')
            else:
                 ax_trends.plot(data_near['Relative_Month'], data_near['Effect'], marker='o', linestyle='-',
                                  label='Near (0-10 km, No CI)', color='blue')
            plot_made = True

    # Line for "Far"
    if 'Far' in models_event_A:
        data_far = extract_event_study_effects(models_event_A['Far'], BASE_PERIOD)
        if not data_far.empty:
            errors_far = [data_far['Effect'] - data_far['Conf_Low'], data_far['Conf_High'] - data_far['Effect']]
            valid_error = ~np.isnan(errors_far[0]) & ~np.isnan(errors_far[1])
            if valid_error.all():
                ax_trends.errorbar(x=data_far['Relative_Month'], y=data_far['Effect'], yerr=errors_far,
                                     fmt='--s', capsize=3, label='Far (>20 km, Control)', color='red')
            else:
                 ax_trends.plot(data_far['Relative_Month'], data_far['Effect'], marker='s', linestyle='--',
                                  label='Far (>20 km, No CI)', color='red')
            plot_made = True

    if plot_made:
        ax_trends.axhline(0, color='black', linestyle='-', linewidth=0.8)
        ax_trends.axvline(x=-0.5, color='grey', linestyle=':', label=f'Acquisition Event (Month 0)')
        ax_trends.set_title("Plan A Event Study: Parallel Trends Check (Acquired PC Clinics, Total ER Rate)", fontsize=14)
        ax_trends.set_xlabel("Months Relative to Nearest PC Clinic Acquisition")
        ax_trends.set_ylabel("Effect vs Month -1 (Change in Daily ER Visits per 1000 Ppl)")
        ax_trends.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
        ax_trends.legend()
        ax_trends.grid(True, axis='y', linestyle=':')
        plt.tight_layout()
        plt.show()
    else:
        print("\nCould not build plot: no data from models.")

    print("\n🎉 --- Parallel Trends Check (Plan A) complete! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from geopy.distance import geodesic
import warnings
import re
import os

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО ПРОВЕРКИ ПАРАЛЛЕЛЬНЫХ ТРЕНДОВ (План А: Acquired PC, Total ER Rate, улучшенная логика радиуса) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ОШИБКА: Не могу найти файл {e.filename}. Останавливаюсь.")
        raise

    # --- 2. Очистка Данных ---
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')

    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Категоризация 'Acquired' и выбор PC клиник ---
    df_acquired_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'Acquired'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']

    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower()
        specialty_name = str(row.get('Specialty', '')).lower()
        for keyword in pc_keywords:
            if keyword.lower() in facility_name or keyword.lower() in specialty_name:
                return True
        return False

    acquired_pc_clinics = df_acquired_clinics_all[df_acquired_clinics_all.apply(is_pc_clinic, axis=1)].copy()
    print(f"\n✅ Найдено {len(acquired_pc_clinics)} Acquired PC/General клиник для анализа.")

    # --- 4. Подготовка основного DataFrame с визитами ---
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['patient_lat', 'patient_lon', 'TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)

    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month', 'patient_lat', 'patient_lon', 'TotPopACS']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    df_agg['Total_ER_Rate'] = (df_agg['Total_Emergency_Visits'] / df_agg['TotPopACS']) * 1000
    print("✅ Агрегированные общие рейты ER визитов рассчитаны.")

    # --- 5. Новая логика: поиск всех клиник в радиусе ≤10 км ---
    def find_first_clinic_within_radius(patient_coords, clinic_group_df, radius_km=10):
        clinics_in_radius = []
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance <= radius_km:
                clinics_in_radius.append((distance, clinic_row['event_date']))
        if clinics_in_radius:
            clinics_in_radius.sort(key=lambda x: x[1])
            return {
                'distance_to_first_clinic_km': clinics_in_radius[0][0],
                'pc_event_date': clinics_in_radius[0][1],
                'num_clinics_in_radius': len(clinics_in_radius)
            }
        else:
            return {}

    treatment_map_A = []
    patient_zips_coords = df_agg[['Filtered_Patient_ZipCode', 'patient_lat', 'patient_lon']].drop_duplicates().dropna()
    print(f"🔍 Поиск клиник в радиусе ≤10 км для {len(patient_zips_coords)} ZIP-кодов...")

    for _, patient_row in patient_zips_coords.iterrows():
        patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        clinic_info = find_first_clinic_within_radius(patient_coords, acquired_pc_clinics, radius_km=10)
        if clinic_info:
            treatment_map_A.append({
                'Filtered_Patient_ZipCode': patient_row['Filtered_Patient_ZipCode'],
                'distance_to_first_clinic_km': clinic_info['distance_to_first_clinic_km'],
                'pc_event_date': clinic_info['pc_event_date'],
                'num_clinics_in_radius': clinic_info['num_clinics_in_radius']
            })

    df_treatment_map_A = pd.DataFrame(treatment_map_A)
    print(f"✅ Найдено ZIP-кодов с клиниками в радиусе ≤10 км: {len(df_treatment_map_A)}")

    # --- 6. Подготовка данных для Event Study ---
    df_event_A = df_agg.merge(df_treatment_map_A, on='Filtered_Patient_ZipCode', how='left')
    df_event_A = df_event_A.dropna(subset=['distance_to_first_clinic_km', 'pc_event_date'])
    df_event_A['event_month'] = pd.to_datetime(df_event_A['pc_event_date']).dt.to_period('M')
    df_event_A['current_month'] = pd.to_datetime(df_event_A['Date']).dt.to_period('M')
    df_event_A['relative_month'] = (df_event_A['current_month'] - df_event_A['event_month']).apply(lambda x: x.n)

    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    df_event_A = df_event_A[df_event_A['relative_month'].between(-EVENT_WINDOW, EVENT_WINDOW)]
    df_event_A['relative_month'] = df_event_A['relative_month'].astype(int)

    # --- 7. Разделение на группы ---
    df_event_A_near = df_event_A[df_event_A['distance_to_first_clinic_km'] <= 10].copy()
    df_event_A_far = df_event_A[df_event_A['distance_to_first_clinic_km'] > 20].copy()

    print(f"\n📊 Подготовлено данных для Event Study:")
    print(f"  Близко (≤10 км): {len(df_event_A_near)} наблюдений")
    print(f"  Далеко (>20 км): {len(df_event_A_far)} наблюдений")
    if 'num_clinics_in_radius' in df_event_A.columns:
        print(f"  Среднее число клиник в радиусе 10 км: {df_event_A['num_clinics_in_radius'].mean():.2f}")


    # --- 8. Запуск ОТДЕЛЬНЫХ регрессий Event Study ---
    models_event_A = {}
    OUTCOME = "Total_ER_Rate"
    # Формула с C(Zip) для контроля базовых различий
    FORMULA_EVENT = f"{OUTCOME} ~ C(relative_month, Treatment(reference={BASE_PERIOD})) + C(Year) + C(Month) + C(Zip)"

    # Модель для группы "Близко"
    if not df_event_A_near.empty:
        try:
            print("...Запуск Event Study для группы 'Близко' (0-10 km)...")
            df_event_A_near = df_event_A_near.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
            model_near = smf.ols(FORMULA_EVENT, data=df_event_A_near)
            models_event_A['Near'] = model_near.fit(cov_type='cluster', cov_kwds={'groups': df_event_A_near['Zip']})
            print("  ✅ Модель 'Близко' рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА при расчете модели 'Близко': {e}")
            if "requires non-empty data" in str(e) or "PerfectSeparationError" in str(e):
                 print("     -> Вероятно, недостаточно данных или вариации в этой группе.")

    # Модель для группы "Далеко"
    if not df_event_A_far.empty:
         try:
            print("...Запуск Event Study для группы 'Далеко' (>20 km)...")
            df_event_A_far = df_event_A_far.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
            model_far = smf.ols(FORMULA_EVENT, data=df_event_A_far)
            models_event_A['Far'] = model_far.fit(cov_type='cluster', cov_kwds={'groups': df_event_A_far['Zip']})
            print("  ✅ Модель 'Далеко' рассчитана.")
         except Exception as e:
            print(f"  ❌ ОШИБКА при расчете модели 'Далеко': {e}")
            if "requires non-empty data" in str(e) or "PerfectSeparationError" in str(e):
                 print("     -> Вероятно, недостаточно данных или вариации в этой группе.")

    # --- 9. Извлечение и Визуализация Результатов ---
    print("\n--- Визуализация Параллельных Трендов (План А) ---")

    # (Функция извлечения эффектов Event Study - та же, что и раньше)
    def extract_event_study_effects(model_result, base_period):
        # ... (Копируем функцию из предыдущего ответа) ...
        params = model_result.params.filter(like="C(relative_month")
        conf = model_result.conf_int().filter(like="C(relative_month", axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        results['Relative_Month'].append(base_period)
        results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
        for idx in params.index:
            try:
                match = re.search(r'\[T\.(-?\d+\.?\d*)\]', idx)
                if match:
                    month = int(float(match.group(1)))
                    if month == base_period: continue
                    results['Relative_Month'].append(month)
                    results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0])
                    results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)


    fig_trends, ax_trends = plt.subplots(figsize=(12, 7))
    plot_made = False

    # Линия для "Близко"
    if 'Near' in models_event_A:
        data_near = extract_event_study_effects(models_event_A['Near'], BASE_PERIOD)
        if not data_near.empty:
            errors_near = [data_near['Effect'] - data_near['Conf_Low'], data_near['Conf_High'] - data_near['Effect']]
            valid_error = ~np.isnan(errors_near[0]) & ~np.isnan(errors_near[1])
            if valid_error.all():
                 ax_trends.errorbar(x=data_near['Relative_Month'], y=data_near['Effect'], yerr=errors_near,
                                   fmt='-o', capsize=3, label='Near (0-10 km)', color='blue')
            else:
                 ax_trends.plot(data_near['Relative_Month'], data_near['Effect'], marker='o', linestyle='-',
                                label='Near (0-10 km, No CI)', color='blue')
            plot_made = True

    # Линия для "Далеко"
    if 'Far' in models_event_A:
        data_far = extract_event_study_effects(models_event_A['Far'], BASE_PERIOD)
        if not data_far.empty:
            errors_far = [data_far['Effect'] - data_far['Conf_Low'], data_far['Conf_High'] - data_far['Effect']]
            valid_error = ~np.isnan(errors_far[0]) & ~np.isnan(errors_far[1])
            if valid_error.all():
                ax_trends.errorbar(x=data_far['Relative_Month'], y=data_far['Effect'], yerr=errors_far,
                                   fmt='--s', capsize=3, label='Far (>20 km, Control)', color='red')
            else:
                 ax_trends.plot(data_far['Relative_Month'], data_far['Effect'], marker='s', linestyle='--',
                                label='Far (>20 km, No CI)', color='red')
            plot_made = True

    if plot_made:
        ax_trends.axhline(0, color='black', linestyle='-', linewidth=0.8)
        ax_trends.axvline(x=-0.5, color='grey', linestyle=':', label=f'Acquisition Event (Month 0)')
        ax_trends.set_title("Plan A Event Study: Parallel Trends Check (Acquired PC Clinics, Total ER Rate)", fontsize=14)
        ax_trends.set_xlabel("Months Relative to Nearest PC Clinic Acquisition")
        ax_trends.set_ylabel("Effect vs Month -1 (Change in Daily ER Visits per 1000 Ppl)")
        ax_trends.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
        ax_trends.legend()
        ax_trends.grid(True, axis='y', linestyle=':')
        plt.tight_layout()
        plt.show()
    else:
        print("\nНе удалось построить график: нет данных из моделей.")

    print("\n🎉 --- Проверка Параллельных Трендов (План А) завершена! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm.notebook import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
# --- УДАЛЕНА ПРОБЛЕМНАЯ СТРОКА ---
# warnings.simplefilter(action='ignore', category=smf.regression.linear_model.MissingDataWarning) # УДАЛЕНО
pd.options.mode.chained_assignment = None
# --- КОНЕЦ ИСПРАВЛЛЕНИЯ ---


print("--- 🚀 НАЧАЛО 'ЧИСТОГО' EVENT STUDY (Метод Радиуса 10 км) ---")

# --- (Остальной код в ячейке остается без изменений) ---
# ... (код начиная с "1. Проверка и Загрузка Данных") ...

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower()
        specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_clinics)} New клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Шаг 1: Создание "Карты Лечения" (Метод Радиуса 10 км) ---
    print("\n--- Шаг 1: Создание Карты Лечения (Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates = {} # Словарь: Patient_Zip -> {'date_new': earliest_date, 'date_acquired': earliest_date}

    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),
                                                          left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

    # Цикл по каждому уникальному ZIP-коду пациента
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']
        patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        
        earliest_date_new = pd.NaT
        earliest_date_acquired = pd.NaT

        # Проверка для New клиник
        nearby_new_dates = []
        for _, clinic_row in df_new_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance <= TREATMENT_RADIUS_KM:
                nearby_new_dates.append(clinic_row['event_date'])
        if nearby_new_dates:
            earliest_date_new = min(nearby_new_dates)

        # Проверка для Acquired PC клиник
        nearby_acquired_dates = []
        for _, clinic_row in df_acquired_pc_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance <= TREATMENT_RADIUS_KM:
                nearby_acquired_dates.append(clinic_row['event_date'])
        if nearby_acquired_dates:
            earliest_date_acquired = min(nearby_acquired_dates)

        treatment_dates[patient_zip] = {
            'treatment_date_NEW': earliest_date_new,
            'treatment_date_ACQUIRED': earliest_date_acquired
        }

    df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
    print(f"\n✅ Карта лечения создана. {len(df_treatment_dates[df_treatment_dates['treatment_date_NEW'].notna()])} ZIP'ов обработаны 'New'.")
    print(f"  {len(df_treatment_dates[df_treatment_dates['treatment_date_ACQUIRED'].notna()])} ZIP'ов обработаны 'Acquired PC'.")

    # --- 5. Шаг 2: Агрегация данных и слияние ---
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    
    # Агрегируем до уровня ZIP-Месяц (для Event Study)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month', 'TotPopACS']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    
    # Рассчитываем лог рейта ER
    # Делаем это ДО слияния, чтобы учесть все наблюдения
    df_agg['Total_ER_Rate'] = (df_agg['Total_Emergency_Visits'] / df_agg['TotPopACS']) * 1000
    df_agg['log_Total_ER_Rate'] = np.log1p(df_agg['Total_ER_Rate'])
    
    # Присоединяем даты лечения
    df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    
    # Создаем фиктивные переменные месяца/года
    df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01') # Дата для расчета time_to_event
    df_analysis['Month_Cat'] = df_analysis['Date'].dt.month.astype(str)
    df_analysis['Year_Cat'] = df_analysis['Date'].dt.year.astype(str)

    print("✅ Данные агрегированы и объединены с датами лечения.")


    # --- 6. Шаг 3: Модель 1 (Эффект 'NEW') ---
    print("\n--- Шаг 3: Запуск Модели 1 (Эффект Доступа - 'NEW') ---")
    
    df_model_new = df_analysis.copy()
    # Удаляем строки, где дата лечения 'New' отсутствует (это наш контроль)
    # df_model_new = df_model_new.dropna(subset=['treatment_date_NEW']) # <-- Неправильно! Контроль нужен!
    
    # Рассчитываем time_to_event_NEW (будет NaN для контроля)
    df_model_new['event_month_new'] = pd.to_datetime(df_model_new['treatment_date_NEW']).dt.to_period('M')
    df_model_new['current_month'] = pd.to_datetime(df_model_new['Date']).dt.to_period('M')
    df_model_new['time_to_event_NEW'] = (df_model_new['current_month'] - df_model_new['event_month_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    
    # Ограничиваем окно и определяем базу
    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    # Отфильтровываем слишком далекие события, но СОХРАНЯЕМ контроль (где time_to_event_NEW is NaN)
    df_model_new_filtered = df_model_new[
        (df_model_new['time_to_event_NEW'].between(-EVENT_WINDOW, EVENT_WINDOW)) |
        (df_model_new['time_to_event_NEW'].isna())
    ].copy()
    df_model_new_filtered['time_to_event_NEW'] = df_model_new_filtered['time_to_event_NEW'].fillna(BASE_PERIOD - 100).astype(int) # Заполняем NaN для модели

    OUTCOME = "log_Total_ER_Rate"
    # Формула для staggered DiD с never-treated
    formula_event_new = f"{OUTCOME} ~ C(time_to_event_NEW, Treatment(reference={BASE_PERIOD})) + C(Filtered_Patient_ZipCode) + C(Year_Month)"

    model_event_new_result = None
    if not df_model_new_filtered.empty and df_model_new_filtered['treatment_date_NEW'].notna().any():
        try:
            print(f"...Запуск модели Event Study для 'New' (Наблюдений: {len(df_model_new_filtered)})...")
            # Переименуем ZIP для совместимости с OLS
            df_model_new_renamed = df_model_new_filtered.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

            # Используем OLS с фикс. эффектами
            model_event = smf.ols(formula_event_new, data=df_model_new_renamed)
            # Кластеризуем ошибки на уровне ZIP
            model_event_new_result = model_event.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_renamed['Zip']})
            print("  ✅ Модель Event Study ('New') рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА при расчете модели Event Study ('New'): {e}")
            # Попробуем без C(Zip), если C(Zip) вызывает проблемы
            try:
                 print("     ... Попытка запуска без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_NEW, Treatment(reference={BASE_PERIOD})) + C(Year_Month)"
                 model_event_no_zip = smf.ols(formula_no_zip, data=df_model_new_renamed)
                 model_event_new_result = model_event_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_renamed['Zip']})
                 print("        ✅ Модель ('New') без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА при расчете модели ('New') без C(Zip): {e2}")

    # --- Визуализация Модели 1 ---
    print("\n--- Визуализация Графика 1 (Эффект 'New') ---")
    
    # (Функция извлечения эффектов Event Study)
    def extract_event_study_effects(model_result, base_period, time_var_name="time_to_event"):
        pattern = rf"C\({time_var_name}.*?Treatment\(reference=.*?\)\)\[T\.(-?\d+\.?\d*)\]"
        params = model_result.params.filter(regex=pattern)
        conf = model_result.conf_int().filter(regex=pattern, axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        results['Relative_Month'].append(base_period); results['Effect'].append(0)
        results['Conf_Low'].append(0); results['Conf_High'].append(0)
        for idx in params.index:
            try:
                match = re.search(pattern, idx)
                if match:
                    month = int(float(match.group(1)))
                    if month == base_period: continue
                    results['Relative_Month'].append(month)
                    results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0])
                    results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)

    if model_event_new_result:
        event_data_new = extract_event_study_effects(model_event_new_result, BASE_PERIOD, "time_to_event_NEW")
        if not event_data_new.empty:
            fig1, ax1 = plt.subplots(figsize=(12, 7))
            errors = [event_data_new['Effect'] - event_data_new['Conf_Low'], event_data_new['Conf_High'] - event_data_new['Effect']]
            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
            if valid_error.all():
                 ax1.errorbar(x=event_data_new['Relative_Month'], y=event_data_new['Effect'], yerr=errors, fmt='-o', capsize=3, label='Effect of New Clinic', color='blue')
            else:
                 ax1.plot(event_data_new['Relative_Month'], event_data_new['Effect'], marker='o', linestyle='-', label='Effect of New Clinic (No CI)', color='blue')
            ax1.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax1.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
            ax1.set_title("Graph 1: Event Study - Effect of 'New' Clinic Entry (Radius Method)", fontsize=14)
            ax1.set_xlabel("Months Relative to First 'New' Clinic within 10km")
            ax1.set_ylabel(f"Effect on {OUTCOME}")
            ax1.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax1.legend()
            ax1.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика 1.")
    else: print("Модель 1 не была рассчитана, график не строится.")


    # --- 7. Шаг 4: Модель 2 (Эффект 'ACQUIRED') ---
    print("\n--- Шаг 4: Запуск Модели 2 (Эффект Воронки - 'ACQUIRED') ---")

    df_model_acq = df_analysis.copy()
    # Рассчитываем time_to_event_ACQUIRED
    df_model_acq['event_month_acq'] = pd.to_datetime(df_model_acq['treatment_date_ACQUIRED']).dt.to_period('M')
    df_model_acq['current_month'] = pd.to_datetime(df_model_acq['Date']).dt.to_period('M') # Пересчитываем, на всякий случай
    df_model_acq['time_to_event_ACQUIRED'] = (df_model_acq['current_month'] - df_model_acq['event_month_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # Ограничиваем окно и определяем базу
    df_model_acq_filtered = df_model_acq[
        (df_model_acq['time_to_event_ACQUIRED'].between(-EVENT_WINDOW, EVENT_WINDOW)) |
        (df_model_acq['time_to_event_ACQUIRED'].isna())
    ].copy()
    # Заполняем NaN для контроля, чтобы модель работала
    df_model_acq_filtered['time_to_event_ACQUIRED'] = df_model_acq_filtered['time_to_event_ACQUIRED'].fillna(BASE_PERIOD - 100).astype(int)

    # Формула для Acquired
    formula_event_acq = f"{OUTCOME} ~ C(time_to_event_ACQUIRED, Treatment(reference={BASE_PERIOD})) + C(Filtered_Patient_ZipCode) + C(Year_Month)"

    model_event_acq_result = None
    if not df_model_acq_filtered.empty and df_model_acq_filtered['treatment_date_ACQUIRED'].notna().any():
        try:
            print(f"...Запуск модели Event Study для 'Acquired' (Наблюдений: {len(df_model_acq_filtered)})...")
            df_model_acq_renamed = df_model_acq_filtered.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

            model_event = smf.ols(formula_event_acq, data=df_model_acq_renamed)
            model_event_acq_result = model_event.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_renamed['Zip']})
            print("  ✅ Модель Event Study ('Acquired') рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА при расчете модели Event Study ('Acquired'): {e}")
            try:
                 print("     ... Попытка запуска без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_ACQUIRED, Treatment(reference={BASE_PERIOD})) + C(Year_Month)"
                 model_event_no_zip = smf.ols(formula_no_zip, data=df_model_acq_renamed)
                 model_event_acq_result = model_event_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_renamed['Zip']})
                 print("        ✅ Модель ('Acquired') без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА при расчете модели ('Acquired') без C(Zip): {e2}")

    # --- Визуализация Модели 2 ---
    print("\n--- Визуализация Графика 2 (Эффект 'Acquired') ---")
    if model_event_acq_result:
        event_data_acq = extract_event_study_effects(model_event_acq_result, BASE_PERIOD, "time_to_event_ACQUIRED")
        if not event_data_acq.empty:
            fig2, ax2 = plt.subplots(figsize=(12, 7))
            errors = [event_data_acq['Effect'] - event_data_acq['Conf_Low'], event_data_acq['Conf_High'] - event_data_acq['Effect']]
            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
            if valid_error.all():
                 ax2.errorbar(x=event_data_acq['Relative_Month'], y=event_data_acq['Effect'], yerr=errors, fmt='-s', capsize=3, label='Effect of Acquired PC Clinic', color='red')
            else:
                 ax2.plot(event_data_acq['Relative_Month'], event_data_acq['Effect'], marker='s', linestyle='-', label='Effect of Acquired PC Clinic (No CI)', color='red')

            ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax2.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
            ax2.set_title("Graph 2: Event Study - Effect of 'Acquired' PC Clinic Entry (Radius Method)", fontsize=14)
            ax2.set_xlabel("Months Relative to First 'Acquired' PC Clinic within 10km")
            ax2.set_ylabel(f"Effect on {OUTCOME}")
            ax2.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax2.legend()
            ax2.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика 2.")
    else: print("Модель 2 не была рассчитана, график не строится.")

    print("\n🎉 --- 'Чистый' Event Study (Метод Радиуса) завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from geopy.distance import geodesic
import warnings
import re
import os

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО ПРОВЕРКИ ПАРАЛЛЕЛЬНЫХ ТРЕНДОВ (План А: улучшенная логика радиуса + корректная контрольная группа) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ОШИБКА: Не могу найти файл {e.filename}. Останавливаюсь.")
        raise

    # --- 2. Очистка Данных ---
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]

    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')

    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()

    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Категоризация 'Acquired' и выбор PC клиник ---
    df_acquired_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'Acquired'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']

    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower()
        specialty_name = str(row.get('Specialty', '')).lower()
        for keyword in pc_keywords:
            if keyword.lower() in facility_name or keyword.lower() in specialty_name:
                return True
        return False

    acquired_pc_clinics = df_acquired_clinics_all[df_acquired_clinics_all.apply(is_pc_clinic, axis=1)].copy()
    print(f"\n✅ Найдено {len(acquired_pc_clinics)} Acquired PC/General клиник для анализа.")

    # --- 4. Подготовка основного DataFrame с визитами ---
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['patient_lat', 'patient_lon', 'TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)

    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Date', 'Year', 'Month', 'patient_lat', 'patient_lon', 'TotPopACS']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    df_agg['Total_ER_Rate'] = (df_agg['Total_Emergency_Visits'] / df_agg['TotPopACS']) * 1000
    print("✅ Агрегированные общие рейты ER визитов рассчитаны.")

    # --- 5. Поиск всех клиник в радиусе ≤10 км ---
    def find_first_clinic_within_radius(patient_coords, clinic_group_df, radius_km=10):
        clinics_in_radius = []
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance <= radius_km:
                clinics_in_radius.append((distance, clinic_row['event_date']))
        if clinics_in_radius:
            clinics_in_radius.sort(key=lambda x: x[1])
            return {
                'distance_to_first_clinic_km': clinics_in_radius[0][0],
                'pc_event_date': clinics_in_radius[0][1],
                'num_clinics_in_radius': len(clinics_in_radius)
            }
        else:
            return {}

    treatment_map_A = []
    patient_zips_coords = df_agg[['Filtered_Patient_ZipCode', 'patient_lat', 'patient_lon']].drop_duplicates().dropna()
    print(f"🔍 Поиск клиник в радиусе ≤10 км для {len(patient_zips_coords)} ZIP-кодов...")

    for _, patient_row in patient_zips_coords.iterrows():
        patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        clinic_info = find_first_clinic_within_radius(patient_coords, acquired_pc_clinics, radius_km=10)
        if clinic_info:
            treatment_map_A.append({
                'Filtered_Patient_ZipCode': patient_row['Filtered_Patient_ZipCode'],
                'distance_to_first_clinic_km': clinic_info['distance_to_first_clinic_km'],
                'pc_event_date': clinic_info['pc_event_date'],
                'num_clinics_in_radius': clinic_info['num_clinics_in_radius']
            })

    df_treatment_map_A = pd.DataFrame(treatment_map_A)
    print(f"✅ Найдено ZIP-кодов с клиниками в радиусе ≤10 км: {len(df_treatment_map_A)}")

    # --- 6. Подготовка данных для Event Study (с контрольной группой) ---
    df_event_A = df_agg.merge(df_treatment_map_A, on='Filtered_Patient_ZipCode', how='left')

    # Маркер наличия клиники поблизости
    df_event_A['has_clinic_within_10km'] = df_event_A['pc_event_date'].notna().astype(int)

    df_event_A['event_month'] = pd.to_datetime(df_event_A['pc_event_date'], errors='coerce').dt.to_period('M')
    df_event_A['current_month'] = pd.to_datetime(df_event_A['Date']).dt.to_period('M')
    df_event_A['relative_month'] = (df_event_A['current_month'] - df_event_A['event_month']).apply(lambda x: x.n if pd.notnull(x) else np.nan)

    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    df_event_A = df_event_A[(df_event_A['relative_month'].between(-EVENT_WINDOW, EVENT_WINDOW)) | (df_event_A['has_clinic_within_10km'] == 0)]
    df_event_A['relative_month'] = df_event_A['relative_month'].fillna(-999).astype(int)

    # --- 7. Разделение на группы ---
    df_event_A_near = df_event_A[df_event_A['distance_to_first_clinic_km'] <= 10].copy()
    df_event_A_far = df_event_A[df_event_A['distance_to_first_clinic_km'] > 20].copy()

    # Добавляем ZIP-коды без клиник вообще как часть контроля
    df_event_A_far = pd.concat([df_event_A_far, df_event_A[df_event_A['has_clinic_within_10km'] == 0]])

    print(f"\n📊 Подготовлено данных для Event Study:")
    print(f"  Близко (≤10 км): {len(df_event_A_near)} наблюдений")
    print(f"  Далеко (>20 км или нет клиник): {len(df_event_A_far)} наблюдений")
    if 'num_clinics_in_radius' in df_event_A.columns:
        print(f"  Среднее число клиник в радиусе 10 км: {df_event_A['num_clinics_in_radius'].mean():.2f}")



    # --- 8. Запуск ОТДЕЛЬНЫХ регрессий Event Study ---
    models_event_A = {}
    OUTCOME = "Total_ER_Rate"
    # Формула с C(Zip) для контроля базовых различий
    FORMULA_EVENT = f"{OUTCOME} ~ C(relative_month, Treatment(reference={BASE_PERIOD})) + C(Year) + C(Month) + C(Zip)"

    # Модель для группы "Близко"
    if not df_event_A_near.empty:
        try:
            print("...Запуск Event Study для группы 'Близко' (0-10 km)...")
            df_event_A_near = df_event_A_near.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
            model_near = smf.ols(FORMULA_EVENT, data=df_event_A_near)
            models_event_A['Near'] = model_near.fit(cov_type='cluster', cov_kwds={'groups': df_event_A_near['Zip']})
            print("  ✅ Модель 'Близко' рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА при расчете модели 'Близко': {e}")
            if "requires non-empty data" in str(e) or "PerfectSeparationError" in str(e):
                 print("     -> Вероятно, недостаточно данных или вариации в этой группе.")

    # Модель для группы "Далеко"
    if not df_event_A_far.empty:
         try:
            print("...Запуск Event Study для группы 'Далеко' (>20 km)...")
            df_event_A_far = df_event_A_far.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
            model_far = smf.ols(FORMULA_EVENT, data=df_event_A_far)
            models_event_A['Far'] = model_far.fit(cov_type='cluster', cov_kwds={'groups': df_event_A_far['Zip']})
            print("  ✅ Модель 'Далеко' рассчитана.")
         except Exception as e:
            print(f"  ❌ ОШИБКА при расчете модели 'Далеко': {e}")
            if "requires non-empty data" in str(e) or "PerfectSeparationError" in str(e):
                 print("     -> Вероятно, недостаточно данных или вариации в этой группе.")

    # --- 9. Извлечение и Визуализация Результатов ---
    print("\n--- Визуализация Параллельных Трендов (План А) ---")

    # (Функция извлечения эффектов Event Study - та же, что и раньше)
    def extract_event_study_effects(model_result, base_period):
        # ... (Копируем функцию из предыдущего ответа) ...
        params = model_result.params.filter(like="C(relative_month")
        conf = model_result.conf_int().filter(like="C(relative_month", axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        results['Relative_Month'].append(base_period)
        results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
        for idx in params.index:
            try:
                match = re.search(r'\[T\.(-?\d+\.?\d*)\]', idx)
                if match:
                    month = int(float(match.group(1)))
                    if month == base_period: continue
                    results['Relative_Month'].append(month)
                    results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0])
                    results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)


    fig_trends, ax_trends = plt.subplots(figsize=(12, 7))
    plot_made = False

    # Линия для "Близко"
    if 'Near' in models_event_A:
        data_near = extract_event_study_effects(models_event_A['Near'], BASE_PERIOD)
        if not data_near.empty:
            errors_near = [data_near['Effect'] - data_near['Conf_Low'], data_near['Conf_High'] - data_near['Effect']]
            valid_error = ~np.isnan(errors_near[0]) & ~np.isnan(errors_near[1])
            if valid_error.all():
                 ax_trends.errorbar(x=data_near['Relative_Month'], y=data_near['Effect'], yerr=errors_near,
                                   fmt='-o', capsize=3, label='Near (0-10 km)', color='blue')
            else:
                 ax_trends.plot(data_near['Relative_Month'], data_near['Effect'], marker='o', linestyle='-',
                                label='Near (0-10 km, No CI)', color='blue')
            plot_made = True

    # Линия для "Далеко"
    if 'Far' in models_event_A:
        data_far = extract_event_study_effects(models_event_A['Far'], BASE_PERIOD)
        if not data_far.empty:
            errors_far = [data_far['Effect'] - data_far['Conf_Low'], data_far['Conf_High'] - data_far['Effect']]
            valid_error = ~np.isnan(errors_far[0]) & ~np.isnan(errors_far[1])
            if valid_error.all():
                ax_trends.errorbar(x=data_far['Relative_Month'], y=data_far['Effect'], yerr=errors_far,
                                   fmt='--s', capsize=3, label='Far (>20 km, Control)', color='red')
            else:
                 ax_trends.plot(data_far['Relative_Month'], data_far['Effect'], marker='s', linestyle='--',
                                label='Far (>20 km, No CI)', color='red')
            plot_made = True

    if plot_made:
        ax_trends.axhline(0, color='black', linestyle='-', linewidth=0.8)
        ax_trends.axvline(x=-0.5, color='grey', linestyle=':', label=f'Acquisition Event (Month 0)')
        ax_trends.set_title("Plan A Event Study: Parallel Trends Check (Acquired PC Clinics, Total ER Rate)", fontsize=14)
        ax_trends.set_xlabel("Months Relative to Nearest PC Clinic Acquisition")
        ax_trends.set_ylabel("Effect vs Month -1 (Change in Daily ER Visits per 1000 Ppl)")
        ax_trends.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
        ax_trends.legend()
        ax_trends.grid(True, axis='y', linestyle=':')
        plt.tight_layout()
        plt.show()
    else:
        print("\nНе удалось построить график: нет данных из моделей.")

    print("\n🎉 --- Проверка Параллельных Трендов (План А) завершена! ---")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm.notebook import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
# warnings.simplefilter(action='ignore', category=smf.regression.linear_model.MissingDataWarning) # Убрали
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО 'ЧИСТОГО' EVENT STUDY (Метод Радиуса 10 км) - ИСПРАВЛЕННАЯ ВЕРСИЯ ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_clinics)} New клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Шаг 1: Создание "Карты Лечения" (Метод Радиуса 10 км) ---
    print("\n--- Шаг 1: Создание Карты Лечения (Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new = pd.NaT; earliest_date_acquired = pd.NaT
        nearby_new_dates = []; nearby_acquired_dates = []
        for _, clinic_row in df_new_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_dates.append(clinic_row['event_date'])
        if nearby_new_dates: earliest_date_new = min(nearby_new_dates)
        for _, clinic_row in df_acquired_pc_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_dates.append(clinic_row['event_date'])
        if nearby_acquired_dates: earliest_date_acquired = min(nearby_acquired_dates)
        treatment_dates[patient_zip] = {'treatment_date_NEW': earliest_date_new, 'treatment_date_ACQUIRED': earliest_date_acquired}
    df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
    print(f"\n✅ Карта лечения создана.") # Убрали вывод количества

    # --- 5. Шаг 2: Агрегация данных и слияние ---
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month', 'TotPopACS']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    df_agg['Total_ER_Rate'] = (df_agg['Total_Emergency_Visits'] / df_agg['TotPopACS']) * 1000
    df_agg['log_Total_ER_Rate'] = np.log1p(df_agg['Total_ER_Rate'])
    df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
    df_analysis['Month_Cat'] = df_analysis['Date'].dt.month # Используем число месяца
    df_analysis['Year_Cat'] = df_analysis['Date'].dt.year   # Используем число года
    print("✅ Данные агрегированы и объединены с датами лечения.")


    # --- 6. Шаг 3: Модель 1 (Эффект 'NEW') ---
    # ... (код для Модели 1 остается без изменений, т.к. график был почти нормальным) ...
    print("\n--- Шаг 3: Запуск Модели 1 (Эффект Доступа - 'NEW') ---")
    df_model_new = df_analysis.copy()
    df_model_new['event_month_new'] = pd.to_datetime(df_model_new['treatment_date_NEW']).dt.to_period('M')
    df_model_new['current_month'] = pd.to_datetime(df_model_new['Date']).dt.to_period('M')
    df_model_new['time_to_event_NEW'] = (df_model_new['current_month'] - df_model_new['event_month_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    EVENT_WINDOW = 12; BASE_PERIOD = -1
    df_model_new_filtered = df_model_new[ (df_model_new['time_to_event_NEW'].between(-EVENT_WINDOW, EVENT_WINDOW)) | (df_model_new['time_to_event_NEW'].isna()) ].copy()
    # Заполняем NaN **после** фильтрации
    df_model_new_filtered['time_to_event_NEW_cat'] = df_model_new_filtered['time_to_event_NEW'].fillna(9999).astype(int) # Используем фиктивное число для контроля

    OUTCOME = "log_Total_ER_Rate"
    # Формула с явным указанием контроля через фиктивное число
    formula_event_new = f"{OUTCOME} ~ C(time_to_event_NEW_cat, Treatment(reference={BASE_PERIOD})) + C(Filtered_Patient_ZipCode) + C(Year_Cat) + C(Month_Cat)"

    model_event_new_result = None
    if not df_model_new_filtered.empty and df_model_new_filtered['treatment_date_NEW'].notna().any():
        try:
            print(f"...Запуск модели Event Study для 'New' (Наблюдений: {len(df_model_new_filtered)})...")
            df_model_new_renamed = df_model_new_filtered.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
            model_event = smf.ols(formula_event_new, data=df_model_new_renamed)
            model_event_new_result = model_event.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_renamed['Zip']})
            print("  ✅ Модель Event Study ('New') рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА ('New'): {e}")
            # ... (оставим попытку без C(Zip)) ...
            try:
                 print("     ... Попытка ('New') без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_NEW_cat, Treatment(reference={BASE_PERIOD})) + C(Year_Cat) + C(Month_Cat)"
                 model_event_no_zip = smf.ols(formula_no_zip, data=df_model_new_renamed)
                 model_event_new_result = model_event_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_renamed['Zip']})
                 print("        ✅ Модель ('New') без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА ('New') без C(Zip): {e2}")

    # --- Визуализация Модели 1 ---
    print("\n--- Визуализация Графика 1 (Эффект 'New') ---")
    def extract_event_study_effects(model_result, base_period, time_var_name="time_to_event", control_val=9999):
        # Ищем коэффициенты C(time_var_name, ...)[T.month]
        pattern = rf"C\({time_var_name}.*?Treatment\(reference=.*?\)\)\[T\.(-?\d+)\]" # Месяц должен быть целым
        params = model_result.params.filter(regex=pattern)
        conf = model_result.conf_int().filter(regex=pattern, axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        # Добавляем базовый период
        results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
        # Извлекаем остальные
        for idx in params.index:
            try:
                match = re.search(pattern, idx)
                if match:
                    month = int(match.group(1))
                    # Исключаем базовый период и фиктивное значение контроля
                    if month == base_period or month == control_val: continue
                    results['Relative_Month'].append(month)
                    results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0])
                    results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)

    if model_event_new_result:
        event_data_new = extract_event_study_effects(model_event_new_result, BASE_PERIOD, "time_to_event_NEW_cat")
        if not event_data_new.empty:
            fig1, ax1 = plt.subplots(figsize=(12, 7))
            errors = [event_data_new['Effect'] - event_data_new['Conf_Low'], event_data_new['Conf_High'] - event_data_new['Effect']]
            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
            # Строим только если доверительные интервалы корректны
            if valid_error.all():
                 ax1.errorbar(x=event_data_new['Relative_Month'], y=event_data_new['Effect'], yerr=errors, fmt='-o', capsize=3, label='Effect of New Clinic', color='blue')
            else:
                 ax1.plot(event_data_new['Relative_Month'], event_data_new['Effect'], marker='o', linestyle='-', label='Effect of New Clinic (No CI)', color='blue')
            ax1.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax1.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
            ax1.set_title("Graph 1: Event Study - Effect of 'New' Clinic Entry (Radius Method)", fontsize=14)
            ax1.set_xlabel("Months Relative to First 'New' Clinic within 10km")
            ax1.set_ylabel(f"Effect on {OUTCOME}")
            ax1.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax1.legend()
            ax1.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика 1.")
    else: print("Модель 1 не была рассчитана, график не строится.")


    # --- 7. Шаг 4: Модель 2 (Эффект 'ACQUIRED') ---
    print("\n--- Шаг 4: Запуск Модели 2 (Эффект Воронки - 'ACQUIRED') ---")

    df_model_acq = df_analysis.copy()
    # Рассчитываем time_to_event_ACQUIRED
    df_model_acq['event_month_acq'] = pd.to_datetime(df_model_acq['treatment_date_ACQUIRED']).dt.to_period('M')
    df_model_acq['current_month'] = pd.to_datetime(df_model_acq['Date']).dt.to_period('M')
    df_model_acq['time_to_event_ACQUIRED'] = (df_model_acq['current_month'] - df_model_acq['event_month_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # --- ИСПРАВЛЕНИЕ: Используем тот же метод фильтрации и заполнения NaN ---
    df_model_acq_filtered = df_model_acq[
        (df_model_acq['time_to_event_ACQUIRED'].between(-EVENT_WINDOW, EVENT_WINDOW)) |
        (df_model_acq['time_to_event_ACQUIRED'].isna())
    ].copy()
    df_model_acq_filtered['time_to_event_ACQ_cat'] = df_model_acq_filtered['time_to_event_ACQUIRED'].fillna(9999).astype(int) # Используем другое имя!

    # Формула для Acquired
    # --- ИСПРАВЛЕНИЕ: Используем time_to_event_ACQ_cat ---
    formula_event_acq = f"{OUTCOME} ~ C(time_to_event_ACQ_cat, Treatment(reference={BASE_PERIOD})) + C(Filtered_Patient_ZipCode) + C(Year_Cat) + C(Month_Cat)"

    model_event_acq_result = None
    if not df_model_acq_filtered.empty and df_model_acq_filtered['treatment_date_ACQUIRED'].notna().any():
        try:
            print(f"...Запуск модели Event Study для 'Acquired' (Наблюдений: {len(df_model_acq_filtered)})...")
            df_model_acq_renamed = df_model_acq_filtered.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

            model_event = smf.ols(formula_event_acq, data=df_model_acq_renamed)
            model_event_acq_result = model_event.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_renamed['Zip']})
            print("  ✅ Модель Event Study ('Acquired') рассчитана.")
            # --- ОТЛАДКА: Выводим имена коэффициентов ---
            # print("Коэффициенты модели 'Acquired':")
            # print(model_event_acq_result.params.index.tolist())
        except Exception as e:
            print(f"  ❌ ОШИБКА ('Acquired'): {e}")
            # ... (оставим попытку без C(Zip)) ...
            try:
                 print("     ... Попытка ('Acquired') без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_ACQ_cat, Treatment(reference={BASE_PERIOD})) + C(Year_Cat) + C(Month_Cat)"
                 model_event_no_zip = smf.ols(formula_no_zip, data=df_model_acq_renamed)
                 model_event_acq_result = model_event_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_renamed['Zip']})
                 print("        ✅ Модель ('Acquired') без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА ('Acquired') без C(Zip): {e2}")


    # --- Визуализация Модели 2 ---
    print("\n--- Визуализация Графика 2 (Эффект 'Acquired') ---")
    if model_event_acq_result:
        # --- ИСПРАВЛЕНИЕ: Передаем правильное имя переменной времени ---
        event_data_acq = extract_event_study_effects(model_event_acq_result, BASE_PERIOD, "time_to_event_ACQ_cat")
        if not event_data_acq.empty:
            fig2, ax2 = plt.subplots(figsize=(12, 7))
            errors = [event_data_acq['Effect'] - event_data_acq['Conf_Low'], event_data_acq['Conf_High'] - event_data_acq['Effect']]
            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
            # Строим только если доверительные интервалы корректны
            if valid_error.all():
                 ax2.errorbar(x=event_data_acq['Relative_Month'], y=event_data_acq['Effect'], yerr=errors, fmt='-s', capsize=3, label='Effect of Acquired PC Clinic', color='red')
            else:
                 ax2.plot(event_data_acq['Relative_Month'], event_data_acq['Effect'], marker='s', linestyle='-', label='Effect of Acquired PC Clinic (No CI)', color='red')

            ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax2.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
            ax2.set_title("Graph 2: Event Study - Effect of 'Acquired' PC Clinic Entry (Radius Method)", fontsize=14)
            ax2.set_xlabel("Months Relative to First 'Acquired' PC Clinic within 10km")
            ax2.set_ylabel(f"Effect on {OUTCOME}")
            ax2.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax2.legend()
            ax2.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика 2.")
    else: print("Модель 2 не была рассчитана, график не строится.")

    print("\n🎉 --- 'Чистый' Event Study (Метод Радиуса) завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Используем обычный tqdm

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
# warnings.simplefilter(action='ignore', category=smf.regression.linear_model.MissingDataWarning) # Убрали
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО 'ЧИСТОГО' EVENT STUDY (Метод Радиуса 10 км) - Версия 3 ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_clinics)} New клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Шаг 1: Создание "Карты Лечения" (Метод Радиуса 10 км) ---
    print("\n--- Шаг 1: Создание Карты Лечения (Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов"):
        # ... (логика поиска дат без изменений) ...
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new = pd.NaT; earliest_date_acquired = pd.NaT
        nearby_new_dates = []; nearby_acquired_dates = []
        for _, clinic_row in df_new_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_dates.append(clinic_row['event_date'])
        if nearby_new_dates: earliest_date_new = min(nearby_new_dates)
        for _, clinic_row in df_acquired_pc_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_dates.append(clinic_row['event_date'])
        if nearby_acquired_dates: earliest_date_acquired = min(nearby_acquired_dates)
        treatment_dates[patient_zip] = {'treatment_date_NEW': earliest_date_new, 'treatment_date_ACQUIRED': earliest_date_acquired}

    df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
    print(f"\n✅ Карта лечения создана.")

    # --- 5. Шаг 2: Агрегация данных и слияние ---
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month', 'TotPopACS']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    df_agg['Total_ER_Rate'] = (df_agg['Total_Emergency_Visits'] / df_agg['TotPopACS']) * 1000
    df_agg['log_Total_ER_Rate'] = np.log1p(df_agg['Total_ER_Rate'])
    df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
    df_analysis['Month_Cat'] = df_analysis['Date'].dt.month # Используем число месяца
    df_analysis['Year_Cat'] = df_analysis['Date'].dt.year   # Используем число года
    # --- ИСПРАВЛЕНИЕ: Переименовываем ZIP здесь ---
    df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    print("✅ Данные агрегированы и объединены с датами лечения.")


    # --- 6. Шаг 3: Модель 1 (Эффект 'NEW') ---
    print("\n--- Шаг 3: Запуск Модели 1 (Эффект Доступа - 'NEW') ---")
    df_model_new = df_analysis.copy()
    df_model_new['event_month_new'] = pd.to_datetime(df_model_new['treatment_date_NEW']).dt.to_period('M')
    df_model_new['current_month'] = pd.to_datetime(df_model_new['Date']).dt.to_period('M')
    df_model_new['time_to_event_NEW'] = (df_model_new['current_month'] - df_model_new['event_month_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    EVENT_WINDOW = 12; BASE_PERIOD = -1

    # --- ИСПРАВЛЕНИЕ: Отфильтровываем NaN ДО создания категории ---
    df_model_new_filtered = df_model_new[
        df_model_new['time_to_event_NEW'].between(-EVENT_WINDOW, EVENT_WINDOW)
        # Убираем контроль (NaN) из данных для модели, он будет базой по умолчанию
        # т.к. C(Zip) и C(Year_Month) поглотят их базовые уровни
    ].copy()
    # Убедимся, что time_to_event целое число
    df_model_new_filtered['time_to_event_NEW'] = df_model_new_filtered['time_to_event_NEW'].astype(int)

    OUTCOME = "log_Total_ER_Rate"
    # --- ИСПРАВЛЕНИЕ: Правильная формула с C(Zip) ---
    formula_event_new = f"{OUTCOME} ~ C(time_to_event_NEW, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year_Cat) + C(Month_Cat)"

    model_event_new_result = None
    if not df_model_new_filtered.empty and df_model_new_filtered['treatment_date_NEW'].notna().any(): # Проверка, что есть treated данные
        try:
            print(f"...Запуск модели Event Study для 'New' (Наблюдений: {len(df_model_new_filtered)})...")
            # ZIP уже переименован
            model_event = smf.ols(formula_event_new, data=df_model_new_filtered)
            model_event_new_result = model_event.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_filtered['Zip']})
            print("  ✅ Модель Event Study ('New') с C(Zip) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА ('New') с C(Zip): {e}")
            # ... (оставим попытку без C(Zip)) ...
            try:
                 print("     ... Попытка ('New') без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_NEW, Treatment(reference={BASE_PERIOD})) + C(Year_Cat) + C(Month_Cat)"
                 model_event_no_zip = smf.ols(formula_no_zip, data=df_model_new_filtered)
                 model_event_new_result = model_event_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_filtered['Zip']})
                 print("        ✅ Модель ('New') без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА ('New') без C(Zip): {e2}")
    else:
        print("  ⚠️ Недостаточно данных для запуска модели 'New'.")

    # --- Визуализация Модели 1 ---
    print("\n--- Визуализация Графика 1 (Эффект 'New') ---")
    # --- ИСПРАВЛЕНИЕ: Обновленная функция извлечения ---
    def extract_event_study_effects_v3(model_result, base_period, time_var_name):
        pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]" # Ищем целые числа
        params = model_result.params.filter(regex=pattern)
        conf = model_result.conf_int().filter(regex=pattern, axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
        for idx in params.index:
            try:
                match = re.search(pattern, idx)
                if match:
                    month = int(match.group(1)) # Уже int
                    if month == base_period: continue
                    results['Relative_Month'].append(month)
                    results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0])
                    results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)

    if model_event_new_result:
        # Используем новое имя переменной
        event_data_new = extract_event_study_effects_v3(model_event_new_result, BASE_PERIOD, "time_to_event_NEW")
        if not event_data_new.empty:
            fig1, ax1 = plt.subplots(figsize=(12, 7))
            errors = [event_data_new['Effect'] - event_data_new['Conf_Low'], event_data_new['Conf_High'] - event_data_new['Effect']]
            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
            if valid_error.all():
                 ax1.errorbar(x=event_data_new['Relative_Month'], y=event_data_new['Effect'], yerr=errors, fmt='-o', capsize=3, label='Effect of New Clinic', color='blue')
            else:
                 ax1.plot(event_data_new['Relative_Month'], event_data_new['Effect'], marker='o', linestyle='-', label='Effect of New Clinic (No CI)', color='blue')
            ax1.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax1.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
            ax1.set_title("Graph 1: Event Study - Effect of 'New' Clinic Entry (Radius Method)", fontsize=14)
            ax1.set_xlabel("Months Relative to First 'New' Clinic within 10km")
            ax1.set_ylabel(f"Effect on {OUTCOME}")
            ax1.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax1.legend()
            ax1.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика 1.")
    else: print("Модель 1 не была рассчитана, график не строится.")


    # --- 7. Шаг 4: Модель 2 (Эффект 'ACQUIRED') ---
    print("\n--- Шаг 4: Запуск Модели 2 (Эффект Воронки - 'ACQUIRED') ---")
    df_model_acq = df_analysis.copy()
    df_model_acq['event_month_acq'] = pd.to_datetime(df_model_acq['treatment_date_ACQUIRED']).dt.to_period('M')
    df_model_acq['current_month'] = pd.to_datetime(df_model_acq['Date']).dt.to_period('M')
    df_model_acq['time_to_event_ACQUIRED'] = (df_model_acq['current_month'] - df_model_acq['event_month_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # --- ИСПРАВЛЕНИЕ: Отфильтровываем NaN ДО создания категории ---
    df_model_acq_filtered = df_model_acq[
        df_model_acq['time_to_event_ACQUIRED'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_acq_filtered['time_to_event_ACQUIRED'] = df_model_acq_filtered['time_to_event_ACQUIRED'].astype(int)

    # --- ИСПРАВЛЕНИЕ: Правильная формула с C(Zip) ---
    formula_event_acq = f"{OUTCOME} ~ C(time_to_event_ACQUIRED, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year_Cat) + C(Month_Cat)"

    model_event_acq_result = None
    # Проверка, что есть treated данные для Acquired
    if not df_model_acq_filtered.empty and df_model_acq_filtered['treatment_date_ACQUIRED'].notna().any():
        try:
            print(f"...Запуск модели Event Study для 'Acquired' (Наблюдений: {len(df_model_acq_filtered)})...")
            # ZIP уже переименован ранее в df_analysis
            model_event = smf.ols(formula_event_acq, data=df_model_acq_filtered)
            model_event_acq_result = model_event.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_filtered['Zip']})
            print("  ✅ Модель Event Study ('Acquired') с C(Zip) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА ('Acquired') с C(Zip): {e}")
            # ... (оставим попытку без C(Zip)) ...
            try:
                 print("     ... Попытка ('Acquired') без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_ACQUIRED, Treatment(reference={BASE_PERIOD})) + C(Year_Cat) + C(Month_Cat)"
                 model_event_no_zip = smf.ols(formula_no_zip, data=df_model_acq_filtered)
                 model_event_acq_result = model_event_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_filtered['Zip']})
                 print("        ✅ Модель ('Acquired') без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА ('Acquired') без C(Zip): {e2}")
    else:
        print("  ⚠️ Недостаточно данных для запуска модели 'Acquired'.")


    # --- Визуализация Модели 2 ---
    print("\n--- Визуализация Графика 2 (Эффект 'Acquired') ---")
    if model_event_acq_result:
        # Используем правильное имя переменной
        event_data_acq = extract_event_study_effects_v3(model_event_acq_result, BASE_PERIOD, "time_to_event_ACQUIRED")
        if not event_data_acq.empty:
            fig2, ax2 = plt.subplots(figsize=(12, 7))
            errors = [event_data_acq['Effect'] - event_data_acq['Conf_Low'], event_data_acq['Conf_High'] - event_data_acq['Effect']]
            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
            if valid_error.all():
                 ax2.errorbar(x=event_data_acq['Relative_Month'], y=event_data_acq['Effect'], yerr=errors, fmt='-s', capsize=3, label='Effect of Acquired PC Clinic', color='red')
            else:
                 ax2.plot(event_data_acq['Relative_Month'], event_data_acq['Effect'], marker='s', linestyle='-', label='Effect of Acquired PC Clinic (No CI)', color='red')

            ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax2.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
            ax2.set_title("Graph 2: Event Study - Effect of 'Acquired' PC Clinic Entry (Radius Method)", fontsize=14)
            ax2.set_xlabel("Months Relative to First 'Acquired' PC Clinic within 10km")
            ax2.set_ylabel(f"Effect on {OUTCOME}")
            ax2.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax2.legend()
            ax2.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика 2.")
    else: print("Модель 2 не была рассчитана, график не строится.")

    print("\n🎉 --- 'Чистый' Event Study (Метод Радиуса) завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
# warnings.simplefilter(action='ignore', category=smf.regression.linear_model.MissingDataWarning) # Убрали
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО ТЕСТИРОВАНИЯ 'ГИПОТЕЗЫ ИЗ АБСТРАКТА' (Log Ratio ER/Regular) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация 'New' клиник ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    print(f"Идентифицировано: {len(df_new_clinics)} New клиник.")

    # --- 4. Шаг 1: Создание "Карты Лечения" для 'New' (Метод Радиуса 10 км) ---
    print("\n--- Шаг 1: Создание Карты Лечения для 'New' (Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates_new = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new = pd.NaT
        nearby_new_dates = []
        for _, clinic_row in df_new_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM:
                nearby_new_dates.append(clinic_row['event_date'])
        if nearby_new_dates:
            earliest_date_new = min(nearby_new_dates)
        treatment_dates_new[patient_zip] = {'treatment_date_NEW': earliest_date_new}

    df_treatment_dates_new = pd.DataFrame.from_dict(treatment_dates_new, orient='index')
    print(f"\n✅ Карта лечения ('New') создана.")

    # --- 5. Шаг 1 (продолжение): Агрегация данных и создание Y_Ratio ---
    df_panel = final_data.copy()
    # Нужна популяция? Нет, для соотношения не нужна.

    # Считаем визиты по типам
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int) # Все не-ER считаем регулярными

    # Агрегируем до уровня ZIP-Месяц
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg_ratio = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        # Суммируем EncounterCount только для нужного типа визита
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
    ).reset_index()

    # --- Создаем Y_Ratio = log((ER+1)/(Regular+1)) ---
    # Используем np.log1p(x) которое эквивалентно np.log(x + 1)
    df_agg_ratio['Y_Ratio'] = np.log1p(df_agg_ratio['Total_Emergency_Visits']) - np.log1p(df_agg_ratio['Total_Regular_Visits'])
    print("✅ Новая переменная Y_Ratio (log ER+1 / Regular+1) создана.")

    # Присоединяем даты лечения 'New'
    df_analysis_ratio = df_agg_ratio.merge(df_treatment_dates_new, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')

    # Создаем фиктивные переменные месяца/года и дату
    df_analysis_ratio['Date'] = pd.to_datetime(df_analysis_ratio['Year_Month'] + '-01')
    df_analysis_ratio['Month_Cat'] = df_analysis_ratio['Date'].dt.month
    df_analysis_ratio['Year_Cat'] = df_analysis_ratio['Date'].dt.year
    df_analysis_ratio = df_analysis_ratio.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
    print("✅ Данные агрегированы и объединены с датами лечения 'New'.")


    # --- 6. Шаг 2: Подготовка и Запуск Модели Event Study ---
    df_model_ratio = df_analysis_ratio.copy()
    # Рассчитываем time_to_event_NEW
    df_model_ratio['event_month_new'] = pd.to_datetime(df_model_ratio['treatment_date_NEW']).dt.to_period('M')
    df_model_ratio['current_month'] = pd.to_datetime(df_model_ratio['Date']).dt.to_period('M')
    df_model_ratio['time_to_event_NEW'] = (df_model_ratio['current_month'] - df_model_ratio['event_month_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # Ограничиваем окно и отфильтровываем контроль (NaN) ПЕРЕД созданием категории
    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    df_model_ratio_filtered = df_model_ratio[
        df_model_ratio['time_to_event_NEW'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    # Убедимся, что time_to_event целое число
    df_model_ratio_filtered['time_to_event_NEW'] = df_model_ratio_filtered['time_to_event_NEW'].astype(int)

    # Определяем формулу
    OUTCOME = "Y_Ratio"
    # Используем полные фиксированные эффекты
    formula_event_ratio = f"{OUTCOME} ~ C(time_to_event_NEW, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year_Cat) + C(Month_Cat)"

    model_event_ratio_result = None
    if not df_model_ratio_filtered.empty and df_model_ratio_filtered['treatment_date_NEW'].notna().any(): # Проверка, что есть treated данные
        try:
            print(f"\n...Запуск модели Event Study для Y_Ratio (Наблюдений: {len(df_model_ratio_filtered)})...")
            # ZIP уже переименован
            model_event = smf.ols(formula_event_ratio, data=df_model_ratio_filtered)
            model_event_ratio_result = model_event.fit(cov_type='cluster', cov_kwds={'groups': df_model_ratio_filtered['Zip']})
            print("  ✅ Модель Event Study (Y_Ratio) с C(Zip) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА (Y_Ratio) с C(Zip): {e}")
            # ... (попытка без C(Zip)) ...
            try:
                 print("     ... Попытка (Y_Ratio) без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_NEW, Treatment(reference={BASE_PERIOD})) + C(Year_Cat) + C(Month_Cat)"
                 model_event_no_zip = smf.ols(formula_no_zip, data=df_model_ratio_filtered)
                 model_event_ratio_result = model_event_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_ratio_filtered['Zip']})
                 print("        ✅ Модель (Y_Ratio) без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА (Y_Ratio) без C(Zip): {e2}")
    else:
        print("  ⚠️ Недостаточно данных для запуска модели Y_Ratio.")


    # --- 7. Шаг 3: Визуализация ---
    print("\n--- Визуализация Графика для Y_Ratio ---")

    # (Используем ту же функцию извлечения эффектов v3)
    def extract_event_study_effects_v3(model_result, base_period, time_var_name):
        pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]"
        params = model_result.params.filter(regex=pattern)
        conf = model_result.conf_int().filter(regex=pattern, axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
        for idx in params.index:
            try:
                match = re.search(pattern, idx)
                if match:
                    month = int(match.group(1))
                    if month == base_period: continue
                    results['Relative_Month'].append(month)
                    results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0])
                    results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)

    if model_event_ratio_result:
        event_data_ratio = extract_event_study_effects_v3(model_event_ratio_result, BASE_PERIOD, "time_to_event_NEW")
        if not event_data_ratio.empty:
            fig_ratio, ax_ratio = plt.subplots(figsize=(12, 7))
            errors = [event_data_ratio['Effect'] - event_data_ratio['Conf_Low'], event_data_ratio['Conf_High'] - event_data_ratio['Effect']]
            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
            if valid_error.all():
                 ax_ratio.errorbar(x=event_data_ratio['Relative_Month'], y=event_data_ratio['Effect'], yerr=errors, fmt='-o', capsize=3, label='Effect on Log Ratio (ER/Regular)', color='purple') # Новый цвет
            else:
                 ax_ratio.plot(event_data_ratio['Relative_Month'], event_data_ratio['Effect'], marker='o', linestyle='-', label='Effect on Log Ratio (ER/Regular, No CI)', color='purple')

            ax_ratio.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax_ratio.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
            ax_ratio.set_title("Effect of 'New' Clinics on Log Ratio of ER/Regular Visits", fontsize=14) # Обновленный заголовок
            ax_ratio.set_xlabel("Months Relative to First 'New' Clinic within 10km")
            ax_ratio.set_ylabel(f"Effect on {OUTCOME}")
            ax_ratio.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax_ratio.legend()
            ax_ratio.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика Y_Ratio.")
    else: print("Модель Y_Ratio не была рассчитана, график не строится.")

    print("\n🎉 --- Тестирование 'Гипотезы из Абстракта' завершено! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm.notebook import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
# warnings.simplefilter(action='ignore', category=smf.regression.linear_model.MissingDataWarning) # Убрали
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО АНАЛИЗА С ВЗВЕШЕННЫМ ВОЗДЕЙСТВИЕМ (Exposure Score) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    # --- Добавляем open_month для клиник ---
    df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_clinics)} New клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Шаг 1-4: Расчет Exposure Score ---
    print("\n--- Расчет Exposure Score (Может занять несколько минут) ---")

    # Создаем df_zip_months
    df_zip_months = final_data[['Filtered_Patient_ZipCode', 'Date']].drop_duplicates()
    df_zip_months['Date'] = pd.to_datetime(df_zip_months['Date']) # Убедимся, что Date это datetime
    df_zip_months['month_period'] = df_zip_months['Date'].dt.to_period('M')
    df_zip_months = df_zip_months.merge(zip_coords.rename(columns={'IntPtLat': 'zip_lat', 'IntPtLon': 'zip_lon'}),
                                         left_on='Filtered_Patient_ZipCode', right_index=True, how='inner') # inner join убирает ZIPы без координат

    # Функция расчета exposure
    def calculate_exposure(zip_row, clinics_df):
        exposure = 0.0
        zip_coords_tuple = (zip_row['zip_lat'], zip_row['zip_lon'])
        current_month = zip_row['month_period']

        # Фильтруем клиники, открытые к этому месяцу
        open_clinics = clinics_df[clinics_df['open_month'] <= current_month]

        if not open_clinics.empty:
            for _, clinic_row in open_clinics.iterrows():
                clinic_coords_tuple = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
                distance_km = geodesic(zip_coords_tuple, clinic_coords_tuple).kilometers
                # Вес: 1 / (1 + расстояние). Добавляем малое число, чтобы избежать деления на ноль, если dist=0
                weight = 1.0 / (1.0 + distance_km + 1e-6)
                exposure += weight
        return exposure

    # Применяем функцию (используем tqdm)
    tqdm.pandas(desc="Расчет exposure_new")
    df_zip_months['exposure_new'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_new_clinics), axis=1)

    tqdm.pandas(desc="Расчет exposure_acquired")
    df_zip_months['exposure_acquired'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_acquired_pc_clinics), axis=1)

    print("✅ Exposure scores рассчитаны.")
    # print(df_zip_months[['exposure_new', 'exposure_acquired']].describe()) # Посмотреть статистику

    # --- 5. Агрегация данных и слияние ---
    df_panel = final_data.copy()
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg_exp = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
    ).reset_index()
    df_agg_exp['Y_Ratio'] = np.log1p(df_agg_exp['Total_Emergency_Visits']) - np.log1p(df_agg_exp['Total_Regular_Visits'])
    df_agg_exp['Date'] = pd.to_datetime(df_agg_exp['Year_Month'] + '-01')

    # Присоединяем exposure scores
    df_analysis_exp = df_agg_exp.merge(
        df_zip_months[['Filtered_Patient_ZipCode', 'Date', 'exposure_new', 'exposure_acquired']],
        on=['Filtered_Patient_ZipCode', 'Date'],
        how='left'
    )
    # Заполняем NaN в exposure нулями (означает отсутствие воздействия)
    df_analysis_exp['exposure_new'] = df_analysis_exp['exposure_new'].fillna(0)
    df_analysis_exp['exposure_acquired'] = df_analysis_exp['exposure_acquired'].fillna(0)

    # Добавляем фиктивные переменные времени и переименовываем ZIP
    df_analysis_exp['Month_Cat'] = df_analysis_exp['Date'].dt.month
    df_analysis_exp['Year_Cat'] = df_analysis_exp['Date'].dt.year
    df_analysis_exp = df_analysis_exp.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

    print("✅ Данные агрегированы и объединены с exposure scores.")


    # --- 6. Запуск Регрессий ---
    OUTCOME = "Y_Ratio"
    FIXED_EFFECTS = "C(Zip) + C(Year_Cat) + C(Month_Cat)"

    # Модель 1: Эффект New Exposure
    formula_exp_new = f"{OUTCOME} ~ exposure_new + {FIXED_EFFECTS}"
    model_exp_new_result = None
    print("\n--- Запуск Модели 1 (Эффект exposure_new) ---")
    try:
        model = smf.ols(formula_exp_new, data=df_analysis_exp)
        model_exp_new_result = model.fit(cov_type='cluster', cov_kwds={'groups': df_analysis_exp['Zip']})
        print("✅ Модель 1 рассчитана.")
        # Выводим только интересующий коэффициент
        print("\nРезультат для exposure_new:")
        print(model_exp_new_result.summary().tables[1].as_html()) # Выводим таблицу коэффициентов
    except Exception as e:
        print(f"❌ ОШИБКА при расчете Модели 1: {e}")


    # Модель 2: Эффект Acquired Exposure
    formula_exp_acq = f"{OUTCOME} ~ exposure_acquired + {FIXED_EFFECTS}"
    model_exp_acq_result = None
    print("\n--- Запуск Модели 2 (Эффект exposure_acquired) ---")
    try:
        model = smf.ols(formula_exp_acq, data=df_analysis_exp)
        model_exp_acq_result = model.fit(cov_type='cluster', cov_kwds={'groups': df_analysis_exp['Zip']})
        print("✅ Модель 2 рассчитана.")
        # Выводим только интересующий коэффициент
        print("\nРезультат для exposure_acquired:")
        print(model_exp_acq_result.summary().tables[1].as_html()) # Выводим таблицу коэффициентов
    except Exception as e:
        print(f"❌ ОШИБКА при расчете Модели 2: {e}")


    # --- 7. Визуализация (Не Event Study, а просто эффект) ---
    # Мы можем построить график, показывающий коэффициент и его доверительный интервал
    print("\n--- Визуализация Коэффициентов ---")

    fig_coeffs, (ax_new, ax_acq) = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))

    # График для New
    if model_exp_new_result:
        coef_new = model_exp_new_result.params['exposure_new']
        conf_new = model_exp_new_result.conf_int().loc['exposure_new']
        ax_new.errorbar(x=[0], y=[coef_new], yerr=[[coef_new - conf_new[0]], [conf_new[1] - coef_new]],
                        fmt='o', capsize=5, color='blue', label='Effect of New Clinic Exposure')
        ax_new.axhline(0, color='black', linestyle='--')
        ax_new.set_xticks([])
        ax_new.set_ylabel(f"Effect on {OUTCOME}")
        ax_new.set_title("Effect of Weighted 'New' Exposure")
        ax_new.text(0, coef_new, f' Coef={coef_new:.3f}\n P={model_exp_new_result.pvalues["exposure_new"]:.3f}', va='bottom', ha='center')
    else:
        ax_new.text(0.5, 0.5, "Model Failed", ha='center', va='center')
        ax_new.set_title("Effect of Weighted 'New' Exposure")


    # График для Acquired
    if model_exp_acq_result:
        coef_acq = model_exp_acq_result.params['exposure_acquired']
        conf_acq = model_exp_acq_result.conf_int().loc['exposure_acquired']
        ax_acq.errorbar(x=[0], y=[coef_acq], yerr=[[coef_acq - conf_acq[0]], [conf_acq[1] - coef_acq]],
                        fmt='s', capsize=5, color='red', label='Effect of Acquired Clinic Exposure')
        ax_acq.axhline(0, color='black', linestyle='--')
        ax_acq.set_xticks([])
        ax_acq.set_ylabel(f"Effect on {OUTCOME}")
        ax_acq.set_title("Effect of Weighted 'Acquired' Exposure")
        ax_acq.text(0, coef_acq, f' Coef={coef_acq:.3f}\n P={model_exp_acq_result.pvalues["exposure_acquired"]:.3f}', va='bottom', ha='center')
    else:
        ax_acq.text(0.5, 0.5, "Model Failed", ha='center', va='center')
        ax_acq.set_title("Effect of Weighted 'Acquired' Exposure")

    plt.tight_layout()
    plt.show()

    print("\n🎉 --- Анализ с Взвешенным Воздействием завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Используем обычный tqdm

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО АНАЛИЗА С ВЗВЕШЕННЫМ ВОЗДЕЙСТВИЕМ (Exposure Score) - Исправлено ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ОШИБКА: Не могу найти файл {e.filename}. Останавливаюсь.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_clinics)} New клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Шаг 1-4: Расчет Exposure Score ---
    print("\n--- Расчет Exposure Score (Может занять несколько минут) ---")
    df_zip_months = final_data[['Filtered_Patient_ZipCode', 'Date']].drop_duplicates()
    df_zip_months['Date'] = pd.to_datetime(df_zip_months['Date'])
    df_zip_months['month_period'] = df_zip_months['Date'].dt.to_period('M')
    df_zip_months = df_zip_months.merge(zip_coords.rename(columns={'IntPtLat': 'zip_lat', 'IntPtLon': 'zip_lon'}),
                                         left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    def calculate_exposure(zip_row, clinics_df):
        exposure = 0.0
        zip_coords_tuple = (zip_row['zip_lat'], zip_row['zip_lon'])
        current_month = zip_row['month_period']
        open_clinics = clinics_df[clinics_df['open_month'] <= current_month]
        if not open_clinics.empty:
            for _, clinic_row in open_clinics.iterrows():
                clinic_coords_tuple = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
                distance_km = geodesic(zip_coords_tuple, clinic_coords_tuple).kilometers
                weight = 1.0 / (1.0 + distance_km + 1e-6)
                exposure += weight
        return exposure
    tqdm.pandas(desc="Расчет exposure_new")
    df_zip_months['exposure_new'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_new_clinics), axis=1)
    tqdm.pandas(desc="Расчет exposure_acquired")
    df_zip_months['exposure_acquired'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_acquired_pc_clinics), axis=1)
    print("✅ Exposure scores рассчитаны.")

    # --- 5. Агрегация данных и слияние ---
    df_panel = final_data.copy()
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg_exp = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
    ).reset_index()
    df_agg_exp['Y_Ratio'] = np.log1p(df_agg_exp['Total_Emergency_Visits']) - np.log1p(df_agg_exp['Total_Regular_Visits'])
    df_agg_exp['Date'] = pd.to_datetime(df_agg_exp['Year_Month'] + '-01')
    df_analysis_exp = df_agg_exp.merge(
        df_zip_months[['Filtered_Patient_ZipCode', 'Date', 'exposure_new', 'exposure_acquired']],
        on=['Filtered_Patient_ZipCode', 'Date'],
        how='left'
    )
    df_analysis_exp['exposure_new'] = df_analysis_exp['exposure_new'].fillna(0)
    df_analysis_exp['exposure_acquired'] = df_analysis_exp['exposure_acquired'].fillna(0)
    df_analysis_exp['Month_Cat'] = df_analysis_exp['Date'].dt.month
    df_analysis_exp['Year_Cat'] = df_analysis_exp['Date'].dt.year
    df_analysis_exp = df_analysis_exp.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    print("✅ Данные агрегированы и объединены с exposure scores.")

    # --- 6. Запуск Регрессий ---
    OUTCOME = "Y_Ratio"
    FIXED_EFFECTS = "C(Zip) + C(Year_Cat) + C(Month_Cat)"
    model_exp_new_result = None; model_exp_acq_result = None

    # Модель 1: Эффект New Exposure
    formula_exp_new = f"{OUTCOME} ~ exposure_new + {FIXED_EFFECTS}"
    print("\n--- Запуск Модели 1 (Эффект exposure_new) ---")
    try:
        # --- ИСПРАВЛЕНИЕ: Удаляем NaN и сбрасываем индекс ---
        df_model1_clean = df_analysis_exp[['Zip', 'Year_Cat', 'Month_Cat', OUTCOME, 'exposure_new']].dropna().reset_index(drop=True)
        print(f"  Используется {len(df_model1_clean)} наблюдений для Модели 1.")
        model = smf.ols(formula_exp_new, data=df_model1_clean)
        # Передаем очищенный массив Zip для кластеризации
        model_exp_new_result = model.fit(cov_type='cluster', cov_kwds={'groups': df_model1_clean['Zip'].values})
        print("✅ Модель 1 рассчитана.")
        print("\nРезультат для exposure_new:")
        print(model_exp_new_result.summary().tables[1].as_html())
    except Exception as e:
        print(f"❌ ОШИБКА при расчете Модели 1: {e}")


    # Модель 2: Эффект Acquired Exposure
    formula_exp_acq = f"{OUTCOME} ~ exposure_acquired + {FIXED_EFFECTS}"
    print("\n--- Запуск Модели 2 (Эффект exposure_acquired) ---")
    try:
        # --- ИСПРАВЛЕНИЕ: Удаляем NaN и сбрасываем индекс ---
        df_model2_clean = df_analysis_exp[['Zip', 'Year_Cat', 'Month_Cat', OUTCOME, 'exposure_acquired']].dropna().reset_index(drop=True)
        print(f"  Используется {len(df_model2_clean)} наблюдений для Модели 2.")
        model = smf.ols(formula_exp_acq, data=df_model2_clean)
        # Передаем очищенный массив Zip для кластеризации
        model_exp_acq_result = model.fit(cov_type='cluster', cov_kwds={'groups': df_model2_clean['Zip'].values})
        print("✅ Модель 2 рассчитана.")
        print("\nРезультат для exposure_acquired:")
        print(model_exp_acq_result.summary().tables[1].as_html())
    except Exception as e:
        print(f"❌ ОШИБКА при расчете Модели 2: {e}")


    # --- 7. Визуализация Коэффициентов ---
    # ... (код визуализации без изменений) ...
    print("\n--- Визуализация Коэффициентов ---")
    sns.set_style("whitegrid")
    fig_coeffs, (ax_new, ax_acq) = plt.subplots(nrows=1, ncols=2, figsize=(12, 5), sharey=True)

    # График для New
    ax_new.set_title("Эффект Взвешенного Воздействия ('New')")
    if model_exp_new_result:
        coef_new = model_exp_new_result.params['exposure_new']
        conf_new = model_exp_new_result.conf_int().loc['exposure_new']
        p_val_new = model_exp_new_result.pvalues["exposure_new"]
        ax_new.errorbar(x=[0], y=[coef_new], yerr=[[coef_new - conf_new[0]], [conf_new[1] - coef_new]], fmt='o', capsize=5, color='blue')
        ax_new.axhline(0, color='black', linestyle='--'); ax_new.set_xticks([]); ax_new.set_ylabel(f"Эффект на {OUTCOME}")
        ax_new.text(0.05, 0.95, f' Коэфф.={coef_new:.3f}\n P={p_val_new:.3f}', transform=ax_new.transAxes, ha='left', va='top', fontsize=10, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
    else: ax_new.text(0.5, 0.5, "Модель не сошлась", ha='center', va='center')

    # График для Acquired
    ax_acq.set_title("Эффект Взвешенного Воздействия ('Acquired')")
    if model_exp_acq_result:
        coef_acq = model_exp_acq_result.params['exposure_acquired']
        conf_acq = model_exp_acq_result.conf_int().loc['exposure_acquired']
        p_val_acq = model_exp_acq_result.pvalues["exposure_acquired"]
        ax_acq.errorbar(x=[0], y=[coef_acq], yerr=[[coef_acq - conf_acq[0]], [conf_acq[1] - coef_acq]], fmt='s', capsize=5, color='red')
        ax_acq.axhline(0, color='black', linestyle='--'); ax_acq.set_xticks([])
        ax_acq.text(0.05, 0.95, f' Коэфф.={coef_acq:.3f}\n P={p_val_acq:.3f}', transform=ax_acq.transAxes, ha='left', va='top', fontsize=10, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
    else: ax_acq.text(0.5, 0.5, "Модель не сошлась", ha='center', va='center')

    plt.tight_layout(); plt.show()
    print("\n🎉 --- Анализ с Взвешенным Воздействием завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Используем обычный tqdm

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО АНАЛИЗА С ВЗВЕШЕННЫМ ВОЗДЕЙСТВИЕМ (Exposure Score) - ДИАГНОСТИКА ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ОШИБКА: Не могу найти файл {e.filename}. Останавливаюсь.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS'])
    zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce')
    zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_clinics)} New клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Шаг 1-4: Расчет Exposure Score ---
    print("\n--- Расчет Exposure Score (Может занять несколько минут) ---")
    df_zip_months = final_data[['Filtered_Patient_ZipCode', 'Date']].drop_duplicates()
    df_zip_months['Date'] = pd.to_datetime(df_zip_months['Date'])
    df_zip_months['month_period'] = df_zip_months['Date'].dt.to_period('M')
    df_zip_months = df_zip_months.merge(zip_coords.rename(columns={'IntPtLat': 'zip_lat', 'IntPtLon': 'zip_lon'}),
                                         left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    def calculate_exposure(zip_row, clinics_df):
        exposure = 0.0
        zip_coords_tuple = (zip_row['zip_lat'], zip_row['zip_lon'])
        current_month = zip_row['month_period']
        open_clinics = clinics_df[clinics_df['open_month'] <= current_month]
        if not open_clinics.empty:
            for _, clinic_row in open_clinics.iterrows():
                clinic_coords_tuple = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
                distance_km = geodesic(zip_coords_tuple, clinic_coords_tuple).kilometers
                weight = 1.0 / (1.0 + distance_km + 1e-6)
                exposure += weight
        return exposure
    tqdm.pandas(desc="Расчет exposure_new")
    df_zip_months['exposure_new'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_new_clinics), axis=1)
    tqdm.pandas(desc="Расчет exposure_acquired")
    df_zip_months['exposure_acquired'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_acquired_pc_clinics), axis=1)
    print("✅ Exposure scores рассчитаны.")

    # --- 5. Агрегация данных и слияние ---
    df_panel = final_data.copy()
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg_exp = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
    ).reset_index()
    df_agg_exp['Y_Ratio'] = np.log1p(df_agg_exp['Total_Emergency_Visits']) - np.log1p(df_agg_exp['Total_Regular_Visits'])
    df_agg_exp['Date'] = pd.to_datetime(df_agg_exp['Year_Month'] + '-01')
    df_analysis_exp = df_agg_exp.merge(
        df_zip_months[['Filtered_Patient_ZipCode', 'Date', 'exposure_new', 'exposure_acquired']],
        on=['Filtered_Patient_ZipCode', 'Date'],
        how='left'
    )
    df_analysis_exp['exposure_new'] = df_analysis_exp['exposure_new'].fillna(0)
    df_analysis_exp['exposure_acquired'] = df_analysis_exp['exposure_acquired'].fillna(0)
    df_analysis_exp['Month_Cat'] = df_analysis_exp['Date'].dt.month # Используем число
    df_analysis_exp['Year_Cat'] = df_analysis_exp['Date'].dt.year   # Используем число
    df_analysis_exp = df_analysis_exp.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    print("✅ Данные агрегированы и объединены с exposure scores.")

    # --- 6. Запуск Регрессий с ДИАГНОСТИКОЙ ---
    OUTCOME = "Y_Ratio"
    # --- ИСПРАВЛЕНИЕ: Преобразуем FE в строки, чтобы избежать проблем с NaN/категориями ---
    df_analysis_exp['Zip'] = df_analysis_exp['Zip'].astype(str)
    df_analysis_exp['Year_Cat_Str'] = df_analysis_exp['Year_Cat'].astype(str)
    df_analysis_exp['Month_Cat_Str'] = df_analysis_exp['Month_Cat'].astype(str)
    
    FIXED_EFFECTS = "C(Zip) + C(Year_Cat_Str) + C(Month_Cat_Str)" # Используем строковые версии
    model_exp_new_result = None; model_exp_acq_result = None

    # --- Модель 1: Эффект New Exposure ---
    formula_exp_new = f"{OUTCOME} ~ exposure_new + {FIXED_EFFECTS}"
    print("\n--- Запуск Модели 1 (Эффект exposure_new) ---")
    try:
        # Явно удаляем NaN по ВСЕМ переменным формулы
        cols_for_model1 = ['Zip', 'Year_Cat_Str', 'Month_Cat_Str', OUTCOME, 'exposure_new']
        df_model1_clean = df_analysis_exp[cols_for_model1].dropna().reset_index(drop=True)
        
        # --- ДИАГНОСТИКА ---
        print(f"  --- Диагностика для Модели 1 ---")
        print(f"  Размер данных ПОСЛЕ dropna: {df_model1_clean.shape}")
        print(f"  Кол-во уникальных Zip в данных: {df_model1_clean['Zip'].nunique()}")
        groups_array1 = df_model1_clean['Zip'].values
        print(f"  Длина массива groups для кластеризации: {len(groups_array1)}")
        print(f"  Проверка на NaN в группах: {pd.isna(groups_array1).sum()}")
        # ------------------

        if df_model1_clean.empty or len(groups_array1) == 0:
             print("  ⚠️ ОШИБКА: Нет данных после очистки NaN для Модели 1.")
        else:
            model = smf.ols(formula_exp_new, data=df_model1_clean)
            model_exp_new_result = model.fit(cov_type='cluster', cov_kwds={'groups': groups_array1}) # Передаем массив
            print("✅ Модель 1 рассчитана.")
            print("\nРезультат для exposure_new:")
            print(model_exp_new_result.summary().tables[1].as_html())

    except ValueError as ve:
         print(f"❌ ОШИБКА ValueError при расчете Модели 1: {ve}")
         if "must be associated with non-singleton clusters" in str(ve):
              print("   -> Проблема: Некоторые ZIP-коды имеют только одно наблюдение после очистки.")
              # Попробуем без кластеризации как запасной вариант
              try:
                   print("      ... Попытка запуска без кластеризации ...")
                   model = smf.ols(formula_exp_new, data=df_model1_clean)
                   model_exp_new_result = model.fit() # Обычный fit
                   print("      ✅ Модель 1 без кластеризации рассчитана (станд. ошибки могут быть некорректны!).")
                   print("\nРезультат для exposure_new (БЕЗ кластеризации):")
                   print(model_exp_new_result.summary().tables[1].as_html())
              except Exception as e2:
                   print(f"      ❌ ОШИБКА при запуске без кластеризации: {e2}")

         elif "The weights and list don't have the same length" in str(ve):
              print("   -> Проблема: Несоответствие длины все еще присутствует. Возможно, OLS внутренне удаляет еще строки.")
         else:
              print(f"   -> Неожиданная ValueError: {ve}")

    except Exception as e:
        print(f"❌ НЕОЖИДАННАЯ ОШИБКА при расчете Модели 1: {e}")


    # --- Модель 2: Эффект Acquired Exposure ---
    formula_exp_acq = f"{OUTCOME} ~ exposure_acquired + {FIXED_EFFECTS}"
    print("\n--- Запуск Модели 2 (Эффект exposure_acquired) ---")
    try:
        # Явно удаляем NaN по ВСЕМ переменным формулы
        cols_for_model2 = ['Zip', 'Year_Cat_Str', 'Month_Cat_Str', OUTCOME, 'exposure_acquired']
        df_model2_clean = df_analysis_exp[cols_for_model2].dropna().reset_index(drop=True)

        # --- ДИАГНОСТИКА ---
        print(f"  --- Диагностика для Модели 2 ---")
        print(f"  Размер данных ПОСЛЕ dropna: {df_model2_clean.shape}")
        print(f"  Кол-во уникальных Zip в данных: {df_model2_clean['Zip'].nunique()}")
        groups_array2 = df_model2_clean['Zip'].values
        print(f"  Длина массива groups для кластеризации: {len(groups_array2)}")
        print(f"  Проверка на NaN в группах: {pd.isna(groups_array2).sum()}")
        # ------------------

        if df_model2_clean.empty or len(groups_array2) == 0:
             print("  ⚠️ ОШИБКА: Нет данных после очистки NaN для Модели 2.")
        else:
            model = smf.ols(formula_exp_acq, data=df_model2_clean)
            model_exp_acq_result = model.fit(cov_type='cluster', cov_kwds={'groups': groups_array2}) # Передаем массив
            print("✅ Модель 2 рассчитана.")
            print("\nРезультат для exposure_acquired:")
            print(model_exp_acq_result.summary().tables[1].as_html())

    except ValueError as ve:
         print(f"❌ ОШИБКА ValueError при расчете Модели 2: {ve}")
         if "must be associated with non-singleton clusters" in str(ve):
              print("   -> Проблема: Некоторые ZIP-коды имеют только одно наблюдение после очистки.")
              # Попробуем без кластеризации
              try:
                   print("      ... Попытка запуска без кластеризации ...")
                   model = smf.ols(formula_exp_acq, data=df_model2_clean)
                   model_exp_acq_result = model.fit()
                   print("      ✅ Модель 2 без кластеризации рассчитана (станд. ошибки могут быть некорректны!).")
                   print("\nРезультат для exposure_acquired (БЕЗ кластеризации):")
                   print(model_exp_acq_result.summary().tables[1].as_html())
              except Exception as e2:
                   print(f"      ❌ ОШИБКА при запуске без кластеризации: {e2}")

         elif "The weights and list don't have the same length" in str(ve):
              print("   -> Проблема: Несоответствие длины все еще присутствует.")
         else:
              print(f"   -> Неожиданная ValueError: {ve}")

    except Exception as e:
        print(f"❌ НЕОЖИДАННАЯ ОШИБКА при расчете Модели 2: {e}")


    # --- 7. Визуализация Коэффициентов ---
    # ... (код визуализации без изменений) ...
    print("\n--- Визуализация Коэффициентов ---")
    sns.set_style("whitegrid")
    fig_coeffs, (ax_new, ax_acq) = plt.subplots(nrows=1, ncols=2, figsize=(12, 5), sharey=True)
    # График для New
    ax_new.set_title("Эффект Взвешенного Воздействия ('New')")
    if model_exp_new_result:
        # Проверяем наличие 'exposure_new' в параметрах (на случай запасной модели без кластера)
        if 'exposure_new' in model_exp_new_result.params:
             coef_new = model_exp_new_result.params['exposure_new']
             conf_new = model_exp_new_result.conf_int().loc['exposure_new']
             p_val_new = model_exp_new_result.pvalues["exposure_new"]
             ax_new.errorbar(x=[0], y=[coef_new], yerr=[[coef_new - conf_new[0]], [conf_new[1] - coef_new]], fmt='o', capsize=5, color='blue')
             ax_new.text(0.05, 0.95, f' Коэфф.={coef_new:.3f}\n P={p_val_new:.3f}', transform=ax_new.transAxes, ha='left', va='top', fontsize=10, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
        else: ax_new.text(0.5, 0.5, "Коэфф. не найден", ha='center', va='center')
        ax_new.axhline(0, color='black', linestyle='--'); ax_new.set_xticks([]); ax_new.set_ylabel(f"Эффект на {OUTCOME}")
    else: ax_new.text(0.5, 0.5, "Модель не сошлась", ha='center', va='center')
    # График для Acquired
    ax_acq.set_title("Эффект Взвешенного Воздействия ('Acquired')")
    if model_exp_acq_result:
         if 'exposure_acquired' in model_exp_acq_result.params:
             coef_acq = model_exp_acq_result.params['exposure_acquired']
             conf_acq = model_exp_acq_result.conf_int().loc['exposure_acquired']
             p_val_acq = model_exp_acq_result.pvalues["exposure_acquired"]
             ax_acq.errorbar(x=[0], y=[coef_acq], yerr=[[coef_acq - conf_acq[0]], [conf_acq[1] - coef_acq]], fmt='s', capsize=5, color='red')
             ax_acq.text(0.05, 0.95, f' Коэфф.={coef_acq:.3f}\n P={p_val_acq:.3f}', transform=ax_acq.transAxes, ha='left', va='top', fontsize=10, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
         else: ax_acq.text(0.5, 0.5, "Коэфф. не найден", ha='center', va='center')
         ax_acq.axhline(0, color='black', linestyle='--'); ax_acq.set_xticks([])
    else: ax_acq.text(0.5, 0.5, "Модель не сошлась", ha='center', va='center')

    plt.tight_layout(); plt.show()
    print("\n🎉 --- Анализ с Взвешенным Воздействием завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Используем обычный tqdm

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО АНАЛИЗА С ВЗВЕШЕННЫМ ВОЗДЕЙСТВИЕМ (Exposure Score) - ДИАГНОСТИКА v2 ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ОШИБКА: Не могу найти файл {e.filename}. Останавливаюсь.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy()
    zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS']); zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ---
    df_new_clinics = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_clinics)} New клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Шаг 1-4: Расчет Exposure Score ---
    print("\n--- Расчет Exposure Score (Может занять несколько минут) ---")
    df_zip_months = final_data[['Filtered_Patient_ZipCode', 'Date']].drop_duplicates()
    df_zip_months['Date'] = pd.to_datetime(df_zip_months['Date'])
    df_zip_months['month_period'] = df_zip_months['Date'].dt.to_period('M')
    df_zip_months = df_zip_months.merge(zip_coords.rename(columns={'IntPtLat': 'zip_lat', 'IntPtLon': 'zip_lon'}),
                                         left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    def calculate_exposure(zip_row, clinics_df):
        exposure = 0.0; zip_coords_tuple = (zip_row['zip_lat'], zip_row['zip_lon']); current_month = zip_row['month_period']
        open_clinics = clinics_df[clinics_df['open_month'] <= current_month]
        if not open_clinics.empty:
            for _, clinic_row in open_clinics.iterrows():
                clinic_coords_tuple = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
                distance_km = geodesic(zip_coords_tuple, clinic_coords_tuple).kilometers
                weight = 1.0 / (1.0 + distance_km + 1e-6); exposure += weight
        return exposure
    tqdm.pandas(desc="Расчет exposure_new"); df_zip_months['exposure_new'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_new_clinics), axis=1)
    tqdm.pandas(desc="Расчет exposure_acquired"); df_zip_months['exposure_acquired'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_acquired_pc_clinics), axis=1)
    print("✅ Exposure scores рассчитаны.")

    # --- 5. Агрегация данных и слияние ---
    df_panel = final_data.copy(); df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int); df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg_exp = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
    ).reset_index()
    df_agg_exp['Y_Ratio'] = np.log1p(df_agg_exp['Total_Emergency_Visits']) - np.log1p(df_agg_exp['Total_Regular_Visits'])
    df_agg_exp['Date'] = pd.to_datetime(df_agg_exp['Year_Month'] + '-01')
    df_analysis_exp = df_agg_exp.merge(df_zip_months[['Filtered_Patient_ZipCode', 'Date', 'exposure_new', 'exposure_acquired']], on=['Filtered_Patient_ZipCode', 'Date'], how='left')
    df_analysis_exp['exposure_new'] = df_analysis_exp['exposure_new'].fillna(0); df_analysis_exp['exposure_acquired'] = df_analysis_exp['exposure_acquired'].fillna(0)
    df_analysis_exp['Month_Cat_Str'] = df_analysis_exp['Month'].astype(str); df_analysis_exp['Year_Cat_Str'] = df_analysis_exp['Year'].astype(str)
    df_analysis_exp = df_analysis_exp.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    df_analysis_exp['Zip'] = df_analysis_exp['Zip'].astype(str) # Убедимся, что Zip - строка
    print("✅ Данные агрегированы и объединены с exposure scores.")

    # --- 6. Запуск Регрессий с ДИАГНОСТИКОЙ ---
    OUTCOME = "Y_Ratio"
    FIXED_EFFECTS = "C(Zip) + C(Year_Cat_Str) + C(Month_Cat_Str)"
    model_exp_new_result_clustered = None; model_exp_acq_result_clustered = None

    # --- Модель 1: Эффект New Exposure ---
    formula_exp_new = f"{OUTCOME} ~ exposure_new + {FIXED_EFFECTS}"
    print("\n--- Запуск Модели 1 (Эффект exposure_new) ---")
    try:
        cols_for_model1 = ['Zip', 'Year_Cat_Str', 'Month_Cat_Str', OUTCOME, 'exposure_new']
        df_model1_clean = df_analysis_exp[cols_for_model1].dropna().reset_index(drop=True)
        groups_array1 = df_model1_clean['Zip'].values
        
        print(f"  Данные ДО fit(): {df_model1_clean.shape[0]} строк, {len(groups_array1)} групп.")

        # Сначала запускаем БЕЗ кластеризации, чтобы получить model_fit_no_cluster
        print("  ... Запуск без кластеризации для диагностики ...")
        model_no_cluster1 = smf.ols(formula_exp_new, data=df_model1_clean)
        model_fit_no_cluster1 = model_no_cluster1.fit()
        
        # --- ДИАГНОСТИКА ---
        print(f"  --- Диагностика для Модели 1 ---")
        df_used_by_model1 = model_fit_no_cluster1.model.data.frame # Данные, которые реально использовались
        n_obs_used1 = len(df_used_by_model1)
        print(f"  Размер данных, ФАКТИЧЕСКИ использованных OLS: {n_obs_used1} строк")
        
        # Получаем группы Zip из ФАКТИЧЕСКИ использованных данных
        groups_actually_used1 = df_used_by_model1['Zip'].values
        n_groups_actually_used1 = len(groups_actually_used1)
        print(f"  Длина массива групп из ФАКТИЧЕСКИ использованных данных: {n_groups_actually_used1}")
        
        # Сравниваем длины
        if n_obs_used1 == len(groups_array1):
            print("  ✅ Длины совпадают! Попытка кластеризации...")
            # Передаем массив групп из ОЧИЩЕННЫХ данных (groups_array1)
            model_exp_new_result_clustered = model_no_cluster1.fit(cov_type='cluster', cov_kwds={'groups': groups_array1})
            print("     ✅ Модель 1 с кластеризацией рассчитана.")
            print("\nРезультат для exposure_new (с кластеризацией):")
            print(model_exp_new_result_clustered.summary().tables[1].as_html())
        else:
            print(f"  ❌ ОШИБКА: Несоответствие длин! Данные до fit: {len(groups_array1)}, Данные после fit: {n_obs_used1}")
            print("     -> OLS удалил строки из-за фикс. эффектов или коллинеарности.")
            print("     -> Запускаем без кластеризации (станд. ошибки могут быть некорректны!).")
            model_exp_new_result_clustered = model_fit_no_cluster1 # Используем результат без кластера
            print("\nРезультат для exposure_new (БЕЗ кластеризации):")
            print(model_exp_new_result_clustered.summary().tables[1].as_html())
            
            # --- Дополнительная диагностика синглтонов ---
            print("\n     --- Проверка на синглтоны в ИСПОЛЬЗОВАННЫХ данных ---")
            zip_counts = df_used_by_model1['Zip'].value_counts()
            singletons = zip_counts[zip_counts == 1]
            if not singletons.empty:
                print(f"     Найдено {len(singletons)} ZIP-кодов с одним наблюдением (синглтоны):")
                # print(singletons.index.tolist()) # Раскомментируйте, чтобы увидеть список ZIP-кодов
            else:
                print("     Синглтонов не найдено.")
            # ---------------------------------------------


    except Exception as e:
        print(f"❌ НЕОЖИДАННАЯ ОШИБКА при расчете Модели 1: {e}")


    # --- Модель 2: Эффект Acquired Exposure ---
    formula_exp_acq = f"{OUTCOME} ~ exposure_acquired + {FIXED_EFFECTS}"
    print("\n--- Запуск Модели 2 (Эффект exposure_acquired) ---")
    try:
        cols_for_model2 = ['Zip', 'Year_Cat_Str', 'Month_Cat_Str', OUTCOME, 'exposure_acquired']
        df_model2_clean = df_analysis_exp[cols_for_model2].dropna().reset_index(drop=True)
        groups_array2 = df_model2_clean['Zip'].values
        
        print(f"  Данные ДО fit(): {df_model2_clean.shape[0]} строк, {len(groups_array2)} групп.")

        # Сначала без кластеризации
        print("  ... Запуск без кластеризации для диагностики ...")
        model_no_cluster2 = smf.ols(formula_exp_acq, data=df_model2_clean)
        model_fit_no_cluster2 = model_no_cluster2.fit()
        
        # --- ДИАГНОСТИКА ---
        print(f"  --- Диагностика для Модели 2 ---")
        df_used_by_model2 = model_fit_no_cluster2.model.data.frame
        n_obs_used2 = len(df_used_by_model2)
        print(f"  Размер данных, ФАКТИЧЕСКИ использованных OLS: {n_obs_used2} строк")
        groups_actually_used2 = df_used_by_model2['Zip'].values
        n_groups_actually_used2 = len(groups_actually_used2)
        print(f"  Длина массива групп из ФАКТИЧЕСКИ использованных данных: {n_groups_actually_used2}")

        if n_obs_used2 == len(groups_array2):
            print("  ✅ Длины совпадают! Попытка кластеризации...")
            model_exp_acq_result_clustered = model_no_cluster2.fit(cov_type='cluster', cov_kwds={'groups': groups_array2})
            print("     ✅ Модель 2 с кластеризацией рассчитана.")
            print("\nРезультат для exposure_acquired (с кластеризацией):")
            print(model_exp_acq_result_clustered.summary().tables[1].as_html())
        else:
            print(f"  ❌ ОШИБКА: Несоответствие длин! Данные до fit: {len(groups_array2)}, Данные после fit: {n_obs_used2}")
            print("     -> OLS удалил строки.")
            print("     -> Запускаем без кластеризации.")
            model_exp_acq_result_clustered = model_fit_no_cluster2
            print("\nРезультат для exposure_acquired (БЕЗ кластеризации):")
            print(model_exp_acq_result_clustered.summary().tables[1].as_html())
            
            # --- Дополнительная диагностика синглтонов ---
            print("\n     --- Проверка на синглтоны в ИСПОЛЬЗОВАННЫХ данных ---")
            zip_counts2 = df_used_by_model2['Zip'].value_counts()
            singletons2 = zip_counts2[zip_counts2 == 1]
            if not singletons2.empty:
                print(f"     Найдено {len(singletons2)} ZIP-кодов с одним наблюдением (синглтоны):")
            else:
                print("     Синглтонов не найдено.")
            # ---------------------------------------------

    except Exception as e:
        print(f"❌ НЕОЖИДАННАЯ ОШИБКА при расчете Модели 2: {e}")


    # --- 7. Визуализация Коэффициентов ---
    # Используем результаты _clustered (которые могут быть без кластера, если он не сработал)
    print("\n--- Визуализация Коэффициентов ---")
    sns.set_style("whitegrid")
    fig_coeffs, (ax_new, ax_acq) = plt.subplots(nrows=1, ncols=2, figsize=(12, 5), sharey=True)

    # График для New
    ax_new.set_title("Эффект Взвешенного Воздействия ('New')")
    if model_exp_new_result_clustered:
        if 'exposure_new' in model_exp_new_result_clustered.params:
            coef_new = model_exp_new_result_clustered.params['exposure_new']
            conf_new = model_exp_new_result_clustered.conf_int().loc['exposure_new']
            p_val_new = model_exp_new_result_clustered.pvalues["exposure_new"]
            ax_new.errorbar(x=[0], y=[coef_new], yerr=[[coef_new - conf_new[0]], [conf_new[1] - conf_new]], fmt='o', capsize=5, color='blue')
            ax_new.text(0.05, 0.95, f' Коэфф.={coef_new:.3f}\n P={p_val_new:.3f}', transform=ax_new.transAxes, ha='left', va='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
            # Добавим подпись о кластеризации
            if hasattr(model_exp_new_result_clustered, 'cov_type') and 'cluster' in model_exp_new_result_clustered.cov_type.lower():
                 ax_new.text(0.05, 0.05, 'Кластер. SE', transform=ax_new.transAxes, ha='left', va='bottom', fontsize=8, alpha=0.7)
            else:
                 ax_new.text(0.05, 0.05, 'Без кластера!', transform=ax_new.transAxes, ha='left', va='bottom', fontsize=8, color='orange')

        else: ax_new.text(0.5, 0.5, "Коэфф. не найден", ha='center', va='center')
        ax_new.axhline(0, color='black', linestyle='--'); ax_new.set_xticks([]); ax_new.set_ylabel(f"Эффект на {OUTCOME}")
    else: ax_new.text(0.5, 0.5, "Модель не сошлась", ha='center', va='center')

    # График для Acquired
    ax_acq.set_title("Эффект Взвешенного Воздействия ('Acquired')")
    if model_exp_acq_result_clustered:
         if 'exposure_acquired' in model_exp_acq_result_clustered.params:
             coef_acq = model_exp_acq_result_clustered.params['exposure_acquired']
             conf_acq = model_exp_acq_result_clustered.conf_int().loc['exposure_acquired']
             p_val_acq = model_exp_acq_result_clustered.pvalues["exposure_acquired"]
             ax_acq.errorbar(x=[0], y=[coef_acq], yerr=[[coef_acq - conf_acq[0]], [conf_acq[1] - coef_acq]], fmt='s', capsize=5, color='red')
             ax_acq.text(0.05, 0.95, f' Коэфф.={coef_acq:.3f}\n P={p_val_acq:.3f}', transform=ax_acq.transAxes, ha='left', va='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
             # Добавим подпись о кластеризации
             if hasattr(model_exp_acq_result_clustered, 'cov_type') and 'cluster' in model_exp_acq_result_clustered.cov_type.lower():
                  ax_acq.text(0.05, 0.05, 'Кластер. SE', transform=ax_acq.transAxes, ha='left', va='bottom', fontsize=8, alpha=0.7)
             else:
                  ax_acq.text(0.05, 0.05, 'Без кластера!', transform=ax_acq.transAxes, ha='left', va='bottom', fontsize=8, color='orange')
         else: ax_acq.text(0.5, 0.5, "Коэфф. не найден", ha='center', va='center')
         ax_acq.axhline(0, color='black', linestyle='--'); ax_acq.set_xticks([])
    else: ax_acq.text(0.5, 0.5, "Модель не сошлась", ha='center', va='center')

    plt.tight_layout(); plt.show()
    print("\n🎉 --- Анализ с Взвешенным Воздействием завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import warnings

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🩺 ДИАГНОСТИКА ОШИБКИ КЛАСТЕРИЗАЦИИ ---")

# --- 1. Проверка наличия df_analysis_exp ---
if 'df_analysis_exp' not in locals():
    print("❌ ОШИБКА: DataFrame 'df_analysis_exp' не найден.")
    print("   Пожалуйста, убедитесь, что ячейка с расчетом exposure score была успешно выполнена.")
else:
    print("✅ DataFrame 'df_analysis_exp' найден.")

    # --- 2. Подготовка к Диагностике ---
    OUTCOME = "Y_Ratio"
    # Убедимся, что FE - строки
    if not pd.api.types.is_string_dtype(df_analysis_exp['Zip']):
         df_analysis_exp['Zip'] = df_analysis_exp['Zip'].astype(str)
    if 'Year_Cat_Str' not in df_analysis_exp.columns or not pd.api.types.is_string_dtype(df_analysis_exp['Year_Cat_Str']):
         df_analysis_exp['Year_Cat_Str'] = df_analysis_exp['Year_Cat'].astype(str)
    if 'Month_Cat_Str' not in df_analysis_exp.columns or not pd.api.types.is_string_dtype(df_analysis_exp['Month_Cat_Str']):
         df_analysis_exp['Month_Cat_Str'] = df_analysis_exp['Month_Cat'].astype(str)

    FIXED_EFFECTS = "C(Zip) + C(Year_Cat_Str) + C(Month_Cat_Str)"

    # --- 3. Диагностика для Модели 1 (exposure_new) ---
    formula_exp_new = f"{OUTCOME} ~ exposure_new + {FIXED_EFFECTS}"
    print(f"\n--- Диагностика для Модели 1: {formula_exp_new} ---")
    try:
        # 3.1 Подготовка данных ДО OLS
        cols_for_model1 = ['Zip', 'Year_Cat_Str', 'Month_Cat_Str', OUTCOME, 'exposure_new']
        # Создаем копию перед удалением NaN, чтобы не менять df_analysis_exp
        df_model1_input = df_analysis_exp[cols_for_model1].dropna().reset_index(drop=True)
        groups_array1 = df_model1_input['Zip'].values
        n_input_rows = len(df_model1_input)
        n_input_groups = len(groups_array1)
        print(f"  Данные ПЕРЕД OLS.fit(): {n_input_rows} строк, {n_input_groups} групп (длина массива Zip).")

        # 3.2 Запуск OLS БЕЗ кластеризации
        print("  ... Запуск OLS.fit() без кластеризации ...")
        model_no_cluster1 = smf.ols(formula_exp_new, data=df_model1_input)
        model_fit_no_cluster1 = model_no_cluster1.fit()
        print("  ... OLS.fit() завершен ...")

        # 3.3 Извлечение данных ПОСЛЕ OLS
        df_used_by_model1 = model_fit_no_cluster1.model.data.frame
        n_obs_used1 = len(df_used_by_model1)
        print(f"  Данные, ФАКТИЧЕСКИ использованные OLS: {n_obs_used1} строк")

        # 3.4 Сравнение
        if n_obs_used1 == n_input_groups:
            print("  ✅ Длины совпадают. Проблема кластеризации не в удалении строк OLS.")
            # Попробуем запустить кластеризацию еще раз на этих данных
            try:
                print("     ... Повторная попытка кластеризации ...")
                # Важно: используем ДАННЫЕ, которые были переданы в fit (df_model1_input)
                # и соответствующий массив групп (groups_array1)
                clustered_fit = model_no_cluster1.fit(cov_type='cluster', cov_kwds={'groups': groups_array1})
                print("     ✅ Кластеризация удалась!")
            except ValueError as ve_cluster:
                 print(f"     ❌ ОШИБКА ValueError при повторной кластеризации: {ve_cluster}")
                 if "must be associated with non-singleton clusters" in str(ve_cluster):
                      print("        -> Проблема: Синглтоны в данных, переданных в fit.")
                      zip_counts_input = df_model1_input['Zip'].value_counts()
                      singletons_input = zip_counts_input[zip_counts_input == 1]
                      print(f"        Найдено {len(singletons_input)} синглтонов в данных ДО fit.")
                 else:
                      print(f"        -> Неожиданная ValueError: {ve_cluster}")
            except Exception as e_cluster:
                 print(f"     ❌ НЕОЖИДАННАЯ ОШИБКА при повторной кластеризации: {e_cluster}")

        else:
            print(f"  ❌ Несоответствие длин! ДО OLS: {n_input_groups}, ПОСЛЕ OLS: {n_obs_used1}")
            rows_dropped = n_input_rows - n_obs_used1
            print(f"     -> OLS внутренне удалил {rows_dropped} строк.")

            # 3.5 Проверка на синглтоны в ИСПОЛЬЗОВАННЫХ данных
            print("\n     --- Проверка на синглтоны в ИСПОЛЬЗОВАННЫХ OLS данных ---")
            zip_counts_used = df_used_by_model1['Zip'].value_counts()
            singletons_used = zip_counts_used[zip_counts_used == 1]
            if not singletons_used.empty:
                print(f"     Найдено {len(singletons_used)} ZIP-кодов с одним наблюдением (синглтоны):")
                # print(singletons_used.index.tolist()) # Раскомментируйте для списка ZIP
            else:
                print("     Синглтонов в использованных данных не найдено.")

            # 3.6 Сравнение списков ZIP-кодов
            zips_before = set(df_model1_input['Zip'].unique())
            zips_after = set(df_used_by_model1['Zip'].unique())
            zips_dropped = zips_before - zips_after
            if zips_dropped:
                 print(f"\n     ZIP-коды, полностью удаленные OLS ({len(zips_dropped)} шт.):")
                 # print(list(zips_dropped)) # Раскомментируйте для списка ZIP
            else:
                 print("\n     OLS не удалял ZIP-коды полностью, только отдельные наблюдения.")

    except Exception as e:
        print(f"❌ НЕОЖИДАННАЯ ОШИБКА при диагностике Модели 1: {e}")


    # --- 4. Диагностика для Модели 2 (exposure_acquired) ---
    formula_exp_acq = f"{OUTCOME} ~ exposure_acquired + {FIXED_EFFECTS}"
    print(f"\n--- Диагностика для Модели 2: {formula_exp_acq} ---")
    try:
        # 4.1 Подготовка данных ДО OLS
        cols_for_model2 = ['Zip', 'Year_Cat_Str', 'Month_Cat_Str', OUTCOME, 'exposure_acquired']
        df_model2_input = df_analysis_exp[cols_for_model2].dropna().reset_index(drop=True)
        groups_array2 = df_model2_input['Zip'].values
        n_input_rows2 = len(df_model2_input)
        n_input_groups2 = len(groups_array2)
        print(f"  Данные ПЕРЕД OLS.fit(): {n_input_rows2} строк, {n_input_groups2} групп.")

        # 4.2 Запуск OLS БЕЗ кластеризации
        print("  ... Запуск OLS.fit() без кластеризации ...")
        model_no_cluster2 = smf.ols(formula_exp_acq, data=df_model2_input)
        model_fit_no_cluster2 = model_no_cluster2.fit()
        print("  ... OLS.fit() завершен ...")

        # 4.3 Извлечение данных ПОСЛЕ OLS
        df_used_by_model2 = model_fit_no_cluster2.model.data.frame
        n_obs_used2 = len(df_used_by_model2)
        print(f"  Данные, ФАКТИЧЕСКИ использованные OLS: {n_obs_used2} строк")

        # 4.4 Сравнение
        if n_obs_used2 == n_input_groups2:
            print("  ✅ Длины совпадают. Проблема кластеризации не в удалении строк OLS.")
            # Попробуем кластеризацию
            try:
                print("     ... Повторная попытка кластеризации ...")
                clustered_fit2 = model_no_cluster2.fit(cov_type='cluster', cov_kwds={'groups': groups_array2})
                print("     ✅ Кластеризация удалась!")
            except ValueError as ve_cluster2:
                 print(f"     ❌ ОШИБКА ValueError при повторной кластеризации: {ve_cluster2}")
                 if "must be associated with non-singleton clusters" in str(ve_cluster2):
                      print("        -> Проблема: Синглтоны в данных, переданных в fit.")
                      zip_counts_input2 = df_model2_input['Zip'].value_counts()
                      singletons_input2 = zip_counts_input2[zip_counts_input2 == 1]
                      print(f"        Найдено {len(singletons_input2)} синглтонов в данных ДО fit.")
                 else:
                      print(f"        -> Неожиданная ValueError: {ve_cluster2}")
            except Exception as e_cluster2:
                 print(f"     ❌ НЕОЖИДАННАЯ ОШИБКА при повторной кластеризации: {e_cluster2}")

        else:
            print(f"  ❌ Несоответствие длин! ДО OLS: {n_input_groups2}, ПОСЛЕ OLS: {n_obs_used2}")
            rows_dropped2 = n_input_rows2 - n_obs_used2
            print(f"     -> OLS внутренне удалил {rows_dropped2} строк.")

            # 4.5 Проверка на синглтоны в ИСПОЛЬЗОВАННЫХ данных
            print("\n     --- Проверка на синглтоны в ИСПОЛЬЗОВАННЫХ OLS данных ---")
            zip_counts_used2 = df_used_by_model2['Zip'].value_counts()
            singletons_used2 = zip_counts_used2[zip_counts_used2 == 1]
            if not singletons_used2.empty:
                print(f"     Найдено {len(singletons_used2)} ZIP-кодов с одним наблюдением (синглтоны):")
            else:
                print("     Синглтонов в использованных данных не найдено.")

            # 4.6 Сравнение списков ZIP-кодов
            zips_before2 = set(df_model2_input['Zip'].unique())
            zips_after2 = set(df_used_by_model2['Zip'].unique())
            zips_dropped2 = zips_before2 - zips_after2
            if zips_dropped2:
                 print(f"\n     ZIP-коды, полностью удаленные OLS ({len(zips_dropped2)} шт.).")
            else:
                 print("\n     OLS не удалял ZIP-коды полностью.")

    except Exception as e:
        print(f"❌ НЕОЖИДАННАЯ ОШИБКА при диагностике Модели 2: {e}")

    print("\n🏁 --- Диагностика завершена ---")

In [ ]:
import pandas as pd
import numpy as np
# --- ИЗМЕНЕНИЕ: Импортируем PanelOLS ---
from linearmodels.panel import PanelOLS
import statsmodels.api as sm # Для add_constant, если понадобится
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
# warnings.simplefilter(action='ignore', category=PanelOLS.CovarianceWarning) # Можно добавить, если будут предупреждения
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО АНАЛИЗА С ВЗВЕШЕННЫМ ВОЗДЕЙСТВИЕМ (linearmodels.PanelOLS) ---")

# --- 1. Проверка наличия df_analysis_exp ---
if 'df_analysis_exp' not in locals() or df_analysis_exp.empty:
    print("❌ ОШИБКА: DataFrame 'df_analysis_exp' не найден или пуст.")
    print("   Пожалуйста, убедитесь, что ячейка с расчетом exposure score была успешно выполнена.")
else:
    print("✅ DataFrame 'df_analysis_exp' найден.")

    # --- 2. Подготовка данных для PanelOLS ---
    OUTCOME = "Y_Ratio"
    df_panel_lm = df_analysis_exp.copy()

    # Убедимся, что Date - это datetime
    if not pd.api.types.is_datetime64_any_dtype(df_panel_lm['Date']):
        df_panel_lm['Date'] = pd.to_datetime(df_panel_lm['Date'])

    # --- Создаем МультиИндекс: Entity (Zip) и Time (Date) ---
    df_panel_lm = df_panel_lm.set_index(['Zip', 'Date'])

    print(f"Подготовлено {len(df_panel_lm)} наблюдений для PanelOLS.")

    # --- 3. Запуск Моделей с PanelOLS ---
    model_lm_new_result = None
    model_lm_acq_result = None

    # --- Модель 1: Эффект New Exposure ---
    print("\n--- Запуск Модели 1 (Эффект exposure_new) с PanelOLS ---")
    try:
        # Определяем экзогенные переменные (не включая FE)
        exog_vars_new = ['exposure_new']
        # Добавляем константу, PanelOLS требует ее явно, если нет тренда
        exog_new = sm.add_constant(df_panel_lm[exog_vars_new])

        # Создаем модель с Entity (Zip) и Time (Date) эффектами
        # Включаем Month и Year эффекты через TimeEffects=True
        mod_new = PanelOLS(df_panel_lm[OUTCOME], exog_new, entity_effects=True, time_effects=True, check_rank=False)

        # Расчет с кластеризованными ошибками на уровне Entity (Zip)
        model_lm_new_result = mod_new.fit(cov_type='clustered', cluster_entity=True)

        print("✅ Модель 1 (PanelOLS) рассчитана.")
        print("\nРезультат для exposure_new:")
        # Выводим только интересующий коэффициент
        print(model_lm_new_result.params.to_frame('coefficient').join(model_lm_new_result.std_errors.to_frame('std_error')).join(model_lm_new_result.pvalues.to_frame('p_value')))

    except Exception as e:
        print(f"❌ ОШИБКА при расчете Модели 1 с PanelOLS: {e}")


    # --- Модель 2: Эффект Acquired Exposure ---
    print("\n--- Запуск Модели 2 (Эффект exposure_acquired) с PanelOLS ---")
    try:
        exog_vars_acq = ['exposure_acquired']
        exog_acq = sm.add_constant(df_panel_lm[exog_vars_acq])

        mod_acq = PanelOLS(df_panel_lm[OUTCOME], exog_acq, entity_effects=True, time_effects=True, check_rank=False)
        model_lm_acq_result = mod_acq.fit(cov_type='clustered', cluster_entity=True)

        print("✅ Модель 2 (PanelOLS) рассчитана.")
        print("\nРезультат для exposure_acquired:")
        print(model_lm_acq_result.params.to_frame('coefficient').join(model_lm_acq_result.std_errors.to_frame('std_error')).join(model_lm_acq_result.pvalues.to_frame('p_value')))

    except Exception as e:
        print(f"❌ ОШИБКА при расчете Модели 2 с PanelOLS: {e}")


    # --- 4. Визуализация Коэффициентов ---
    print("\n--- Визуализация Коэффициентов (PanelOLS) ---")
    sns.set_style("whitegrid")
    fig_coeffs, (ax_new, ax_acq) = plt.subplots(nrows=1, ncols=2, figsize=(12, 5), sharey=True)

    # График для New
    ax_new.set_title("Эффект Взвешенного Воздействия ('New')")
    if model_lm_new_result and 'exposure_new' in model_lm_new_result.params:
        coef_new = model_lm_new_result.params['exposure_new']
        # Используем дов. интервал из модели
        conf_new = model_lm_new_result.conf_int().loc['exposure_new']
        p_val_new = model_lm_new_result.pvalues["exposure_new"]
        ax_new.errorbar(x=[0], y=[coef_new], yerr=[[coef_new - conf_new['lower']], [conf_new['upper'] - coef_new]],
                        fmt='o', capsize=5, color='blue')
        ax_new.axhline(0, color='black', linestyle='--'); ax_new.set_xticks([]); ax_new.set_ylabel(f"Эффект на {OUTCOME}")
        ax_new.text(0.05, 0.95, f' Коэфф.={coef_new:.3f}\n P={p_val_new:.3f}', transform=ax_new.transAxes, ha='left', va='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
        ax_new.text(0.05, 0.05, 'Кластер. SE (PanelOLS)', transform=ax_new.transAxes, ha='left', va='bottom', fontsize=8, alpha=0.7)
    else:
        ax_new.text(0.5, 0.5, "Модель не сошлась", ha='center', va='center')

    # График для Acquired
    ax_acq.set_title("Эффект Взвешенного Воздействия ('Acquired')")
    if model_lm_acq_result and 'exposure_acquired' in model_lm_acq_result.params:
        coef_acq = model_lm_acq_result.params['exposure_acquired']
        conf_acq = model_lm_acq_result.conf_int().loc['exposure_acquired']
        p_val_acq = model_lm_acq_result.pvalues["exposure_acquired"]
        ax_acq.errorbar(x=[0], y=[coef_acq], yerr=[[coef_acq - conf_acq['lower']], [conf_acq['upper'] - coef_acq]],
                        fmt='s', capsize=5, color='red')
        ax_acq.axhline(0, color='black', linestyle='--'); ax_acq.set_xticks([])
        ax_acq.text(0.05, 0.95, f' Коэфф.={coef_acq:.3f}\n P={p_val_acq:.3f}', transform=ax_acq.transAxes, ha='left', va='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
        ax_acq.text(0.05, 0.05, 'Кластер. SE (PanelOLS)', transform=ax_acq.transAxes, ha='left', va='bottom', fontsize=8, alpha=0.7)
    else:
        ax_acq.text(0.5, 0.5, "Модель не сошлась", ha='center', va='center')

    plt.tight_layout(); plt.show()
    print("\n🎉 --- Анализ с Взвешенным Воздействием (PanelOLS) завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО ФИНАЛЬНОЙ ПРОВЕРКИ ГИПОТЕЗЫ ИЗ АБСТРАКТА ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ОШИБКА: Не могу найти файл {e.filename}. Останавливаюсь.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
    print("✅ Данные загружены и очищены.")

    # --- 3. Шаг 1: Изолировать "Настоящее Лечение" (New PC/Urgent/FP Clinics) ---
    df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    
    # Ключевые слова для первичной/неотложной помощи
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine'] # Добавили Family Medicine

    def is_new_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower()
        specialty_name = str(row.get('Specialty', '')).lower()
        # Ищем точное совпадение или вхождение ключевых слов
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)

    df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_new_pc_urgent, axis=1)].copy()
    print(f"\n✅ Идентифицировано {len(df_new_PC_clinics)} 'New' PC/Urgent/FP клиник для анализа:")
    print(df_new_PC_clinics[['clinic_zip', 'Facility', 'event_date']].to_string())

    if df_new_PC_clinics.empty:
         raise ValueError("Не найдено 'New' PC/Urgent/FP клиник. Проверьте ключевые слова и данные.")


    # --- 4. Шаг 2: Пересчитать "Карту Лечения" (Только для New PC) ---
    print("\n--- Шаг 2: Пересчет Карты Лечения (только New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT
        nearby_new_pc_dates = []
        # --- Используем ОТФИЛЬТРОВАННЫЙ список df_new_PC_clinics ---
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM:
                nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates:
            earliest_date_new_pc = min(nearby_new_pc_dates)
        treatment_dates_new_pc[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc}

    df_treatment_dates_new_pc = pd.DataFrame.from_dict(treatment_dates_new_pc, orient='index')
    print(f"\n✅ Карта лечения ('New PC') создана. {len(df_treatment_dates_new_pc[df_treatment_dates_new_pc['treatment_date_NEW_PC'].notna()])} ZIP'ов обработаны.")


    # --- 5. Шаг 1 (продолжение): Агрегация данных и создание Y-переменных ---
    df_panel = final_data.copy()
    # Нужна популяция? Да, для Log(Regular Visits+1) и для контроля в модели
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg_final = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month', 'TotPopACS']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
    ).reset_index()

    # Создаем обе Y-переменные
    df_agg_final['Y_Ratio'] = np.log1p(df_agg_final['Total_Emergency_Visits']) - np.log1p(df_agg_final['Total_Regular_Visits'])
    df_agg_final['Y_Regular'] = np.log1p(df_agg_final['Total_Regular_Visits'])
    print("✅ Y_Ratio и Y_Regular созданы.")

    # Присоединяем ТОЛЬКО даты лечения 'New PC'
    df_analysis_final = df_agg_final.merge(df_treatment_dates_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')

    # Создаем фиктивные переменные времени и дату
    df_analysis_final['Date'] = pd.to_datetime(df_analysis_final['Year_Month'] + '-01')
    df_analysis_final['Month_Cat'] = df_analysis_final['Date'].dt.month
    df_analysis_final['Year_Cat'] = df_analysis_final['Date'].dt.year
    df_analysis_final = df_analysis_final.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
    print("✅ Данные агрегированы и объединены с датами лечения 'New PC'.")

    # --- 6. Подготовка данных для Event Study ---
    df_model_final = df_analysis_final.copy()
    # Рассчитываем time_to_event_NEW_PC
    df_model_final['event_month_new_pc'] = pd.to_datetime(df_model_final['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_final['current_month'] = pd.to_datetime(df_model_final['Date']).dt.to_period('M')
    df_model_final['time_to_event_NEW_PC'] = (df_model_final['current_month'] - df_model_final['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # Ограничиваем окно и отфильтровываем контроль (NaN) ПЕРЕД созданием категории
    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    # Отбираем ТОЛЬКО обработанные наблюдения для Event Study
    df_model_final_filtered = df_model_final[
        df_model_final['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_final_filtered['time_to_event_NEW_PC'] = df_model_final_filtered['time_to_event_NEW_PC'].astype(int)

    if df_model_final_filtered.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска моделей.")
    else:
        print(f"Подготовлено {len(df_model_final_filtered)} наблюдений для Event Study.")

        # --- 7. Шаг 3: Запуск Плана А (Y_Ratio) ---
        OUTCOME_A = "Y_Ratio"
        formula_event_A = f"{OUTCOME_A} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year_Cat) + C(Month_Cat)"
        model_event_A_result = None
        print(f"\n--- Шаг 3: Запуск Модели Плана А ({OUTCOME_A}) ---")
        try:
            model_event_A = smf.ols(formula_event_A, data=df_model_final_filtered)
            model_event_A_result = model_event_A.fit(cov_type='cluster', cov_kwds={'groups': df_model_final_filtered['Zip']})
            print("  ✅ Модель Плана А с C(Zip) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА Плана А с C(Zip): {e}")
            try:
                 print("     ... Попытка Плана А без C(Zip) ...")
                 formula_no_zip_A = f"{OUTCOME_A} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + C(Year_Cat) + C(Month_Cat)"
                 model_A_no_zip = smf.ols(formula_no_zip_A, data=df_model_final_filtered)
                 model_event_A_result = model_A_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_final_filtered['Zip']})
                 print("        ✅ Модель Плана А без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА Плана А без C(Zip): {e2}")

        # --- 8. Шаг 4: Запуск Плана Б (Y_Regular) ---
        OUTCOME_B = "Y_Regular"
        formula_event_B = f"{OUTCOME_B} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year_Cat) + C(Month_Cat)"
        model_event_B_result = None
        print(f"\n--- Шаг 4: Запуск Модели Плана Б ({OUTCOME_B}) ---")
        try:
            model_event_B = smf.ols(formula_event_B, data=df_model_final_filtered)
            model_event_B_result = model_event_B.fit(cov_type='cluster', cov_kwds={'groups': df_model_final_filtered['Zip']})
            print("  ✅ Модель Плана Б с C(Zip) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА Плана Б с C(Zip): {e}")
            try:
                 print("     ... Попытка Плана Б без C(Zip) ...")
                 formula_no_zip_B = f"{OUTCOME_B} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + C(Year_Cat) + C(Month_Cat)"
                 model_B_no_zip = smf.ols(formula_no_zip_B, data=df_model_final_filtered)
                 model_event_B_result = model_B_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_final_filtered['Zip']})
                 print("        ✅ Модель Плана Б без C(Zip) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА Плана Б без C(Zip): {e2}")

        # --- 9. Визуализация ---
        print("\n--- Визуализация Результатов ---")

        # (Функция извлечения эффектов Event Study v3 - без изменений)
        def extract_event_study_effects_v3(model_result, base_period, time_var_name):
            pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]"
            if model_result is None: return pd.DataFrame() # Добавлена проверка
            params = model_result.params.filter(regex=pattern)
            conf = model_result.conf_int().filter(regex=pattern, axis=0)
            results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
            results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
            for idx in params.index:
                try:
                    match = re.search(pattern, idx)
                    if match:
                        month = int(match.group(1));
                        if month == base_period: continue
                        results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                        results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
                except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
            df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
            return df.sort_values('Relative_Month').reset_index(drop=True)

        # График А (Y_Ratio)
        print("\n--- График А: Эффект на Соотношение ER/Regular ---")
        event_data_A = extract_event_study_effects_v3(model_event_A_result, BASE_PERIOD, "time_to_event_NEW_PC")
        if not event_data_A.empty:
            fig_A, ax_A = plt.subplots(figsize=(12, 7))
            errors_A = [event_data_A['Effect'] - event_data_A['Conf_Low'], event_data_A['Conf_High'] - event_data_A['Effect']]
            valid_error_A = ~np.isnan(errors_A[0]) & ~np.isnan(errors_A[1])
            if valid_error_A.all():
                 ax_A.errorbar(x=event_data_A['Relative_Month'], y=event_data_A['Effect'], yerr=errors_A, fmt='-o', capsize=3, label='Эффект на Log(ER/Reg)', color='purple')
            else:
                 ax_A.plot(event_data_A['Relative_Month'], event_data_A['Effect'], marker='o', linestyle='-', label='Эффект на Log(ER/Reg) (без ДИ)', color='purple')
            ax_A.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax_A.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
            ax_A.set_title("График А: Эффект 'New' PC Клиник на Соотношение ER/Regular Визитов", fontsize=14)
            ax_A.set_xlabel("Месяцы относительно открытия первой 'New' PC клиники в радиусе 10 км")
            ax_A.set_ylabel(f"Эффект на {OUTCOME_A}")
            ax_A.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax_A.legend()
            ax_A.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика А.")

        # График Б (Y_Regular)
        print("\n--- График Б: Эффект на Регулярные Визиты ---")
        event_data_B = extract_event_study_effects_v3(model_event_B_result, BASE_PERIOD, "time_to_event_NEW_PC")
        if not event_data_B.empty:
            fig_B, ax_B = plt.subplots(figsize=(12, 7))
            errors_B = [event_data_B['Effect'] - event_data_B['Conf_Low'], event_data_B['Conf_High'] - event_data_B['Effect']]
            valid_error_B = ~np.isnan(errors_B[0]) & ~np.isnan(errors_B[1])
            if valid_error_B.all():
                 ax_B.errorbar(x=event_data_B['Relative_Month'], y=event_data_B['Effect'], yerr=errors_B, fmt='-o', capsize=3, label='Эффект на Log(Regular)', color='green') # Другой цвет
            else:
                 ax_B.plot(event_data_B['Relative_Month'], event_data_B['Effect'], marker='o', linestyle='-', label='Эффект на Log(Regular) (без ДИ)', color='green')
            ax_B.axhline(0, color='black', linestyle='-', linewidth=0.8)
            ax_B.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
            ax_B.set_title("График Б: Эффект 'New' PC Клиник на Регулярные Визиты", fontsize=14)
            ax_B.set_xlabel("Месяцы относительно открытия первой 'New' PC клиники в радиусе 10 км")
            ax_B.set_ylabel(f"Эффект на {OUTCOME_B}")
            ax_B.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
            ax_B.legend()
            ax_B.grid(True, axis='y', linestyle=':')
            plt.tight_layout()
            plt.show()
        else: print("Не удалось извлечь данные для графика Б.")

    print("\n🎉 --- Финальная Проверка Гипотезы из Абстракта завершена! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО EVENT STUDY С КОНТРОЛЕМ НА EXPOSURE SCORE ---")

# --- 1. Проверка и Загрузка Данных ---
# ... (Предполагаем, что final_data существует и данные загружены как раньше) ...
if 'final_data' not in locals():
    print("❌ ERROR: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy(); zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS']); zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ('New PC' и 'Acquired PC') ---
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)

    df_new_PC_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'New') & (df_clinics_info_full.apply(is_pc_urgent, axis=1))].copy()
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_urgent, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Расчет Exposure Score (Как раньше) ---
    print("\n--- Расчет Exposure Score (Может занять несколько минут) ---")
    df_zip_months = final_data[['Filtered_Patient_ZipCode', 'Date']].drop_duplicates()
    df_zip_months['Date'] = pd.to_datetime(df_zip_months['Date'])
    df_zip_months['month_period'] = df_zip_months['Date'].dt.to_period('M')
    df_zip_months = df_zip_months.merge(zip_coords.rename(columns={'IntPtLat': 'zip_lat', 'IntPtLon': 'zip_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    def calculate_exposure(zip_row, clinics_df):
        exposure = 0.0; zip_coords_tuple = (zip_row['zip_lat'], zip_row['zip_lon']); current_month = zip_row['month_period']
        open_clinics = clinics_df[clinics_df['open_month'] <= current_month]
        if not open_clinics.empty:
            for _, clinic_row in open_clinics.iterrows():
                clinic_coords_tuple = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
                distance_km = geodesic(zip_coords_tuple, clinic_coords_tuple).kilometers
                weight = 1.0 / (1.0 + distance_km + 1e-6); exposure += weight
        return exposure
    # --- Считаем exposure только для нужных групп клиник ---
    tqdm.pandas(desc="Расчет exposure_new_pc")
    df_zip_months['exposure_new_pc'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_new_PC_clinics), axis=1)
    tqdm.pandas(desc="Расчет exposure_acquired_pc")
    df_zip_months['exposure_acquired_pc'] = df_zip_months.progress_apply(lambda row: calculate_exposure(row, df_acquired_pc_clinics), axis=1)
    print("✅ Exposure scores (для PC/Urgent) рассчитаны.")

    # --- 5. Расчет Дат Первого События (Метод Радиуса) ---
    print("\n--- Расчет Дат Первого События (Метод Радиуса 10 км) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (Даты)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT
        nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows(): # Используем отфильтрованный список
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        for _, clinic_row in df_acquired_pc_clinics.iterrows(): # Используем отфильтрованный список
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
        if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
        treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
    df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
    print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

    # --- 6. Агрегация данных и Финальное слияние ---
    df_panel = final_data.copy()
    # df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left') # Население больше не нужно для Y_Ratio
    # df_panel = df_panel.dropna(subset=['TotPopACS']) # Убрали
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int); df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg_final = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
    ).reset_index()
    df_agg_final['Y_Ratio'] = np.log1p(df_agg_final['Total_Emergency_Visits']) - np.log1p(df_agg_final['Total_Regular_Visits'])
    df_agg_final['Date'] = pd.to_datetime(df_agg_final['Year_Month'] + '-01')

    # Присоединяем exposure scores
    df_analysis_final = df_agg_final.merge(
        df_zip_months[['Filtered_Patient_ZipCode', 'Date', 'exposure_new_pc', 'exposure_acquired_pc']],
        on=['Filtered_Patient_ZipCode', 'Date'],
        how='left'
    )
    # Присоединяем даты лечения
    df_analysis_final = df_analysis_final.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')

    # Заполняем NaN в exposure нулями
    df_analysis_final['exposure_new_pc'] = df_analysis_final['exposure_new_pc'].fillna(0)
    df_analysis_final['exposure_acquired_pc'] = df_analysis_final['exposure_acquired_pc'].fillna(0)

    # Создаем фиктивные переменные времени и переименовываем ZIP
    df_analysis_final['Month_Cat'] = df_analysis_final['Date'].dt.month
    df_analysis_final['Year_Cat'] = df_analysis_final['Date'].dt.year
    df_analysis_final = df_analysis_final.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    print("✅ Финальный DataFrame для анализа готов.")


    # --- 7. Подготовка к Event Study ---
    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    OUTCOME = "Y_Ratio"

    # --- Для Модели 1 (New PC) ---
    df_model_new = df_analysis_final.copy()
    df_model_new['event_month_new'] = pd.to_datetime(df_model_new['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_new['current_month'] = pd.to_datetime(df_model_new['Date']).dt.to_period('M')
    df_model_new['time_to_event_NEW_PC'] = (df_model_new['current_month'] - df_model_new['event_month_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    # Отбираем только treated для модели Event Study (контроль будет учтен через FE)
    df_model_new_filtered = df_model_new[df_model_new['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
    df_model_new_filtered['time_to_event_NEW_PC'] = df_model_new_filtered['time_to_event_NEW_PC'].astype(int)

    # --- Для Модели 2 (Acquired PC) ---
    df_model_acq = df_analysis_final.copy()
    df_model_acq['event_month_acq'] = pd.to_datetime(df_model_acq['treatment_date_ACQUIRED_PC']).dt.to_period('M')
    df_model_acq['current_month'] = pd.to_datetime(df_model_acq['Date']).dt.to_period('M')
    df_model_acq['time_to_event_ACQUIRED_PC'] = (df_model_acq['current_month'] - df_model_acq['event_month_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    # Отбираем только treated для модели Event Study
    df_model_acq_filtered = df_model_acq[df_model_acq['time_to_event_ACQUIRED_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
    df_model_acq_filtered['time_to_event_ACQUIRED_PC'] = df_model_acq_filtered['time_to_event_ACQUIRED_PC'].astype(int)

    # --- 8. Запуск Моделей Event Study с Контролем на Exposure ---
    models_final = {}

    # --- Модель 1: Эффект New PC ---
    print(f"\n--- Запуск Модели 1 (Эффект New PC на {OUTCOME}) ---")
    formula_event_new_exp = f"{OUTCOME} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + exposure_new_pc + C(Zip) + C(Year_Cat) + C(Month_Cat)"
    if not df_model_new_filtered.empty:
        try:
            print(f"... Запуск модели с C(Zip) (Наблюдений: {len(df_model_new_filtered)}) ...")
            model = smf.ols(formula_event_new_exp, data=df_model_new_filtered)
            models_final['New_PC'] = model.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_filtered['Zip']})
            print("  ✅ Модель 'New PC' с C(Zip) рассчитана.")
            # Выводим коэффициент exposure
            print("\n  Результат для exposure_new_pc:")
            print(models_final['New_PC'].summary().tables[1].as_html())
        except Exception as e:
            print(f"  ❌ ОШИБКА с C(Zip): {e}")
            try:
                 print("     ... Попытка без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + exposure_new_pc + C(Year_Cat) + C(Month_Cat)"
                 model_no_zip = smf.ols(formula_no_zip, data=df_model_new_filtered)
                 models_final['New_PC'] = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_filtered['Zip']})
                 print("        ✅ Модель 'New PC' без C(Zip) рассчитана.")
                 print("\n  Результат для exposure_new_pc (БЕЗ C(Zip)):")
                 print(models_final['New_PC'].summary().tables[1].as_html())
            except Exception as e2:
                 print(f"        ❌ ОШИБКА без C(Zip): {e2}")
                 models_final['New_PC'] = None # Помечаем, что модель не сошлась
    else:
        print("  ⚠️ Недостаточно данных для запуска модели 'New PC'.")

    # --- Модель 2: Эффект Acquired PC ---
    print(f"\n--- Запуск Модели 2 (Эффект Acquired PC на {OUTCOME}) ---")
    formula_event_acq_exp = f"{OUTCOME} ~ C(time_to_event_ACQUIRED_PC, Treatment(reference={BASE_PERIOD})) + exposure_acquired_pc + C(Zip) + C(Year_Cat) + C(Month_Cat)"
    if not df_model_acq_filtered.empty:
        try:
            print(f"... Запуск модели с C(Zip) (Наблюдений: {len(df_model_acq_filtered)}) ...")
            model = smf.ols(formula_event_acq_exp, data=df_model_acq_filtered)
            models_final['Acquired_PC'] = model.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_filtered['Zip']})
            print("  ✅ Модель 'Acquired PC' с C(Zip) рассчитана.")
            # Выводим коэффициент exposure
            print("\n  Результат для exposure_acquired_pc:")
            print(models_final['Acquired_PC'].summary().tables[1].as_html())
        except Exception as e:
            print(f"  ❌ ОШИБКА с C(Zip): {e}")
            try:
                 print("     ... Попытка без C(Zip) ...")
                 formula_no_zip = f"{OUTCOME} ~ C(time_to_event_ACQUIRED_PC, Treatment(reference={BASE_PERIOD})) + exposure_acquired_pc + C(Year_Cat) + C(Month_Cat)"
                 model_no_zip = smf.ols(formula_no_zip, data=df_model_acq_filtered)
                 models_final['Acquired_PC'] = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_filtered['Zip']})
                 print("        ✅ Модель 'Acquired PC' без C(Zip) рассчитана.")
                 print("\n  Результат для exposure_acquired_pc (БЕЗ C(Zip)):")
                 print(models_final['Acquired_PC'].summary().tables[1].as_html())

            except Exception as e2:
                 print(f"        ❌ ОШИБКА без C(Zip): {e2}")
                 models_final['Acquired_PC'] = None
    else:
        print("  ⚠️ Недостаточно данных для запуска модели 'Acquired PC'.")


    # --- 9. Визуализация Event Study ---
    print("\n--- Визуализация Результатов Event Study ---")

    # (Функция извлечения эффектов Event Study v3 - без изменений)
    def extract_event_study_effects_v3(model_result, base_period, time_var_name):
        # ... (Копируем функцию из предыдущего кода) ...
        pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
        if model_result is None: return pd.DataFrame()
        params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
        for idx in params.index:
            try:
                match = re.search(pattern, idx)
                if match:
                    month = int(match.group(1));
                    if month == base_period: continue
                    results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)

    # График 1 (New PC)
    print("\n--- График 1: Эффект 'New PC' на Соотношение ER/Regular ---")
    event_data_new = extract_event_study_effects_v3(models_final.get('New_PC'), BASE_PERIOD, "time_to_event_NEW_PC")
    if not event_data_new.empty:
        fig1, ax1 = plt.subplots(figsize=(12, 7))
        errors = [event_data_new['Effect'] - event_data_new['Conf_Low'], event_data_new['Conf_High'] - event_data_new['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax1.errorbar(x=event_data_new['Relative_Month'], y=event_data_new['Effect'], yerr=errors, fmt='-o', capsize=3, label='Эффект New PC', color='blue')
        else: ax1.plot(event_data_new['Relative_Month'], event_data_new['Effect'], marker='o', linestyle='-', label='Эффект New PC (без ДИ)', color='blue')
        ax1.axhline(0, color='black', linestyle='-', linewidth=0.8); ax1.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
        ax1.set_title("График 1: Эффект 'New' PC Клиник на Соотношение ER/Regular", fontsize=14)
        ax1.set_xlabel("Месяцы относительно открытия первой 'New' PC клиники в радиусе 10 км"); ax1.set_ylabel(f"Эффект на {OUTCOME}")
        ax1.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2)); ax1.legend(); ax1.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
    else: print("Не удалось извлечь/построить данные для графика 1.")

    # График 2 (Acquired PC)
    print("\n--- График 2: Эффект 'Acquired PC' на Соотношение ER/Regular ---")
    event_data_acq = extract_event_study_effects_v3(models_final.get('Acquired_PC'), BASE_PERIOD, "time_to_event_ACQUIRED_PC")
    if not event_data_acq.empty:
        fig2, ax2 = plt.subplots(figsize=(12, 7))
        errors = [event_data_acq['Effect'] - event_data_acq['Conf_Low'], event_data_acq['Conf_High'] - event_data_acq['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax2.errorbar(x=event_data_acq['Relative_Month'], y=event_data_acq['Effect'], yerr=errors, fmt='-s', capsize=3, label='Эффект Acquired PC', color='red')
        else: ax2.plot(event_data_acq['Relative_Month'], event_data_acq['Effect'], marker='s', linestyle='-', label='Эффект Acquired PC (без ДИ)', color='red')
        ax2.axhline(0, color='black', linestyle='-', linewidth=0.8); ax2.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
        ax2.set_title("График 2: Эффект 'Acquired' PC Клиник на Соотношение ER/Regular", fontsize=14)
        ax2.set_xlabel("Месяцы относительно поглощения первой 'Acquired' PC клиники в радиусе 10 км"); ax2.set_ylabel(f"Эффект на {OUTCOME}")
        ax2.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2)); ax2.legend(); ax2.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
    else: print("Не удалось извлечь/построить данные для графика 2.")

    print("\n🎉 --- Event Study с Контролем на Exposure Score завершен! ---")

In [ ]:
# --- Установка библиотеки RDD ---
try:
    import rdd
    print("Библиотека 'rdd' уже установлена.")
except ImportError:
    print("Устанавливаю библиотеку 'rdd'...")
    import sys
    !{sys.executable} -m pip install rdd
    import rdd
    print("Библиотека 'rdd' успешно установлена.")

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("\n--- 🚀 НАЧАЛО RDD АНАЛИЗА (Acquired PC, Cutoff 10 км) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy(); zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS']); zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация Acquired PC клиник ---
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Подготовка данных для RDD (уровень ZIP) ---
    print("\n--- Подготовка данных для RDD (уровень ZIP) ---")
    # 4.1 Считаем средний log(ER Rate) для каждого ZIP
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['TotPopACS']) # Убираем ZIPы без населения
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    # Агрегируем ПОЛНЫЕ суммы визитов и СРЕДНЕЕ население по ZIP
    df_zip_agg = df_panel.groupby('Filtered_Patient_ZipCode').agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        # Считаем среднее население за период
        Avg_TotPopACS = pd.NamedAgg(column='TotPopACS', aggfunc='mean'),
        # Считаем количество месяцев с данными для этого ZIP
        Num_Months = pd.NamedAgg(column='Date', aggfunc='nunique')
    ).reset_index()

    # Рассчитываем СРЕДНЕМЕСЯЧНЫЙ рейт на 1000 чел для каждого ZIP
    # Делим на кол-во месяцев, чтобы получить среднее за месяц
    df_zip_agg['Avg_Monthly_ER_Rate'] = (df_zip_agg['Total_Emergency_Visits'] / df_zip_agg['Num_Months']) / df_zip_agg['Avg_TotPopACS'] * 1000
    df_zip_agg['avg_log_ER_Rate'] = np.log1p(df_zip_agg['Avg_Monthly_ER_Rate'])

    # 4.2 Считаем расстояние до ближайшей Acquired PC клиники
    patient_zips_coords_df = df_zip_agg[['Filtered_Patient_ZipCode']].merge(
        zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),
        left_on='Filtered_Patient_ZipCode', right_index=True, how='inner'
    )
    zip_distances = []
    def find_nearest_clinic_in_group(patient_coords, clinic_group_df):
        min_dist = np.inf; nearest_info = {}
        if clinic_group_df.empty: return nearest_info
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist: min_dist = distance
        return {'distance_km': min_dist} if min_dist != np.inf else {}

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет расстояний до Acquired PC"):
        patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        nearest_info = find_nearest_clinic_in_group(patient_coords, df_acquired_pc_clinics)
        if nearest_info:
            zip_distances.append({
                'Filtered_Patient_ZipCode': patient_row['Filtered_Patient_ZipCode'],
                'distance_to_nearest_acquired_pc_km': nearest_info['distance_km']
            })
    df_zip_distances = pd.DataFrame(zip_distances)

    # 4.3 Объединяем данные для RDD
    df_rdd_data = df_zip_agg.merge(df_zip_distances, on='Filtered_Patient_ZipCode', how='inner')
    df_rdd_data = df_rdd_data[['Filtered_Patient_ZipCode', 'avg_log_ER_Rate', 'distance_to_nearest_acquired_pc_km']].dropna()

    # --- 5. Фильтрация данных для RDD (окно вокруг cutoff) ---
    CUTOFF = 10
    BANDWIDTH = 5 # Ширина окна по обе стороны от cutoff (можно менять)
    df_rdd_window = df_rdd_data[
        (df_rdd_data['distance_to_nearest_acquired_pc_km'] >= CUTOFF - BANDWIDTH) &
        (df_rdd_data['distance_to_nearest_acquired_pc_km'] <= CUTOFF + BANDWIDTH)
    ].copy()

    print(f"\n✅ Данные для RDD подготовлены. Используется {len(df_rdd_window)} ZIP-кодов в окне [{CUTOFF-BANDWIDTH}, {CUTOFF+BANDWIDTH}] км.")

    # --- 6. Запуск RDD Анализа ---
    X_VAR = 'distance_to_nearest_acquired_pc_km'
    Y_VAR = 'avg_log_ER_Rate'

    try:
        print(f"... Запуск RDD с порогом {CUTOFF} км ...")
        # Используем bandwidth=BANDWIDTH для локальной линейной регрессии
        rdd_result = rdd.rdd(df_rdd_window, X_VAR, Y_VAR, cut=CUTOFF, bandwidth=BANDWIDTH)

        # Выводим оценку эффекта
        rdd_estimate = rdd_result.iloc[0]['Estimate']
        rdd_pvalue = rdd_result.iloc[0]['P>|t|']
        print("\n--- Результаты RDD ---")
        print(rdd_result.to_string())

        # --- 7. Визуализация RDD ---
        print("\n--- Визуализация Графика RDD ---")
        fig_rdd, ax_rdd = plt.subplots(figsize=(10, 7))

        # Стандартный график из библиотеки rdd
        rdd.plot(df_rdd_window, X_VAR, Y_VAR, cut=CUTOFF, bandwidth=BANDWIDTH, ax=ax_rdd)

        # Кастомизация
        ax_rdd.set_title(f"RDD: Эффект Нахождения в Радиусе {CUTOFF} км от Acquired PC Клиники на Log(ER Rate)", fontsize=14)
        ax_rdd.set_xlabel("Расстояние до ближайшей Acquired PC Клиники (км)")
        ax_rdd.set_ylabel(f"Средний {Y_VAR}")
        # Добавляем текст с оценкой
        ax_rdd.text(0.05, 0.95, f'RDD Оценка = {rdd_estimate:.3f}\nP-value = {rdd_pvalue:.3f}',
                    transform=ax_rdd.transAxes, ha='left', va='top', fontsize=11,
                    bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.8))

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ ОШИБКА при запуске RDD анализа: {e}")
        if "no observations" in str(e).lower():
             print("   -> Вероятно, после фильтрации по окну не осталось данных. Попробуйте увеличить BANDWIDTH.")
        elif "singular matrix" in str(e).lower():
             print("   -> Проблема с регрессией внутри RDD. Возможно, нужно другое окно (BANDWIDTH) или меньше полиномов.")


    print("\n🎉 --- RDD Анализ завершен! ---")

In [ ]:
# --- Установка библиотеки rdrobust ---
try:
    import rdrobust
    print("Библиотека 'rdrobust' уже установлена.")
except ImportError:
    print("Устанавливаю библиотеку 'rdrobust'...")
    import sys
    # Используем --user для установки в пользовательское пространство, если нет прав администратора
    !{sys.executable} -m pip install --user rdrobust
    try:
         import rdrobust
         print("Библиотека 'rdrobust' успешно установлена.")
    except ImportError:
         print("❌ ОШИБКА: Не удалось установить 'rdrobust'. Пожалуйста, установите вручную.")
         # Можно остановить выполнение, если установка критична
         # raise

import pandas as pd
import numpy as np
# --- ИЗМЕНЕНИЕ: Импортируем rdrobust и rdplot ---
from rdrobust import rdrobust, rdplot
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("\n--- 🚀 НАЧАЛО RDD АНАЛИЗА (Acquired PC, Cutoff 10 км) с rdrobust ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy(); zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS']); zip_population = zip_population[zip_population['TotPopACS'] > 0]
    zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация Acquired PC клиник ---
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Подготовка данных для RDD (уровень ZIP) ---
    print("\n--- Подготовка данных для RDD (уровень ZIP) ---")
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_zip_agg = df_panel.groupby('Filtered_Patient_ZipCode').agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Avg_TotPopACS = pd.NamedAgg(column='TotPopACS', aggfunc='mean'),
        Num_Months = pd.NamedAgg(column='Date', aggfunc='nunique')
    ).reset_index()
    df_zip_agg['Avg_Monthly_ER_Rate'] = (df_zip_agg['Total_Emergency_Visits'] / df_zip_agg['Num_Months']) / df_zip_agg['Avg_TotPopACS'] * 1000
    df_zip_agg['avg_log_ER_Rate'] = np.log1p(df_zip_agg['Avg_Monthly_ER_Rate'])

    patient_zips_coords_df = df_zip_agg[['Filtered_Patient_ZipCode']].merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    zip_distances = []
    def find_nearest_clinic_in_group(patient_coords, clinic_group_df):
        min_dist = np.inf; nearest_info = {}
        if clinic_group_df.empty: return nearest_info
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon']); distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist: min_dist = distance
        return {'distance_km': min_dist} if min_dist != np.inf else {}

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет расстояний до Acquired PC"):
        patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        nearest_info = find_nearest_clinic_in_group(patient_coords, df_acquired_pc_clinics)
        if nearest_info: zip_distances.append({'Filtered_Patient_ZipCode': patient_row['Filtered_Patient_ZipCode'],'distance_to_nearest_acquired_pc_km': nearest_info['distance_km']})
    df_zip_distances = pd.DataFrame(zip_distances)
    df_rdd_data = df_zip_agg.merge(df_zip_distances, on='Filtered_Patient_ZipCode', how='inner')
    df_rdd_data = df_rdd_data[['Filtered_Patient_ZipCode', 'avg_log_ER_Rate', 'distance_to_nearest_acquired_pc_km']].dropna()

    # --- 5. Фильтрация данных НЕ ТРЕБУЕТСЯ для rdrobust (он сам выбирает окно) ---
    CUTOFF = 10
    X_VAR = 'distance_to_nearest_acquired_pc_km'
    Y_VAR = 'avg_log_ER_Rate'
    print(f"\n✅ Данные для RDD подготовлены. Используется {len(df_rdd_data)} ZIP-кодов.")

    # --- 6. Запуск RDD Анализа с rdrobust ---
    try:
        print(f"... Запуск rdrobust с порогом {CUTOFF} км ...")
        # --- ИЗМЕНЕНИЕ: Используем rdrobust ---
        # Он автоматически выбирает оптимальную ширину окна (bandwidth)
        rdd_result_obj = rdrobust(y=df_rdd_data[Y_VAR], x=df_rdd_data[X_VAR], c=CUTOFF)

        # Извлекаем основные результаты
        rdd_estimate = rdd_result_obj.Estimate[0]
        # P-value для "Conventional" (обычно используется)
        rdd_pvalue_conv = rdd_result_obj.pv[0, 0] # p > |z| для Conventional
        # Можно также посмотреть "Robust" p-value
        rdd_pvalue_robust = rdd_result_obj.pv[0, 2] # p > |z| для Robust

        print("\n--- Результаты rdrobust ---")
        print(rdd_result_obj) # Выводим полный объект с результатами
        print(f"\nОценка RDD (Conventional): {rdd_estimate:.4f}")
        print(f"P-value (Conventional): {rdd_pvalue_conv:.4f}")
        print(f"P-value (Robust): {rdd_pvalue_robust:.4f}")


        # --- 7. Визуализация RDD с rdplot ---
        print("\n--- Визуализация Графика RDD с rdplot ---")
        # --- ИЗМЕНЕНИЕ: Используем rdplot ---
        fig_rdd = rdplot(y=df_rdd_data[Y_VAR], x=df_rdd_data[X_VAR], c=CUTOFF)

        # Добавляем кастомизацию, если rdplot возвращает объект figure/axes
        if fig_rdd and hasattr(fig_rdd, 'axes'):
             ax_rdd = fig_rdd.axes[0] # Получаем оси графика
             ax_rdd.set_title(f"RDD: Эффект Нахождения в Радиусе {CUTOFF} км от Acquired PC Клиники на Log(ER Rate)", fontsize=14)
             ax_rdd.set_xlabel("Расстояние до ближайшей Acquired PC Клиники (км)")
             ax_rdd.set_ylabel(f"Средний {Y_VAR} (в бинах)")
             # Добавляем текст с оценкой
             ax_rdd.text(0.05, 0.95, f'RDD Оценка = {rdd_estimate:.3f}\nP-value (Conv.) = {rdd_pvalue_conv:.3f}',
                         transform=ax_rdd.transAxes, ha='left', va='top', fontsize=11,
                         bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.8))
             plt.show()
        else:
             print("rdplot не вернул объект графика для кастомизации.")


    except NameError:
         print("❌ ОШИБКА: Похоже, 'rdrobust' не импортировался правильно. Убедитесь, что установка прошла успешно.")
    except Exception as e:
        print(f"❌ ОШИБКА при запуске rdrobust/rdplot анализа: {e}")


    print("\n🎉 --- RDD Анализ (rdrobust) завершен! ---")

In [ ]:
# --- Установка/Проверка rdrobust ---
try:
    import rdrobust
    print("Библиотека 'rdrobust' уже установлена.")
except ImportError:
    print("Устанавливаю библиотеку 'rdrobust'...")
    import sys
    !{sys.executable} -m pip install --user rdrobust
    try: import rdrobust; print("Библиотека 'rdrobust' успешно установлена.")
    except ImportError: print("❌ ОШИБКА: Не удалось установить 'rdrobust'.")

import pandas as pd
import numpy as np
from rdrobust import rdrobust, rdplot # Импортируем функции
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("\n--- 🚀 НАЧАЛО RDD АНАЛИЗА (Acquired PC) с НЕСКОЛЬКИМИ CUTOFFS ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    # ... (Весь код Загрузки, Очистки, Идентификации PC клиник, Подготовки данных RDD остается БЕЗ ИЗМЕНЕНИЙ) ...
    # --- Повторяем этот блок для ясности ---
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e: print(f"❌ ERROR: Cannot find file {e.filename}."); raise
    # Очистка
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]; final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy(); zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS']); zip_population = zip_population[zip_population['TotPopACS'] > 0]; zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")
    # Идентификация Acquired PC
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_acquired_pc_clinics)} Acquired PC клиник.")
    # Подготовка данных RDD
    print("\n--- Подготовка данных для RDD (уровень ZIP) ---")
    df_panel = final_data.copy()
    df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_panel = df_panel.dropna(subset=['TotPopACS'])
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_zip_agg = df_panel.groupby('Filtered_Patient_ZipCode').agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Avg_TotPopACS = pd.NamedAgg(column='TotPopACS', aggfunc='mean'), Num_Months = pd.NamedAgg(column='Date', aggfunc='nunique')
    ).reset_index()
    df_zip_agg['Avg_Monthly_ER_Rate'] = (df_zip_agg['Total_Emergency_Visits'] / df_zip_agg['Num_Months']) / df_zip_agg['Avg_TotPopACS'] * 1000
    df_zip_agg['avg_log_ER_Rate'] = np.log1p(df_zip_agg['Avg_Monthly_ER_Rate'])
    patient_zips_coords_df = df_zip_agg[['Filtered_Patient_ZipCode']].merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}), left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    zip_distances = []
    def find_nearest_clinic_in_group(patient_coords, clinic_group_df):
        min_dist = np.inf; nearest_info = {}
        if clinic_group_df.empty: return nearest_info
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon']); distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist: min_dist = distance
        return {'distance_km': min_dist} if min_dist != np.inf else {}
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет расстояний до Acquired PC"):
        patient_coords = (patient_row['patient_lat'], patient_row['patient_lon']); nearest_info = find_nearest_clinic_in_group(patient_coords, df_acquired_pc_clinics)
        if nearest_info: zip_distances.append({'Filtered_Patient_ZipCode': patient_row['Filtered_Patient_ZipCode'],'distance_to_nearest_acquired_pc_km': nearest_info['distance_km']})
    df_zip_distances = pd.DataFrame(zip_distances)
    df_rdd_data = df_zip_agg.merge(df_zip_distances, on='Filtered_Patient_ZipCode', how='inner')
    df_rdd_data = df_rdd_data[['Filtered_Patient_ZipCode', 'avg_log_ER_Rate', 'distance_to_nearest_acquired_pc_km']].dropna()
    X_VAR = 'distance_to_nearest_acquired_pc_km'
    Y_VAR = 'avg_log_ER_Rate'
    print(f"\n✅ Данные для RDD подготовлены. Используется {len(df_rdd_data)} ZIP-кодов.")
    # --- КОНЕЦ БЛОКА ПОДГОТОВКИ ДАННЫХ ---


    # --- 6. Запуск RDD Анализа для РАЗНЫХ Cutoffs ---
    cutoffs_to_try = [5, 10, 15] # Список порогов для проверки

    for cutoff_val in cutoffs_to_try:
        print(f"\n--- === Запуск RDD для Cutoff = {cutoff_val} км === ---")
        try:
            # Запускаем rdrobust
            rdd_result_obj = rdrobust(y=df_rdd_data[Y_VAR], x=df_rdd_data[X_VAR], c=cutoff_val)

            # Извлекаем результаты
            rdd_estimate = rdd_result_obj.Estimate[0]
            rdd_pvalue_conv = rdd_result_obj.pv[0, 0]
            rdd_pvalue_robust = rdd_result_obj.pv[0, 2]

            print(f"--- Результаты rdrobust (Cutoff = {cutoff_val} км) ---")
            print(rdd_result_obj)
            print(f"\nОценка RDD (Conventional): {rdd_estimate:.4f}")
            print(f"P-value (Conventional): {rdd_pvalue_conv:.4f}")
            print(f"P-value (Robust): {rdd_pvalue_robust:.4f}")

            # --- 7. Визуализация RDD ---
            print(f"\n--- Визуализация Графика RDD (Cutoff = {cutoff_val} км) ---")
            try:
                fig_rdd = rdplot(y=df_rdd_data[Y_VAR], x=df_rdd_data[X_VAR], c=cutoff_val)
                if fig_rdd and hasattr(fig_rdd, 'axes'):
                     ax_rdd = fig_rdd.axes[0]
                     ax_rdd.set_title(f"RDD Plot (Cutoff = {cutoff_val} km): Acquired PC Clinics on Log(ER Rate)", fontsize=14)
                     ax_rdd.set_xlabel("Расстояние до ближайшей Acquired PC Клиники (км)")
                     ax_rdd.set_ylabel(f"Средний {Y_VAR} (в бинах)")
                     ax_rdd.text(0.05, 0.95, f'RDD Оценка = {rdd_estimate:.3f}\nP-val (Conv) = {rdd_pvalue_conv:.3f}',
                                 transform=ax_rdd.transAxes, ha='left', va='top', fontsize=11,
                                 bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.8))
                     plt.show()
                else: print("rdplot не вернул объект графика.")
            except Exception as e_plot:
                 print(f"  ⚠️ ОШИБКА при построении графика rdplot для cutoff {cutoff_val}: {e_plot}")

        except NameError:
             print("❌ ОШИБКА: 'rdrobust' не импортировался. Проверьте установку.")
             break # Прерываем цикл, если библиотека не импортирована
        except Exception as e:
            print(f"❌ ОШИБКА при запуске rdrobust для cutoff {cutoff_val}: {e}")
            # Добавим проверку на ошибку, связанную с недостатком данных
            if "Not enough observations" in str(e) or "std non-positive" in str(e):
                 print("   -> Вероятно, недостаточно данных вокруг этой точки отсечения.")
            elif isinstance(e, int) and e == 0: # Перехватываем ошибку '0'
                 print("   -> Неопределенная ошибка (возможно, из-за mass points или коллинеарности).")
            else:
                 print(f"   -> Неожиданная ошибка: {type(e).__name__} - {e}")
        print(f"--- === Завершено для Cutoff = {cutoff_val} км === ---")


    print("\n🎉 --- RDD Анализ (rdrobust) завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None

print("--- 🚀 НАЧАЛО ФИНАЛЬНОЙ ПРОВЕРКИ ГИПОТЕЗЫ ИЗ АБСТРАКТА (Чистый Event Study для New PC) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    # --- Убрали загрузку населения - не нужно для Y_Ratio ---
    # zip_population = ...
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Шаг 1: Изолировать 'New PC/Urgent' клиники ---
    df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_new_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_new_pc_urgent, axis=1)].copy()
    print(f"\n✅ Шаг 1: Идентифицировано {len(df_new_PC_clinics)} 'New' PC/Urgent/FP клиник для анализа:")
    # print(df_new_PC_clinics[['clinic_zip', 'Facility', 'event_date']].to_string()) # Раскомментировать для просмотра списка

    if df_new_PC_clinics.empty:
         raise ValueError("Не найдено 'New' PC/Urgent/FP клиник.")

    # --- 4. Шаг 2: Пересчитать "Карту Лечения" (Только для New PC) ---
    print("\n--- Шаг 2: Пересчет Карты Лечения (только New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; nearby_new_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        treatment_dates_new_pc[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc}
    df_treatment_dates_new_pc = pd.DataFrame.from_dict(treatment_dates_new_pc, orient='index')
    print(f"\n✅ Карта лечения ('New PC') создана. {len(df_treatment_dates_new_pc[df_treatment_dates_new_pc['treatment_date_NEW_PC'].notna()])} ZIP'ов обработаны.")

    # --- 5. Шаг 1 (продолжение): Агрегация данных и создание Y-переменных ---
    print("\n--- Агрегация данных и создание Y-переменных ---")
    df_panel = final_data.copy()
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int); df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    # --- Группируем ТОЛЬКО по ZIP и Месяцу ---
    df_agg_final = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum()),
        Year = pd.NamedAgg(column='Year', aggfunc='first'), # Сохраняем год/месяц
        Month = pd.NamedAgg(column='Month', aggfunc='first')
    ).reset_index()

    # Создаем Y_Ratio и Y_Regular
    df_agg_final['Y_Ratio'] = np.log1p(df_agg_final['Total_Emergency_Visits']) - np.log1p(df_agg_final['Total_Regular_Visits'])
    df_agg_final['Y_Regular'] = np.log1p(df_agg_final['Total_Regular_Visits'])
    print("✅ Y_Ratio и Y_Regular созданы.")

    # Присоединяем ТОЛЬКО даты лечения 'New PC'
    df_analysis_final = df_agg_final.merge(df_treatment_dates_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')

    # Создаем фиктивные переменные времени и дату
    df_analysis_final['Date'] = pd.to_datetime(df_analysis_final['Year_Month'] + '-01')
    # --- Используем числовые Month/Year для FE ---
    # df_analysis_final['Month_Cat'] = df_analysis_final['Date'].dt.month
    # df_analysis_final['Year_Cat'] = df_analysis_final['Date'].dt.year
    df_analysis_final = df_analysis_final.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
    print("✅ Данные агрегированы и объединены с датами лечения 'New PC'.")

    # --- 6. Подготовка данных для Event Study ---
    df_model_final = df_analysis_final.copy()
    df_model_final['event_month_new_pc'] = pd.to_datetime(df_model_final['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_final['current_month'] = pd.to_datetime(df_model_final['Date']).dt.to_period('M')
    df_model_final['time_to_event_NEW_PC'] = (df_model_final['current_month'] - df_model_final['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    # Отбираем ТОЛЬКО обработанные наблюдения для Event Study
    df_model_final_filtered = df_model_final[
        df_model_final['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_final_filtered['time_to_event_NEW_PC'] = df_model_final_filtered['time_to_event_NEW_PC'].astype(int)

    if df_model_final_filtered.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска моделей.")
    else:
        print(f"Подготовлено {len(df_model_final_filtered)} наблюдений для Event Study.")

        # --- 7. Шаг 3: Запуск Плана А (Y_Ratio) ---
        OUTCOME_A = "Y_Ratio"
        # --- Используем числовые Month/Year ---
        formula_event_A = f"{OUTCOME_A} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
        model_event_A_result = None
        print(f"\n--- Шаг 3: Запуск Модели Плана А ({OUTCOME_A}) ---")
        try:
            print(f"... Запуск модели с C(Zip) ...")
            model_event_A = smf.ols(formula_event_A, data=df_model_final_filtered)
            model_event_A_result = model_event_A.fit(cov_type='cluster', cov_kwds={'groups': df_model_final_filtered['Zip']})
            print("  ✅ Модель Плана А с C(Zip) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА Плана А с C(Zip): {e}")
            try:
                 print("     ... Попытка Плана А без C(Zip) ...")
                 formula_no_zip_A = f"{OUTCOME_A} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + C(Year) + C(Month)"
                 model_A_no_zip = smf.ols(formula_no_zip_A, data=df_model_final_filtered)
                 model_event_A_result = model_A_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_final_filtered['Zip']})
                 print("        ✅ Модель Плана А без C(Zip) рассчитана.")
            except Exception as e2: print(f"        ❌ ОШИБКА Плана А без C(Zip): {e2}")

        # --- 8. Шаг 4: Запуск Плана Б (Y_Regular) ---
        OUTCOME_B = "Y_Regular"
        # --- Используем числовые Month/Year ---
        formula_event_B = f"{OUTCOME_B} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
        model_event_B_result = None
        print(f"\n--- Шаг 4: Запуск Модели Плана Б ({OUTCOME_B}) ---")
        try:
            print(f"... Запуск модели с C(Zip) ...")
            model_event_B = smf.ols(formula_event_B, data=df_model_final_filtered)
            model_event_B_result = model_event_B.fit(cov_type='cluster', cov_kwds={'groups': df_model_final_filtered['Zip']})
            print("  ✅ Модель Плана Б с C(Zip) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА Плана Б с C(Zip): {e}")
            try:
                 print("     ... Попытка Плана Б без C(Zip) ...")
                 formula_no_zip_B = f"{OUTCOME_B} ~ C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) + C(Year) + C(Month)"
                 model_B_no_zip = smf.ols(formula_no_zip_B, data=df_model_final_filtered)
                 model_event_B_result = model_B_no_zip.fit(cov_type='cluster', cov_kwds={'groups': df_model_final_filtered['Zip']})
                 print("        ✅ Модель Плана Б без C(Zip) рассчитана.")
            except Exception as e2: print(f"        ❌ ОШИБКА Плана Б без C(Zip): {e2}")

        # --- 9. Визуализация ---
        print("\n--- Визуализация Результатов ---")

        # (Функция извлечения эффектов Event Study v3 - без изменений)
        def extract_event_study_effects_v3(model_result, base_period, time_var_name):
            pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
            if model_result is None: return pd.DataFrame()
            params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
            results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
            results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
            for idx in params.index:
                try:
                    match = re.search(pattern, idx)
                    if match:
                        month = int(match.group(1));
                        if month == base_period: continue
                        results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                        results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
                except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
            df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
            return df.sort_values('Relative_Month').reset_index(drop=True)

        # График А (Y_Ratio)
        print("\n--- График А: Эффект на Соотношение ER/Regular ---")
        event_data_A = extract_event_study_effects_v3(model_event_A_result, BASE_PERIOD, "time_to_event_NEW_PC")
        if not event_data_A.empty:
            fig_A, ax_A = plt.subplots(figsize=(12, 7))
            errors_A = [event_data_A['Effect'] - event_data_A['Conf_Low'], event_data_A['Conf_High'] - event_data_A['Effect']]
            valid_error_A = ~np.isnan(errors_A[0]) & ~np.isnan(errors_A[1])
            if valid_error_A.all(): ax_A.errorbar(x=event_data_A['Relative_Month'], y=event_data_A['Effect'], yerr=errors_A, fmt='-o', capsize=3, label='Эффект на Log(ER/Reg)', color='purple')
            else: ax_A.plot(event_data_A['Relative_Month'], event_data_A['Effect'], marker='o', linestyle='-', label='Эффект на Log(ER/Reg) (без ДИ)', color='purple')
            ax_A.axhline(0, color='black', linestyle='-', linewidth=0.8); ax_A.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
            ax_A.set_title("График А: Эффект 'New' PC Клиник на Соотношение ER/Regular Визитов", fontsize=14)
            ax_A.set_xlabel("Месяцы относительно открытия первой 'New' PC клиники в радиусе 10 км"); ax_A.set_ylabel(f"Эффект на {OUTCOME_A}")
            ax_A.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2)); ax_A.legend(); ax_A.grid(True, axis='y', linestyle=':')
            plt.tight_layout(); plt.show()
        else: print("Не удалось извлечь/построить данные для графика А.")

        # График Б (Y_Regular)
        print("\n--- График Б: Эффект на Регулярные Визиты ---")
        event_data_B = extract_event_study_effects_v3(model_event_B_result, BASE_PERIOD, "time_to_event_NEW_PC")
        if not event_data_B.empty:
            fig_B, ax_B = plt.subplots(figsize=(12, 7))
            errors_B = [event_data_B['Effect'] - event_data_B['Conf_Low'], event_data_B['Conf_High'] - event_data_B['Effect']]
            valid_error_B = ~np.isnan(errors_B[0]) & ~np.isnan(errors_B[1])
            if valid_error_B.all(): ax_B.errorbar(x=event_data_B['Relative_Month'], y=event_data_B['Effect'], yerr=errors_B, fmt='-o', capsize=3, label='Эффект на Log(Regular)', color='green')
            else: ax_B.plot(event_data_B['Relative_Month'], event_data_B['Effect'], marker='o', linestyle='-', label='Эффект на Log(Regular) (без ДИ)', color='green')
            ax_B.axhline(0, color='black', linestyle='-', linewidth=0.8); ax_B.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
            ax_B.set_title("График Б: Эффект 'New' PC Клиник на Регулярные Визиты", fontsize=14)
            ax_B.set_xlabel("Месяцы относительно открытия первой 'New' PC клиники в радиусе 10 км"); ax_B.set_ylabel(f"Эффект на {OUTCOME_B}")
            ax_B.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2)); ax_B.legend(); ax_B.grid(True, axis='y', linestyle=':')
            plt.tight_layout(); plt.show()
        else: print("Не удалось извлечь/построить данные для графика Б.")

    print("\n🎉 --- Финальная Проверка Гипотезы из Абстракта завершена! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
# Стиль для графиков
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО АНАЛИЗА ПО ПЛАНУ 'НАСЫЩЕНИЯ' (5 Регрессий) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (стандартный код очистки) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_population = df_crosswalk[['zcta5', 'TotPopACS']].copy(); zip_population['TotPopACS'] = pd.to_numeric(zip_population['TotPopACS'].str.replace(',', '', regex=False), errors='coerce')
    zip_population = zip_population.dropna(subset=['TotPopACS']); zip_population = zip_population[zip_population['TotPopACS'] > 0]; zip_population = zip_population.set_index('zcta5')
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация групп клиник ('New PC' и 'Acquired PC') ---
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'New') & (df_clinics_info_full.apply(is_pc_urgent, axis=1))].copy()
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_urgent, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

    # --- 4. Расчет Дат Первого События (Метод Радиуса 10 км) ---
    print("\n--- Расчет Дат Первого События (Метод Радиуса 10 км) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (Даты)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        for _, clinic_row in df_acquired_pc_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
        if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
        treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
    df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
    print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

    # --- 5. Агрегация данных и создание Y-переменных ---
    print("\n--- Агрегация данных и создание Y-переменных ---")
    df_panel = final_data.copy()
    # df_panel = df_panel.merge(zip_population, left_on='Filtered_Patient_ZipCode', right_index=True, how='left') # Население не нужно для логарифмов
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int); df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
    ).reset_index()

    # Создаем ВСЕ нужные Y-переменные
    df_agg['log_ER_Rate'] = np.log1p(df_agg['Total_Emergency_Visits']) # Используем log(Visits+1), т.к. нет населения
    df_agg['log_Regular_Visits'] = np.log1p(df_agg['Total_Regular_Visits'])
    df_agg['Y_Ratio'] = np.log1p(df_agg['Total_Emergency_Visits']) - np.log1p(df_agg['Total_Regular_Visits'])
    print("✅ Y-переменные созданы: log_ER_Rate, log_Regular_Visits, Y_Ratio.")

    # Присоединяем даты лечения
    df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
    # Убираем _Cat, используем числовые Month/Year
    # df_analysis['Month_Cat'] = df_analysis['Date'].dt.month
    # df_analysis['Year_Cat'] = df_analysis['Date'].dt.year
    df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
    print("✅ Данные агрегированы и готовы к анализу.")

    # --- Общие параметры для Event Study ---
    EVENT_WINDOW = 12
    BASE_PERIOD = -1

    # --- Функция для извлечения эффектов Event Study ---
    # (Используем ту же функцию v3)
    def extract_event_study_effects_v3(model_result, base_period, time_var_name):
        pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
        if model_result is None: return pd.DataFrame()
        params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
        results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
        results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
        for idx in params.index:
            try:
                match = re.search(pattern, idx)
                if match:
                    month = int(match.group(1));
                    if month == base_period: continue
                    results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                    results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
            except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
        df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
        return df.sort_values('Relative_Month').reset_index(drop=True)

    # --- Функция для построения графика Event Study ---
    def plot_event_study(event_data, outcome_name, title, color, event_window, base_period):
         if not event_data.empty:
            fig, ax = plt.subplots(figsize=(12, 7))
            errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
            if valid_error.all(): ax.errorbar(x=event_data['Relative_Month'], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
            else: ax.plot(event_data['Relative_Month'], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
            ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
            ax.set_title(title, fontsize=14); ax.set_xlabel("Месяцы относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
            ax.set_xticks(range(-event_window, event_window + 1, 2)); ax.legend(); ax.grid(True, axis='y', linestyle=':')
            plt.tight_layout(); plt.show()
         else: print(f"Не удалось извлечь/построить данные для: {title}")


    # =========================================================================
    # --- ШАГ 1 (RQ1 & RQ2): Основные эффекты на log(ER_Rate) ---
    # =========================================================================
    print("\n\n--- ШАГ 1 (RQ1 & RQ2): Основные эффекты на log(ER_Rate) ---")
    OUTCOME_RQ12 = "log_ER_Rate"
    model_results_RQ12 = {'New': None, 'Acquired': None}

    # --- Модель для 'New PC' ---
    df_model_new = df_analysis.copy()
    df_model_new['event_month'] = pd.to_datetime(df_model_new['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_new['current_month'] = pd.to_datetime(df_model_new['Date']).dt.to_period('M')
    df_model_new['time_to_event'] = (df_model_new['current_month'] - df_model_new['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_model_new_filtered = df_model_new[df_model_new['time_to_event'].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
    df_model_new_filtered['time_to_event'] = df_model_new_filtered['time_to_event'].astype(int)
    formula_rq1 = f"{OUTCOME_RQ12} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    print(f"\nЗапуск RQ1 (New PC на {OUTCOME_RQ12})...")
    if not df_model_new_filtered.empty:
         try:
             model = smf.ols(formula_rq1, data=df_model_new_filtered)
             model_results_RQ12['New'] = model.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_filtered['Zip']})
             print("  ✅ Модель RQ1 рассчитана.")
         except Exception as e: print(f"  ❌ ОШИБКА RQ1: {e}")
    else: print("  ⚠️ Нет данных для RQ1.")

    # --- Модель для 'Acquired PC' ---
    df_model_acq = df_analysis.copy()
    df_model_acq['event_month'] = pd.to_datetime(df_model_acq['treatment_date_ACQUIRED_PC']).dt.to_period('M')
    df_model_acq['current_month'] = pd.to_datetime(df_model_acq['Date']).dt.to_period('M')
    df_model_acq['time_to_event'] = (df_model_acq['current_month'] - df_model_acq['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_model_acq_filtered = df_model_acq[df_model_acq['time_to_event'].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
    df_model_acq_filtered['time_to_event'] = df_model_acq_filtered['time_to_event'].astype(int)
    formula_rq2 = f"{OUTCOME_RQ12} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    print(f"\nЗапуск RQ2 (Acquired PC на {OUTCOME_RQ12})...")
    if not df_model_acq_filtered.empty:
        try:
            model = smf.ols(formula_rq2, data=df_model_acq_filtered)
            model_results_RQ12['Acquired'] = model.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_filtered['Zip']})
            print("  ✅ Модель RQ2 рассчитана.")
        except Exception as e: print(f"  ❌ ОШИБКА RQ2: {e}")
    else: print("  ⚠️ Нет данных для RQ2.")

    # --- Визуализация RQ1 & RQ2 ---
    event_data_rq1 = extract_event_study_effects_v3(model_results_RQ12.get('New'), BASE_PERIOD, "time_to_event")
    event_data_rq2 = extract_event_study_effects_v3(model_results_RQ12.get('Acquired'), BASE_PERIOD, "time_to_event")
    plot_event_study(event_data_rq1, OUTCOME_RQ12, "Шаг 1 (RQ1): Эффект 'New PC' на Log ER Rate", 'blue', EVENT_WINDOW, BASE_PERIOD)
    plot_event_study(event_data_rq2, OUTCOME_RQ12, "Шаг 1 (RQ2): Эффект 'Acquired PC' на Log ER Rate", 'red', EVENT_WINDOW, BASE_PERIOD)


    # =========================================================================
    # --- ШАГ 2 (RQ3 - Механизм 'New'): Эффект New PC на log(Regular_Visits) ---
    # =========================================================================
    print("\n\n--- ШАГ 2 (RQ3 - Механизм 'New'): Эффект на log(Regular_Visits) ---")
    OUTCOME_RQ3 = "log_Regular_Visits"
    model_result_RQ3 = None
    formula_rq3 = f"{OUTCOME_RQ3} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    # Используем те же данные, что и для RQ1 (df_model_new_filtered)
    print(f"\nЗапуск RQ3 (New PC на {OUTCOME_RQ3})...")
    if not df_model_new_filtered.empty:
         try:
             model = smf.ols(formula_rq3, data=df_model_new_filtered)
             model_result_RQ3 = model.fit(cov_type='cluster', cov_kwds={'groups': df_model_new_filtered['Zip']})
             print("  ✅ Модель RQ3 рассчитана.")
         except Exception as e: print(f"  ❌ ОШИБКА RQ3: {e}")
    else: print("  ⚠️ Нет данных для RQ3.")

    # --- Визуализация RQ3 ---
    event_data_rq3 = extract_event_study_effects_v3(model_result_RQ3, BASE_PERIOD, "time_to_event")
    plot_event_study(event_data_rq3, OUTCOME_RQ3, "Шаг 2 (RQ3): Эффект 'New PC' на Log Regular Visits", 'green', EVENT_WINDOW, BASE_PERIOD)


    # =============================================================================
    # --- ШАГ 3 (RQ4 - Механизм 'Acquired'): Эффект Acquired PC на log(Regular_Visits) ---
    # =============================================================================
    print("\n\n--- ШАГ 3 (RQ4 - Механизм 'Acquired'): Эффект на log(Regular_Visits) ---")
    OUTCOME_RQ4 = "log_Regular_Visits"
    model_result_RQ4 = None
    formula_rq4 = f"{OUTCOME_RQ4} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    # Используем те же данные, что и для RQ2 (df_model_acq_filtered)
    print(f"\nЗапуск RQ4 (Acquired PC на {OUTCOME_RQ4})...")
    if not df_model_acq_filtered.empty:
        try:
            model = smf.ols(formula_rq4, data=df_model_acq_filtered)
            model_result_RQ4 = model.fit(cov_type='cluster', cov_kwds={'groups': df_model_acq_filtered['Zip']})
            print("  ✅ Модель RQ4 рассчитана.")
        except Exception as e: print(f"  ❌ ОШИБКА RQ4: {e}")
    else: print("  ⚠️ Нет данных для RQ4.")

    # --- Визуализация RQ4 ---
    event_data_rq4 = extract_event_study_effects_v3(model_result_RQ4, BASE_PERIOD, "time_to_event")
    plot_event_study(event_data_rq4, OUTCOME_RQ4, "Шаг 3 (RQ4): Эффект 'Acquired PC' на Log Regular Visits", 'orange', EVENT_WINDOW, BASE_PERIOD)


    # =========================================================================
    # --- ШАГ 4 (Гетерогенность - Абстракт): Y_Ratio ~ time * SVI ---
    # =========================================================================
    print("\n\n--- ШАГ 4 (Гетерогенность - Абстракт): Y_Ratio ~ time * SVI ---")
    OUTCOME_RQ5 = "Y_Ratio"
    model_result_RQ5 = None

    # --- Загрузка и подготовка SVI данных ---
    SVI_FILE = 'svi_data.csv' # ЗАМЕНИТЕ ИМЯ ФАЙЛА, ЕСЛИ НУЖНО
    try:
        print(f"... Загрузка данных SVI из {SVI_FILE} ...")
        df_svi = pd.read_csv(SVI_FILE, dtype={'Zip': str}) # Убедимся, что ZIP читается как строка
        # --- АДАПТИРУЙТЕ ЭТИ СТРОКИ ПОД ВАШ ФАЙЛ SVI ---
        # Предполагаем, что есть колонка 'SVI_Category' с 'High'/'Low' или 'Rural'/'Urban'
        # Или колонка с числовым SVI, которую нужно будет бинаризовать
        if 'SVI_Category' not in df_svi.columns:
             # Пример бинаризации, если есть числовая колонка 'SVI_Score'
             if 'SVI_Score' in df_svi.columns:
                  svi_median = df_svi['SVI_Score'].median()
                  df_svi['SVI_Category'] = np.where(df_svi['SVI_Score'] >= svi_median, 'High_Vulnerability', 'Low_Vulnerability')
                  print(f"  Создана SVI_Category: High >= {svi_median:.2f}, Low < {svi_median:.2f}")
             else:
                  raise ValueError("Колонка 'SVI_Category' или 'SVI_Score' не найдена в файле SVI.")
        else:
             # Убедимся, что есть только две категории, например
             svi_categories = df_svi['SVI_Category'].unique()
             if len(svi_categories) > 2:
                  print(f"  Предупреждение: Найдено более 2 категорий SVI: {svi_categories}. Используются первые две.")
                  # Можно добавить логику для выбора нужных категорий
             elif len(svi_categories) < 2:
                   raise ValueError("Найдено менее 2 категорий SVI. Невозможно провести анализ гетерогенности.")

        # Оставляем только нужные колонки и убираем дубликаты
        df_svi = df_svi[['Zip', 'SVI_Category']].drop_duplicates(subset=['Zip'])
        print(f"  ✅ Данные SVI подготовлены для {len(df_svi)} ZIP-кодов.")

        # --- Присоединение SVI к данным модели RQ1 ---
        # Используем df_model_new_filtered (данные для RQ1)
        df_model_rq5 = df_model_new_filtered.merge(df_svi, on='Zip', how='inner') # INNER join!
        if df_model_rq5.empty:
            print("  ❌ ОШИБКА: После слияния с SVI не осталось данных.")
        else:
            print(f"  Подготовлено {len(df_model_rq5)} наблюдений для анализа гетерогенности.")
            print("  Распределение SVI категорий в данных:")
            print(df_model_rq5['SVI_Category'].value_counts())

            # --- Формула с взаимодействием ---
            # C(time_to_event):C(SVI_Category) - основной эффект гетерогенности
            formula_rq5 = f"{OUTCOME_RQ5} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) * C(SVI_Category) + C(Zip) + C(Year) + C(Month)"
            print(f"\nЗапуск RQ5 (Гетерогенность New PC на {OUTCOME_RQ5})...")

            try:
                model = smf.ols(formula_rq5, data=df_model_rq5)
                model_result_RQ5 = model.fit(cov_type='cluster', cov_kwds={'groups': df_model_rq5['Zip']})
                print("  ✅ Модель RQ5 рассчитана.")
                # print(model_result_RQ5.summary()) # Для детального просмотра

                # --- Извлечение и Визуализация RQ5 ---
                print("\n--- Визуализация RQ5: Эффект по группам SVI ---")
                
                svi_groups = df_model_rq5['SVI_Category'].unique()
                if len(svi_groups) != 2:
                     print("  Предупреждение: Ожидалось 2 группы SVI для сравнения.")
                
                # Извлекаем данные для каждой группы
                event_data_rq5 = {}
                base_svi_group = svi_groups[0] # Первая категория будет базой по умолчанию
                
                # Эффект для базовой группы SVI
                base_effects = extract_event_study_effects_v3(model_result_RQ5, BASE_PERIOD, "time_to_event")
                if not base_effects.empty:
                     event_data_rq5[base_svi_group] = base_effects
                     
                # Эффект для второй группы SVI (База + Взаимодействие)
                if len(svi_groups) > 1:
                     other_svi_group = svi_groups[1]
                     interaction_pattern = rf"C\(time_to_event.*?\)\[T\.(-?\d+)\]:C\(SVI_Category\)\[T\.{other_svi_group}\]"
                     interaction_params = model_result_RQ5.params.filter(regex=interaction_pattern)
                     interaction_conf = model_result_RQ5.conf_int().filter(regex=interaction_pattern, axis=0)
                     
                     results_other = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
                     results_other['Relative_Month'].append(BASE_PERIOD); results_other['Effect'].append(0); results_other['Conf_Low'].append(0); results_other['Conf_High'].append(0)

                     for month in base_effects['Relative_Month']:
                          if month == BASE_PERIOD: continue # Пропускаем базу
                          
                          base_effect_row = base_effects[base_effects['Relative_Month'] == month].iloc[0]
                          interaction_term_name = f"C(time_to_event, Treatment(reference={BASE_PERIOD}))[T.{month}]:C(SVI_Category)[T.{other_svi_group}]"
                          
                          if interaction_term_name in interaction_params.index:
                               # Считаем сумму эффектов и ее ДИ через t_test
                               # Имя базового эффекта: C(time_to_event...)[T.month]
                               base_term_name = f"C(time_to_event, Treatment(reference={BASE_PERIOD}))[T.{month}]"
                               if base_term_name in model_result_RQ5.params.index:
                                    try:
                                         t_test = model_result_RQ5.t_test(f"{base_term_name} + {interaction_term_name}")
                                         results_other['Relative_Month'].append(month)
                                         results_other['Effect'].append(t_test.effect[0])
                                         results_other['Conf_Low'].append(t_test.conf_int()[0][0])
                                         results_other['Conf_High'].append(t_test.conf_int()[0][1])
                                    except Exception as e_ttest:
                                         print(f"    Warning: t_test failed for month {month}: {e_ttest}")
                                         # Если t_test не сработал, просто суммируем оценки (ДИ будет некорректным)
                                         results_other['Relative_Month'].append(month)
                                         results_other['Effect'].append(base_effect_row['Effect'] + interaction_params[interaction_term_name])
                                         results_other['Conf_Low'].append(np.nan)
                                         results_other['Conf_High'].append(np.nan)
                               else: # Если базовый эффект для этого месяца не найден (странно)
                                    results_other['Relative_Month'].append(month); results_other['Effect'].append(np.nan)
                                    results_other['Conf_Low'].append(np.nan); results_other['Conf_High'].append(np.nan)

                          else: # Если взаимодействие не найдено, эффект такой же как у базы
                               results_other['Relative_Month'].append(month); results_other['Effect'].append(base_effect_row['Effect'])
                               results_other['Conf_Low'].append(base_effect_row['Conf_Low']); results_other['Conf_High'].append(base_effect_row['Conf_High'])
                               
                     df_other = pd.DataFrame(results_other).sort_values('Relative_Month').reset_index(drop=True)
                     if not df_other.empty:
                          event_data_rq5[other_svi_group] = df_other

                # Строим график RQ5
                fig_rq5, ax_rq5 = plt.subplots(figsize=(12, 7))
                colors = ['blue', 'red']
                markers = ['o', 's']
                linestyles = ['-', '--']
                plotted = False

                for i, (group_name, group_data) in enumerate(event_data_rq5.items()):
                     if not group_data.empty:
                          errors = [group_data['Effect'] - group_data['Conf_Low'], group_data['Conf_High'] - group_data['Effect']]
                          valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                          if valid_error.all():
                               ax_rq5.errorbar(x=group_data['Relative_Month'], y=group_data['Effect'], yerr=errors,
                                             fmt=f'{linestyles[i]}{markers[i]}', capsize=3, label=f'Эффект для {group_name}', color=colors[i])
                          else:
                               ax_rq5.plot(group_data['Relative_Month'], group_data['Effect'], marker=markers[i], linestyle=linestyles[i],
                                          label=f'Эффект для {group_name} (без ДИ)', color=colors[i])
                          plotted = True

                if plotted:
                    ax_rq5.axhline(0, color='black', linestyle='-', linewidth=0.8)
                    ax_rq5.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                    ax_rq5.set_title("Шаг 4 (RQ5): Гетерогенный Эффект 'New PC' на Y_Ratio по SVI", fontsize=14)
                    ax_rq5.set_xlabel("Месяцы относительно открытия"); ax_rq5.set_ylabel(f"Эффект на {OUTCOME_RQ5}")
                    ax_rq5.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2)); ax_rq5.legend(); ax_rq5.grid(True, axis='y', linestyle=':')
                    plt.tight_layout(); plt.show()
                else: print("Не удалось извлечь данные для графика RQ5.")

            except Exception as e: print(f"  ❌ ОШИБКА при запуске/визуализации RQ5: {e}")

    except FileNotFoundError:
        print(f"❌ ОШИБКА: Файл данных SVI '{SVI_FILE}' не найден. Шаг 4 (RQ5) пропущен.")
    except ValueError as ve:
        print(f"❌ ОШИБКА данных SVI: {ve}. Шаг 4 (RQ5) пропущен.")
    except Exception as e_svi:
        print(f"❌ НЕОЖИДАННАЯ ОШИБКА в Шаге 4 (RQ5): {e_svi}")


    print("\n🎉 --- Анализ по Плану 'Насыщения' завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
# Стиль для графиков
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО ФИНАЛЬНОГО АНАЛИЗА ГЕТЕРОГЕННОСТИ (Medicaid vs Commercial) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
    print("   Пожалуйста, сначала выполните ячейку, которая создает 'final_data'.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (стандартный код очистки) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация 'New PC/Urgent' клиник ---
    df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_new_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_new_pc_urgent, axis=1)].copy()
    print(f"\n✅ Идентифицировано {len(df_new_PC_clinics)} 'New' PC/Urgent/FP клиник.")

    if df_new_PC_clinics.empty:
         raise ValueError("Не найдено 'New' PC/Urgent/FP клиник.")

    # --- 4. Расчет Карты Лечения (Только для New PC) ---
    print("\n--- Расчет Карты Лечения (только New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; nearby_new_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        treatment_dates_new_pc[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc}
    df_treatment_dates_new_pc = pd.DataFrame.from_dict(treatment_dates_new_pc, orient='index')
    print(f"\n✅ Карта лечения ('New PC') создана.")

    # --- 5. Шаг 1: Агрегация данных и создание Y-переменных по плательщику ---
    print("\n--- Агрегация данных и создание Y-переменных по плательщику ---")
    df_panel = final_data.copy()
    # Нужна колонка FinancialClassNM
    if 'FinancialClassNM' not in df_panel.columns:
         raise ValueError("Колонка 'FinancialClassNM' не найдена в 'final_data'.")

    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    # Определяем группы плательщиков
    df_panel['Is_Medicaid'] = (df_panel['FinancialClassNM'] == 'Medicaid').astype(int)
    df_panel['Is_Commercial'] = (df_panel['FinancialClassNM'] == 'Commercial').astype(int)

    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    # Агрегируем ER визиты отдельно для Medicaid и Commercial
    df_agg_payer = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        # Суммируем EncounterCount только для ER И нужного плательщика
        Total_ER_Visits_Medicaid = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[(df_panel.loc[x.index, 'Is_Emergency'] == 1) & (df_panel.loc[x.index, 'Is_Medicaid'] == 1)].sum()),
        Total_ER_Visits_Commercial = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[(df_panel.loc[x.index, 'Is_Emergency'] == 1) & (df_panel.loc[x.index, 'Is_Commercial'] == 1)].sum())
    ).reset_index()

    # Создаем Y-переменные (логарифм + 1)
    df_agg_payer['Y_ER_Medicaid'] = np.log1p(df_agg_payer['Total_ER_Visits_Medicaid'])
    df_agg_payer['Y_ER_Commercial'] = np.log1p(df_agg_payer['Total_ER_Visits_Commercial'])
    print("✅ Y_ER_Medicaid и Y_ER_Commercial созданы.")

    # --- 6. Шаг 2: Присоединить "Карту Лечения" ---
    df_analysis_payer = df_agg_payer.merge(df_treatment_dates_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis_payer['Date'] = pd.to_datetime(df_analysis_payer['Year_Month'] + '-01')
    df_analysis_payer = df_analysis_payer.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
    print("✅ Данные агрегированы и объединены с датами лечения 'New PC'.")

    # --- 7. Подготовка данных для Event Study ---
    df_model_payer = df_analysis_payer.copy()
    df_model_payer['event_month_new_pc'] = pd.to_datetime(df_model_payer['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_payer['current_month'] = pd.to_datetime(df_model_payer['Date']).dt.to_period('M')
    df_model_payer['time_to_event_NEW_PC'] = (df_model_payer['current_month'] - df_model_payer['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    # Отбираем ТОЛЬКО обработанные наблюдения
    df_model_payer_filtered = df_model_payer[
        df_model_payer['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_payer_filtered['time_to_event_NEW_PC'] = df_model_payer_filtered['time_to_event_NEW_PC'].astype(int)

    if df_model_payer_filtered.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска моделей.")
    else:
        print(f"Подготовлено {len(df_model_payer_filtered)} наблюдений для Event Study.")

        # --- 8. Шаг 3: Запуск Моделей ---
        models_payer = {}
        # Формула (общая для обеих моделей)
        formula_event_payer = "{outcome} ~ C(time_to_event_NEW_PC, Treatment(reference={ref})) + C(Zip) + C(Year) + C(Month)"

        # Модель 1 (Medicaid)
        OUTCOME_MCD = "Y_ER_Medicaid"
        print(f"\n--- Запуск Модели 1 ({OUTCOME_MCD}) ---")
        try:
            model_mcd = smf.ols(formula_event_payer.format(outcome=OUTCOME_MCD, ref=BASE_PERIOD), data=df_model_payer_filtered)
            models_payer['Medicaid'] = model_mcd.fit(cov_type='cluster', cov_kwds={'groups': df_model_payer_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_MCD} рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА Модели 1: {e}")
            # Можно добавить попытку без C(Zip) при необходимости

        # Модель 2 (Commercial)
        OUTCOME_COM = "Y_ER_Commercial"
        print(f"\n--- Запуск Модели 2 ({OUTCOME_COM}) ---")
        try:
            model_com = smf.ols(formula_event_payer.format(outcome=OUTCOME_COM, ref=BASE_PERIOD), data=df_model_payer_filtered)
            models_payer['Commercial'] = model_com.fit(cov_type='cluster', cov_kwds={'groups': df_model_payer_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_COM} рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА Модели 2: {e}")
            # Можно добавить попытку без C(Zip)

        # --- 9. Шаг 4: Визуализация ---
        print("\n--- Визуализация Результатов по Плательщику ---")

        # (Функция извлечения эффектов Event Study v3 - без изменений)
        def extract_event_study_effects_v3(model_result, base_period, time_var_name):
            pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
            if model_result is None: return pd.DataFrame()
            params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
            results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
            results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
            for idx in params.index:
                try:
                    match = re.search(pattern, idx)
                    if match:
                        month = int(match.group(1));
                        if month == base_period: continue
                        results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                        results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
                except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
            df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
            return df.sort_values('Relative_Month').reset_index(drop=True)

        # (Функция для построения графика Event Study - без изменений)
        def plot_event_study(event_data, outcome_name, title, color, event_window, base_period):
             if not event_data.empty:
                fig, ax = plt.subplots(figsize=(12, 7))
                errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
                valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                if valid_error.all(): ax.errorbar(x=event_data['Relative_Month'], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
                else: ax.plot(event_data['Relative_Month'], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
                ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax.set_title(title, fontsize=14); ax.set_xlabel("Месяцы относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
                ax.set_xticks(range(-event_window, event_window + 1, 2)); ax.legend(); ax.grid(True, axis='y', linestyle=':')
                plt.tight_layout(); plt.show()
             else: print(f"Не удалось извлечь/построить данные для: {title}")

        # График 1 (Medicaid)
        print("\n--- График 1: Эффект на ER Визиты Medicaid ---")
        event_data_mcd = extract_event_study_effects_v3(models_payer.get('Medicaid'), BASE_PERIOD, "time_to_event_NEW_PC")
        plot_event_study(event_data_mcd, OUTCOME_MCD, "График 1: Эффект 'New PC' Клиник на ER Визиты (Medicaid)", 'darkorange', EVENT_WINDOW, BASE_PERIOD)

        # График 2 (Commercial)
        print("\n--- График 2: Эффект на ER Визиты Commercial ---")
        event_data_com = extract_event_study_effects_v3(models_payer.get('Commercial'), BASE_PERIOD, "time_to_event_NEW_PC")
        plot_event_study(event_data_com, OUTCOME_COM, "График 2: Эффект 'New PC' Клиник на ER Визиты (Commercial)", 'darkcyan', EVENT_WINDOW, BASE_PERIOD)


    print("\n🎉 --- Финальный Анализ Гетерогенности (Medicaid vs Commercial) завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
# Стиль для графиков
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО АНАЛИЗА ГЕТЕРОГЕННОСТИ ПО СРОЧНОСТИ (Low vs High Acuity) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
    print("   Пожалуйста, сначала выполните ячейку, которая создает 'final_data'.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (стандартный код очистки) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация 'New PC/Urgent' клиник ---
    df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_new_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_new_pc_urgent, axis=1)].copy()
    print(f"\n✅ Идентифицировано {len(df_new_PC_clinics)} 'New' PC/Urgent/FP клиник.")

    # --- 4. Расчет Карты Лечения (Только для New PC) ---
    print("\n--- Расчет Карты Лечения (только New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; nearby_new_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        treatment_dates_new_pc[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc}
    df_treatment_dates_new_pc = pd.DataFrame.from_dict(treatment_dates_new_pc, orient='index')
    print(f"\n✅ Карта лечения ('New PC') создана.")

    # --- 5. Шаг 1: Категоризация Визитов в ER по Срочности ---
    print("\n--- Шаг 1: Категоризация ER визитов по срочности ---")
    df_panel_acuity = final_data.copy()

    # Списки ICD кодов (используем ICD_1char)
    # Используем множества для быстрой проверки
    low_acuity_icd = {'R50', 'R05', 'S93'}
    # Добавим общие респираторные J коды (ОРВИ, бронхит и т.п.), но исключим пневмонию (J12-J18)
    j_codes_low = {f'J{i:02d}' for i in range(0, 12)} | {f'J{i:02d}' for i in range(20, 23)} | {f'J{i:02d}' for i in range(30, 48)}
    low_acuity_icd.update(j_codes_low)

    high_acuity_icd = {'I21', 'I63', 'R07'}

    def categorize_acuity(row):
        icd = str(row['ICD_1char']).strip().upper()
        visit_type = str(row['VisitType']).strip()
        # Категоризируем ТОЛЬКО ER визиты
        if visit_type != 'Emergency':
            return 'Not_ER'
        # Проверяем ICD
        if icd in high_acuity_icd:
            return 'High_Acuity'
        elif icd in low_acuity_icd:
            return 'Low_Acuity'
        # Можно добавить EDPrimaryClinicalImpressionDSC, если нужно
        # ed_impression = str(row.get('EDPrimaryClinicalImpressionDSC', '')).lower()
        # if 'chest pain' in ed_impression: return 'High_Acuity'
        # if 'cough' in ed_impression or 'fever' in ed_impression: return 'Low_Acuity'
        else:
            return 'Medium_Acuity' # Все остальные ER визиты

    df_panel_acuity['Acuity_Level'] = df_panel_acuity.apply(categorize_acuity, axis=1)
    print("Распределение ER визитов по категориям срочности:")
    print(df_panel_acuity[df_panel_acuity['Acuity_Level'] != 'Not_ER']['Acuity_Level'].value_counts())

    # --- 6. Шаг 2: Агрегация данных и создание Y-переменных по срочности ---
    print("\n--- Шаг 2: Агрегация и создание Y-переменных по срочности ---")
    df_panel_acuity['Year_Month'] = df_panel_acuity['Date'].dt.strftime('%Y-%m')
    df_agg_acuity = df_panel_acuity.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        # Суммируем EncounterCount только для нужной категории срочности
        Total_ER_Visits_Low = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_acuity.loc[x.index, 'Acuity_Level'] == 'Low_Acuity'].sum()),
        Total_ER_Visits_High = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_acuity.loc[x.index, 'Acuity_Level'] == 'High_Acuity'].sum())
    ).reset_index()

    # Создаем Y-переменные (логарифм + 1)
    df_agg_acuity['Y_ER_Low_Acuity'] = np.log1p(df_agg_acuity['Total_ER_Visits_Low'])
    df_agg_acuity['Y_ER_High_Acuity'] = np.log1p(df_agg_acuity['Total_ER_Visits_High'])
    print("✅ Y_ER_Low_Acuity и Y_ER_High_Acuity созданы.")

    # Присоединяем Карту Лечения 'New PC'
    df_analysis_acuity = df_agg_acuity.merge(df_treatment_dates_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis_acuity['Date'] = pd.to_datetime(df_analysis_acuity['Year_Month'] + '-01')
    df_analysis_acuity = df_analysis_acuity.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
    print("✅ Данные агрегированы и объединены с датами лечения 'New PC'.")

    # --- 7. Подготовка данных для Event Study ---
    df_model_acuity = df_analysis_acuity.copy()
    df_model_acuity['event_month_new_pc'] = pd.to_datetime(df_model_acuity['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_acuity['current_month'] = pd.to_datetime(df_model_acuity['Date']).dt.to_period('M')
    df_model_acuity['time_to_event_NEW_PC'] = (df_model_acuity['current_month'] - df_model_acuity['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    # Отбираем ТОЛЬКО обработанные наблюдения
    df_model_acuity_filtered = df_model_acuity[
        df_model_acuity['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_acuity_filtered['time_to_event_NEW_PC'] = df_model_acuity_filtered['time_to_event_NEW_PC'].astype(int)

    if df_model_acuity_filtered.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска моделей.")
    else:
        print(f"Подготовлено {len(df_model_acuity_filtered)} наблюдений для Event Study по срочности.")

        # --- 8. Шаг 3: Запуск Моделей ---
        models_acuity = {}
        # Формула (общая)
        formula_event_acuity = "{outcome} ~ C(time_to_event_NEW_PC, Treatment(reference={ref})) + C(Zip) + C(Year) + C(Month)"

        # Модель 1 (Low Acuity)
        OUTCOME_LOW = "Y_ER_Low_Acuity"
        print(f"\n--- Запуск Модели 1 ({OUTCOME_LOW}) ---")
        try:
            model_low = smf.ols(formula_event_acuity.format(outcome=OUTCOME_LOW, ref=BASE_PERIOD), data=df_model_acuity_filtered)
            models_acuity['Low'] = model_low.fit(cov_type='cluster', cov_kwds={'groups': df_model_acuity_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_LOW} рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА Модели 1: {e}")
            # Можно добавить попытку без C(Zip)

        # Модель 2 (High Acuity)
        OUTCOME_HIGH = "Y_ER_High_Acuity"
        print(f"\n--- Запуск Модели 2 ({OUTCOME_HIGH}) ---")
        try:
            model_high = smf.ols(formula_event_acuity.format(outcome=OUTCOME_HIGH, ref=BASE_PERIOD), data=df_model_acuity_filtered)
            models_acuity['High'] = model_high.fit(cov_type='cluster', cov_kwds={'groups': df_model_acuity_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_HIGH} рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА Модели 2: {e}")
            # Можно добавить попытку без C(Zip)

        # --- 9. Шаг 4: Визуализация ---
        print("\n--- Визуализация Результатов по Срочности ---")

        # (Функция извлечения эффектов Event Study v3 - без изменений)
        def extract_event_study_effects_v3(model_result, base_period, time_var_name):
            # ... (Копируем функцию из предыдущего кода) ...
            pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
            if model_result is None: return pd.DataFrame()
            params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
            results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
            results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
            for idx in params.index:
                try:
                    match = re.search(pattern, idx);
                    if match:
                        month = int(match.group(1));
                        if month == base_period: continue
                        results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                        results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
                except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
            df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
            return df.sort_values('Relative_Month').reset_index(drop=True)

        # (Функция для построения графика Event Study - без изменений)
        def plot_event_study(event_data, outcome_name, title, color, event_window, base_period):
             # ... (Копируем функцию из предыдущего кода) ...
             if not event_data.empty:
                fig, ax = plt.subplots(figsize=(12, 7))
                errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
                valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                if valid_error.all(): ax.errorbar(x=event_data['Relative_Month'], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
                else: ax.plot(event_data['Relative_Month'], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
                ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax.set_title(title, fontsize=14); ax.set_xlabel("Месяцы относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
                ax.set_xticks(range(-event_window, event_window + 1, 2)); ax.legend(); ax.grid(True, axis='y', linestyle=':')
                plt.tight_layout(); plt.show()
             else: print(f"Не удалось извлечь/построить данные для: {title}")


        # График 1 (Low Acuity)
        print("\n--- График 1: Эффект на ER Визиты Low Acuity ---")
        event_data_low = extract_event_study_effects_v3(models_acuity.get('Low'), BASE_PERIOD, "time_to_event_NEW_PC")
        plot_event_study(event_data_low, OUTCOME_LOW, "График 1: Эффект 'New PC' Клиник на ER Визиты (Low Acuity)", 'darkseagreen', EVENT_WINDOW, BASE_PERIOD)

        
        # График 2 (High Acuity)
        print("\n--- График 2: Эффект на ER Визиты High Acuity ---")
        event_data_high = extract_event_study_effects_v3(models_acuity.get('High'), BASE_PERIOD, "time_to_event_NEW_PC")
        plot_event_study(event_data_high, OUTCOME_HIGH, "График 2: Эффект 'New PC' Клиник на ER Визиты (High Acuity)", 'firebrick', EVENT_WINDOW, BASE_PERIOD)


    print("\n🎉 --- Финальный Анализ Гетерогенности по Срочности завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
# Стиль для графиков
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО АНАЛИЗА ГЕТЕРОГЕННОСТИ ПО СРОЧНОСТИ (Low/Medium/High Acuity) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
    print("   Пожалуйста, сначала выполните ячейку, которая создает 'final_data'.")
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (стандартный код очистки) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация 'New PC/Urgent' клиник ---
    df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_new_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_new_pc_urgent, axis=1)].copy()
    print(f"\n✅ Идентифицировано {len(df_new_PC_clinics)} 'New' PC/Urgent/FP клиник.")

    # --- 4. Расчет Карты Лечения (Только для New PC) ---
    print("\n--- Расчет Карты Лечения (только New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10
    treatment_dates_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; nearby_new_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        treatment_dates_new_pc[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc}
    df_treatment_dates_new_pc = pd.DataFrame.from_dict(treatment_dates_new_pc, orient='index')
    print(f"\n✅ Карта лечения ('New PC') создана.")

    # --- 5. Шаг 1: Категоризация Визитов в ER по Срочности ---
    print("\n--- Шаг 1: Категоризация ER визитов по срочности ---")
    df_panel_acuity = final_data.copy()
    low_acuity_icd = {'R50', 'R05', 'S93'}
    j_codes_low = {f'J{i:02d}' for i in range(0, 12)} | {f'J{i:02d}' for i in range(20, 23)} | {f'J{i:02d}' for i in range(30, 48)}
    low_acuity_icd.update(j_codes_low)
    high_acuity_icd = {'I21', 'I63', 'R07'}
    def categorize_acuity(row):
        icd = str(row['ICD_1char']).strip().upper()
        visit_type = str(row['VisitType']).strip()
        if visit_type != 'Emergency': return 'Not_ER'
        if icd in high_acuity_icd: return 'High_Acuity'
        elif icd in low_acuity_icd: return 'Low_Acuity'
        else: return 'Medium_Acuity' # Остальное - Medium
    df_panel_acuity['Acuity_Level'] = df_panel_acuity.apply(categorize_acuity, axis=1)
    print("Распределение ER визитов по категориям срочности:")
    print(df_panel_acuity[df_panel_acuity['Acuity_Level'] != 'Not_ER']['Acuity_Level'].value_counts())

    # --- 6. Шаг 2: Агрегация данных и создание Y-переменных по срочности ---
    print("\n--- Шаг 2: Агрегация и создание Y-переменных по срочности ---")
    df_panel_acuity['Year_Month'] = df_panel_acuity['Date'].dt.strftime('%Y-%m')
    df_agg_acuity = df_panel_acuity.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_ER_Visits_Low = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_acuity.loc[x.index, 'Acuity_Level'] == 'Low_Acuity'].sum()),
        Total_ER_Visits_High = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_acuity.loc[x.index, 'Acuity_Level'] == 'High_Acuity'].sum()),
        # --- ДОБАВЛЕНО: Агрегация для Medium ---
        Total_ER_Visits_Medium = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_acuity.loc[x.index, 'Acuity_Level'] == 'Medium_Acuity'].sum())
    ).reset_index()

    # Создаем Y-переменные (логарифм + 1)
    df_agg_acuity['Y_ER_Low_Acuity'] = np.log1p(df_agg_acuity['Total_ER_Visits_Low'])
    df_agg_acuity['Y_ER_High_Acuity'] = np.log1p(df_agg_acuity['Total_ER_Visits_High'])
    # --- ДОБАВЛЕНО: Y для Medium ---
    df_agg_acuity['Y_ER_Medium_Acuity'] = np.log1p(df_agg_acuity['Total_ER_Visits_Medium'])
    print("✅ Y_ER_Low_Acuity, Y_ER_High_Acuity и Y_ER_Medium_Acuity созданы.")

    # Присоединяем Карту Лечения 'New PC'
    df_analysis_acuity = df_agg_acuity.merge(df_treatment_dates_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis_acuity['Date'] = pd.to_datetime(df_analysis_acuity['Year_Month'] + '-01')
    df_analysis_acuity = df_analysis_acuity.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
    print("✅ Данные агрегированы и объединены с датами лечения 'New PC'.")

    # --- 7. Подготовка данных для Event Study ---
    df_model_acuity = df_analysis_acuity.copy()
    df_model_acuity['event_month_new_pc'] = pd.to_datetime(df_model_acuity['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_acuity['current_month'] = pd.to_datetime(df_model_acuity['Date']).dt.to_period('M')
    df_model_acuity['time_to_event_NEW_PC'] = (df_model_acuity['current_month'] - df_model_acuity['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    EVENT_WINDOW = 12
    BASE_PERIOD = -1
    df_model_acuity_filtered = df_model_acuity[
        df_model_acuity['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_acuity_filtered['time_to_event_NEW_PC'] = df_model_acuity_filtered['time_to_event_NEW_PC'].astype(int)

    if df_model_acuity_filtered.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска моделей.")
    else:
        print(f"Подготовлено {len(df_model_acuity_filtered)} наблюдений для Event Study по срочности.")

        # --- 8. Шаг 3: Запуск Моделей ---
        models_acuity = {}
        formula_event_acuity = "{outcome} ~ C(time_to_event_NEW_PC, Treatment(reference={ref})) + C(Zip) + C(Year) + C(Month)"

        # Модель 1 (Low Acuity)
        OUTCOME_LOW = "Y_ER_Low_Acuity"
        print(f"\n--- Запуск Модели 1 ({OUTCOME_LOW}) ---")
        try:
            model_low = smf.ols(formula_event_acuity.format(outcome=OUTCOME_LOW, ref=BASE_PERIOD), data=df_model_acuity_filtered)
            models_acuity['Low'] = model_low.fit(cov_type='cluster', cov_kwds={'groups': df_model_acuity_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_LOW} рассчитана.")
        except Exception as e: print(f"  ❌ ОШИБКА Модели 1: {e}")

        # Модель 2 (High Acuity)
        OUTCOME_HIGH = "Y_ER_High_Acuity"
        print(f"\n--- Запуск Модели 2 ({OUTCOME_HIGH}) ---")
        try:
            model_high = smf.ols(formula_event_acuity.format(outcome=OUTCOME_HIGH, ref=BASE_PERIOD), data=df_model_acuity_filtered)
            models_acuity['High'] = model_high.fit(cov_type='cluster', cov_kwds={'groups': df_model_acuity_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_HIGH} рассчитана.")
        except Exception as e: print(f"  ❌ ОШИБКА Модели 2: {e}")

        # --- ДОБАВЛЕНО: Модель 3 (Medium Acuity) ---
        OUTCOME_MEDIUM = "Y_ER_Medium_Acuity"
        print(f"\n--- Запуск Модели 3 ({OUTCOME_MEDIUM}) ---")
        try:
            model_medium = smf.ols(formula_event_acuity.format(outcome=OUTCOME_MEDIUM, ref=BASE_PERIOD), data=df_model_acuity_filtered)
            models_acuity['Medium'] = model_medium.fit(cov_type='cluster', cov_kwds={'groups': df_model_acuity_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_MEDIUM} рассчитана.")
        except Exception as e: print(f"  ❌ ОШИБКА Модели 3: {e}")


        # --- 9. Шаг 4: Визуализация ---
        print("\n--- Визуализация Результатов по Срочности ---")

        # (Функции extract_event_study_effects_v3 и plot_event_study - без изменений)
        def extract_event_study_effects_v3(model_result, base_period, time_var_name):
            pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
            if model_result is None: return pd.DataFrame()
            params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
            results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
            results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
            for idx in params.index:
                try:
                    match = re.search(pattern, idx);
                    if match:
                        month = int(match.group(1));
                        if month == base_period: continue
                        results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                        results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
                except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
            df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
            return df.sort_values('Relative_Month').reset_index(drop=True)

        def plot_event_study(event_data, outcome_name, title, color, event_window, base_period):
             if not event_data.empty:
                fig, ax = plt.subplots(figsize=(12, 7))
                errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
                valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                if valid_error.all(): ax.errorbar(x=event_data['Relative_Month'], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
                else: ax.plot(event_data['Relative_Month'], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
                ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax.set_title(title, fontsize=14); ax.set_xlabel("Месяцы относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
                ax.set_xticks(range(-event_window, event_window + 1, 2)); ax.legend(); ax.grid(True, axis='y', linestyle=':')
                plt.tight_layout(); plt.show()
             else: print(f"Не удалось извлечь/построить данные для: {title}")


        # График 1 (Low Acuity)
        print("\n--- График 1: Эффект на ER Визиты Low Acuity ---")
        event_data_low = extract_event_study_effects_v3(models_acuity.get('Low'), BASE_PERIOD, "time_to_event_NEW_PC")
        plot_event_study(event_data_low, OUTCOME_LOW, "График 1: Эффект 'New PC' Клиник на ER Визиты (Low Acuity)", 'darkseagreen', EVENT_WINDOW, BASE_PERIOD)

        # --- ДОБАВЛЕНО: График 3 (Medium Acuity) ---
        print("\n--- График 3: Эффект на ER Визиты Medium Acuity ---")
        event_data_medium = extract_event_study_effects_v3(models_acuity.get('Medium'), BASE_PERIOD, "time_to_event_NEW_PC")
        plot_event_study(event_data_medium, OUTCOME_MEDIUM, "График 3: Эффект 'New PC' Клиник на ER Визиты (Medium Acuity)", 'skyblue', EVENT_WINDOW, BASE_PERIOD) # Новый цвет

        # График 2 (High Acuity)
        print("\n--- График 2: Эффект на ER Визиты High Acuity ---")
        event_data_high = extract_event_study_effects_v3(models_acuity.get('High'), BASE_PERIOD, "time_to_event_NEW_PC")
        plot_event_study(event_data_high, OUTCOME_HIGH, "График 2: Эффект 'New PC' Клиник на ER Визиты (High Acuity)", 'firebrick', EVENT_WINDOW, BASE_PERIOD)


    print("\n🎉 --- Финальный Анализ Гетерогенности по Срочности завершен! ---")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

print("--- 🚀 НАЧАЛО АНАЛИЗА ПО СРОЧНОСТИ (ПО КВАРТАЛАМ) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.")
else:
    # ... (Загрузка clinics_info и crosswalk - без изменений) ...
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
        raise

    # --- 2. Очистка Данных ---
    # ... (код очистки без изменений) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.")

    # --- 3. Идентификация 'New PC/Urgent' клиник ---
    df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_new_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_new_pc_urgent, axis=1)].copy()
    print(f"\n✅ Идентифицировано {len(df_new_PC_clinics)} 'New' PC/Urgent/FP клиник.")

    # --- 4. Расчет Карты Лечения (Только для New PC) ---
    print("\n--- Расчет Карты Лечения (только New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10; treatment_dates_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; nearby_new_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        treatment_dates_new_pc[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc}
    df_treatment_dates_new_pc = pd.DataFrame.from_dict(treatment_dates_new_pc, orient='index')
    print(f"\n✅ Карта лечения ('New PC') создана.")

    # --- 5. Категоризация Визитов в ER по Срочности ---
    print("\n--- Категоризация ER визитов по срочности ---")
    df_panel_acuity = final_data.copy()
    low_acuity_icd = {'R50', 'R05', 'S93'}; j_codes_low = {f'J{i:02d}' for i in range(0, 12)} | {f'J{i:02d}' for i in range(20, 23)} | {f'J{i:02d}' for i in range(30, 48)}; low_acuity_icd.update(j_codes_low)
    high_acuity_icd = {'I21', 'I63', 'R07'}
    def categorize_acuity(row):
        icd = str(row['ICD_1char']).strip().upper(); visit_type = str(row['VisitType']).strip()
        if visit_type != 'Emergency': return 'Not_ER'
        if icd in high_acuity_icd: return 'High_Acuity'
        elif icd in low_acuity_icd: return 'Low_Acuity'
        else: return 'Medium_Acuity'
    df_panel_acuity['Acuity_Level'] = df_panel_acuity.apply(categorize_acuity, axis=1)
    print(df_panel_acuity[df_panel_acuity['Acuity_Level'] != 'Not_ER']['Acuity_Level'].value_counts())

    # --- 6. Агрегация данных ПО КВАРТАЛАМ ---
    print("\n--- Агрегация по КВАРТАЛАМ и создание Y-переменных ---")
    # --- ИЗМЕНЕНИЕ: Используем Quarter ---
    df_panel_acuity['Quarter'] = pd.to_datetime(df_panel_acuity['Date']).dt.to_period('Q')
    df_agg_acuity_q = df_panel_acuity.groupby(['Filtered_Patient_ZipCode', 'Quarter']).agg(
        Total_ER_Visits_Low = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_acuity.loc[x.index, 'Acuity_Level'] == 'Low_Acuity'].sum()),
        Total_ER_Visits_High = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_acuity.loc[x.index, 'Acuity_Level'] == 'High_Acuity'].sum()),
        Total_ER_Visits_Medium = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_acuity.loc[x.index, 'Acuity_Level'] == 'Medium_Acuity'].sum()),
        # Сохраняем год/квартал для FE
        Year = pd.NamedAgg(column='Year', aggfunc='first'),
        Qtr = pd.NamedAgg(column='Quarter', aggfunc=lambda x: x.iloc[0].quarter) # Получаем номер квартала 1-4
    ).reset_index()

    # Создаем Y-переменные (логарифм + 1)
    df_agg_acuity_q['Y_ER_Low_Acuity'] = np.log1p(df_agg_acuity_q['Total_ER_Visits_Low'])
    df_agg_acuity_q['Y_ER_High_Acuity'] = np.log1p(df_agg_acuity_q['Total_ER_Visits_High'])
    df_agg_acuity_q['Y_ER_Medium_Acuity'] = np.log1p(df_agg_acuity_q['Total_ER_Visits_Medium'])
    print("✅ Y-переменные по срочности (квартальные) созданы.")

    # Присоединяем Карту Лечения 'New PC'
    df_analysis_acuity_q = df_agg_acuity_q.merge(df_treatment_dates_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis_acuity_q = df_analysis_acuity_q.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
    print("✅ Данные агрегированы (квартально) и объединены с датами лечения 'New PC'.")

    # --- 7. Подготовка данных для Event Study (ПО КВАРТАЛАМ) ---
    df_model_acuity_q = df_analysis_acuity_q.copy()
    # --- ИЗМЕНЕНИЕ: Считаем время в кварталах ---
    df_model_acuity_q['event_quarter_new_pc'] = pd.to_datetime(df_model_acuity_q['treatment_date_NEW_PC']).dt.to_period('Q')
    # df_model_acuity_q['current_quarter'] = df_model_acuity_q['Quarter'] # Уже есть Period[Q]
    df_model_acuity_q['time_to_event_QUARTER'] = (df_model_acuity_q['Quarter'] - df_model_acuity_q['event_quarter_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # --- ИЗМЕНЕНИЕ: Окно в кварталах (+/- 1 год = +/- 4 квартала) ---
    EVENT_WINDOW_Q = 4
    BASE_PERIOD_Q = -1 # Квартал перед событием

    df_model_acuity_q_filtered = df_model_acuity_q[
        df_model_acuity_q['time_to_event_QUARTER'].between(-EVENT_WINDOW_Q, EVENT_WINDOW_Q)
    ].copy()
    df_model_acuity_q_filtered['time_to_event_QUARTER'] = df_model_acuity_q_filtered['time_to_event_QUARTER'].astype(int)

    if df_model_acuity_q_filtered.empty:
         print("⚠️ Недостаточно данных в пределах окна [-4, 4] кварталов для запуска моделей.")
    else:
        print(f"Подготовлено {len(df_model_acuity_q_filtered)} наблюдений (ZIP-Квартал) для Event Study.")

        # --- 8. Шаг 3: Запуск Моделей (ПО КВАРТАЛАМ) ---
        models_acuity_q = {}
        # --- ИЗМЕНЕНИЕ: Формула с квартальной переменной и FE ---
        formula_event_acuity_q = "{outcome} ~ C(time_to_event_QUARTER, Treatment(reference={ref})) + C(Zip) + C(Year) + C(Qtr)"

        # Модель 1 (Low Acuity)
        OUTCOME_LOW = "Y_ER_Low_Acuity"
        print(f"\n--- Запуск Модели 1 ({OUTCOME_LOW}, Квартально) ---")
        try:
            model_low_q = smf.ols(formula_event_acuity_q.format(outcome=OUTCOME_LOW, ref=BASE_PERIOD_Q), data=df_model_acuity_q_filtered)
            models_acuity_q['Low'] = model_low_q.fit(cov_type='cluster', cov_kwds={'groups': df_model_acuity_q_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_LOW} (Квартально) рассчитана.")
        except Exception as e: print(f"  ❌ ОШИБКА Модели 1: {e}")

        # Модель 2 (High Acuity)
        OUTCOME_HIGH = "Y_ER_High_Acuity"
        print(f"\n--- Запуск Модели 2 ({OUTCOME_HIGH}, Квартально) ---")
        try:
            model_high_q = smf.ols(formula_event_acuity_q.format(outcome=OUTCOME_HIGH, ref=BASE_PERIOD_Q), data=df_model_acuity_q_filtered)
            models_acuity_q['High'] = model_high_q.fit(cov_type='cluster', cov_kwds={'groups': df_model_acuity_q_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_HIGH} (Квартально) рассчитана.")
        except Exception as e: print(f"  ❌ ОШИБКА Модели 2: {e}")

        # Модель 3 (Medium Acuity)
        OUTCOME_MEDIUM = "Y_ER_Medium_Acuity"
        print(f"\n--- Запуск Модели 3 ({OUTCOME_MEDIUM}, Квартально) ---")
        try:
            model_medium_q = smf.ols(formula_event_acuity_q.format(outcome=OUTCOME_MEDIUM, ref=BASE_PERIOD_Q), data=df_model_acuity_q_filtered)
            models_acuity_q['Medium'] = model_medium_q.fit(cov_type='cluster', cov_kwds={'groups': df_model_acuity_q_filtered['Zip']})
            print(f"  ✅ Модель для {OUTCOME_MEDIUM} (Квартально) рассчитана.")
        except Exception as e: print(f"  ❌ ОШИБКА Модели 3: {e}")


        # --- 9. Шаг 4: Визуализация (ПО КВАРТАЛАМ) ---
        print("\n--- Визуализация Результатов по Срочности (Квартально) ---")

        # --- ИЗМЕНЕНИЕ: Функция извлечения для кварталов ---
        def extract_event_study_effects_quarterly(model_result, base_period, time_var_name):
            pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
            if model_result is None: return pd.DataFrame()
            params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
            results = {'Relative_Quarter': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []} # Изменено имя колонки
            results['Relative_Quarter'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
            for idx in params.index:
                try:
                    match = re.search(pattern, idx);
                    if match:
                        qtr = int(match.group(1)); # Теперь это квартал
                        if qtr == base_period: continue
                        results['Relative_Quarter'].append(qtr); results['Effect'].append(params[idx])
                        results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
                except Exception as e: print(f"  Warning: Could not parse quarter from '{idx}': {e}")
            df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Quarter'])
            return df.sort_values('Relative_Quarter').reset_index(drop=True)

        # --- ИЗМЕНЕНИЕ: Функция построения графика для кварталов ---
        def plot_event_study_quarterly(event_data, outcome_name, title, color, event_window_q, base_period_q):
             if not event_data.empty:
                fig, ax = plt.subplots(figsize=(12, 7))
                errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
                valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                # Ось X теперь 'Relative_Quarter'
                if valid_error.all(): ax.errorbar(x=event_data['Relative_Quarter'], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
                else: ax.plot(event_data['Relative_Quarter'], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
                ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Квартал 0)') # Линия между -1 и 0
                ax.set_title(title, fontsize=14); ax.set_xlabel("Кварталы относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
                # Метки оси X для кварталов
                ax.set_xticks(range(-event_window_q, event_window_q + 1, 1));
                ax.legend(); ax.grid(True, axis='y', linestyle=':')
                plt.tight_layout(); plt.show()
             else: print(f"Не удалось извлечь/построить данные для: {title}")


        # График 1 (Low Acuity)
        print("\n--- График 1: Эффект на ER Визиты Low Acuity (Квартально) ---")
        event_data_low_q = extract_event_study_effects_quarterly(models_acuity_q.get('Low'), BASE_PERIOD_Q, "time_to_event_QUARTER")
        plot_event_study_quarterly(event_data_low_q, OUTCOME_LOW, "График 1: Эффект 'New PC' Клиник на ER Визиты (Low Acuity, Квартально)", 'darkseagreen', EVENT_WINDOW_Q, BASE_PERIOD_Q)

        # График 3 (Medium Acuity)
        print("\n--- График 3: Эффект на ER Визиты Medium Acuity (Квартально) ---")
        event_data_medium_q = extract_event_study_effects_quarterly(models_acuity_q.get('Medium'), BASE_PERIOD_Q, "time_to_event_QUARTER")
        plot_event_study_quarterly(event_data_medium_q, OUTCOME_MEDIUM, "График 3: Эффект 'New PC' Клиник на ER Визиты (Medium Acuity, Квартально)", 'skyblue', EVENT_WINDOW_Q, BASE_PERIOD_Q)

        # График 2 (High Acuity)
        print("\n--- График 2: Эффект на ER Визиты High Acuity (Квартально) ---")
        event_data_high_q = extract_event_study_effects_quarterly(models_acuity_q.get('High'), BASE_PERIOD_Q, "time_to_event_QUARTER")
        plot_event_study_quarterly(event_data_high_q, OUTCOME_HIGH, "График 2: Эффект 'New PC' Клиник на ER Визиты (High Acuity, Квартально)", 'firebrick', EVENT_WINDOW_Q, BASE_PERIOD_Q)


    print("\n🎉 --- Анализ Гетерогенности по Срочности (Квартально) завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО АНАЛИЗА ПО ПЛАНУ 'НАСЫЩЕНИЯ' (5 Регрессий) ---")

# --- 1. Проверка и Загрузка Данных ---
# Используем final_data, если он есть, иначе пытаемся загрузить агрегированные данные
# Это нужно, чтобы скрипт был самодостаточным, если запускается отдельно
if 'final_data' not in locals():
    print("Предупреждение: 'final_data' не найден. Пытаюсь загрузить 'aggregated_data.csv'")
    # ПРИМЕЧАНИЕ: Предполагается, что у вас есть файл с агрегированными данными
    # Если его нет, нужно сначала запустить код, который создает final_data
    try:
        final_data = pd.read_csv('aggregated_data.csv', dtype={'Filtered_Patient_ZipCode': str, 'Zip': str}) # Убедимся что Zip - строка
        # Преобразуем Date обратно, если нужно
        if 'Date' in final_data.columns and not pd.api.types.is_datetime64_any_dtype(final_data['Date']):
             final_data['Date'] = pd.to_datetime(final_data['Date'])
        # Переименуем Zip обратно для консистентности
        if 'Zip' in final_data.columns and 'Filtered_Patient_ZipCode' not in final_data.columns:
            final_data = final_data.rename(columns={'Zip': 'Filtered_Patient_ZipCode'})

        print("  ✅ 'aggregated_data.csv' загружен.")
    except FileNotFoundError:
        print("❌ ОШИБКА: DataFrame 'final_data' не найден и 'aggregated_data.csv' тоже.")
        raise FileNotFoundError("Необходим 'final_data' или 'aggregated_data.csv'")

# Загрузка остальных файлов
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (стандартный код очистки) ...
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
print("✅ Данные загружены и очищены.")

# --- 3. Идентификация групп клиник ('New PC' и 'Acquired PC') ---
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_pc_urgent, axis=1)].copy()
df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_urgent, axis=1))].copy()
print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

# --- 4. Расчет Дат Первого События (Метод Радиуса 10 км) ---
print("\n--- Расчет Дат Первого События (Метод Радиуса 10 км) ---")
TREATMENT_RADIUS_KM = 10; treatment_dates = {}
patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет Дат Лечения"):
    patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
    earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
    for _, clinic_row in df_new_PC_clinics.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
    if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
    for _, clinic_row in df_acquired_pc_clinics.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
    if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
    treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

# --- 5. Агрегация данных и создание Y-переменных ---
print("\n--- Агрегация данных и создание Y-переменных ---")
df_panel = final_data.copy()
df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int); df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
    Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
    Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum())
).reset_index()
df_agg['log_ER_Rate'] = np.log1p(df_agg['Total_Emergency_Visits'])
df_agg['log_Regular_Visits'] = np.log1p(df_agg['Total_Regular_Visits'])
df_agg['Y_Ratio'] = np.log1p(df_agg['Total_Emergency_Visits']) - np.log1p(df_agg['Total_Regular_Visits'])
print("✅ Y-переменные созданы: log_ER_Rate, log_Regular_Visits, Y_Ratio.")
df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'}) # Переименовываем ZIP
print("✅ Данные агрегированы и готовы к анализу.")

# --- Общие параметры и функции ---
EVENT_WINDOW = 12
BASE_PERIOD = -1

# Функция извлечения эффектов
def extract_event_study_effects_v3(model_result, base_period, time_var_name):
    # ... (копируем функцию v3 из предыдущего кода) ...
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                month = int(match.group(1));
                if month == base_period: continue
                results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
    return df.sort_values('Relative_Month').reset_index(drop=True)

# Функция построения графика
def plot_event_study(event_data, outcome_name, title, color, event_window, base_period):
     # ... (копируем функцию из предыдущего кода) ...
     if not event_data.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax.errorbar(x=event_data['Relative_Month'], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
        else: ax.plot(event_data['Relative_Month'], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
        ax.set_title(title, fontsize=14); ax.set_xlabel("Месяцы относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
        ax.set_xticks(range(-event_window, event_window + 1, 2)); ax.legend(); ax.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
     else: print(f"Не удалось извлечь/построить данные для: {title}")

# Функция для запуска и обработки ошибок модели
def run_event_study_model(formula, data, cluster_var='Zip'):
    model_result = None
    if not data.empty:
        try:
            print(f"... Запуск модели с C({cluster_var}) ...")
            model = smf.ols(formula, data=data)
            model_result = model.fit(cov_type='cluster', cov_kwds={'groups': data[cluster_var]})
            print(f"  ✅ Модель с C({cluster_var}) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА с C({cluster_var}): {e}")
            # Откат к модели без C(Zip)
            try:
                 formula_no_zip = formula.replace(f"+ C({cluster_var})", "") # Убираем FE Zip
                 print(f"     ... Попытка без C({cluster_var}) ...")
                 model_no_zip = smf.ols(formula_no_zip, data=data)
                 model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': data[cluster_var]}) # Кластеризуем все равно
                 print(f"        ✅ Модель без C({cluster_var}) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}")
                 model_result = None # Модель не сошлась
    else:
        print("  ⚠️ Нет данных для запуска модели.")
    return model_result

# =========================================================================
# --- ШАГ 1 (RQ1 & RQ2): Основные эффекты на log_ER_Rate ---
# =========================================================================
print("\n\n" + "="*70)
print("--- ШАГ 1 (RQ1 & RQ2): Основные эффекты на log_ER_Rate ---")
print("="*70)
OUTCOME_RQ12 = "log_ER_Rate"
model_results_RQ12 = {'New': None, 'Acquired': None}

# --- Данные для 'New PC' ---
df_model_new = df_analysis.copy()
df_model_new['event_month'] = pd.to_datetime(df_model_new['treatment_date_NEW_PC']).dt.to_period('M')
df_model_new['current_month'] = pd.to_datetime(df_model_new['Date']).dt.to_period('M')
df_model_new['time_to_event'] = (df_model_new['current_month'] - df_model_new['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
df_model_new_filtered = df_model_new[df_model_new['time_to_event'].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
df_model_new_filtered['time_to_event'] = df_model_new_filtered['time_to_event'].astype(int)
formula_rq1 = f"{OUTCOME_RQ12} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
print(f"\nЗапуск RQ1 (New PC на {OUTCOME_RQ12})...")
model_results_RQ12['New'] = run_event_study_model(formula_rq1, df_model_new_filtered)

# --- Данные для 'Acquired PC' ---
df_model_acq = df_analysis.copy()
df_model_acq['event_month'] = pd.to_datetime(df_model_acq['treatment_date_ACQUIRED_PC']).dt.to_period('M')
df_model_acq['current_month'] = pd.to_datetime(df_model_acq['Date']).dt.to_period('M')
df_model_acq['time_to_event'] = (df_model_acq['current_month'] - df_model_acq['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
df_model_acq_filtered = df_model_acq[df_model_acq['time_to_event'].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
df_model_acq_filtered['time_to_event'] = df_model_acq_filtered['time_to_event'].astype(int)
formula_rq2 = f"{OUTCOME_RQ12} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
print(f"\nЗапуск RQ2 (Acquired PC на {OUTCOME_RQ12})...")
model_results_RQ12['Acquired'] = run_event_study_model(formula_rq2, df_model_acq_filtered)

# --- Визуализация RQ1 & RQ2 ---
print("\n--- Визуализация Шага 1 ---")
event_data_rq1 = extract_event_study_effects_v3(model_results_RQ12.get('New'), BASE_PERIOD, "time_to_event")
event_data_rq2 = extract_event_study_effects_v3(model_results_RQ12.get('Acquired'), BASE_PERIOD, "time_to_event")
plot_event_study(event_data_rq1, OUTCOME_RQ12, "Шаг 1 (RQ1): Эффект 'New PC' на Log ER Visits", 'blue', EVENT_WINDOW, BASE_PERIOD)
plot_event_study(event_data_rq2, OUTCOME_RQ12, "Шаг 1 (RQ2): Эффект 'Acquired PC' на Log ER Visits", 'red', EVENT_WINDOW, BASE_PERIOD)


# =========================================================================
# --- ШАГ 2 (RQ3 - Механизм 'New'): Эффект New PC на log_Regular_Visits ---
# =========================================================================
print("\n\n" + "="*70)
print("--- ШАГ 2 (RQ3 - Механизм 'New'): Эффект на log_Regular_Visits ---")
print("="*70)
OUTCOME_RQ3 = "log_Regular_Visits"
formula_rq3 = f"{OUTCOME_RQ3} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
print(f"\nЗапуск RQ3 (New PC на {OUTCOME_RQ3})...")
# Используем те же данные, что и для RQ1 (df_model_new_filtered)
model_result_RQ3 = run_event_study_model(formula_rq3, df_model_new_filtered)

# --- Визуализация RQ3 ---
print("\n--- Визуализация Шага 2 ---")
event_data_rq3 = extract_event_study_effects_v3(model_result_RQ3, BASE_PERIOD, "time_to_event")
plot_event_study(event_data_rq3, OUTCOME_RQ3, "Шаг 2 (RQ3): Эффект 'New PC' на Log Regular Visits", 'green', EVENT_WINDOW, BASE_PERIOD)


# =============================================================================
# --- ШАГ 3 (RQ4 - Механизм 'Acquired'): Эффект Acquired PC на log_Regular_Visits ---
# =============================================================================
print("\n\n" + "="*70)
print("--- ШАГ 3 (RQ4 - Механизм 'Acquired'): Эффект на log_Regular_Visits ---")
print("="*70)
OUTCOME_RQ4 = "log_Regular_Visits"
formula_rq4 = f"{OUTCOME_RQ4} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
print(f"\nЗапуск RQ4 (Acquired PC на {OUTCOME_RQ4})...")
# Используем те же данные, что и для RQ2 (df_model_acq_filtered)
model_result_RQ4 = run_event_study_model(formula_rq4, df_model_acq_filtered)

# --- Визуализация RQ4 ---
print("\n--- Визуализация Шага 3 ---")
event_data_rq4 = extract_event_study_effects_v3(model_result_RQ4, BASE_PERIOD, "time_to_event")
plot_event_study(event_data_rq4, OUTCOME_RQ4, "Шаг 3 (RQ4): Эффект 'Acquired PC' на Log Regular Visits", 'orange', EVENT_WINDOW, BASE_PERIOD)


# =========================================================================
# --- ШАГ 4 (RQ5 - Гетерогенность - Абстракт): Y_Ratio ~ time * SVI ---
# =========================================================================
print("\n\n" + "="*70)
print("--- ШАГ 4 (RQ5 - Гетерогенность - Абстракт): Y_Ratio ~ time * SVI ---")
print("="*70)
OUTCOME_RQ5 = "Y_Ratio"
model_result_RQ5 = None

# --- Загрузка и подготовка SVI данных ---
SVI_FILE = 'final_filtered_dataset_reordered.csv' # Используем исходный файл
# --- ИЗМЕНЕНИЕ: Выбираем колонку SVI ---
SVI_COLUMN_NAME = 'X..Uninsured.Adults_zip' # Пример - процент незастрахованных
# SVI_COLUMN_NAME = 'MedianHHInc' # Другой пример - медианный доход (если есть)

try:
    print(f"... Загрузка данных SVI ({SVI_COLUMN_NAME}) из {SVI_FILE} ...")
    # Загружаем только ZIP и нужную колонку SVI
    df_svi_raw = pd.read_csv(SVI_FILE, usecols=['Filtered_Patient_ZipCode', SVI_COLUMN_NAME],
                             dtype={'Filtered_Patient_ZipCode': str}, low_memory=False)
    # Очищаем ZIP
    df_svi_raw['Filtered_Patient_ZipCode'] = df_svi_raw['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_svi_raw = df_svi_raw.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

    # Оставляем уникальные значения по ZIP, берем первое (предполагая, что SVI не меняется по времени в наших данных)
    df_svi = df_svi_raw.drop_duplicates(subset=['Zip']).dropna(subset=[SVI_COLUMN_NAME])

    # Преобразуем SVI в числовое значение (если нужно)
    if not pd.api.types.is_numeric_dtype(df_svi[SVI_COLUMN_NAME]):
        df_svi[SVI_COLUMN_NAME] = pd.to_numeric(df_svi[SVI_COLUMN_NAME], errors='coerce')
        df_svi = df_svi.dropna(subset=[SVI_COLUMN_NAME])

    # --- Бинаризация SVI по медиане ---
    svi_median = df_svi[SVI_COLUMN_NAME].median()
    df_svi['SVI_Category'] = np.where(df_svi[SVI_COLUMN_NAME] >= svi_median, 'High_Value', 'Low_Value')
    # Переименуем для ясности (например, High_Uninsured / Low_Uninsured)
    group_high_name = f'High_{SVI_COLUMN_NAME.split("_")[0].replace("X..","")}' # e.g., High_Uninsured
    group_low_name = f'Low_{SVI_COLUMN_NAME.split("_")[0].replace("X..","")}'   # e.g., Low_Uninsured
    df_svi['SVI_Category'] = df_svi['SVI_Category'].replace({'High_Value': group_high_name, 'Low_Value': group_low_name})


    print(f"  ✅ Данные SVI подготовлены для {len(df_svi)} ZIP-кодов.")
    print(f"     Разделение по медиане ({svi_median:.2f}):")
    print(df_svi['SVI_Category'].value_counts())

    # --- Присоединение SVI к данным модели RQ1 ---
    # Используем df_model_new_filtered (данные для RQ1)
    df_model_rq5 = df_model_new_filtered.merge(df_svi[['Zip', 'SVI_Category']], on='Zip', how='inner') # INNER join!

    if df_model_rq5.empty or df_model_rq5['SVI_Category'].nunique() < 2:
        print("  ❌ ОШИБКА: После слияния с SVI не осталось данных или только одна категория SVI.")
    else:
        print(f"  Подготовлено {len(df_model_rq5)} наблюдений для анализа гетерогенности.")
        print("  Распределение SVI категорий в данных модели:")
        print(df_model_rq5['SVI_Category'].value_counts())

        # --- Формула с взаимодействием ---
        # Важно: C(SVI_Category) должна быть последней во взаимодействии для правильного извлечения
        formula_rq5 = f"{OUTCOME_RQ5} ~ C(time_to_event, Treatment(reference={BASE_PERIOD})) * C(SVI_Category) + C(Zip) + C(Year) + C(Month)"
        print(f"\nЗапуск RQ5 (Гетерогенность New PC на {OUTCOME_RQ5} по SVI)...")

        model_result_RQ5 = run_event_study_model(formula_rq5, df_model_rq5) # Используем общую функцию запуска

        # --- Извлечение и Визуализация RQ5 ---
        print("\n--- Визуализация RQ5: Эффект по группам SVI ---")

        if model_result_RQ5:
            svi_groups = sorted(df_model_rq5['SVI_Category'].unique()) # Сортируем для консистентности
            event_data_rq5 = {}
            base_svi_group = svi_groups[0] # Первая категория будет базой

            # Эффект для базовой группы SVI
            base_effects = extract_event_study_effects_v3(model_result_RQ5, BASE_PERIOD, "time_to_event")
            if not base_effects.empty:
                 event_data_rq5[base_svi_group] = base_effects

            # Эффект для второй группы SVI (База + Взаимодействие)
            if len(svi_groups) > 1:
                 other_svi_group = svi_groups[1]
                 results_other = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
                 results_other['Relative_Month'].append(BASE_PERIOD); results_other['Effect'].append(0); results_other['Conf_Low'].append(0); results_other['Conf_High'].append(0)

                 for month in base_effects['Relative_Month']:
                      if month == BASE_PERIOD: continue
                      base_effect_row = base_effects[base_effects['Relative_Month'] == month].iloc[0]
                      base_term_name = f"C(time_to_event, Treatment(reference={BASE_PERIOD}))[T.{month}]"
                      # Имя взаимодействия может немного отличаться, пробуем оба варианта
                      interaction_term_name1 = f"C(time_to_event, Treatment(reference={BASE_PERIOD}))[T.{month}]:C(SVI_Category)[T.{other_svi_group}]"
                      interaction_term_name2 = f"C(SVI_Category)[T.{other_svi_group}]:C(time_to_event, Treatment(reference={BASE_PERIOD}))[T.{month}]"

                      interaction_term_name = None
                      if interaction_term_name1 in model_result_RQ5.params.index:
                           interaction_term_name = interaction_term_name1
                      elif interaction_term_name2 in model_result_RQ5.params.index:
                           interaction_term_name = interaction_term_name2

                      if interaction_term_name and base_term_name in model_result_RQ5.params.index:
                           try:
                                t_test = model_result_RQ5.t_test(f"{base_term_name} + {interaction_term_name}")
                                results_other['Relative_Month'].append(month); results_other['Effect'].append(t_test.effect[0])
                                results_other['Conf_Low'].append(t_test.conf_int()[0][0]); results_other['Conf_High'].append(t_test.conf_int()[0][1])
                           except Exception as e_ttest:
                                print(f"    Warning: t_test failed for month {month}: {e_ttest}")
                                # Откат к простой сумме (ДИ будет некорректным)
                                results_other['Relative_Month'].append(month); results_other['Effect'].append(base_effect_row['Effect'] + model_result_RQ5.params[interaction_term_name])
                                results_other['Conf_Low'].append(np.nan); results_other['Conf_High'].append(np.nan)
                      else: # Если взаимодействие не найдено, эффект = базовому
                           results_other['Relative_Month'].append(month); results_other['Effect'].append(base_effect_row['Effect'])
                           results_other['Conf_Low'].append(base_effect_row['Conf_Low']); results_other['Conf_High'].append(base_effect_row['Conf_High'])

                 df_other = pd.DataFrame(results_other).sort_values('Relative_Month').reset_index(drop=True)
                 if not df_other.empty: event_data_rq5[other_svi_group] = df_other

            # Строим график RQ5
            fig_rq5, ax_rq5 = plt.subplots(figsize=(12, 7))
            colors = plt.cm.viridis(np.linspace(0, 1, len(event_data_rq5))) # Разные цвета
            markers = ['o', 's', '^', 'd']
            linestyles = ['-', '--', ':', '-.']
            plotted = False

            for i, (group_name, group_data) in enumerate(event_data_rq5.items()):
                 if not group_data.empty:
                      errors = [group_data['Effect'] - group_data['Conf_Low'], group_data['Conf_High'] - group_data['Effect']]
                      valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                      fmt = f'{linestyles[i%len(linestyles)]}{markers[i%len(markers)]}'
                      if valid_error.all():
                           ax_rq5.errorbar(x=group_data['Relative_Month'], y=group_data['Effect'], yerr=errors,
                                         fmt=fmt, capsize=3, label=f'Эффект для {group_name}', color=colors[i])
                      else:
                           ax_rq5.plot(group_data['Relative_Month'], group_data['Effect'], marker=markers[i%len(markers)], linestyle=linestyles[i%len(linestyles)],
                                      label=f'Эффект для {group_name} (без ДИ)', color=colors[i])
                      plotted = True

            if plotted:
                ax_rq5.axhline(0, color='black', linestyle='-', linewidth=0.8)
                ax_rq5.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax_rq5.set_title(f"Шаг 4 (RQ5): Гетерогенный Эффект 'New PC' на Y_Ratio по {SVI_COLUMN_NAME}", fontsize=14)
                ax_rq5.set_xlabel("Месяцы относительно открытия"); ax_rq5.set_ylabel(f"Эффект на {OUTCOME_RQ5}")
                ax_rq5.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2)); ax_rq5.legend(); ax_rq5.grid(True, axis='y', linestyle=':')
                plt.tight_layout(); plt.show()
            else: print("Не удалось извлечь данные для графика RQ5.")
        else:
             print("Модель RQ5 не была рассчитана, график не строится.")

except FileNotFoundError:
    print(f"❌ ОШИБКА: Файл данных SVI '{SVI_FILE}' не найден. Шаг 4 (RQ5) пропущен.")
except KeyError as ke:
     print(f"❌ ОШИБКА: Колонка '{ke}' не найдена в файле SVI '{SVI_FILE}'. Проверьте '{SVI_COLUMN_NAME}'. Шаг 4 (RQ5) пропущен.")
except ValueError as ve:
    print(f"❌ ОШИБКА данных SVI: {ve}. Шаг 4 (RQ5) пропущен.")
except Exception as e_svi:
    print(f"❌ НЕОЖИДАННАЯ ОШИБКА в Шаге 4 (RQ5): {e_svi}")


print("\n🎉 --- Анализ по Плану 'Насыщения' завершен! ---")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

print("🚀 Перерасчёт и сглаженный Event Study с поправкой на население...")

# 📌 Настройки
EVENT_WINDOW = 12
BASE_PERIOD = -1
GROUP_WIDTH = 3  # Окно сглаживания в месяцах (по 3 месяца в группе)

# ✅ 1. Пересчёт лог-рейтов ER с поправкой на население
df_analysis['ER_Rate_per_1000'] = (df_analysis['Total_Emergency_Visits'] / df_analysis['Total_Regular_Visits'].replace(0, np.nan)) * 1000
df_analysis['log_ER_Rate_per_1000'] = np.log1p(df_analysis['ER_Rate_per_1000'])

# 🔁 Группировка месяцев в окна
def group_relative_month(x, width=GROUP_WIDTH):
    if x == -999:
        return np.nan
    return int(np.floor(x / width) * width)

# --- Для New PC
df_new = df_analysis.copy()
df_new['event_month'] = pd.to_datetime(df_new['treatment_date_NEW_PC']).dt.to_period('M')
df_new['current_month'] = df_new['Date'].dt.to_period('M')
df_new['relative_month'] = (df_new['current_month'] - df_new['event_month']).apply(lambda x: x.n if pd.notnull(x) else np.nan)
df_new = df_new[df_new['relative_month'].between(-EVENT_WINDOW, EVENT_WINDOW)]
df_new['month_group'] = df_new['relative_month'].apply(lambda x: group_relative_month(x, width=GROUP_WIDTH))

# --- Для Acquired PC
df_acq = df_analysis.copy()
df_acq['event_month'] = pd.to_datetime(df_acq['treatment_date_ACQUIRED_PC']).dt.to_period('M')
df_acq['current_month'] = df_acq['Date'].dt.to_period('M')
df_acq['relative_month'] = (df_acq['current_month'] - df_acq['event_month']).apply(lambda x: x.n if pd.notnull(x) else np.nan)
df_acq = df_acq[df_acq['relative_month'].between(-EVENT_WINDOW, EVENT_WINDOW)]
df_acq['month_group'] = df_acq['relative_month'].apply(lambda x: group_relative_month(x, width=GROUP_WIDTH))

# ✅ Функция агрегации и сглаживания
def aggregate_and_plot(df, outcome_var, title, color):
    grouped = df.groupby('month_group').agg(
        mean_effect=(outcome_var, 'mean'),
        std_effect=(outcome_var, 'std'),
        n=('Zip', 'nunique')
    ).reset_index()

    grouped['se'] = grouped['std_effect'] / np.sqrt(grouped['n'])
    grouped['ci_low'] = grouped['mean_effect'] - 1.96 * grouped['se']
    grouped['ci_high'] = grouped['mean_effect'] + 1.96 * grouped['se']

    # 📈 LOWESS сглаживание
    smoothed = lowess(grouped['mean_effect'], grouped['month_group'], frac=0.3, return_sorted=False)

    # 📊 Построение графика
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(grouped['month_group'], smoothed, color=color, label='Сглаженный эффект', linewidth=2)
    ax.fill_between(grouped['month_group'], grouped['ci_low'], grouped['ci_high'], color=color, alpha=0.2, label='95% ДИ')
    ax.axhline(0, linestyle='--', color='black', linewidth=0.8)
    ax.axvline(0, linestyle=':', color='grey', label='Событие (Месяц 0)')
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Месяцы относительно события (группы по 3 мес.)")
    ax.set_ylabel(f"Средний эффект: {outcome_var}")
    ax.legend()
    ax.grid(True, linestyle=':')
    plt.tight_layout()
    plt.show()

# 📉 Построение графиков
aggregate_and_plot(df_new, 'log_ER_Rate_per_1000', "Сглаженный эффект New PC на log(ER Rate per 1000)", 'blue')
aggregate_and_plot(df_acq, 'log_ER_Rate_per_1000', "Сглаженный эффект Acquired PC на log(ER Rate per 1000)", 'red')


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО EVENT STUDY С ВЗАИМОДЕЙСТВИЕМ ПО РАССТОЯНИЮ (Acquired PC) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.") # Сообщение на русском
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.") # Сообщение на английском для traceback
        raise

    # --- 2. Очистка Данных ---
    # ... (стандартный код очистки) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.") # Сообщение на русском

    # --- 3. Идентификация 'Acquired PC' клиник ---
    pc_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_acquired_pc_clinics)} Acquired PC клиник.") # Сообщение на русском

    # --- 4. Расчет Карты Лечения и Расстояния (Только для Acquired PC) ---
    print("\n--- Расчет Карты Лечения и Расстояния (Acquired PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10; treatment_info_acq_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    
    # Функция для нахождения ближайшей клиники *из группы* и ее расстояния
    def find_nearest_clinic_details(patient_coords, clinic_group_df):
        min_dist = np.inf; nearest_info = {}
        if clinic_group_df.empty: return nearest_info
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon']); distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist:
                min_dist = distance
                nearest_info = {'distance_km': min_dist, 'event_date': clinic_row['event_date']}
        return nearest_info

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (Acquired PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_acquired_pc = pd.NaT; nearby_acquired_pc_clinics = []
        
        # Находим ВСЕ Acquired PC клиники в радиусе
        for _, clinic_row in df_acquired_pc_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance <= TREATMENT_RADIUS_KM:
                nearby_acquired_pc_clinics.append({'date': clinic_row['event_date'], 'distance': distance})
                
        # Находим самую раннюю дату среди клиник в радиусе
        if nearby_acquired_pc_clinics:
            earliest_date_acquired_pc = min(c['date'] for c in nearby_acquired_pc_clinics)
            
        # Находим расстояние до САМОЙ БЛИЖАЙШЕЙ Acquired PC клиники (даже если она дальше 10 км)
        nearest_clinic_details = find_nearest_clinic_details(patient_coords, df_acquired_pc_clinics)
        distance_to_nearest = nearest_clinic_details.get('distance_km', np.nan)

        treatment_info_acq_pc[patient_zip] = {
            'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc,
            'distance_to_nearest_ACQUIRED_PC_km': distance_to_nearest # Это расстояние используем для категорий
        }

    df_treatment_info_acq_pc = pd.DataFrame.from_dict(treatment_info_acq_pc, orient='index')
    print(f"\n✅ Карта лечения и расстояния ('Acquired PC') создана.")

    # --- 5. Агрегация данных и создание Y-переменной ---
    print("\n--- Агрегация данных и создание Y-переменной ---")
    df_panel = final_data.copy()
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    df_agg['log_ER_Rate'] = np.log1p(df_agg['Total_Emergency_Visits']) # Используем log(Visits+1)
    print("✅ Y-переменная создана: log_ER_Rate.")

    # Присоединяем информацию о лечении и расстоянии
    df_analysis_acq_dist = df_agg.merge(df_treatment_info_acq_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis_acq_dist['Date'] = pd.to_datetime(df_analysis_acq_dist['Year_Month'] + '-01')
    df_analysis_acq_dist = df_analysis_acq_dist.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

    # --- 6. Создание Категорий Расстояния и Переменных Event Study ---
    # Используем 'distance_to_nearest_ACQUIRED_PC_km'
    bins_dist = [-np.inf, 5, 10, 20, np.inf]
    labels_dist = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    # Базовая категория для взаимодействия
    REF_DISTANCE = '20+ km'
    df_analysis_acq_dist['distance_category'] = pd.cut(
        df_analysis_acq_dist['distance_to_nearest_ACQUIRED_PC_km'],
        bins=bins_dist, labels=labels_dist, right=False
    )
    # Устанавливаем базовую категорию явно
    df_analysis_acq_dist['distance_category'] = pd.Categorical(df_analysis_acq_dist['distance_category'], categories=labels_dist, ordered=True)
    # df_analysis_acq_dist['distance_category'] = df_analysis_acq_dist['distance_category'].cat.relevel({REF_DISTANCE}) # Не работает в старом pandas
    print("\nКатегории расстояния созданы:")
    print(df_analysis_acq_dist['distance_category'].value_counts())

    # Рассчитываем time_to_event_ACQUIRED_PC
    df_analysis_acq_dist['event_month_acq_pc'] = pd.to_datetime(df_analysis_acq_dist['treatment_date_ACQUIRED_PC']).dt.to_period('M')
    df_analysis_acq_dist['current_month'] = pd.to_datetime(df_analysis_acq_dist['Date']).dt.to_period('M')
    df_analysis_acq_dist['time_to_event_ACQUIRED_PC'] = (df_analysis_acq_dist['current_month'] - df_analysis_acq_dist['event_month_acq_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # Отбираем ТОЛЬКО обработанные наблюдения для Event Study
    EVENT_WINDOW = 12; BASE_PERIOD = -1
    df_model_acq_dist = df_analysis_acq_dist[
        df_analysis_acq_dist['time_to_event_ACQUIRED_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_acq_dist['time_to_event_ACQUIRED_PC'] = df_model_acq_dist['time_to_event_ACQUIRED_PC'].astype(int)

    # Убираем строки с пропущенной категорией расстояния (если есть)
    df_model_acq_dist = df_model_acq_dist.dropna(subset=['distance_category'])

    if df_model_acq_dist.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска модели.")
    else:
        print(f"Подготовлено {len(df_model_acq_dist)} наблюдений для Event Study с взаимодействием.")

        # --- 7. Запуск Модели с Взаимодействием ---
        OUTCOME = "log_ER_Rate"
        # --- ИЗМЕНЕНИЕ: Формула с взаимодействием и Treatment() для расстояния ---
        formula_event_dist = (f"{OUTCOME} ~ "
                              f"C(time_to_event_ACQUIRED_PC, Treatment(reference={BASE_PERIOD})) * C(distance_category, Treatment(reference='{REF_DISTANCE}')) + "
                              f"C(Zip) + C(Year) + C(Month)")

        model_event_acq_dist_result = None
        print(f"\n--- Запуск Модели Event Study с взаимодействием по расстоянию ({OUTCOME}) ---")
        try:
            print(f"... Запуск модели с C(Zip) ...")
            # --- Используем данные БЕЗ пропусков по категориям ---
            data_for_model = df_model_acq_dist.dropna(subset=['distance_category', 'time_to_event_ACQUIRED_PC', OUTCOME, 'Zip', 'Year', 'Month'])

            if data_for_model.empty or data_for_model['distance_category'].nunique() < 2 or data_for_model['time_to_event_ACQUIRED_PC'].nunique() < 2:
                 print("  ⚠️ Недостаточно данных/вариации после очистки NaN для запуска.")
            else:
                 model_event_dist = smf.ols(formula_event_dist, data=data_for_model)
                 model_event_acq_dist_result = model_event_dist.fit(cov_type='cluster', cov_kwds={'groups': data_for_model['Zip']})
                 print("  ✅ Модель с взаимодействием по расстоянию рассчитана.")
                 # print(model_event_acq_dist_result.summary()) # Для детального просмотра

        except Exception as e:
            print(f"  ❌ ОШИБКА Модели с взаимодействием: {e}")
            # Можно добавить откат к модели без C(Zip), если нужно

        # --- 8. Извлечение и Визуализация Результатов ---
        print("\n--- Визуализация Event Study с взаимодействием по расстоянию ---")

        # --- НОВАЯ Функция извлечения эффектов с взаимодействием ---
        def extract_event_study_interaction_effects(model_result, base_period, time_var_name, dist_var_name, dist_categories, ref_dist_category):
            if model_result is None: return {}
            
            all_effects = {}
            time_points = sorted([p for p in range(-EVENT_WINDOW, EVENT_WINDOW + 1) if p != base_period])
            
            # Эффект для референсной категории расстояния
            ref_effects_list = {'Relative_Month': [base_period], 'Effect': [0], 'Conf_Low': [0], 'Conf_High': [0]}
            time_pattern_base = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.({('|').join(map(str, time_points))})\](?!:)" # Только основной эффект времени
            
            base_params = model_result.params.filter(regex=time_pattern_base)
            base_conf = model_result.conf_int().filter(regex=time_pattern_base, axis=0)
            
            for k in time_points:
                term_name = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                if term_name in base_params.index:
                    ref_effects_list['Relative_Month'].append(k)
                    ref_effects_list['Effect'].append(base_params[term_name])
                    ref_effects_list['Conf_Low'].append(base_conf.loc[term_name, 0])
                    ref_effects_list['Conf_High'].append(base_conf.loc[term_name, 1])

            df_ref = pd.DataFrame(ref_effects_list).sort_values('Relative_Month').reset_index(drop=True)
            if not df_ref.empty:
                 all_effects[ref_dist_category] = df_ref

            # Эффекты для остальных категорий расстояния
            other_dist_categories = [d for d in dist_categories if d != ref_dist_category]
            for dist_cat in other_dist_categories:
                dist_effects_list = {'Relative_Month': [base_period], 'Effect': [0], 'Conf_Low': [0], 'Conf_High': [0]}
                for k in time_points:
                    base_term = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                    # Ищем правильное имя взаимодействия
                    interact_term1 = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]:C({dist_var_name}, Treatment(reference='{ref_dist_category}'))[T.{dist_cat}]"
                    interact_term2 = f"C({dist_var_name}, Treatment(reference='{ref_dist_category}'))[T.{dist_cat}]:C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                    
                    interaction_term_name = None
                    if interact_term1 in model_result.params.index: interaction_term_name = interact_term1
                    elif interact_term2 in model_result.params.index: interaction_term_name = interact_term2

                    if interaction_term_name and base_term in model_result.params.index:
                        try:
                            # Суммарный эффект = База + Взаимодействие
                            t_test = model_result.t_test(f"{base_term} + {interaction_term_name}")
                            dist_effects_list['Relative_Month'].append(k)
                            dist_effects_list['Effect'].append(t_test.effect[0])
                            dist_effects_list['Conf_Low'].append(t_test.conf_int()[0][0])
                            dist_effects_list['Conf_High'].append(t_test.conf_int()[0][1])
                        except Exception as e_ttest:
                             print(f"    Warning: t_test failed for month {k}, dist {dist_cat}: {e_ttest}")
                             # Если t_test не сработал, эффект = NaN
                             dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(np.nan)
                             dist_effects_list['Conf_Low'].append(np.nan); dist_effects_list['Conf_High'].append(np.nan)
                    # Если взаимодействия нет, эффект = базовому (для реф. расстояния)
                    elif base_term in model_result.params.index:
                         base_row = df_ref[df_ref['Relative_Month'] == k]
                         if not base_row.empty:
                              dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(base_row['Effect'].iloc[0])
                              dist_effects_list['Conf_Low'].append(base_row['Conf_Low'].iloc[0]); dist_effects_list['Conf_High'].append(base_row['Conf_High'].iloc[0])


                df_dist = pd.DataFrame(dist_effects_list).sort_values('Relative_Month').reset_index(drop=True)
                if not df_dist.empty:
                     all_effects[dist_cat] = df_dist
                     
            return all_effects

        if model_event_acq_dist_result:
            # Извлекаем эффекты для каждой группы
            all_event_data = extract_event_study_interaction_effects(
                model_event_acq_dist_result,
                BASE_PERIOD,
                "time_to_event_ACQUIRED_PC",
                "distance_category",
                labels_dist, # Список всех категорий
                REF_DISTANCE # Референсная категория
            )

            if all_event_data:
                fig_dist, ax_dist = plt.subplots(figsize=(12, 7))
                colors = plt.cm.viridis(np.linspace(0, 0.8, len(labels_dist) -1 )) # Цвета для не-референсных групп
                markers = ['o', 's', '^']
                
                # Строим линии для 0-5, 5-10, 10-20 км
                plot_categories = [d for d in labels_dist if d != REF_DISTANCE]
                for i, dist_cat in enumerate(plot_categories):
                    if dist_cat in all_event_data:
                         group_data = all_event_data[dist_cat]
                         errors = [group_data['Effect'] - group_data['Conf_Low'], group_data['Conf_High'] - group_data['Effect']]
                         valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                         if valid_error.all():
                              ax_dist.errorbar(x=group_data['Relative_Month'], y=group_data['Effect'], yerr=errors,
                                             fmt=f'-{markers[i%len(markers)]}', capsize=3, label=f'Эффект для {dist_cat}', color=colors[i])
                         else:
                              ax_dist.plot(group_data['Relative_Month'], group_data['Effect'], marker=markers[i%len(markers)], linestyle='-',
                                           label=f'Эффект для {dist_cat} (без ДИ)', color=colors[i])

                ax_dist.axhline(0, color='black', linestyle='-', linewidth=0.8)
                ax_dist.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax_dist.set_title("Event Study: Эффект 'Acquired PC' на Log ER Rate по Расстоянию", fontsize=14)
                ax_dist.set_xlabel("Месяцы относительно поглощения")
                ax_dist.set_ylabel(f"Эффект на {OUTCOME} (относительно месяца -1)")
                ax_dist.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
                ax_dist.legend(title="Расстояние до клиники")
                ax_dist.grid(True, axis='y', linestyle=':')
                plt.tight_layout()
                plt.show()
            else:
                print("Не удалось извлечь данные для графика взаимодействия.")
        else:
            print("Модель с взаимодействием не была рассчитана, график не строится.")

    print("\n🎉 --- Event Study с Взаимодействием по Расстоянию завершен! ---")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО EVENT STUDY С ВЗАИМОДЕЙСТВИЕМ ПО РАССТОЯНИЮ (New PC) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.") 
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.") 
        raise

    # --- 2. Очистка Данных ---
    # ... (стандартный код очистки) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.") 

    # --- 3. Идентификация 'New PC' клиник ---
    # Используем определение из "помогающего" скрипта
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine', 'Internal Medicine', 'IM']
    
    def is_new_pc_clinic(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        # Добавим специальную проверку для 'IM' (Internal Medicine) как в первом скрипте
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        # Проверка остальных ключевых слов
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords if keyword not in ['Internal Medicine', 'IM'])

    df_new_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'New') & (df_clinics_info_full.apply(is_new_pc_clinic, axis=1))].copy()
    print(f"Идентифицировано: {len(df_new_pc_clinics)} New PC/Urgent клиник.") 

    # --- 4. Расчет Карты Лечения и Расстояния (Только для New PC) ---
    print("\n--- Расчет Карты Лечения и Расстояния (New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10; treatment_info_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    
    # Функция для нахождения ближайшей клиники *из группы* и ее расстояния
    def find_nearest_clinic_details(patient_coords, clinic_group_df):
        min_dist = np.inf; nearest_info = {}
        if clinic_group_df.empty: return nearest_info
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon']); distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist:
                min_dist = distance
                nearest_info = {'distance_km': min_dist, 'event_date': clinic_row['event_date']}
        return nearest_info

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; nearby_new_pc_clinics = []
        
        # Находим ВСЕ New PC клиники в радиусе
        for _, clinic_row in df_new_pc_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance <= TREATMENT_RADIUS_KM:
                nearby_new_pc_clinics.append({'date': clinic_row['event_date'], 'distance': distance})
                
        # Находим самую раннюю дату среди клиник в радиусе
        if nearby_new_pc_clinics:
            earliest_date_new_pc = min(c['date'] for c in nearby_new_pc_clinics)
            
        # Находим расстояние до САМОЙ БЛИЖАЙШЕЙ New PC клиники (даже если она дальше 10 км)
        nearest_clinic_details = find_nearest_clinic_details(patient_coords, df_new_pc_clinics)
        distance_to_nearest = nearest_clinic_details.get('distance_km', np.nan)

        treatment_info_new_pc[patient_zip] = {
            'treatment_date_NEW_PC': earliest_date_new_pc,
            'distance_to_nearest_NEW_PC_km': distance_to_nearest # Это расстояние используем для категорий
        }

    df_treatment_info_new_pc = pd.DataFrame.from_dict(treatment_info_new_pc, orient='index')
    print(f"\n✅ Карта лечения и расстояния ('New PC') создана.")

    # --- 5. Агрегация данных и создание Y-переменной ---
    print("\n--- Агрегация данных и создание Y-переменной ---")
    df_panel = final_data.copy()
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    df_agg['log_ER_Rate'] = np.log1p(df_agg['Total_Emergency_Visits']) # Используем log(Visits+1)
    print("✅ Y-переменная создана: log_ER_Rate.")

    # Присоединяем информацию о лечении и расстоянии
    df_analysis_new_dist = df_agg.merge(df_treatment_info_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis_new_dist['Date'] = pd.to_datetime(df_analysis_new_dist['Year_Month'] + '-01')
    df_analysis_new_dist = df_analysis_new_dist.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

    # --- 6. Создание Категорий Расстояния и Переменных Event Study ---
    # Используем 'distance_to_nearest_NEW_PC_km'
    bins_dist = [-np.inf, 5, 10, 20, np.inf]
    labels_dist = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    # Базовая категория для взаимодействия
    REF_DISTANCE = '20+ km' 
    df_analysis_new_dist['distance_category'] = pd.cut(
        df_analysis_new_dist['distance_to_nearest_NEW_PC_km'],
        bins=bins_dist, labels=labels_dist, right=False
    )
    # Устанавливаем базовую категорию явно
    df_analysis_new_dist['distance_category'] = pd.Categorical(df_analysis_new_dist['distance_category'], categories=labels_dist, ordered=True)
    print("\nКатегории расстояния созданы:")
    print(df_analysis_new_dist['distance_category'].value_counts())

    # Рассчитываем time_to_event_NEW_PC
    df_analysis_new_dist['event_month_new_pc'] = pd.to_datetime(df_analysis_new_dist['treatment_date_NEW_PC']).dt.to_period('M')
    df_analysis_new_dist['current_month'] = pd.to_datetime(df_analysis_new_dist['Date']).dt.to_period('M')
    df_analysis_new_dist['time_to_event_NEW_PC'] = (df_analysis_new_dist['current_month'] - df_analysis_new_dist['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # Отбираем ТОЛЬКО обработанные наблюдения для Event Study
    EVENT_WINDOW = 12; BASE_PERIOD = -1
    df_model_new_dist = df_analysis_new_dist[
        df_analysis_new_dist['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_new_dist['time_to_event_NEW_PC'] = df_model_new_dist['time_to_event_NEW_PC'].astype(int)

    # Убираем строки с пропущенной категорией расстояния (если есть)
    df_model_new_dist = df_model_new_dist.dropna(subset=['distance_category'])

    if df_model_new_dist.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска модели.")
    else:
        print(f"Подготовлено {len(df_model_new_dist)} наблюдений для Event Study с взаимодействием.")

        # --- 7. Запуск Модели с Взаимодействием ---
        OUTCOME = "log_ER_Rate"
        # --- ИЗМЕНЕНИЕ: Формула с time_to_event_NEW_PC ---
        formula_event_dist = (f"{OUTCOME} ~ "
                              f"C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) * C(distance_category, Treatment(reference='{REF_DISTANCE}')) + "
                              f"C(Zip) + C(Year) + C(Month)")

        model_event_new_dist_result = None
        print(f"\n--- Запуск Модели Event Study с взаимодействием по расстоянию ({OUTCOME}) ---")
        try:
            print(f"... Запуск модели с C(Zip) ...")
            # --- Используем данные БЕЗ пропусков по категориям ---
            data_for_model = df_model_new_dist.dropna(subset=['distance_category', 'time_to_event_NEW_PC', OUTCOME, 'Zip', 'Year', 'Month'])

            if data_for_model.empty or data_for_model['distance_category'].nunique() < 2 or data_for_model['time_to_event_NEW_PC'].nunique() < 2:
                 print("  ⚠️ Недостаточно данных/вариации после очистки NaN для запуска.")
            else:
                model_event_dist = smf.ols(formula_event_dist, data=data_for_model)
                model_event_new_dist_result = model_event_dist.fit(cov_type='cluster', cov_kwds={'groups': data_for_model['Zip']})
                print("  ✅ Модель с взаимодействием по расстоянию рассчитана.")
                # print(model_event_new_dist_result.summary()) # Для детального просмотра

        except Exception as e:
            print(f"  ❌ ОШИБКА Модели с взаимодействием: {e}")
            # Можно добавить откат к модели без C(Zip), если нужно

        # --- 8. Извлечение и Визуализация Результатов ---
        print("\n--- Визуализация Event Study с взаимодействием по расстоянию ---")

        # --- Функция извлечения эффектов (остается без изменений) ---
        def extract_event_study_interaction_effects(model_result, base_period, time_var_name, dist_var_name, dist_categories, ref_dist_category):
            if model_result is None: return {}
            
            all_effects = {}
            time_points = sorted([p for p in range(-EVENT_WINDOW, EVENT_WINDOW + 1) if p != base_period])
            
            # Эффект для референсной категории расстояния
            ref_effects_list = {'Relative_Month': [base_period], 'Effect': [0], 'Conf_Low': [0], 'Conf_High': [0]}
            time_pattern_base = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.({('|').join(map(str, time_points))})\](?!:)" # Только основной эффект времени
            
            base_params = model_result.params.filter(regex=time_pattern_base)
            base_conf = model_result.conf_int().filter(regex=time_pattern_base, axis=0)
            
            for k in time_points:
                term_name = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                if term_name in base_params.index:
                    ref_effects_list['Relative_Month'].append(k)
                    ref_effects_list['Effect'].append(base_params[term_name])
                    ref_effects_list['Conf_Low'].append(base_conf.loc[term_name, 0])
                    ref_effects_list['Conf_High'].append(base_conf.loc[term_name, 1])

            df_ref = pd.DataFrame(ref_effects_list).sort_values('Relative_Month').reset_index(drop=True)
            if not df_ref.empty:
                    all_effects[ref_dist_category] = df_ref

            # Эффекты для остальных категорий расстояния
            other_dist_categories = [d for d in dist_categories if d != ref_dist_category]
            for dist_cat in other_dist_categories:
                dist_effects_list = {'Relative_Month': [base_period], 'Effect': [0], 'Conf_Low': [0], 'Conf_High': [0]}
                for k in time_points:
                    base_term = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                    # Ищем правильное имя взаимодействия
                    interact_term1 = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]:C({dist_var_name}, Treatment(reference='{ref_dist_category}'))[T.{dist_cat}]"
                    interact_term2 = f"C({dist_var_name}, Treatment(reference='{ref_dist_category}'))[T.{dist_cat}]:C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                    
                    interaction_term_name = None
                    if interact_term1 in model_result.params.index: interaction_term_name = interact_term1
                    elif interact_term2 in model_result.params.index: interaction_term_name = interact_term2

                    if interaction_term_name and base_term in model_result.params.index:
                        try:
                            # Суммарный эффект = База + Взаимодействие
                            t_test = model_result.t_test(f"{base_term} + {interaction_term_name}")
                            dist_effects_list['Relative_Month'].append(k)
                            dist_effects_list['Effect'].append(t_test.effect[0])
                            dist_effects_list['Conf_Low'].append(t_test.conf_int()[0][0])
                            dist_effects_list['Conf_High'].append(t_test.conf_int()[0][1])
                        except Exception as e_ttest:
                            print(f"    Warning: t_test failed for month {k}, dist {dist_cat}: {e_ttest}")
                            # Если t_test не сработал, эффект = NaN
                            dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(np.nan)
                            dist_effects_list['Conf_Low'].append(np.nan); dist_effects_list['Conf_High'].append(np.nan)
                    # Если взаимодействия нет, эффект = базовому (для реф. расстояния)
                    elif base_term in model_result.params.index:
                         base_row = df_ref[df_ref['Relative_Month'] == k]
                         if not base_row.empty:
                                dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(base_row['Effect'].iloc[0])
                                dist_effects_list['Conf_Low'].append(base_row['Conf_Low'].iloc[0]); dist_effects_list['Conf_High'].append(base_row['Conf_High'].iloc[0])


                df_dist = pd.DataFrame(dist_effects_list).sort_values('Relative_Month').reset_index(drop=True)
                if not df_dist.empty:
                        all_effects[dist_cat] = df_dist
                        
            return all_effects

        if model_event_new_dist_result:
            # Извлекаем эффекты для каждой группы
            all_event_data = extract_event_study_interaction_effects(
                model_event_new_dist_result,
                BASE_PERIOD,
                "time_to_event_NEW_PC", # <-- Изменение
                "distance_category",
                labels_dist, # Список всех категорий
                REF_DISTANCE # Референсная категория
            )

            if all_event_data:
                fig_dist, ax_dist = plt.subplots(figsize=(12, 7))
                colors = plt.cm.viridis(np.linspace(0, 0.8, len(labels_dist) -1 )) # Цвета для не-референсных групп
                markers = ['o', 's', '^']
                
                # Строим линии для 0-5, 5-10, 10-20 км
                plot_categories = [d for d in labels_dist if d != REF_DISTANCE]
                for i, dist_cat in enumerate(plot_categories):
                    if dist_cat in all_event_data:
                            group_data = all_event_data[dist_cat]
                            errors = [group_data['Effect'] - group_data['Conf_Low'], group_data['Conf_High'] - group_data['Effect']]
                            valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                            if valid_error.all():
                                ax_dist.errorbar(x=group_data['Relative_Month'], y=group_data['Effect'], yerr=errors,
                                                 fmt=f'-{markers[i%len(markers)]}', capsize=3, label=f'Эффект для {dist_cat}', color=colors[i])
                            else:
                                ax_dist.plot(group_data['Relative_Month'], group_data['Effect'], marker=markers[i%len(markers)], linestyle='-',
                                             label=f'Эффект для {dist_cat} (без ДИ)', color=colors[i])

                ax_dist.axhline(0, color='black', linestyle='-', linewidth=0.8)
                ax_dist.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax_dist.set_title("Event Study: Эффект 'New PC' на Log ER Rate по Расстоянию", fontsize=14) # <-- Изменение
                ax_dist.set_xlabel("Месяцы относительно открытия") # <-- Изменение
                ax_dist.set_ylabel(f"Эффект на {OUTCOME} (относительно месяца -1)")
                ax_dist.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
                ax_dist.legend(title="Расстояние до клиники")
                ax_dist.grid(True, axis='y', linestyle=':')
                plt.tight_layout()
                plt.show()
            else:
                print("Не удалось извлечь данные для графика взаимодействия.")
        else:
            print("Модель с взаимодействием не была рассчитана, график не строится.")

    print("\n🎉 --- Event Study (New PC) с Взаимодействием по Расстоянию завершен! ---")

In [ ]:

# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО EVENT STUDY С ВЗАИМОДЕЙСТВИЕМ ПО РАССТОЯНИЮ (New PC) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.") # Сообщение на русском
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.") # Сообщение на английском для traceback
        raise

    # --- 2. Очистка Данных ---
    # ... (стандартный код очистки) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.") # Сообщение на русском

    # --- 3. Идентификация 'New PC' клиник ---
    df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_pc_urgent, axis=1)].copy()
    print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent клиник.") # Сообщение на русском
    if df_new_PC_clinics.empty: raise ValueError("Не найдено 'New' PC/Urgent/FP клиник.") # Сообщение на русском

    # --- 4. Расчет Карты Лечения и Расстояния (Только для New PC) ---
    print("\n--- Расчет Карты Лечения и Расстояния (New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10; treatment_info_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

    # Функция для нахождения ближайшей клиники *из группы* и ее расстояния
    def find_nearest_clinic_details(patient_coords, clinic_group_df):
        min_dist = np.inf; nearest_info = {}
        if clinic_group_df.empty: return nearest_info
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon']); distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist:
                min_dist = distance
                nearest_info = {'distance_km': min_dist, 'event_date': clinic_row['event_date']}
        return nearest_info

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; nearby_new_pc_clinics = []
        # Находим ВСЕ New PC клиники в радиусе
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance <= TREATMENT_RADIUS_KM:
                nearby_new_pc_clinics.append({'date': clinic_row['event_date'], 'distance': distance})
        # Находим самую раннюю дату
        if nearby_new_pc_clinics:
            earliest_date_new_pc = min(c['date'] for c in nearby_new_pc_clinics)
        # Находим расстояние до САМОЙ БЛИЖАЙШЕЙ New PC клиники
        nearest_clinic_details = find_nearest_clinic_details(patient_coords, df_new_PC_clinics)
        distance_to_nearest = nearest_clinic_details.get('distance_km', np.nan)

        treatment_info_new_pc[patient_zip] = {
            'treatment_date_NEW_PC': earliest_date_new_pc,
            'distance_to_nearest_NEW_PC_km': distance_to_nearest
        }

    df_treatment_info_new_pc = pd.DataFrame.from_dict(treatment_info_new_pc, orient='index')
    print(f"\n✅ Карта лечения и расстояния ('New PC') создана.")

    # --- 5. Агрегация данных и создание Y-переменной ---
    print("\n--- Агрегация данных и создание Y-переменной ---")
    df_panel = final_data.copy()
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    # --- Используем Y = log(Total_ER_Visits + 1) ---
    df_agg['log_ER_Visits'] = np.log1p(df_agg['Total_Emergency_Visits'])
    print("✅ Y-переменная создана: log_ER_Visits.") # Изменили сообщение

    # Присоединяем информацию о лечении и расстоянии
    df_analysis_new_dist = df_agg.merge(df_treatment_info_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis_new_dist['Date'] = pd.to_datetime(df_analysis_new_dist['Year_Month'] + '-01')
    df_analysis_new_dist = df_analysis_new_dist.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

    # --- 6. Создание Категорий Расстояния и Переменных Event Study ---
    # Используем 'distance_to_nearest_NEW_PC_km'
    bins_dist = [-np.inf, 5, 10, 20, np.inf]
    labels_dist = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    REF_DISTANCE = '20+ km' # Базовая категория
    df_analysis_new_dist['distance_category'] = pd.cut(
        df_analysis_new_dist['distance_to_nearest_NEW_PC_km'],
        bins=bins_dist, labels=labels_dist, right=False
    )
    df_analysis_new_dist['distance_category'] = pd.Categorical(df_analysis_new_dist['distance_category'], categories=labels_dist, ordered=True)
    print("\nКатегории расстояния созданы:")
    print(df_analysis_new_dist['distance_category'].value_counts())

    # Рассчитываем time_to_event_NEW_PC
    df_analysis_new_dist['event_month_new_pc'] = pd.to_datetime(df_analysis_new_dist['treatment_date_NEW_PC']).dt.to_period('M')
    df_analysis_new_dist['current_month'] = pd.to_datetime(df_analysis_new_dist['Date']).dt.to_period('M')
    df_analysis_new_dist['time_to_event_NEW_PC'] = (df_analysis_new_dist['current_month'] - df_analysis_new_dist['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # Отбираем ТОЛЬКО обработанные наблюдения для Event Study
    EVENT_WINDOW = 12; BASE_PERIOD = -1
    df_model_new_dist = df_analysis_new_dist[
        df_analysis_new_dist['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_new_dist['time_to_event_NEW_PC'] = df_model_new_dist['time_to_event_NEW_PC'].astype(int)
    df_model_new_dist = df_model_new_dist.dropna(subset=['distance_category']) # Убираем NaN категории

    if df_model_new_dist.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска модели.")
    else:
        print(f"Подготовлено {len(df_model_new_dist)} наблюдений для Event Study с взаимодействием.")

        # --- 7. Запуск Модели с Взаимодействием ---
        # --- Используем Y = log_ER_Visits ---
        OUTCOME = "log_ER_Visits"
        # Формула с взаимодействием и референсной категорией расстояния
        formula_event_dist_new = (f"{OUTCOME} ~ "
                                  f"C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) * C(distance_category, Treatment(reference='{REF_DISTANCE}')) + "
                                  f"C(Zip) + C(Year) + C(Month)")

        model_event_new_dist_result = None
        print(f"\n--- Запуск Модели Event Study ('New PC') с взаимодействием по расстоянию ({OUTCOME}) ---")
        try:
            print(f"... Запуск модели с C(Zip) ...")
            # Используем dropna для всех колонок в формуле
            formula_vars = ['time_to_event_NEW_PC', 'distance_category', 'Zip', 'Year', 'Month', OUTCOME]
            data_for_model = df_model_new_dist[formula_vars].dropna().reset_index(drop=True)

            if data_for_model.empty or data_for_model['distance_category'].nunique() < 2 or data_for_model['time_to_event_NEW_PC'].nunique() < 2:
                 print("  ⚠️ Недостаточно данных/вариации после очистки NaN для запуска.")
            else:
                 present_categories = data_for_model['distance_category'].unique()
                 current_ref_distance = REF_DISTANCE
                 if REF_DISTANCE not in present_categories:
                      alt_ref_distance = present_categories[-1]
                      print(f"  ⚠️ Референсная категория '{REF_DISTANCE}' отсутствует. Используем '{alt_ref_distance}' как референсную.")
                      formula_event_dist_new = formula_event_dist_new.replace(f"Treatment(reference='{REF_DISTANCE}')", f"Treatment(reference='{alt_ref_distance}')")
                      current_ref_distance = alt_ref_distance # Обновляем для извлечения

                 model_event_dist = smf.ols(formula_event_dist_new, data=data_for_model)
                 model_event_new_dist_result = model_event_dist.fit(cov_type='cluster', cov_kwds={'groups': data_for_model['Zip']})
                 print("  ✅ Модель с взаимодействием по расстоянию рассчитана.")
                 # print(model_event_new_dist_result.summary())

        except Exception as e:
            print(f"  ❌ ОШИБКА Модели с взаимодействием: {e}")
            # Можно добавить откат к модели без C(Zip)

        # --- 8. Извлечение и Визуализация Результатов ---
        print("\n--- Визуализация Event Study ('New PC') с взаимодействием по расстоянию ---")

        # Функция извлечения эффектов взаимодействия (без изменений)
        def extract_event_study_interaction_effects(model_result, base_period, time_var_name, dist_var_name, dist_categories, ref_dist_category):
            # ... (Копируем функцию из предыдущего кода) ...
            if model_result is None: return {}
            all_effects = {}; time_points = sorted([p for p in range(-EVENT_WINDOW, EVENT_WINDOW + 1) if p != base_period])
            ref_effects_list = {'Relative_Month': [base_period], 'Effect': [0], 'Conf_Low': [0], 'Conf_High': [0]}
            time_pattern_base = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.({('|').join(map(str, time_points))})\](?!:)"
            base_params = model_result.params.filter(regex=time_pattern_base); base_conf = model_result.conf_int().filter(regex=time_pattern_base, axis=0)
            for k in time_points:
                term_name = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                if term_name in base_params.index:
                    ref_effects_list['Relative_Month'].append(k); ref_effects_list['Effect'].append(base_params[term_name])
                    ref_effects_list['Conf_Low'].append(base_conf.loc[term_name, 0]); ref_effects_list['Conf_High'].append(base_conf.loc[term_name, 1])
            df_ref = pd.DataFrame(ref_effects_list).sort_values('Relative_Month').reset_index(drop=True)
            if not df_ref.empty: all_effects[ref_dist_category] = df_ref
            other_dist_categories = [d for d in dist_categories if d != ref_dist_category]
            for dist_cat in other_dist_categories:
                dist_effects_list = {'Relative_Month': [base_period], 'Effect': [0], 'Conf_Low': [0], 'Conf_High': [0]}
                for k in time_points:
                    base_term = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                    interact_term1 = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]:C({dist_var_name}, Treatment(reference='{ref_dist_category}'))[T.{dist_cat}]"
                    interact_term2 = f"C({dist_var_name}, Treatment(reference='{ref_dist_category}'))[T.{dist_cat}]:C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                    interaction_term_name = None
                    if interact_term1 in model_result.params.index: interaction_term_name = interact_term1
                    elif interact_term2 in model_result.params.index: interaction_term_name = interact_term2
                    if interaction_term_name and base_term in model_result.params.index:
                        try:
                            t_test = model_result.t_test(f"{base_term} + {interaction_term_name}")
                            dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(t_test.effect[0])
                            dist_effects_list['Conf_Low'].append(t_test.conf_int()[0][0]); dist_effects_list['Conf_High'].append(t_test.conf_int()[0][1])
                        except Exception as e_ttest:
                             print(f"    Warning: t_test failed for month {k}, dist {dist_cat}: {e_ttest}")
                             dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(np.nan)
                             dist_effects_list['Conf_Low'].append(np.nan); dist_effects_list['Conf_High'].append(np.nan)
                    elif base_term in model_result.params.index:
                         base_row = df_ref[df_ref['Relative_Month'] == k]
                         if not base_row.empty:
                              dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(base_row['Effect'].iloc[0])
                              dist_effects_list['Conf_Low'].append(base_row['Conf_Low'].iloc[0]); dist_effects_list['Conf_High'].append(base_row['Conf_High'].iloc[0])
                df_dist = pd.DataFrame(dist_effects_list).sort_values('Relative_Month').reset_index(drop=True)
                if not df_dist.empty: all_effects[dist_cat] = df_dist
            return all_effects

        if model_event_new_dist_result:
            # Используем обновленную референсную категорию, если она изменилась
            current_ref_distance = REF_DISTANCE
            all_event_data_new = extract_event_study_interaction_effects(
                model_event_new_dist_result,
                BASE_PERIOD,
                "time_to_event_NEW_PC",
                "distance_category",
                labels_dist,
                current_ref_distance # Используем актуальную референсную
            )

            if all_event_data_new:
                fig_dist_new, ax_dist_new = plt.subplots(figsize=(12, 7))
                colors = plt.cm.viridis(np.linspace(0, 0.8, len(labels_dist) -1 ))
                markers = ['o', 's', '^']
                plot_categories = [d for d in labels_dist if d != current_ref_distance] # Используем актуальную референсную

                for i, dist_cat in enumerate(plot_categories):
                    if dist_cat in all_event_data_new:
                         group_data = all_event_data_new[dist_cat]
                         errors = [group_data['Effect'] - group_data['Conf_Low'], group_data['Conf_High'] - group_data['Effect']]
                         valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                         if valid_error.all():
                              ax_dist_new.errorbar(x=group_data['Relative_Month'], y=group_data['Effect'], yerr=errors,
                                             fmt=f'-{markers[i%len(markers)]}', capsize=3, label=f'Эффект для {dist_cat}', color=colors[i])
                         else:
                              ax_dist_new.plot(group_data['Relative_Month'], group_data['Effect'], marker=markers[i%len(markers)], linestyle='-',
                                           label=f'Эффект для {dist_cat} (без ДИ)', color=colors[i])

                ax_dist_new.axhline(0, color='black', linestyle='-', linewidth=0.8)
                ax_dist_new.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax_dist_new.set_title("Event Study: Эффект 'New PC' на Log ER Visits по Расстоянию", fontsize=14) # Обновлено
                ax_dist_new.set_xlabel("Месяцы относительно открытия")
                ax_dist_new.set_ylabel(f"Эффект на {OUTCOME} (относительно месяца -1 и группы {current_ref_distance})") # Обновлено
                ax_dist_new.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
                ax_dist_new.legend(title="Расстояние до клиники")
                ax_dist_new.grid(True, axis='y', linestyle=':')
                plt.tight_layout()
                plt.show()
            else:
                print("Не удалось извлечь данные для графика взаимодействия ('New PC').")
        else:
            print("Модель с взаимодействием ('New PC') не была рассчитана, график не строится.")

    print("\n🎉 --- Event Study ('New PC') с Взаимодействием по Расстоянию завершен! ---")



In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО EVENT STUDY С ВЗАИМОДЕЙСТВИЕМ ПО РАССТОЯНИЮ (New PC) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' не найден.") # Сообщение на русском
else:
    try:
        df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
        df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    except FileNotFoundError as e:
        print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.") # Сообщение на английском для traceback
        raise

    # --- 2. Очистка Данных ---
    # ... (стандартный код очистки) ...
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
    zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
    zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
    zip_coords = zip_coords.dropna()
    df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
    df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
    df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
    df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
    print("✅ Данные загружены и очищены.") # Сообщение на русском

    # --- 3. Идентификация 'New PC' клиник ---
    df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_pc_urgent, axis=1)].copy()
    print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent клиник.") # Сообщение на русском
    if df_new_PC_clinics.empty: raise ValueError("Не найдено 'New' PC/Urgent/FP клиник.") # Сообщение на русском

    # --- 4. Расчет Карты Лечения и Расстояния (Только для New PC) ---
    print("\n--- Расчет Карты Лечения и Расстояния (New PC, Метод Радиуса) ---")
    TREATMENT_RADIUS_KM = 10; treatment_info_new_pc = {}
    patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
    patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

    # Функция для нахождения ближайшей клиники *из группы* и ее расстояния
    def find_nearest_clinic_details(patient_coords, clinic_group_df):
        min_dist = np.inf; nearest_info = {}
        if clinic_group_df.empty: return nearest_info
        for _, clinic_row in clinic_group_df.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon']); distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance < min_dist:
                min_dist = distance
                nearest_info = {'distance_km': min_dist, 'event_date': clinic_row['event_date']}
        return nearest_info

    for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Обработка ZIP-кодов (New PC)"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; nearby_new_pc_clinics = []
        # Находим ВСЕ New PC клиники в радиусе
        for _, clinic_row in df_new_PC_clinics.iterrows():
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            distance = geodesic(patient_coords, clinic_coords).kilometers
            if distance <= TREATMENT_RADIUS_KM:
                nearby_new_pc_clinics.append({'date': clinic_row['event_date'], 'distance': distance})
        # Находим самую раннюю дату
        if nearby_new_pc_clinics:
            earliest_date_new_pc = min(c['date'] for c in nearby_new_pc_clinics)
        # Находим расстояние до САМОЙ БЛИЖАЙШЕЙ New PC клиники
        nearest_clinic_details = find_nearest_clinic_details(patient_coords, df_new_PC_clinics)
        distance_to_nearest = nearest_clinic_details.get('distance_km', np.nan)

        treatment_info_new_pc[patient_zip] = {
            'treatment_date_NEW_PC': earliest_date_new_pc,
            'distance_to_nearest_NEW_PC_km': distance_to_nearest
        }

    df_treatment_info_new_pc = pd.DataFrame.from_dict(treatment_info_new_pc, orient='index')
    print(f"\n✅ Карта лечения и расстояния ('New PC') создана.")

    # --- 5. Агрегация данных и создание Y-переменной ---
    print("\n--- Агрегация данных и создание Y-переменной ---")
    df_panel = final_data.copy()
    df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
    df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
    df_agg = df_panel.groupby(['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum())
    ).reset_index()
    df_agg['log_ER_Rate'] = np.log1p(df_agg['Total_Emergency_Visits'])
    print("✅ Y-переменная создана: log_ER_Rate.")

    # Присоединяем информацию о лечении и расстоянии
    df_analysis_new_dist = df_agg.merge(df_treatment_info_new_pc, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis_new_dist['Date'] = pd.to_datetime(df_analysis_new_dist['Year_Month'] + '-01')
    df_analysis_new_dist = df_analysis_new_dist.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

    # --- 6. Создание Категорий Расстояния и Переменных Event Study ---
    # Используем 'distance_to_nearest_NEW_PC_km'
    bins_dist = [-np.inf, 5, 10, 20, np.inf]
    labels_dist = ['0-5 km', '5-10 km', '10-20 km', '20+ km']
    REF_DISTANCE = '20+ km' # Базовая категория
    df_analysis_new_dist['distance_category'] = pd.cut(
        df_analysis_new_dist['distance_to_nearest_NEW_PC_km'],
        bins=bins_dist, labels=labels_dist, right=False
    )
    df_analysis_new_dist['distance_category'] = pd.Categorical(df_analysis_new_dist['distance_category'], categories=labels_dist, ordered=True)
    print("\nКатегории расстояния созданы:")
    print(df_analysis_new_dist['distance_category'].value_counts())

    # Рассчитываем time_to_event_NEW_PC
    df_analysis_new_dist['event_month_new_pc'] = pd.to_datetime(df_analysis_new_dist['treatment_date_NEW_PC']).dt.to_period('M')
    df_analysis_new_dist['current_month'] = pd.to_datetime(df_analysis_new_dist['Date']).dt.to_period('M')
    df_analysis_new_dist['time_to_event_NEW_PC'] = (df_analysis_new_dist['current_month'] - df_analysis_new_dist['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # Отбираем ТОЛЬКО обработанные наблюдения для Event Study
    EVENT_WINDOW = 12; BASE_PERIOD = -1
    df_model_new_dist = df_analysis_new_dist[
        df_analysis_new_dist['time_to_event_NEW_PC'].between(-EVENT_WINDOW, EVENT_WINDOW)
    ].copy()
    df_model_new_dist['time_to_event_NEW_PC'] = df_model_new_dist['time_to_event_NEW_PC'].astype(int)
    df_model_new_dist = df_model_new_dist.dropna(subset=['distance_category']) # Убираем NaN категории

    if df_model_new_dist.empty:
         print("⚠️ Недостаточно данных в пределах окна [-12, 12] месяцев для запуска модели.")
    else:
        print(f"Подготовлено {len(df_model_new_dist)} наблюдений для Event Study с взаимодействием.")

        # --- 7. Запуск Модели с Взаимодействием ---
        OUTCOME = "log_ER_Rate"
        # Формула с взаимодействием и референсной категорией расстояния
        formula_event_dist_new = (f"{OUTCOME} ~ "
                                  f"C(time_to_event_NEW_PC, Treatment(reference={BASE_PERIOD})) * C(distance_category, Treatment(reference='{REF_DISTANCE}')) + "
                                  f"C(Zip) + C(Year) + C(Month)")

        model_event_new_dist_result = None
        print(f"\n--- Запуск Модели Event Study ('New PC') с взаимодействием по расстоянию ({OUTCOME}) ---")
        try:
            print(f"... Запуск модели с C(Zip) ...")
            data_for_model = df_model_new_dist.dropna(subset=['distance_category', 'time_to_event_NEW_PC', OUTCOME, 'Zip', 'Year', 'Month'])
            if data_for_model.empty or data_for_model['distance_category'].nunique() < 2 or data_for_model['time_to_event_NEW_PC'].nunique() < 2:
                 print("  ⚠️ Недостаточно данных/вариации после очистки NaN для запуска.")
            else:
                 # Проверим, все ли категории расстояния присутствуют
                 present_categories = data_for_model['distance_category'].unique()
                 if REF_DISTANCE not in present_categories:
                      # Если референсной категории нет в данных, модель не сойдется.
                      # Можно выбрать другую референсную категорию из присутствующих.
                      alt_ref_distance = present_categories[-1] # Берем самую дальнюю из присутствующих
                      print(f"  ⚠️ Референсная категория '{REF_DISTANCE}' отсутствует. Используем '{alt_ref_distance}' как референсную.")
                      formula_event_dist_new = formula_event_dist_new.replace(f"Treatment(reference='{REF_DISTANCE}')", f"Treatment(reference='{alt_ref_distance}')")
                      REF_DISTANCE = alt_ref_distance # Обновляем для извлечения

                 model_event_dist = smf.ols(formula_event_dist_new, data=data_for_model)
                 model_event_new_dist_result = model_event_dist.fit(cov_type='cluster', cov_kwds={'groups': data_for_model['Zip']})
                 print("  ✅ Модель с взаимодействием по расстоянию рассчитана.")
                 # print(model_event_new_dist_result.summary())

        except Exception as e:
            print(f"  ❌ ОШИБКА Модели с взаимодействием: {e}")
            # Можно добавить откат к модели без C(Zip)

        # --- 8. Извлечение и Визуализация Результатов ---
        print("\n--- Визуализация Event Study ('New PC') с взаимодействием по расстоянию ---")

        # --- Используем ТУ ЖЕ функцию извлечения эффектов с взаимодействием ---
        def extract_event_study_interaction_effects(model_result, base_period, time_var_name, dist_var_name, dist_categories, ref_dist_category):
            # ... (Копируем функцию из предыдущего кода, она универсальна) ...
            if model_result is None: return {}
            all_effects = {}; time_points = sorted([p for p in range(-EVENT_WINDOW, EVENT_WINDOW + 1) if p != base_period])
            ref_effects_list = {'Relative_Month': [base_period], 'Effect': [0], 'Conf_Low': [0], 'Conf_High': [0]}
            time_pattern_base = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.({('|').join(map(str, time_points))})\](?!:)"
            base_params = model_result.params.filter(regex=time_pattern_base); base_conf = model_result.conf_int().filter(regex=time_pattern_base, axis=0)
            for k in time_points:
                term_name = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                if term_name in base_params.index:
                    ref_effects_list['Relative_Month'].append(k); ref_effects_list['Effect'].append(base_params[term_name])
                    ref_effects_list['Conf_Low'].append(base_conf.loc[term_name, 0]); ref_effects_list['Conf_High'].append(base_conf.loc[term_name, 1])
            df_ref = pd.DataFrame(ref_effects_list).sort_values('Relative_Month').reset_index(drop=True)
            if not df_ref.empty: all_effects[ref_dist_category] = df_ref
            other_dist_categories = [d for d in dist_categories if d != ref_dist_category]
            for dist_cat in other_dist_categories:
                dist_effects_list = {'Relative_Month': [base_period], 'Effect': [0], 'Conf_Low': [0], 'Conf_High': [0]}
                for k in time_points:
                    base_term = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                    interact_term1 = f"C({time_var_name}, Treatment(reference={base_period}))[T.{k}]:C({dist_var_name}, Treatment(reference='{ref_dist_category}'))[T.{dist_cat}]"
                    interact_term2 = f"C({dist_var_name}, Treatment(reference='{ref_dist_category}'))[T.{dist_cat}]:C({time_var_name}, Treatment(reference={base_period}))[T.{k}]"
                    interaction_term_name = None
                    if interact_term1 in model_result.params.index: interaction_term_name = interact_term1
                    elif interact_term2 in model_result.params.index: interaction_term_name = interact_term2
                    if interaction_term_name and base_term in model_result.params.index:
                        try:
                            t_test = model_result.t_test(f"{base_term} + {interaction_term_name}")
                            dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(t_test.effect[0])
                            dist_effects_list['Conf_Low'].append(t_test.conf_int()[0][0]); dist_effects_list['Conf_High'].append(t_test.conf_int()[0][1])
                        except Exception as e_ttest:
                             print(f"    Warning: t_test failed for month {k}, dist {dist_cat}: {e_ttest}")
                             dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(np.nan)
                             dist_effects_list['Conf_Low'].append(np.nan); dist_effects_list['Conf_High'].append(np.nan)
                    elif base_term in model_result.params.index:
                         base_row = df_ref[df_ref['Relative_Month'] == k]
                         if not base_row.empty:
                              dist_effects_list['Relative_Month'].append(k); dist_effects_list['Effect'].append(base_row['Effect'].iloc[0])
                              dist_effects_list['Conf_Low'].append(base_row['Conf_Low'].iloc[0]); dist_effects_list['Conf_High'].append(base_row['Conf_High'].iloc[0])
                df_dist = pd.DataFrame(dist_effects_list).sort_values('Relative_Month').reset_index(drop=True)
                if not df_dist.empty: all_effects[dist_cat] = df_dist
            return all_effects

        if model_event_new_dist_result:
            # Извлекаем эффекты для каждой группы
            all_event_data_new = extract_event_study_interaction_effects(
                model_event_new_dist_result,
                BASE_PERIOD,
                "time_to_event_NEW_PC", # Правильное имя переменной времени
                "distance_category",
                labels_dist, # Список всех категорий
                REF_DISTANCE # Референсная категория (может быть обновлена)
            )

            if all_event_data_new:
                fig_dist_new, ax_dist_new = plt.subplots(figsize=(12, 7))
                # Цвета и маркеры для 0-5, 5-10, 10-20
                colors = plt.cm.viridis(np.linspace(0, 0.8, len(labels_dist) -1 ))
                markers = ['o', 's', '^']
                plot_categories = [d for d in labels_dist if d != REF_DISTANCE]

                for i, dist_cat in enumerate(plot_categories):
                    if dist_cat in all_event_data_new:
                         group_data = all_event_data_new[dist_cat]
                         errors = [group_data['Effect'] - group_data['Conf_Low'], group_data['Conf_High'] - group_data['Effect']]
                         valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
                         if valid_error.all():
                              ax_dist_new.errorbar(x=group_data['Relative_Month'], y=group_data['Effect'], yerr=errors,
                                             fmt=f'-{markers[i%len(markers)]}', capsize=3, label=f'Эффект для {dist_cat}', color=colors[i])
                         else:
                              ax_dist_new.plot(group_data['Relative_Month'], group_data['Effect'], marker=markers[i%len(markers)], linestyle='-',
                                           label=f'Эффект для {dist_cat} (без ДИ)', color=colors[i])

                ax_dist_new.axhline(0, color='black', linestyle='-', linewidth=0.8)
                ax_dist_new.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax_dist_new.set_title("Event Study: Эффект 'New PC' на Log ER Rate по Расстоянию", fontsize=14) # Изменен заголовок
                ax_dist_new.set_xlabel("Месяцы относительно открытия") # Изменена подпись
                ax_dist_new.set_ylabel(f"Эффект на {OUTCOME} (относительно месяца -1 и группы {REF_DISTANCE})") # Обновлена подпись
                ax_dist_new.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2))
                ax_dist_new.legend(title="Расстояние до клиники")
                ax_dist_new.grid(True, axis='y', linestyle=':')
                plt.tight_layout()
                plt.show()
            else:
                print("Не удалось извлечь данные для графика взаимодействия ('New PC').")
        else:
            print("Модель с взаимодействием ('New PC') не была рассчитана, график не строится.")

    print("\n🎉 --- Event Study ('New PC') с Взаимодействием по Расстоянию завершен! ---")


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО МАССИРОВАННОГО АНАЛИЗА ГЕТЕРОГЕННОСТИ ('НАСЫЩЕНИЕ') ---")

# --- 1. Проверка и Загрузка Данных ---
# Используем final_data, если он есть, иначе пытаемся загрузить агрегированные данные
if 'final_data' not in locals():
    print("Предупреждение: 'final_data' не найден. Пытаюсь загрузить 'aggregated_data.csv'")
    try:
        final_data = pd.read_csv('aggregated_data.csv', dtype={'Filtered_Patient_ZipCode': str, 'Zip': str})
        if 'Date' in final_data.columns and not pd.api.types.is_datetime64_any_dtype(final_data['Date']):
             final_data['Date'] = pd.to_datetime(final_data['Date'])
        if 'Zip' in final_data.columns and 'Filtered_Patient_ZipCode' not in final_data.columns:
            final_data = final_data.rename(columns={'Zip': 'Filtered_Patient_ZipCode'})
        print("  ✅ 'aggregated_data.csv' загружен.")
    except FileNotFoundError:
        print("❌ ОШИБКА: DataFrame 'final_data' не найден и 'aggregated_data.csv' тоже.")
        raise FileNotFoundError("Необходим 'final_data' или 'aggregated_data.csv'")

# Загрузка остальных файлов
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    # Загружаем ИСХОДНЫЙ файл для SVI и Department
    print("... Загрузка исходного файла для SVI и Department ...")
    raw_svi_dept_cols = ['Filtered_Patient_ZipCode', 'Department'] + \
                        [col for col in pd.read_csv("final_filtered_dataset_reordered.csv", nrows=0).columns if col.startswith('X..')]
    df_raw_data_svi_dept = pd.read_csv(
        "final_filtered_dataset_reordered.csv",
        usecols=raw_svi_dept_cols,
        dtype={'Filtered_Patient_ZipCode': str},
        low_memory=False
    )
    df_raw_data_svi_dept['Filtered_Patient_ZipCode'] = df_raw_data_svi_dept['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)

except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (стандартный код очистки) ...
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
print("✅ Данные загружены и очищены.")

# --- 3. Шаг 1: Разделение 'New' клиник на группы ---
print("\n--- Шаг 1: Разделение 'New' клиник на группы ---")
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()

# Исключаем конфаундер и слишком новые
confounder_zip = '61364'; confounder_date = pd.to_datetime('2021-09-01')
date_cutoff = pd.to_datetime('2023-01-01')

df_new_clinics_filtered = df_new_clinics_all[
    ~((df_new_clinics_all['clinic_zip'] == confounder_zip) & (df_new_clinics_all['event_date'] == confounder_date)) &
    (df_new_clinics_all['event_date'] < date_cutoff)
].copy()
print(f"Исключен конфаундер Streator ER и клиники после {date_cutoff.date()}. Осталось {len(df_new_clinics_filtered)} 'New' клиник.")

# Определяем ключевые слова для каждой группы
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
cardio_keywords = ['Cardiology', 'CV']
ortho_keywords = ['Ortho', 'Sport', 'Podiatry', 'ENT', 'Surgery'] # Включили ENT и Surgery сюда
telehealth_keywords = ['Telehealth']

# Функция категоризации
def categorize_new_clinic(row):
    facility_name = str(row.get('Facility', '')).lower()
    specialty_name = str(row.get('Specialty', '')).lower()
    combined_text = facility_name + " " + specialty_name

    if any(keyword.lower() in combined_text for keyword in pc_urgent_keywords): return 'New_PC_Urgent'
    if any(keyword.lower() in combined_text for keyword in cardio_keywords): return 'New_Cardio'
    if any(keyword.lower() in combined_text for keyword in ortho_keywords): return 'New_Ortho_Surgery'
    if any(keyword.lower() in combined_text for keyword in telehealth_keywords): return 'New_Telehealth'
    return 'New_Other' # Остальные

df_new_clinics_filtered['Group'] = df_new_clinics_filtered.apply(categorize_new_clinic, axis=1)

# Создаем словари DataFrame'ов для каждой группы
clinic_groups = {group: df_new_clinics_filtered[df_new_clinics_filtered['Group'] == group]
                 for group in df_new_clinics_filtered['Group'].unique() if group != 'New_Other'}

# Отдельно для Acquired PC (для контрольного графика)
pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
def is_acq_pc(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
    if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_acq_pc, axis=1))].copy()
clinic_groups['Acquired_PC'] = df_acquired_pc_clinics # Добавляем Acquired PC

print("\nКлиники разделены на группы:")
for group_name, group_df in clinic_groups.items():
    print(f"  - {group_name}: {len(group_df)} клиник")


# --- 4. Расчет Карт Лечения для КАЖДОЙ группы (Метод Радиуса 10 км) ---
print("\n--- Расчет Карт Лечения для каждой группы клиник ---")
TREATMENT_RADIUS_KM = 10
treatment_dates_all_groups = {} # Словарь для хранения дат по группам

patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

for group_name, group_df in clinic_groups.items():
    treatment_dates_group = {}
    date_col_name = f'treatment_date_{group_name}' # Напр., treatment_date_New_PC_Urgent
    
    if group_df.empty:
        print(f"  Пропуск группы '{group_name}': нет клиник.")
        # Создаем пустую колонку дат для этой группы
        for zip_code in patient_zips_coords_df['Filtered_Patient_ZipCode']:
             treatment_dates_group[zip_code] = {date_col_name: pd.NaT}

    else:
        for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc=f"Даты для {group_name}"):
            patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
            earliest_date = pd.NaT; nearby_dates = []
            for _, clinic_row in group_df.iterrows():
                clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
                if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM:
                    nearby_dates.append(clinic_row['event_date'])
            if nearby_dates:
                earliest_date = min(nearby_dates)
            treatment_dates_group[patient_zip] = {date_col_name: earliest_date}

    treatment_dates_all_groups[group_name] = pd.DataFrame.from_dict(treatment_dates_group, orient='index')
    print(f"  ✅ Карта лечения для '{group_name}' создана ({len(treatment_dates_all_groups[group_name].dropna())} ZIP'ов обработаны).")

# Объединяем все DataFrame'ы с датами лечения
df_treatment_dates_combined = pd.concat(treatment_dates_all_groups.values(), axis=1)


# --- 5. Шаг 2: Агрегация данных и создание Y-переменных ---
print("\n--- Шаг 2: Агрегация данных и создание Y-переменных ---")
df_panel = final_data.copy()
df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
df_panel['ICD_Chapter'] = df_panel['ICD_1char'].str[0].fillna('Unknown')

# Определяем топ-5 сущностей для гетерогенности (ДО агрегации)
top_fin_classes = df_panel['FinancialClassNM'].astype(str).value_counts().nlargest(5).index.tolist()
top_departments_er = df_panel[df_panel['Is_Emergency'] == 1]['Department'].astype(str).value_counts().nlargest(5).index.tolist()
top_chapters_er = df_panel[df_panel['Is_Emergency'] == 1]['ICD_Chapter'].astype(str).value_counts().nlargest(5).index.tolist()

# Создаем колонки ДО агрегации
df_panel['ER_Visits'] = df_panel['EncounterCount'] * df_panel['Is_Emergency']
df_panel['Regular_Visits'] = df_panel['EncounterCount'] * df_panel['Is_Regular']
# Плановые PC визиты в госпитале
pc_hosp_depts = ['Family Practice', 'Internal Medicine'] # Примерные названия, проверьте ваши
df_panel['Regular_PC_Hosp_Visits'] = df_panel['Regular_Visits'] * df_panel['Department'].isin(pc_hosp_depts)
# ER по главам
for chapter in top_chapters_er: df_panel[f'ER_Chapter_{chapter}'] = df_panel['ER_Visits'] * (df_panel['ICD_Chapter'] == chapter)
# ER по спец диагнозам
df_panel['ER_Low_J'] = df_panel['ER_Visits'] * (df_panel['ICD_Chapter'] == 'J') # Пример: Глава J
df_panel['ER_Low_R'] = df_panel['ER_Visits'] * (df_panel['ICD_Chapter'] == 'R') # Пример: Глава R
df_panel['ER_Cardio'] = df_panel['ER_Visits'] * (df_panel['ICD_Chapter'] == 'I')
df_panel['ER_Ortho'] = df_panel['ER_Visits'] * (df_panel['ICD_Chapter'] == 'M')
df_panel['ER_Mental'] = df_panel['ER_Visits'] * (df_panel['ICD_Chapter'] == 'F')
# Плановые визиты в спец департаменты
df_panel['Dept_Cardio_Visits'] = df_panel['Regular_Visits'] * (df_panel['Department'] == 'Cardiology') # Проверьте точное имя
df_panel['Dept_Ortho_Visits'] = df_panel['Regular_Visits'] * (df_panel['Department'] == 'Orthopedics') # Проверьте точное имя

# Быстрая агрегация
grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
agg_dict_opt = {
    'ER_Visits': 'sum', 'Regular_Visits': 'sum', 'Regular_PC_Hosp_Visits': 'sum',
    'ER_Low_J': 'sum', 'ER_Low_R': 'sum', 'ER_Cardio': 'sum', 'ER_Ortho': 'sum', 'ER_Mental': 'sum',
    'Dept_Cardio_Visits': 'sum', 'Dept_Ortho_Visits': 'sum'
}
print("... Выполняю быструю агрегацию ...")
df_agg_matrix = df_panel.groupby(grouping_cols).agg(agg_dict_opt).reset_index()

# Создаем Y-переменные (логарифмы)
df_agg_matrix['Y_ER_Total'] = np.log1p(df_agg_matrix['ER_Visits'])
df_agg_matrix['Y_Regular_Total'] = np.log1p(df_agg_matrix['Regular_Visits'])
df_agg_matrix['Y_Regular_PC_at_HOSPITAL'] = np.log1p(df_agg_matrix['Regular_PC_Hosp_Visits'])
df_agg_matrix['Y_ER_Low_Acuity_J'] = np.log1p(df_agg_matrix['ER_Low_J'])
df_agg_matrix['Y_ER_Low_Acuity_R'] = np.log1p(df_agg_matrix['ER_Low_R'])
df_agg_matrix['Y_ER_Cardio'] = np.log1p(df_agg_matrix['ER_Cardio'])
df_agg_matrix['Y_ER_Ortho'] = np.log1p(df_agg_matrix['ER_Ortho'])
df_agg_matrix['Y_ER_Mental'] = np.log1p(df_agg_matrix['ER_Mental'])
df_agg_matrix['Y_Dept_Cardio'] = np.log1p(df_agg_matrix['Dept_Cardio_Visits'])
df_agg_matrix['Y_Dept_Ortho'] = np.log1p(df_agg_matrix['Dept_Ortho_Visits'])
print("✅ Y-переменные для матрицы созданы.")

# Присоединяем все даты лечения
df_analysis_matrix = df_agg_matrix.merge(df_treatment_dates_combined, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis_matrix['Date'] = pd.to_datetime(df_analysis_matrix['Year_Month'] + '-01')
df_analysis_matrix = df_analysis_matrix.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
print("✅ Данные агрегированы и готовы к матрице анализа.")


# --- 6. Шаг 3: Запуск "Матрицы" Event Study ---
print("\n\n" + "="*70); print("--- ШАГ 3: Запуск 'Матрицы' Event Study ---"); print("="*70)

# Словарь для хранения результатов моделей
matrix_models = {}
# Список пар (Группа клиник -> колонка даты, Список Y-переменных для этой группы)
analysis_pairs = [
    ('New_PC_Urgent', 'treatment_date_New_PC_Urgent', ['Y_ER_Total', 'Y_Regular_Total', 'Y_Regular_PC_at_HOSPITAL', 'Y_ER_Low_Acuity_J', 'Y_ER_Low_Acuity_R']),
    ('New_Cardio', 'treatment_date_New_Cardio', ['Y_ER_Cardio', 'Y_Dept_Cardio']),
    ('New_Ortho_Surgery', 'treatment_date_New_Ortho_Surgery', ['Y_ER_Ortho', 'Y_Dept_Ortho']),
    ('New_Telehealth', 'treatment_date_New_Telehealth', ['Y_ER_Mental']),
    ('Acquired_PC', 'treatment_date_Acquired_PC', ['Y_ER_Total']) # Контрольный график воронки
]

# Общие параметры
EVENT_WINDOW = 12; BASE_PERIOD = -1

# Функция запуска и обработки ошибок (из предыдущего кода)
def run_event_study_model(formula, data, cluster_var='Zip'):
    # ... (Копируем функцию run_event_study_model из предыдущего кода) ...
    model_result = None
    outcome_col = formula.split('~')[0].strip()
    required_cols_run = [outcome_col, time_var, cluster_var, 'Year', 'Month']
    if not all(col in data.columns for col in required_cols_run):
         print(f"  ⚠️ Пропуск: Отсутствуют необходимые колонки для '{outcome_col}'.")
         return None
    if not data.empty and data[outcome_col].notna().any() and data[time_var].nunique() >= 2:
        try:
            # print(f"... Запуск модели с C({cluster_var}) для {outcome_col}...") # Убрали для краткости
            cols_exist = [col for col in required_cols_run if col in data.columns]
            data_clean = data[cols_exist].dropna().reset_index(drop=True)
            if data_clean.empty or data_clean[time_var].nunique() < 2:
                 # print(f"  ⚠️ Пропуск: Недостаточно данных после очистки NaN для '{outcome_col}'.") # Убрали для краткости
                 return None
            model = smf.ols(formula, data=data_clean)
            model_result = model.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]})
            # print(f"  ✅ Модель с C({cluster_var}) рассчитана.") # Убрали для краткости
        except Exception as e:
            # print(f"  ❌ ОШИБКА с C({cluster_var}): {e}") # Убрали для краткости
            try:
                 formula_no_zip = formula.replace(f"+ C({cluster_var})", "")
                 # print(f"     ... Попытка без C({cluster_var}) ...") # Убрали
                 cols_nozip = [col for col in cols_exist if col != cluster_var]
                 data_clean_nozip = data[cols_nozip].dropna().reset_index(drop=True)
                 cluster_groups_nozip = data.loc[data_clean_nozip.index, cluster_var]
                 if data_clean_nozip.empty or data_clean_nozip[time_var].nunique() < 2: return None
                 model_no_zip = smf.ols(formula_no_zip, data=data_clean_nozip)
                 model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': cluster_groups_nozip})
                 # print(f"        ✅ Модель без C({cluster_var}) рассчитана.") # Убрали
            except Exception as e2:
                 # print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}") # Убрали
                 model_result = None
    return model_result


# Цикл по парам Группа-Y
for group_name, date_col, outcome_list in analysis_pairs:
    print(f"\n--- Анализ для группы клиник: {group_name} ---")
    if date_col not in df_analysis_matrix.columns:
        print(f"  ⚠️ Пропуск: Колонка даты '{date_col}' не найдена (возможно, группа пуста).")
        continue

    # Готовим данные для этой группы
    df_model_group = df_analysis_matrix.copy()
    # Рассчитываем time_to_event для ЭТОЙ группы
    df_model_group['event_month'] = pd.to_datetime(df_model_group[date_col]).dt.to_period('M')
    df_model_group['current_month'] = pd.to_datetime(df_model_group['Date']).dt.to_period('M')
    # Используем уникальное имя для time_to_event
    time_var = f'time_to_event_{group_name}'
    df_model_group[time_var] = (df_model_group['current_month'] - df_model_group['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # Фильтруем по окну
    df_model_filtered = df_model_group[df_model_group[time_var].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
    df_model_filtered[time_var] = df_model_filtered[time_var].astype(int)

    if df_model_filtered.empty:
        print(f"  ⚠️ Пропуск: Нет данных в окне [-{EVENT_WINDOW}, {EVENT_WINDOW}] для группы '{group_name}'.")
        continue

    # Запускаем модели для каждого Y из списка
    for outcome in outcome_list:
        if outcome not in df_model_filtered.columns:
             print(f"  ⚠️ Пропуск: Колонка исхода '{outcome}' не найдена.")
             continue

        formula = f"{outcome} ~ C({time_var}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
        print(f"\n  Запуск модели: {group_name} -> {outcome}")
        model_key = f"{group_name}_{outcome}" # Уникальный ключ для словаря
        matrix_models[model_key] = run_event_study_model(formula, df_model_filtered)


# --- 7. Шаг 3 (продолжение): Визуализация Матрицы ---
print("\n\n" + "="*70); print("--- ВИЗУАЛИЗАЦИЯ МАТРИЦЫ РЕЗУЛЬТАТОВ ---"); print("="*70)

# Цвета для графиков
colors = plt.cm.tab10(np.linspace(0, 1, 10)) # Используем tab10 для различимых цветов
color_map = {
    'Y_ER_Total': colors[0], 'Y_Regular_Total': colors[1], 'Y_Regular_PC_at_HOSPITAL': colors[2],
    'Y_ER_Low_Acuity_J': colors[3], 'Y_ER_Low_Acuity_R': colors[4],
    'Y_ER_Cardio': colors[5], 'Y_Dept_Cardio': colors[6],
    'Y_ER_Ortho': colors[7], 'Y_Dept_Ortho': colors[8],
    'Y_ER_Mental': colors[9]
}

# Цикл по парам для построения графиков
for group_name, date_col, outcome_list in analysis_pairs:
    time_var = f'time_to_event_{group_name}' # Правильное имя переменной времени
    print(f"\n--- Графики для группы клиник: {group_name} ---")

    for outcome in outcome_list:
        model_key = f"{group_name}_{outcome}"
        model_result = matrix_models.get(model_key) # Получаем результат модели

        event_data = extract_event_study_effects_v3(model_result, BASE_PERIOD, time_var)
        plot_event_study(
            event_data,
            outcome,
            f"Эффект '{group_name}' на {outcome}",
            color_map.get(outcome, 'grey'), # Используем цвет из карты или серый
            EVENT_WINDOW,
            BASE_PERIOD
        )

# --- Анализ и Визуализация Плана 4 (SVI) ---
# ... (Копируем весь блок Плана 4 из предыдущего скрипта, начиная с print и try/except) ...
print("\n\n" + "="*70); print("--- ПЛАН 4: Анализ по 'Лучшей' SVI-Колонке ---"); print("="*70)
OUTCOME_RQ5 = "Y_Ratio" # Возвращаемся к Y_Ratio
model_result_RQ5 = None
SVI_FILE = 'final_filtered_dataset_reordered.csv'
SVI_COLUMN_NAME = 'X..Uninsured.Adults_zip' # Пример

try:
    print(f"... Анализ SVI-подобных колонок в {SVI_FILE} ...")
    svi_cols_to_load = ['Filtered_Patient_ZipCode'] + [col for col in pd.read_csv(SVI_FILE, nrows=0).columns if col.startswith('X..')]
    df_svi_raw_full = pd.read_csv(SVI_FILE, usecols=svi_cols_to_load, dtype={'Filtered_Patient_ZipCode': str}, low_memory=False)
    df_svi_raw_full['Filtered_Patient_ZipCode'] = df_svi_raw_full['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_svi_raw_full = df_svi_raw_full.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    df_svi_unique = df_svi_raw_full.drop_duplicates(subset=['Zip']).set_index('Zip')
    svi_stats = []
    for col in df_svi_unique.columns:
        if col.startswith('X..'):
            nan_pct = df_svi_unique[col].isna().mean() * 100
            numeric_col = pd.to_numeric(df_svi_unique[col], errors='coerce')
            nunique = numeric_col.nunique()
            svi_stats.append({'Column': col, 'NaN_Pct': nan_pct, 'N_Unique_Numeric': nunique})
    df_svi_stats = pd.DataFrame(svi_stats).sort_values(by=['NaN_Pct', 'N_Unique_Numeric'], ascending=[True, False])
    print("\nСтатистика SVI-подобных колонок (Топ 10 по качеству):"); print(df_svi_stats.head(10).to_string())
    best_svi_col = df_svi_stats.iloc[0]['Column']; print(f"\nВыбрана 'Лучшая' SVI колонка: {best_svi_col}")
    df_svi = df_svi_unique[[best_svi_col]].copy(); df_svi[best_svi_col] = pd.to_numeric(df_svi[best_svi_col], errors='coerce')
    df_svi = df_svi.dropna(subset=[best_svi_col]); svi_median = df_svi[best_svi_col].median()
    df_svi['SVI_Category'] = np.where(df_svi[best_svi_col] >= svi_median, 'High_Value', 'Low_Value')
    group_high_name = f'Выс_{best_svi_col.split("_")[0].replace("X..","").replace(".","_")}'; group_low_name = f'Низ_{best_svi_col.split("_")[0].replace("X..","").replace(".","_")}'
    df_svi['SVI_Category'] = df_svi['SVI_Category'].replace({'High_Value': group_high_name, 'Low_Value': group_low_name})
    print(f"  Разделение по медиане ({svi_median:.2f}):"); print(df_svi['SVI_Category'].value_counts())

    # --- Присоединение SVI к данным модели 'New PC' (df_model_new_filtered из Шага 1) ---
    # Пересоздаем df_model_new_filtered, если его нет
    time_var_rq5 = 'time_to_event_New_PC_Urgent' # Убедимся, что это правильное имя
    date_col_rq5 = 'treatment_date_New_PC_Urgent'
    if time_var_rq5 not in df_analysis_matrix.columns:
         # Рассчитываем time_to_event для New PC Urgent
         df_analysis_matrix['event_month_new_pc'] = pd.to_datetime(df_analysis_matrix[date_col_rq5]).dt.to_period('M')
         df_analysis_matrix['current_month'] = pd.to_datetime(df_analysis_matrix['Date']).dt.to_period('M')
         df_analysis_matrix[time_var_rq5] = (df_analysis_matrix['current_month'] - df_analysis_matrix['event_month_new_pc']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    df_model_rq5_base = df_analysis_matrix[df_analysis_matrix[time_var_rq5].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
    df_model_rq5_base[time_var_rq5] = df_model_rq5_base[time_var_rq5].astype(int)

    df_model_rq5 = df_model_rq5_base.merge(df_svi[['SVI_Category']], on='Zip', how='inner')

    if df_model_rq5.empty or df_model_rq5['SVI_Category'].nunique() < 2: print("  ❌ ОШИБКА: Нет данных для анализа гетерогенности по SVI.")
    else:
        print(f"  Подготовлено {len(df_model_rq5)} наблюдений для анализа гетерогенности по SVI.")
        print("  Распределение SVI категорий:\n", df_model_rq5['SVI_Category'].value_counts())
        formula_rq5 = f"{OUTCOME_RQ5} ~ C({time_var_rq5}, Treatment(reference={BASE_PERIOD})) * C(SVI_Category) + C(Zip) + C(Year) + C(Month)"
        print(f"\nЗапуск RQ5 (Гетерогенность New PC на {OUTCOME_RQ5} по SVI)...")
        model_result_RQ5 = run_event_study_model(formula_rq5, df_model_rq5) # Используем общую функцию

        print("\n--- Визуализация RQ5: Эффект по группам SVI ---")
        if model_result_RQ5:
            # ... (код извлечения и построения графика для RQ5 без изменений) ...
            svi_groups = sorted(df_model_rq5['SVI_Category'].unique()); event_data_rq5 = {}; base_svi_group = svi_groups[0]
            base_effects = extract_event_study_effects_v3(model_result_RQ5, BASE_PERIOD, time_var_rq5)
            if not base_effects.empty: event_data_rq5[base_svi_group] = base_effects
            if len(svi_groups) > 1:
                other_svi_group = svi_groups[1]; results_other = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
                results_other['Relative_Month'].append(BASE_PERIOD); results_other['Effect'].append(0); results_other['Conf_Low'].append(0); results_other['Conf_High'].append(0)
                for month in base_effects['Relative_Month']:
                    if month == BASE_PERIOD: continue
                    base_effect_row = base_effects[base_effects['Relative_Month'] == month].iloc[0]; base_term_name = f"C({time_var_rq5}, Treatment(reference={BASE_PERIOD}))[T.{month}]"
                    interaction_term_name1 = f"C({time_var_rq5}, Treatment(reference={BASE_PERIOD}))[T.{month}]:C(SVI_Category)[T.{other_svi_group}]"; interaction_term_name2 = f"C(SVI_Category)[T.{other_svi_group}]:C({time_var_rq5}, Treatment(reference={BASE_PERIOD}))[T.{month}]"
                    interaction_term_name = None;
                    if interaction_term_name1 in model_result_RQ5.params.index: interaction_term_name = interaction_term_name1
                    elif interaction_term_name2 in model_result_RQ5.params.index: interaction_term_name = interaction_term_name2
                    if interaction_term_name and base_term_name in model_result_RQ5.params.index:
                        try:
                            t_test = model_result_RQ5.t_test(f"{base_term_name} + {interaction_term_name}")
                            results_other['Relative_Month'].append(month); results_other['Effect'].append(t_test.effect[0]); results_other['Conf_Low'].append(t_test.conf_int()[0][0]); results_other['Conf_High'].append(t_test.conf_int()[0][1])
                        except Exception as e_ttest: results_other['Relative_Month'].append(month); results_other['Effect'].append(base_effect_row['Effect'] + model_result_RQ5.params[interaction_term_name]); results_other['Conf_Low'].append(np.nan); results_other['Conf_High'].append(np.nan)
                    else: results_other['Relative_Month'].append(month); results_other['Effect'].append(base_effect_row['Effect']); results_other['Conf_Low'].append(base_effect_row['Conf_Low']); results_other['Conf_High'].append(base_effect_row['Conf_High'])
                df_other = pd.DataFrame(results_other).sort_values('Relative_Month').reset_index(drop=True);
                if not df_other.empty: event_data_rq5[other_svi_group] = df_other
            fig_rq5, ax_rq5 = plt.subplots(figsize=(12, 7)); colors = plt.cm.viridis(np.linspace(0, 1, len(event_data_rq5))); markers = ['o', 's', '^', 'd']; linestyles = ['-', '--', ':', '-.']; plotted = False
            for i, (group_name, group_data) in enumerate(event_data_rq5.items()):
                 if not group_data.empty:
                      errors = [group_data['Effect'] - group_data['Conf_Low'], group_data['Conf_High'] - group_data['Effect']]; valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1]); fmt = f'{linestyles[i%len(linestyles)]}{markers[i%len(markers)]}'
                      if valid_error.all(): ax_rq5.errorbar(x=group_data['Relative_Month'], y=group_data['Effect'], yerr=errors, fmt=fmt, capsize=3, label=f'Эффект для {group_name}', color=colors[i])
                      else: ax_rq5.plot(group_data['Relative_Month'], group_data['Effect'], marker=markers[i%len(markers)], linestyle=linestyles[i%len(linestyles)], label=f'Эффект для {group_name} (без ДИ)', color=colors[i])
                      plotted = True
            if plotted:
                ax_rq5.axhline(0, color='black', linestyle='-', linewidth=0.8); ax_rq5.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие (Месяц 0)')
                ax_rq5.set_title(f"Шаг 4 (RQ5): Гетерогенный Эффект 'New PC' на Y_Ratio по {best_svi_col}", fontsize=14)
                ax_rq5.set_xlabel("Месяцы относительно открытия"); ax_rq5.set_ylabel(f"Эффект на {OUTCOME_RQ5}")
                ax_rq5.set_xticks(range(-EVENT_WINDOW, EVENT_WINDOW + 1, 2)); ax_rq5.legend(); ax_rq5.grid(True, axis='y', linestyle=':')
                plt.tight_layout(); plt.show()
            else: print("Не удалось извлечь данные для графика RQ5.")
        else: print("Модель RQ5 не была рассчитана, график не строится.")
except (FileNotFoundError, KeyError, ValueError, Exception) as e_svi: print(f"❌ ОШИБКА в Шаге 4 (RQ5): {e_svi}. Шаг пропущен.")

print("\n🎉 --- Массированный Анализ Гетерогенности завершен! ---")


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО ФИНАЛЬНЫХ ПРОВЕРОК ('НАСЫЩЕНИЕ' + Кварталы + Таблицы) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("Предупреждение: 'final_data' не найден. Пытаюсь загрузить 'aggregated_data.csv'")
    try:
        final_data = pd.read_csv('aggregated_data.csv', dtype={'Filtered_Patient_ZipCode': str, 'Zip': str})
        if 'Date' in final_data.columns and not pd.api.types.is_datetime64_any_dtype(final_data['Date']):
             final_data['Date'] = pd.to_datetime(final_data['Date'])
        if 'Zip' in final_data.columns and 'Filtered_Patient_ZipCode' not in final_data.columns:
            final_data = final_data.rename(columns={'Zip': 'Filtered_Patient_ZipCode'})
        print("  ✅ 'aggregated_data.csv' загружен.")
    except FileNotFoundError:
        print("❌ ОШИБКА: DataFrame 'final_data' не найден и 'aggregated_data.csv' тоже.")
        raise FileNotFoundError("Необходим 'final_data' или 'aggregated_data.csv'")

# Загрузка остальных файлов
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    print("... Загрузка исходного файла для Department ...")
    df_raw_data_svi_dept = pd.read_csv(
        "final_filtered_dataset_reordered.csv",
        usecols=['Filtered_Patient_ZipCode', 'Department'], # Загружаем только Department
        dtype={'Filtered_Patient_ZipCode': str},
        low_memory=False
    )
    df_raw_data_svi_dept['Filtered_Patient_ZipCode'] = df_raw_data_svi_dept['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)

except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (стандартный код очистки) ...
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
print("✅ Данные загружены и очищены.")

# --- 3. Идентификация групп клиник ('New PC' и 'Acquired PC') ---
# ... (код идентификации без изменений) ...
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_pc_urgent, axis=1)].copy()
# Acquired PC
pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
def is_acq_pc(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
    if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_acq_pc, axis=1))].copy()
print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

# --- 4. Расчет Карт Лечения (Метод Радиуса 10 км) ---
print("\n--- Расчет Карт Лечения (Метод Радиуса 10 км) ---")
TREATMENT_RADIUS_KM = 10; treatment_dates = {}
patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет Дат Лечения"):
    patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
    earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
    for _, clinic_row in df_new_PC_clinics.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
    if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
    for _, clinic_row in df_acquired_pc_clinics.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
    if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
    treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

# --- 5. Агрегация данных и создание Y-переменных ---
print("\n--- Агрегация данных и создание Y-переменных ---")
df_panel = final_data.copy()
# --- Присоединяем Department из сырых данных ---
df_raw_dept_map = df_raw_data_svi_dept[['Filtered_Patient_ZipCode', 'Department']].drop_duplicates()
# Если один ZIP может быть связан с разными Department, берем самый частый, например
# Или просто оставляем как есть, если агрегация ниже это учтет
# Пока просто присоединяем
# df_panel = df_panel.merge(df_raw_dept_map, on='Filtered_Patient_ZipCode', how='left') # Это может создать дубликаты, если не агрегировать df_raw_dept_map

# --- Альтернатива: Используем Department из final_data, если он там есть ---
if 'Department' not in df_panel.columns:
    print("Предупреждение: Колонка 'Department' не найдена в 'final_data'. Расчет Y_Regular_PC_at_HOSPITAL может быть неточным.")
    df_panel['Department'] = 'Unknown' # Добавляем заглушку

df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int); df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')

# Агрегация (Упрощенная, считаем только нужные Y)
grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
df_agg = df_panel.groupby(grouping_cols).agg(
    Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
    Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum()),
    # Считаем Regular PC Visits в госпитальных департаментах
    Regular_PC_Hosp_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[
        (df_panel.loc[x.index, 'Is_Regular'] == 1) &
        (df_panel.loc[x.index, 'Department'].isin(['Family Practice', 'Internal Medicine'])) # Проверьте имена!
    ].sum())
).reset_index()

# Создаем Y-переменные для моделей
df_agg['Y_ER_Total'] = np.log1p(df_agg['Total_Emergency_Visits'])
df_agg['Y_Regular_PC_at_HOSPITAL'] = np.log1p(df_agg['Regular_PC_Hosp_Visits'])
print("✅ Y-переменные Y_ER_Total и Y_Regular_PC_at_HOSPITAL созданы.")

# Присоединяем даты лечения
df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
print("✅ Данные агрегированы и готовы к анализу.")

# --- Общие параметры и функции ---
EVENT_WINDOW_MONTHLY = 12
EVENT_WINDOW_QUARTERLY = 4 # +/- 1 год
BASE_PERIOD = -1

# Функция извлечения эффектов Event Study (Месяцы)
def extract_event_study_effects_monthly(model_result, base_period, time_var_name):
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                month = int(match.group(1));
                if month == base_period: continue
                results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
    return df.sort_values('Relative_Month').reset_index(drop=True)

# Функция извлечения эффектов Event Study (Кварталы)
def extract_event_study_effects_quarterly(model_result, base_period, time_var_name):
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Quarter': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Quarter'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                qtr = int(match.group(1));
                if qtr == base_period: continue
                results['Relative_Quarter'].append(qtr); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse quarter from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Quarter'])
    return df.sort_values('Relative_Quarter').reset_index(drop=True)


# Функция построения графика Event Study (Общая)
def plot_event_study(event_data, time_col, outcome_name, title, color, event_window, base_period, time_unit="Месяцы"):
     if not event_data.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax.errorbar(x=event_data[time_col], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
        else: ax.plot(event_data[time_col], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие ({time_unit} 0)')
        ax.set_title(title, fontsize=14); ax.set_xlabel(f"{time_unit} относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
        # Устанавливаем метки оси X
        if time_unit == "Кварталы":
             ax.set_xticks(range(-event_window, event_window + 1, 1));
        else: # Месяцы
             ax.set_xticks(range(-event_window, event_window + 1, 2));
        ax.legend(); ax.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
     else: print(f"Не удалось извлечь/построить данные для: {title}")

# Функция запуска модели
def run_event_study_model(formula, data, cluster_var='Zip', time_var='time_to_event'): # Добавили time_var
    # ... (Копируем функцию run_event_study_model из предыдущего кода, добавив time_var) ...
    model_result = None
    outcome_col = formula.split('~')[0].strip()
    required_cols_run = [outcome_col, time_var, cluster_var, 'Year', 'Month'] # Month может быть Qtr
    # Адаптируем проверку для кварталов
    if 'Qtr' in formula: required_cols_run = [outcome_col, time_var, cluster_var, 'Year', 'Qtr']

    if not all(col in data.columns for col in required_cols_run):
         print(f"  ⚠️ Пропуск: Отсутствуют необходимые колонки ({required_cols_run}) для '{outcome_col}'.")
         return None
    # Проверка вариации
    if not data.empty and data[outcome_col].notna().any() and data[time_var].nunique() >= 2:
        try:
            cols_exist = [col for col in required_cols_run if col in data.columns]
            data_clean = data[cols_exist].dropna().reset_index(drop=True)
            if data_clean.empty or data_clean[time_var].nunique() < 2: return None

            print(f"... Запуск модели с C({cluster_var}) для {outcome_col} (N={len(data_clean)})...")
            model = smf.ols(formula, data=data_clean)
            model_result = model.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]})
            print(f"  ✅ Модель с C({cluster_var}) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА с C({cluster_var}): {e}")
            try:
                 formula_no_zip = formula.replace(f"+ C({cluster_var})", "")
                 print(f"     ... Попытка без C({cluster_var}) ...")
                 cols_nozip = [col for col in cols_exist if col != cluster_var]
                 data_clean_nozip = data[cols_nozip].dropna().reset_index(drop=True)
                 cluster_groups_nozip = data.loc[data_clean_nozip.index, cluster_var] # Группы для оставшихся
                 if data_clean_nozip.empty or data_clean_nozip[time_var].nunique() < 2: return None

                 model_no_zip = smf.ols(formula_no_zip, data=data_clean_nozip)
                 model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': cluster_groups_nozip})
                 print(f"        ✅ Модель без C({cluster_var}) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}")
                 model_result = None
    else:
        print(f"  ⚠️ Нет данных или вариации для запуска модели '{outcome_col}'.")
    return model_result


    # =========================================================================
    # --- ЗАДАЧА 1: Агрегация по Кварталам ---
    # =========================================================================
    print("\n\n" + "="*70); print("--- ЗАДАЧА 1: Анализ по Кварталам ---"); print("="*70)

    # --- 1.1 Подготовка данных по кварталам ---
    df_analysis_q = df_analysis.copy() # Используем df_analysis с месячными Y
    df_analysis_q['Quarter'] = pd.to_datetime(df_analysis_q['Date']).dt.to_period('Q')

    # Агрегируем до уровня ZIP-Квартал, СУММИРУЯ месячные визиты
    # Важно: Сначала суммируем визиты, потом считаем логарифм
    grouping_cols_q = ['Zip', 'Quarter', 'Year'] # Год нужен для FE
    agg_dict_q = {
        'Total_Emergency_Visits': 'sum',
        'Regular_PC_Hosp_Visits': 'sum',
        # Добавляем даты лечения (берем первую, т.к. она одна на ZIP)
        'treatment_date_NEW_PC': 'first',
        'treatment_date_ACQUIRED_PC': 'first'
    }
    df_agg_q = df_analysis_q.groupby(grouping_cols_q).agg(agg_dict_q).reset_index()

    # Считаем квартальные Y (лог от суммы + 1)
    df_agg_q['Y_ER_Total_Q'] = np.log1p(df_agg_q['Total_Emergency_Visits'])
    df_agg_q['Y_Regular_PC_Hosp_Q'] = np.log1p(df_agg_q['Regular_PC_Hosp_Visits'])
    df_agg_q['Qtr'] = df_agg_q['Quarter'].dt.quarter # Номер квартала для FE

    # Рассчитываем время до события в КВАРТАЛАХ
    df_agg_q['event_quarter_new'] = pd.to_datetime(df_agg_q['treatment_date_NEW_PC']).dt.to_period('Q')
    df_agg_q['time_to_event_Q_NEW'] = (df_agg_q['Quarter'] - df_agg_q['event_quarter_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_agg_q['event_quarter_acq'] = pd.to_datetime(df_agg_q['treatment_date_ACQUIRED_PC']).dt.to_period('Q')
    df_agg_q['time_to_event_Q_ACQ'] = (df_agg_q['Quarter'] - df_agg_q['event_quarter_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # --- 1.2 Модель 1 (Замещение, Кварталы) ---
    print("\n--- Модель 1 (Замещение, Кварталы): Y_Regular_PC_Hosp_Q ~ New_PC ---")
    OUTCOME_M1_Q = "Y_Regular_PC_Hosp_Q"
    TIME_VAR_M1_Q = 'time_to_event_Q_NEW'
    # Фильтруем данные для модели
    df_model_M1_Q = df_agg_q[df_agg_q[TIME_VAR_M1_Q].between(-EVENT_WINDOW_QUARTERLY, EVENT_WINDOW_QUARTERLY)].copy()
    df_model_M1_Q[TIME_VAR_M1_Q] = df_model_M1_Q[TIME_VAR_M1_Q].astype(int)
    formula_M1_Q = f"{OUTCOME_M1_Q} ~ C({TIME_VAR_M1_Q}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Qtr)"
    model_result_M1_Q = run_event_study_model(formula_M1_Q, df_model_M1_Q, time_var=TIME_VAR_M1_Q)

    # Визуализация 1 (Кварталы)
    event_data_M1_Q = extract_event_study_effects_quarterly(model_result_M1_Q, BASE_PERIOD, TIME_VAR_M1_Q)
    plot_event_study(event_data_M1_Q, 'Relative_Quarter', OUTCOME_M1_Q, "Модель 1 (Кварталы): Эффект 'New PC' на Log Regular PC Visits (Hospital)", 'blue', EVENT_WINDOW_QUARTERLY, BASE_PERIOD, time_unit="Кварталы")


    # --- 1.3 Модель 2 (Воронка, Кварталы) ---
    print("\n--- Модель 2 (Воронка, Кварталы): Y_ER_Total_Q ~ Acquired_PC ---")
    OUTCOME_M2_Q = "Y_ER_Total_Q"
    TIME_VAR_M2_Q = 'time_to_event_Q_ACQ'
    # Фильтруем данные для модели
    df_model_M2_Q = df_agg_q[df_agg_q[TIME_VAR_M2_Q].between(-EVENT_WINDOW_QUARTERLY, EVENT_WINDOW_QUARTERLY)].copy()
    df_model_M2_Q[TIME_VAR_M2_Q] = df_model_M2_Q[TIME_VAR_M2_Q].astype(int)
    formula_M2_Q = f"{OUTCOME_M2_Q} ~ C({TIME_VAR_M2_Q}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Qtr)"
    model_result_M2_Q = run_event_study_model(formula_M2_Q, df_model_M2_Q, time_var=TIME_VAR_M2_Q)

    # Визуализация 2 (Кварталы)
    event_data_M2_Q = extract_event_study_effects_quarterly(model_result_M2_Q, BASE_PERIOD, TIME_VAR_M2_Q)
    plot_event_study(event_data_M2_Q, 'Relative_Quarter', OUTCOME_M2_Q, "Модель 2 (Кварталы): Эффект 'Acquired PC' на Log Total ER Visits", 'red', EVENT_WINDOW_QUARTERLY, BASE_PERIOD, time_unit="Кварталы")


    # =========================================================================
    # --- ЗАДАЧА 2: Таблицы Регрессий (Месячный формат) ---
    # =========================================================================
    print("\n\n" + "="*70); print("--- ЗАДАЧА 2: Таблицы Регрессий (Месячный формат) ---"); print("="*70)

    # --- Модель 1 (Замещение, Месяцы) ---
    print("\n--- Таблица для Модели 1 (Замещение, Месяцы): Y_Regular_PC_at_HOSPITAL ~ New_PC ---")
    OUTCOME_M1_M = "Y_Regular_PC_at_HOSPITAL"
    TIME_VAR_M1_M = 'time_to_event_NEW_PC'
    # Используем df_analysis с месячными данными
    df_model_M1_M = df_analysis.copy()
    df_model_M1_M['event_month'] = pd.to_datetime(df_model_M1_M['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_M1_M['current_month'] = pd.to_datetime(df_model_M1_M['Date']).dt.to_period('M')
    df_model_M1_M[TIME_VAR_M1_M] = (df_model_M1_M['current_month'] - df_model_M1_M['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_model_M1_M_filtered = df_model_M1_M[df_model_M1_M[TIME_VAR_M1_M].between(-EVENT_WINDOW_MONTHLY, EVENT_WINDOW_MONTHLY)].copy()
    df_model_M1_M_filtered[TIME_VAR_M1_M] = df_model_M1_M_filtered[TIME_VAR_M1_M].astype(int)
    # Создаем Y_Regular_PC_at_HOSPITAL, если его еще нет
    if OUTCOME_M1_M not in df_model_M1_M_filtered.columns:
         if 'Regular_PC_Hosp_Visits' in df_model_M1_M_filtered.columns:
              df_model_M1_M_filtered[OUTCOME_M1_M] = np.log1p(df_model_M1_M_filtered['Regular_PC_Hosp_Visits'])
         else:
              print(f"  ⚠️ Предупреждение: Колонка 'Regular_PC_Hosp_Visits' не найдена для расчета {OUTCOME_M1_M}")
              df_model_M1_M_filtered[OUTCOME_M1_M] = 0 # Заглушка

    formula_M1_M = f"{OUTCOME_M1_M} ~ C({TIME_VAR_M1_M}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    model_result_M1_M = run_event_study_model(formula_M1_M, df_model_M1_M_filtered, time_var=TIME_VAR_M1_M)

    # Вывод таблицы
    if model_result_M1_M:
        print("\nПолная таблица регрессии:")
        try:
             # Выводим только коэффициенты time_to_event
             summary_table = model_result_M1_M.summary().tables[1]
             print(summary_table)
             # Альтернативно, фильтруем параметры
             # event_params = model_result_M1_M.params[model_result_M1_M.params.index.str.contains(f'C\({TIME_VAR_M1_M}')]
             # event_pvalues = model_result_M1_M.pvalues[model_result_M1_M.pvalues.index.str.contains(f'C\({TIME_VAR_M1_M}')]
             # event_summary = pd.DataFrame({'Coef.': event_params, 'P>|t|': event_pvalues})
             # print(event_summary.to_string())
        except Exception as e_summary:
             print(f"  Не удалось вывести таблицу summary: {e_summary}")
    else:
        print("  Модель не была рассчитана.")


    # --- Модель 2 (Воронка, Месяцы) ---
    print("\n--- Таблица для Модели 2 (Воронка, Месяцы): Y_ER_Total ~ Acquired_PC ---")
    OUTCOME_M2_M = "Y_ER_Total"
    TIME_VAR_M2_M = 'time_to_event_ACQUIRED_PC'
    # Используем df_analysis с месячными данными
    df_model_M2_M = df_analysis.copy()
    df_model_M2_M['event_month'] = pd.to_datetime(df_model_M2_M['treatment_date_ACQUIRED_PC']).dt.to_period('M')
    df_model_M2_M['current_month'] = pd.to_datetime(df_model_M2_M['Date']).dt.to_period('M')
    df_model_M2_M[TIME_VAR_M2_M] = (df_model_M2_M['current_month'] - df_model_M2_M['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_model_M2_M_filtered = df_model_M2_M[df_model_M2_M[TIME_VAR_M2_M].between(-EVENT_WINDOW_MONTHLY, EVENT_WINDOW_MONTHLY)].copy()
    df_model_M2_M_filtered[TIME_VAR_M2_M] = df_model_M2_M_filtered[TIME_VAR_M2_M].astype(int)
    # Создаем Y_ER_Total, если его еще нет
    if OUTCOME_M2_M not in df_model_M2_M_filtered.columns:
         if 'Total_Emergency_Visits' in df_model_M2_M_filtered.columns:
              df_model_M2_M_filtered[OUTCOME_M2_M] = np.log1p(df_model_M2_M_filtered['Total_Emergency_Visits'])
         else:
              print(f"  ⚠️ Предупреждение: Колонка 'Total_Emergency_Visits' не найдена для расчета {OUTCOME_M2_M}")
              df_model_M2_M_filtered[OUTCOME_M2_M] = 0

    formula_M2_M = f"{OUTCOME_M2_M} ~ C({TIME_VAR_M2_M}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    model_result_M2_M = run_event_study_model(formula_M2_M, df_model_M2_M_filtered, time_var=TIME_VAR_M2_M)

    # Вывод таблицы
    if model_result_M2_M:
        print("\nПолная таблица регрессии:")
        try:
             summary_table2 = model_result_M2_M.summary().tables[1]
             print(summary_table2)
        except Exception as e_summary2:
             print(f"  Не удалось вывести таблицу summary: {e_summary2}")

    else:
        print("  Модель не была рассчитана.")


    print("\n🎉 --- Финальные Проверки завершены! ---")

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО ФИНАЛЬНЫХ ПРОВЕРОК ('НАСЫЩЕНИЕ' + Кварталы + Таблицы) ---")

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("Предупреждение: 'final_data' не найден. Пытаюсь загрузить 'aggregated_data.csv'")
    try:
        final_data = pd.read_csv('aggregated_data.csv', dtype={'Filtered_Patient_ZipCode': str, 'Zip': str})
        if 'Date' in final_data.columns and not pd.api.types.is_datetime64_any_dtype(final_data['Date']):
             final_data['Date'] = pd.to_datetime(final_data['Date'])
        if 'Zip' in final_data.columns and 'Filtered_Patient_ZipCode' not in final_data.columns:
            final_data = final_data.rename(columns={'Zip': 'Filtered_Patient_ZipCode'})
        print("  ✅ 'aggregated_data.csv' загружен.")
    except FileNotFoundError:
        print("❌ ОШИБКА: DataFrame 'final_data' не найден и 'aggregated_data.csv' тоже.")
        raise FileNotFoundError("Необходим 'final_data' или 'aggregated_data.csv'")

# Загрузка остальных файлов
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    print("... Загрузка исходного файла для Department ...")
    df_raw_data_svi_dept = pd.read_csv(
        "final_filtered_dataset_reordered.csv",
        usecols=['Filtered_Patient_ZipCode', 'Department'], # Загружаем только Department
        dtype={'Filtered_Patient_ZipCode': str},
        low_memory=False
    )
    df_raw_data_svi_dept['Filtered_Patient_ZipCode'] = df_raw_data_svi_dept['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)

except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (стандартный код очистки) ...
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
print("✅ Данные загружены и очищены.")

# --- 3. Идентификация групп клиник ('New PC' и 'Acquired PC') ---
# ... (код идентификации без изменений) ...
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_pc_urgent, axis=1)].copy()
# Acquired PC
pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
def is_acq_pc(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
    if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_acq_pc, axis=1))].copy()
print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

# --- 4. Расчет Карт Лечения (Метод Радиуса 10 км) ---
print("\n--- Расчет Карт Лечения (Метод Радиуса 10 км) ---")
TREATMENT_RADIUS_KM = 10; treatment_dates = {}
patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет Дат Лечения"):
    patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
    earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
    for _, clinic_row in df_new_PC_clinics.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
    if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
    for _, clinic_row in df_acquired_pc_clinics.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
    if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
    treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

# --- 5. Агрегация данных и создание Y-переменных ---
print("\n--- Агрегация данных и создание Y-переменных ---")
df_panel = final_data.copy()
# --- Присоединяем Department из сырых данных ---
df_raw_dept_map = df_raw_data_svi_dept[['Filtered_Patient_ZipCode', 'Department']].drop_duplicates()
# Если один ZIP может быть связан с разными Department, берем самый частый, например
# Или просто оставляем как есть, если агрегация ниже это учтет
# Пока просто присоединяем
# df_panel = df_panel.merge(df_raw_dept_map, on='Filtered_Patient_ZipCode', how='left') # Это может создать дубликаты, если не агрегировать df_raw_dept_map

# --- Альтернатива: Используем Department из final_data, если он там есть ---
if 'Department' not in df_panel.columns:
    print("Предупреждение: Колонка 'Department' не найдена в 'final_data'. Расчет Y_Regular_PC_at_HOSPITAL может быть неточным.")
    df_panel['Department'] = 'Unknown' # Добавляем заглушку

df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int); df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')

# Агрегация (Упрощенная, считаем только нужные Y)
grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
df_agg = df_panel.groupby(grouping_cols).agg(
    Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
    Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum()),
    # Считаем Regular PC Visits в госпитальных департаментах
    Regular_PC_Hosp_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[
        (df_panel.loc[x.index, 'Is_Regular'] == 1) &
        (df_panel.loc[x.index, 'Department'].isin(['Family Practice', 'Internal Medicine'])) # Проверьте имена!
    ].sum())
).reset_index()

# Создаем Y-переменные для моделей
df_agg['Y_ER_Total'] = np.log1p(df_agg['Total_Emergency_Visits'])
df_agg['Y_Regular_PC_at_HOSPITAL'] = np.log1p(df_agg['Regular_PC_Hosp_Visits'])
print("✅ Y-переменные Y_ER_Total и Y_Regular_PC_at_HOSPITAL созданы.")

# Присоединяем даты лечения
df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
print("✅ Данные агрегированы и готовы к анализу.")

# --- Общие параметры и функции ---
EVENT_WINDOW_MONTHLY = 12
EVENT_WINDOW_QUARTERLY = 4 # +/- 1 год
BASE_PERIOD = -1

# Функция извлечения эффектов Event Study (Месяцы)
def extract_event_study_effects_monthly(model_result, base_period, time_var_name):
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                month = int(match.group(1));
                if month == base_period: continue
                results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
    return df.sort_values('Relative_Month').reset_index(drop=True)

# Функция извлечения эффектов Event Study (Кварталы)
def extract_event_study_effects_quarterly(model_result, base_period, time_var_name):
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Quarter': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Quarter'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                qtr = int(match.group(1));
                if qtr == base_period: continue
                results['Relative_Quarter'].append(qtr); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse quarter from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Quarter'])
    return df.sort_values('Relative_Quarter').reset_index(drop=True)


# Функция построения графика Event Study (Общая)
def plot_event_study(event_data, time_col, outcome_name, title, color, event_window, base_period, time_unit="Месяцы"):
     if not event_data.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax.errorbar(x=event_data[time_col], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
        else: ax.plot(event_data[time_col], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие ({time_unit} 0)')
        ax.set_title(title, fontsize=14); ax.set_xlabel(f"{time_unit} относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
        # Устанавливаем метки оси X
        if time_unit == "Кварталы":
             ax.set_xticks(range(-event_window, event_window + 1, 1));
        else: # Месяцы
             ax.set_xticks(range(-event_window, event_window + 1, 2));
        ax.legend(); ax.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
     else: print(f"Не удалось извлечь/построить данные для: {title}")

# Функция запуска модели
def run_event_study_model(formula, data, cluster_var='Zip', time_var='time_to_event'): # Добавили time_var
    # ... (Копируем функцию run_event_study_model из предыдущего кода, добавив time_var) ...
    model_result = None
    outcome_col = formula.split('~')[0].strip()
    required_cols_run = [outcome_col, time_var, cluster_var, 'Year', 'Month'] # Month может быть Qtr
    # Адаптируем проверку для кварталов
    if 'Qtr' in formula: required_cols_run = [outcome_col, time_var, cluster_var, 'Year', 'Qtr']

    if not all(col in data.columns for col in required_cols_run):
         print(f"  ⚠️ Пропуск: Отсутствуют необходимые колонки ({required_cols_run}) для '{outcome_col}'.")
         return None
    # Проверка вариации
    if not data.empty and data[outcome_col].notna().any() and data[time_var].nunique() >= 2:
        try:
            cols_exist = [col for col in required_cols_run if col in data.columns]
            data_clean = data[cols_exist].dropna().reset_index(drop=True)
            if data_clean.empty or data_clean[time_var].nunique() < 2: return None

            print(f"... Запуск модели с C({cluster_var}) для {outcome_col} (N={len(data_clean)})...")
            model = smf.ols(formula, data=data_clean)
            model_result = model.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]})
            print(f"  ✅ Модель с C({cluster_var}) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА с C({cluster_var}): {e}")
            try:
                 formula_no_zip = formula.replace(f"+ C({cluster_var})", "")
                 print(f"     ... Попытка без C({cluster_var}) ...")
                 cols_nozip = [col for col in cols_exist if col != cluster_var]
                 data_clean_nozip = data[cols_nozip].dropna().reset_index(drop=True)
                 cluster_groups_nozip = data.loc[data_clean_nozip.index, cluster_var] # Группы для оставшихся
                 if data_clean_nozip.empty or data_clean_nozip[time_var].nunique() < 2: return None

                 model_no_zip = smf.ols(formula_no_zip, data=data_clean_nozip)
                 model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': cluster_groups_nozip})
                 print(f"        ✅ Модель без C({cluster_var}) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}")
                 model_result = None
    else:
        print(f"  ⚠️ Нет данных или вариации для запуска модели '{outcome_col}'.")
    return model_result


    # =========================================================================
    # --- ЗАДАЧА 1: Агрегация по Кварталам ---
    # =========================================================================
    print("\n\n" + "="*70); print("--- ЗАДАЧА 1: Анализ по Кварталам ---"); print("="*70)

    # --- 1.1 Подготовка данных по кварталам ---
    df_analysis_q = df_analysis.copy() # Используем df_analysis с месячными Y
    df_analysis_q['Quarter'] = pd.to_datetime(df_analysis_q['Date']).dt.to_period('Q')

    # Агрегируем до уровня ZIP-Квартал, СУММИРУЯ месячные визиты
    # Важно: Сначала суммируем визиты, потом считаем логарифм
    grouping_cols_q = ['Zip', 'Quarter', 'Year'] # Год нужен для FE
    agg_dict_q = {
        'Total_Emergency_Visits': 'sum',
        'Regular_PC_Hosp_Visits': 'sum',
        # Добавляем даты лечения (берем первую, т.к. она одна на ZIP)
        'treatment_date_NEW_PC': 'first',
        'treatment_date_ACQUIRED_PC': 'first'
    }
    df_agg_q = df_analysis_q.groupby(grouping_cols_q).agg(agg_dict_q).reset_index()

    # Считаем квартальные Y (лог от суммы + 1)
    df_agg_q['Y_ER_Total_Q'] = np.log1p(df_agg_q['Total_Emergency_Visits'])
    df_agg_q['Y_Regular_PC_Hosp_Q'] = np.log1p(df_agg_q['Regular_PC_Hosp_Visits'])
    df_agg_q['Qtr'] = df_agg_q['Quarter'].dt.quarter # Номер квартала для FE

    # Рассчитываем время до события в КВАРТАЛАХ
    df_agg_q['event_quarter_new'] = pd.to_datetime(df_agg_q['treatment_date_NEW_PC']).dt.to_period('Q')
    df_agg_q['time_to_event_Q_NEW'] = (df_agg_q['Quarter'] - df_agg_q['event_quarter_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_agg_q['event_quarter_acq'] = pd.to_datetime(df_agg_q['treatment_date_ACQUIRED_PC']).dt.to_period('Q')
    df_agg_q['time_to_event_Q_ACQ'] = (df_agg_q['Quarter'] - df_agg_q['event_quarter_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)

    # --- 1.2 Модель 1 (Замещение, Кварталы) ---
    print("\n--- Модель 1 (Замещение, Кварталы): Y_Regular_PC_Hosp_Q ~ New_PC ---")
    OUTCOME_M1_Q = "Y_Regular_PC_Hosp_Q"
    TIME_VAR_M1_Q = 'time_to_event_Q_NEW'
    # Фильтруем данные для модели
    df_model_M1_Q = df_agg_q[df_agg_q[TIME_VAR_M1_Q].between(-EVENT_WINDOW_QUARTERLY, EVENT_WINDOW_QUARTERLY)].copy()
    df_model_M1_Q[TIME_VAR_M1_Q] = df_model_M1_Q[TIME_VAR_M1_Q].astype(int)
    formula_M1_Q = f"{OUTCOME_M1_Q} ~ C({TIME_VAR_M1_Q}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Qtr)"
    model_result_M1_Q = run_event_study_model(formula_M1_Q, df_model_M1_Q, time_var=TIME_VAR_M1_Q)

    # Визуализация 1 (Кварталы)
    event_data_M1_Q = extract_event_study_effects_quarterly(model_result_M1_Q, BASE_PERIOD, TIME_VAR_M1_Q)
    plot_event_study(event_data_M1_Q, 'Relative_Quarter', OUTCOME_M1_Q, "Модель 1 (Кварталы): Эффект 'New PC' на Log Regular PC Visits (Hospital)", 'blue', EVENT_WINDOW_QUARTERLY, BASE_PERIOD, time_unit="Кварталы")


    # --- 1.3 Модель 2 (Воронка, Кварталы) ---
    print("\n--- Модель 2 (Воронка, Кварталы): Y_ER_Total_Q ~ Acquired_PC ---")
    OUTCOME_M2_Q = "Y_ER_Total_Q"
    TIME_VAR_M2_Q = 'time_to_event_Q_ACQ'
    # Фильтруем данные для модели
    df_model_M2_Q = df_agg_q[df_agg_q[TIME_VAR_M2_Q].between(-EVENT_WINDOW_QUARTERLY, EVENT_WINDOW_QUARTERLY)].copy()
    df_model_M2_Q[TIME_VAR_M2_Q] = df_model_M2_Q[TIME_VAR_M2_Q].astype(int)
    formula_M2_Q = f"{OUTCOME_M2_Q} ~ C({TIME_VAR_M2_Q}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Qtr)"
    model_result_M2_Q = run_event_study_model(formula_M2_Q, df_model_M2_Q, time_var=TIME_VAR_M2_Q)

    # Визуализация 2 (Кварталы)
    event_data_M2_Q = extract_event_study_effects_quarterly(model_result_M2_Q, BASE_PERIOD, TIME_VAR_M2_Q)
    plot_event_study(event_data_M2_Q, 'Relative_Quarter', OUTCOME_M2_Q, "Модель 2 (Кварталы): Эффект 'Acquired PC' на Log Total ER Visits", 'red', EVENT_WINDOW_QUARTERLY, BASE_PERIOD, time_unit="Кварталы")


    # =========================================================================
    # --- ЗАДАЧА 2: Таблицы Регрессий (Месячный формат) ---
    # =========================================================================
    print("\n\n" + "="*70); print("--- ЗАДАЧА 2: Таблицы Регрессий (Месячный формат) ---"); print("="*70)

    # --- Модель 1 (Замещение, Месяцы) ---
    print("\n--- Таблица для Модели 1 (Замещение, Месяцы): Y_Regular_PC_at_HOSPITAL ~ New_PC ---")
    OUTCOME_M1_M = "Y_Regular_PC_at_HOSPITAL"
    TIME_VAR_M1_M = 'time_to_event_NEW_PC'
    # Используем df_analysis с месячными данными
    df_model_M1_M = df_analysis.copy()
    df_model_M1_M['event_month'] = pd.to_datetime(df_model_M1_M['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_M1_M['current_month'] = pd.to_datetime(df_model_M1_M['Date']).dt.to_period('M')
    df_model_M1_M[TIME_VAR_M1_M] = (df_model_M1_M['current_month'] - df_model_M1_M['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_model_M1_M_filtered = df_model_M1_M[df_model_M1_M[TIME_VAR_M1_M].between(-EVENT_WINDOW_MONTHLY, EVENT_WINDOW_MONTHLY)].copy()
    df_model_M1_M_filtered[TIME_VAR_M1_M] = df_model_M1_M_filtered[TIME_VAR_M1_M].astype(int)
    # Создаем Y_Regular_PC_at_HOSPITAL, если его еще нет
    if OUTCOME_M1_M not in df_model_M1_M_filtered.columns:
         if 'Regular_PC_Hosp_Visits' in df_model_M1_M_filtered.columns:
              df_model_M1_M_filtered[OUTCOME_M1_M] = np.log1p(df_model_M1_M_filtered['Regular_PC_Hosp_Visits'])
         else:
              print(f"  ⚠️ Предупреждение: Колонка 'Regular_PC_Hosp_Visits' не найдена для расчета {OUTCOME_M1_M}")
              df_model_M1_M_filtered[OUTCOME_M1_M] = 0 # Заглушка

    formula_M1_M = f"{OUTCOME_M1_M} ~ C({TIME_VAR_M1_M}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    model_result_M1_M = run_event_study_model(formula_M1_M, df_model_M1_M_filtered, time_var=TIME_VAR_M1_M)

    # Вывод таблицы
    if model_result_M1_M:
        print("\nПолная таблица регрессии:")
        try:
             # Выводим только коэффициенты time_to_event
             summary_table = model_result_M1_M.summary().tables[1]
             print(summary_table)
             # Альтернативно, фильтруем параметры
             # event_params = model_result_M1_M.params[model_result_M1_M.params.index.str.contains(f'C\({TIME_VAR_M1_M}')]
             # event_pvalues = model_result_M1_M.pvalues[model_result_M1_M.pvalues.index.str.contains(f'C\({TIME_VAR_M1_M}')]
             # event_summary = pd.DataFrame({'Coef.': event_params, 'P>|t|': event_pvalues})
             # print(event_summary.to_string())
        except Exception as e_summary:
             print(f"  Не удалось вывести таблицу summary: {e_summary}")
    else:
        print("  Модель не была рассчитана.")


    # --- Модель 2 (Воронка, Месяцы) ---
    print("\n--- Таблица для Модели 2 (Воронка, Месяцы): Y_ER_Total ~ Acquired_PC ---")
    OUTCOME_M2_M = "Y_ER_Total"
    TIME_VAR_M2_M = 'time_to_event_ACQUIRED_PC'
    # Используем df_analysis с месячными данными
    df_model_M2_M = df_analysis.copy()
    df_model_M2_M['event_month'] = pd.to_datetime(df_model_M2_M['treatment_date_ACQUIRED_PC']).dt.to_period('M')
    df_model_M2_M['current_month'] = pd.to_datetime(df_model_M2_M['Date']).dt.to_period('M')
    df_model_M2_M[TIME_VAR_M2_M] = (df_model_M2_M['current_month'] - df_model_M2_M['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_model_M2_M_filtered = df_model_M2_M[df_model_M2_M[TIME_VAR_M2_M].between(-EVENT_WINDOW_MONTHLY, EVENT_WINDOW_MONTHLY)].copy()
    df_model_M2_M_filtered[TIME_VAR_M2_M] = df_model_M2_M_filtered[TIME_VAR_M2_M].astype(int)
    # Создаем Y_ER_Total, если его еще нет
    if OUTCOME_M2_M not in df_model_M2_M_filtered.columns:
         if 'Total_Emergency_Visits' in df_model_M2_M_filtered.columns:
              df_model_M2_M_filtered[OUTCOME_M2_M] = np.log1p(df_model_M2_M_filtered['Total_Emergency_Visits'])
         else:
              print(f"  ⚠️ Предупреждение: Колонка 'Total_Emergency_Visits' не найдена для расчета {OUTCOME_M2_M}")
              df_model_M2_M_filtered[OUTCOME_M2_M] = 0

    formula_M2_M = f"{OUTCOME_M2_M} ~ C({TIME_VAR_M2_M}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    model_result_M2_M = run_event_study_model(formula_M2_M, df_model_M2_M_filtered, time_var=TIME_VAR_M2_M)

    # Вывод таблицы
    if model_result_M2_M:
        print("\nПолная таблица регрессии:")
        try:
             summary_table2 = model_result_M2_M.summary().tables[1]
             print(summary_table2)
        except Exception as e_summary2:
             print(f"  Не удалось вывести таблицу summary: {e_summary2}")

    else:
        print("  Модель не была рассчитана.")


    print("\n🎉 --- Финальные Проверки завершены! ---")

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО ФИНАЛЬНЫХ ПРОВЕРОК ('НАСЫЩЕНИЕ' + Кварталы + Таблицы +/- 1 год) ---") # Изменен заголовок

# --- 1. Проверка и Загрузка Данных ---
if 'final_data' not in locals():
    print("Предупреждение: 'final_data' не найден. Пытаюсь загрузить 'aggregated_data.csv'")
    try:
        # Убедимся что Zip читается как строка при загрузке
        final_data = pd.read_csv('aggregated_data.csv', dtype={'Filtered_Patient_ZipCode': str, 'Zip': str})
        if 'Date' in final_data.columns and not pd.api.types.is_datetime64_any_dtype(final_data['Date']):
             final_data['Date'] = pd.to_datetime(final_data['Date'])
        if 'Zip' in final_data.columns and 'Filtered_Patient_ZipCode' not in final_data.columns:
            final_data = final_data.rename(columns={'Zip': 'Filtered_Patient_ZipCode'})
        print("  ✅ 'aggregated_data.csv' загружен.")
    except FileNotFoundError:
        print("❌ ОШИБКА: DataFrame 'final_data' не найден и 'aggregated_data.csv' тоже.")
        raise FileNotFoundError("Необходим 'final_data' или 'aggregated_data.csv'")

# Загрузка остальных файлов
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    # --- УБРАЛИ ЗАГРУЗКУ final_filtered_dataset_reordered.csv для Department ---
    # print("... Загрузка исходного файла для Department ...")
    # df_raw_data_svi_dept = pd.read_csv(...)

except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (стандартный код очистки) ...
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
df_clinics_info_full['open_month'] = df_clinics_info_full['event_date'].dt.to_period('M')
print("✅ Данные загружены и очищены.")

# --- 3. Идентификация групп клиник ('New PC' и 'Acquired PC') ---
# ... (код идентификации без изменений) ...
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_pc_urgent, axis=1)].copy()
# Acquired PC
pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
def is_acq_pc(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
    if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_acq_pc, axis=1))].copy()
print(f"Идентифицировано: {len(df_new_PC_clinics)} New PC/Urgent клиник, {len(df_acquired_pc_clinics)} Acquired PC клиник.")

# --- 4. Расчет Карт Лечения (Метод Радиуса 10 км) ---
print("\n--- Расчет Карт Лечения (Метод Радиуса 10 км) ---")
TREATMENT_RADIUS_KM = 10; treatment_dates = {}
patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет Дат Лечения"):
    patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
    earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
    for _, clinic_row in df_new_PC_clinics.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
    if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
    for _, clinic_row in df_acquired_pc_clinics.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
    if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
    treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

# --- 5. Агрегация данных и создание Y-переменных ---
print("\n--- Агрегация данных и создание Y-переменных ---")
df_panel = final_data.copy()

# --- ИСПРАВЛЕНИЕ: Обрабатываем Department в df_panel (из final_data) ---
if 'Department' not in df_panel.columns:
    print("Предупреждение: Колонка 'Department' не найдена в 'final_data'. Расчет Y_Regular_PC_at_HOSPITAL будет неточным.")
    df_panel['Department'] = 'Unknown' # Добавляем заглушку
else:
    # Заполняем пропуски, если они есть
    df_panel['Department'] = df_panel['Department'].fillna('Unknown')
    # Убедимся, что это строка
    df_panel['Department'] = df_panel['Department'].astype(str)

# --- УБРАЛИ НЕПРАВИЛЬНЫЙ MERGE ---
# df_raw_dept_map = ...
# df_panel = df_panel.merge(df_raw_dept_map, ...)

df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int); df_panel['Is_Regular'] = (df_panel['VisitType'] != 'Emergency').astype(int)
df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')

# Агрегация (Используем исправленный df_panel)
grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
print("... Выполняю агрегацию ...")
df_agg = df_panel.groupby(grouping_cols).agg(
    Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Emergency'] == 1].sum()),
    Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel.loc[x.index, 'Is_Regular'] == 1].sum()),
    # Считаем Regular PC Visits в госпитальных департаментах
    # Используем Department из агрегируемой группы (x)
    Regular_PC_Hosp_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[
        (df_panel.loc[x.index, 'Is_Regular'] == 1) &
        (df_panel.loc[x.index, 'Department'].isin(['Family Practice', 'Internal Medicine'])) # Проверьте имена!
    ].sum()) # Используем sum() вместо ['EncounterCount'].sum() т.к. x это уже EncounterCount
).reset_index()
print("... Агрегация завершена ...")


# Создаем Y-переменные для моделей
df_agg['Y_ER_Total'] = np.log1p(df_agg['Total_Emergency_Visits'])
df_agg['Y_Regular_PC_at_HOSPITAL'] = np.log1p(df_agg['Regular_PC_Hosp_Visits'])
print("✅ Y-переменные Y_ER_Total и Y_Regular_PC_at_HOSPITAL созданы.")

# Присоединяем даты лечения
df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
print("✅ Данные агрегированы и готовы к анализу.")

# --- Общие параметры и функции ---
EVENT_WINDOW_MONTHLY = 12
EVENT_WINDOW_QUARTERLY = 4 # +/- 1 год
BASE_PERIOD = -1

# Функция извлечения эффектов Event Study (Месяцы)
def extract_event_study_effects_monthly(model_result, base_period, time_var_name):
    # ... (код без изменений) ...
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                month = int(match.group(1));
                if month == base_period: continue
                results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
    return df.sort_values('Relative_Month').reset_index(drop=True)

# Функция извлечения эффектов Event Study (Кварталы)
def extract_event_study_effects_quarterly(model_result, base_period, time_var_name):
    # ... (код без изменений) ...
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\]";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Quarter': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Quarter'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                qtr = int(match.group(1));
                if qtr == base_period: continue
                results['Relative_Quarter'].append(qtr); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse quarter from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Quarter'])
    return df.sort_values('Relative_Quarter').reset_index(drop=True)

# Функция построения графика Event Study (Общая)
def plot_event_study(event_data, time_col, outcome_name, title, color, event_window, base_period, time_unit="Месяцы"):
     # ... (код без изменений, используем стандартные метки X) ...
     if not event_data.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax.errorbar(x=event_data[time_col], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Эффект на {outcome_name}', color=color)
        else: ax.plot(event_data[time_col], event_data['Effect'], marker='o', linestyle='-', label=f'Эффект на {outcome_name} (без ДИ)', color=color)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Событие ({time_unit} 0)')
        ax.set_title(title, fontsize=14); ax.set_xlabel(f"{time_unit} относительно события"); ax.set_ylabel(f"Эффект на {outcome_name}")
        # --- Стандартные метки X ---
        if time_unit == "Кварталы":
             ax.set_xticks(range(-event_window, event_window + 1, 1));
        else: # Месяцы
             ax.set_xticks(range(-event_window, event_window + 1, 2)); # Каждые 2 месяца для +/-12
        ax.legend(); ax.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
     else: print(f"Не удалось извлечь/построить данные для: {title}")

# Функция запуска модели
def run_event_study_model(formula, data, cluster_var='Zip', time_var='time_to_event'):
    # ... (код без изменений) ...
    model_result = None
    outcome_col = formula.split('~')[0].strip()
    required_cols_run = [outcome_col, time_var, cluster_var, 'Year', 'Month'] # Month может быть Qtr
    if 'Qtr' in formula: required_cols_run = [outcome_col, time_var, cluster_var, 'Year', 'Qtr']
    # Добавляем проверку на существование колонок перед доступом
    if not all(col in data.columns for col in required_cols_run):
         print(f"  ⚠️ Пропуск: Отсутствуют необходимые колонки ({[col for col in required_cols_run if col not in data.columns]}) для '{outcome_col}'.")
         return None
    # Проверка вариации
    if not data.empty and data[outcome_col].notna().any() and data[time_var].nunique() >= 2:
        try:
            cols_exist = [col for col in required_cols_run if col in data.columns]
            # --- Убедимся, что time_var существует перед dropna ---
            if time_var not in data.columns: return None
            data_clean = data[cols_exist].dropna().reset_index(drop=True)
            if data_clean.empty or data_clean[time_var].nunique() < 2: return None
            print(f"... Запуск модели с C({cluster_var}) для {outcome_col} (N={len(data_clean)})...")
            model = smf.ols(formula, data=data_clean)
            model_result = model.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]})
            print(f"  ✅ Модель с C({cluster_var}) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА с C({cluster_var}): {e}")
            try:
                 formula_no_zip = formula.replace(f"+ C({cluster_var})", "")
                 print(f"     ... Попытка без C({cluster_var}) ...")
                 cols_nozip = [col for col in cols_exist if col != cluster_var]
                 # --- Убедимся, что time_var существует перед dropna ---
                 if time_var not in data.columns: return None
                 data_clean_nozip = data[cols_nozip].dropna().reset_index(drop=True)
                 cluster_groups_nozip = data.loc[data_clean_nozip.index, cluster_var] # Группы для оставшихся
                 if data_clean_nozip.empty or data_clean_nozip[time_var].nunique() < 2: return None
                 model_no_zip = smf.ols(formula_no_zip, data=data_clean_nozip)
                 model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': cluster_groups_nozip})
                 print(f"        ✅ Модель без C({cluster_var}) рассчитана.")
            except Exception as e2:
                 print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}")
                 model_result = None
    else:
        print(f"  ⚠️ Нет данных или вариации для запуска модели '{outcome_col}'.")
    return model_result


    # =========================================================================
    # --- ЗАДАЧА 1: Анализ по Кварталам (Графики) ---
    # =========================================================================
    print("\n\n" + "="*70); print("--- ЗАДАЧА 1: Анализ по Кварталам (Графики) ---"); print("="*70)
    # --- (Код для квартального анализа без изменений) ---
    df_analysis_q = df_analysis.copy(); df_analysis_q['Quarter'] = pd.to_datetime(df_analysis_q['Date']).dt.to_period('Q')
    grouping_cols_q = ['Zip', 'Quarter', 'Year']; agg_dict_q = {'Total_Emergency_Visits': 'sum','Regular_PC_Hosp_Visits': 'sum','treatment_date_NEW_PC': 'first','treatment_date_ACQUIRED_PC': 'first'}
    df_agg_q = df_analysis_q.groupby(grouping_cols_q).agg(agg_dict_q).reset_index()
    df_agg_q['Y_ER_Total_Q'] = np.log1p(df_agg_q['Total_Emergency_Visits']); df_agg_q['Y_Regular_PC_Hosp_Q'] = np.log1p(df_agg_q['Regular_PC_Hosp_Visits'])
    df_agg_q['Qtr'] = df_agg_q['Quarter'].dt.quarter
    df_agg_q['event_quarter_new'] = pd.to_datetime(df_agg_q['treatment_date_NEW_PC']).dt.to_period('Q'); df_agg_q['time_to_event_Q_NEW'] = (df_agg_q['Quarter'] - df_agg_q['event_quarter_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_agg_q['event_quarter_acq'] = pd.to_datetime(df_agg_q['treatment_date_ACQUIRED_PC']).dt.to_period('Q'); df_agg_q['time_to_event_Q_ACQ'] = (df_agg_q['Quarter'] - df_agg_q['event_quarter_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    # Модель 1 (Кварталы)
    print("\n--- Модель 1 (Замещение, Кварталы): Y_Regular_PC_Hosp_Q ~ New_PC ---")
    OUTCOME_M1_Q = "Y_Regular_PC_Hosp_Q"; TIME_VAR_M1_Q = 'time_to_event_Q_NEW'
    df_model_M1_Q = df_agg_q[df_agg_q[TIME_VAR_M1_Q].between(-EVENT_WINDOW_QUARTERLY, EVENT_WINDOW_QUARTERLY)].copy(); df_model_M1_Q[TIME_VAR_M1_Q] = df_model_M1_Q[TIME_VAR_M1_Q].astype(int)
    formula_M1_Q = f"{OUTCOME_M1_Q} ~ C({TIME_VAR_M1_Q}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Qtr)"
    model_result_M1_Q = run_event_study_model(formula_M1_Q, df_model_M1_Q, time_var=TIME_VAR_M1_Q)
    event_data_M1_Q = extract_event_study_effects_quarterly(model_result_M1_Q, BASE_PERIOD, TIME_VAR_M1_Q)
    plot_event_study(event_data_M1_Q, 'Relative_Quarter', OUTCOME_M1_Q, "Модель 1 (Кварталы): Эффект 'New PC' на Log Regular PC Visits (Hospital)", 'blue', EVENT_WINDOW_QUARTERLY, BASE_PERIOD, time_unit="Кварталы")
    # Модель 2 (Кварталы)
    print("\n--- Модель 2 (Воронка, Кварталы): Y_ER_Total_Q ~ Acquired_PC ---")
    OUTCOME_M2_Q = "Y_ER_Total_Q"; TIME_VAR_M2_Q = 'time_to_event_Q_ACQ'
    df_model_M2_Q = df_agg_q[df_agg_q[TIME_VAR_M2_Q].between(-EVENT_WINDOW_QUARTERLY, EVENT_WINDOW_QUARTERLY)].copy(); df_model_M2_Q[TIME_VAR_M2_Q] = df_model_M2_Q[TIME_VAR_M2_Q].astype(int)
    formula_M2_Q = f"{OUTCOME_M2_Q} ~ C({TIME_VAR_M2_Q}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Qtr)"
    model_result_M2_Q = run_event_study_model(formula_M2_Q, df_model_M2_Q, time_var=TIME_VAR_M2_Q)
    event_data_M2_Q = extract_event_study_effects_quarterly(model_result_M2_Q, BASE_PERIOD, TIME_VAR_M2_Q)
    plot_event_study(event_data_M2_Q, 'Relative_Quarter', OUTCOME_M2_Q, "Модель 2 (Кварталы): Эффект 'Acquired PC' на Log Total ER Visits", 'red', EVENT_WINDOW_QUARTERLY, BASE_PERIOD, time_unit="Кварталы")


    # =========================================================================
    # --- ЗАДАЧА 2: Таблицы Регрессий (Месячный формат, ОКНО +/- 12) ---
    # =========================================================================
    print("\n\n" + "="*70); print("--- ЗАДАЧА 2: Таблицы Регрессий (Месячный формат, ОКНО +/- 12) ---"); print("="*70) # Изменен заголовок

    # --- Модель 1 (Замещение, Месяцы, +/- 12) ---
    print("\n--- Таблица для Модели 1 (Замещение, Месяцы, +/- 12): Y_Regular_PC_at_HOSPITAL ~ New_PC ---")
    OUTCOME_M1_M = "Y_Regular_PC_at_HOSPITAL"
    TIME_VAR_M1_M = 'time_to_event_NEW_PC'
    df_model_M1_M = df_analysis.copy()
    df_model_M1_M['event_month'] = pd.to_datetime(df_model_M1_M['treatment_date_NEW_PC']).dt.to_period('M')
    df_model_M1_M['current_month'] = pd.to_datetime(df_model_M1_M['Date']).dt.to_period('M')
    df_model_M1_M[TIME_VAR_M1_M] = (df_model_M1_M['current_month'] - df_model_M1_M['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_model_M1_M_filtered = df_model_M1_M[df_model_M1_M[TIME_VAR_M1_M].between(-EVENT_WINDOW_MONTHLY, EVENT_WINDOW_MONTHLY)].copy()
    df_model_M1_M_filtered[TIME_VAR_M1_M] = df_model_M1_M_filtered[TIME_VAR_M1_M].astype(int)
    if OUTCOME_M1_M not in df_model_M1_M_filtered.columns:
         if 'Regular_PC_Hosp_Visits' in df_model_M1_M_filtered.columns: df_model_M1_M_filtered[OUTCOME_M1_M] = np.log1p(df_model_M1_M_filtered['Regular_PC_Hosp_Visits'])
         else: print(f"  ⚠️ Предупреждение: Колонка 'Regular_PC_Hosp_Visits' не найдена."); df_model_M1_M_filtered[OUTCOME_M1_M] = 0
    formula_M1_M = f"{OUTCOME_M1_M} ~ C({TIME_VAR_M1_M}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    print(f"Запуск модели с окном +/- {EVENT_WINDOW_MONTHLY} месяцев...")
    model_result_M1_M = run_event_study_model(formula_M1_M, df_model_M1_M_filtered, time_var=TIME_VAR_M1_M)
    # Вывод таблицы
    if model_result_M1_M:
        print("\nПолная таблица регрессии (Месяцы, +/- 12):")
        try:
             summary_table = model_result_M1_M.summary().tables[1]
             print(summary_table)
        except Exception as e_summary: print(f"  Не удалось вывести таблицу summary: {e_summary}")
    else: print("  Модель не была рассчитана.")

    # --- Модель 2 (Воронка, Месяцы, +/- 12) ---
    print("\n--- Таблица для Модели 2 (Воронка, Месяцы, +/- 12): Y_ER_Total ~ Acquired_PC ---")
    OUTCOME_M2_M = "Y_ER_Total"
    TIME_VAR_M2_M = 'time_to_event_ACQUIRED_PC'
    df_model_M2_M = df_analysis.copy()
    df_model_M2_M['event_month'] = pd.to_datetime(df_model_M2_M['treatment_date_ACQUIRED_PC']).dt.to_period('M')
    df_model_M2_M['current_month'] = pd.to_datetime(df_model_M2_M['Date']).dt.to_period('M')
    df_model_M2_M[TIME_VAR_M2_M] = (df_model_M2_M['current_month'] - df_model_M2_M['event_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)
    df_model_M2_M_filtered = df_model_M2_M[df_model_M2_M[TIME_VAR_M2_M].between(-EVENT_WINDOW_MONTHLY, EVENT_WINDOW_MONTHLY)].copy()
    df_model_M2_M_filtered[TIME_VAR_M2_M] = df_model_M2_M_filtered[TIME_VAR_M2_M].astype(int)
    if OUTCOME_M2_M not in df_model_M2_M_filtered.columns:
         if 'Total_Emergency_Visits' in df_model_M2_M_filtered.columns: df_model_M2_M_filtered[OUTCOME_M2_M] = np.log1p(df_model_M2_M_filtered['Total_Emergency_Visits'])
         else: print(f"  ⚠️ Предупреждение: Колонка 'Total_Emergency_Visits' не найдена."); df_model_M2_M_filtered[OUTCOME_M2_M] = 0
    formula_M2_M = f"{OUTCOME_M2_M} ~ C({TIME_VAR_M2_M}, Treatment(reference={BASE_PERIOD})) + C(Zip) + C(Year) + C(Month)"
    print(f"Запуск модели с окном +/- {EVENT_WINDOW_MONTHLY} месяцев...")
    model_result_M2_M = run_event_study_model(formula_M2_M, df_model_M2_M_filtered, time_var=TIME_VAR_M2_M)
    # Вывод таблицы
    if model_result_M2_M:
        print("\nПолная таблица регрессии (Месяцы, +/- 12):")
        try:
             summary_table2 = model_result_M2_M.summary().tables[1]
             print(summary_table2)
        except Exception as e_summary2: print(f"  Не удалось вывести таблицу summary: {e_summary2}")
    else: print("  Модель не была рассчитана.")


    print("\n🎉 --- Финальные Проверки завершены! ---")



In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
import io # Для перехвата вывода info()

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")

print("--- 🚀 Генерация Данных для Слайда 6 ---")

# --- 1. Проверка и Загрузка Данных ---
# Загружаем агрегированные данные (final_data)
# Используем final_data, если он есть, иначе пытаемся загрузить
if 'final_data' not in locals():
    print("Предупреждение: 'final_data' не найден. Пытаюсь загрузить 'aggregated_data.csv'")
    try:
        final_data = pd.read_csv('aggregated_data.csv', dtype={'Filtered_Patient_ZipCode': str, 'Zip': str})
        if 'Date' in final_data.columns and not pd.api.types.is_datetime64_any_dtype(final_data['Date']):
             final_data['Date'] = pd.to_datetime(final_data['Date'])
        if 'Zip' in final_data.columns and 'Filtered_Patient_ZipCode' not in final_data.columns:
            final_data = final_data.rename(columns={'Zip': 'Filtered_Patient_ZipCode'})
        print("  ✅ 'aggregated_data.csv' загружен.")
    except FileNotFoundError:
        print("❌ ОШИБКА: DataFrame 'final_data'/'aggregated_data.csv' не найден.")
        raise FileNotFoundError("Необходим 'final_data' или 'aggregated_data.csv'")

# Загружаем остальные файлы
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
    # Загружаем ИСХОДНЫЙ файл для Описательной Статистики
    print("... Загрузка исходного файла для описательной статистики ...")
    # Определяем колонки SVI, которые могут понадобиться (пример)
    svi_cols_potential = ['X..Uninsured.Adults_zip', 'MedianHHInc', 'Poverty_Rate'] # Добавьте/измените по необходимости
    cols_to_load_raw = ['Filtered_Patient_ZipCode', 'Date', 'EncounterCount',
                        'VisitType', 'FinancialClassNM', 'Department'] + svi_cols_potential
    # Убираем дубликаты и несуществующие колонки
    all_cols_in_raw = pd.read_csv("final_filtered_dataset_reordered.csv", nrows=0).columns
    cols_to_load_raw = [col for col in list(set(cols_to_load_raw)) if col in all_cols_in_raw]

    df_raw_data = pd.read_csv(
        "final_filtered_dataset_reordered.csv",
        usecols=cols_to_load_raw,
        dtype={'Filtered_Patient_ZipCode': str},
        parse_dates=['Date'],
        low_memory=False
    )
    print(f"  ✅ Исходный файл '{'final_filtered_dataset_reordered.csv'}' загружен ({len(df_raw_data)} строк).")

except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise
except ValueError as e:
    print(f"❌ ERROR: Проблема с колонками в исходном файле: {e}. Проверьте список 'svi_cols_potential'.")
    raise


# --- 2. Очистка Данных ---
# Очистка координат и клиник (как раньше)
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True)
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
print("✅ Базовые данные очищены.")

# Очистка сырых данных для статистики
df_raw_data['Filtered_Patient_ZipCode'] = df_raw_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
df_raw_data = df_raw_data[~df_raw_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_raw_data = df_raw_data.dropna(subset=['Date', 'EncounterCount', 'VisitType', 'FinancialClassNM', 'Filtered_Patient_ZipCode']) # Убираем пропуски в ключевых колонках
print("✅ Сырые данные для статистики очищены.")


# --- 3. Подготовка финального агрегированного датасета (df_analysis) ---
#    (Нужно для Задачи 1 - info())
print("\n--- Подготовка финального агрегированного датасета (df_analysis) ---")
# Идентификация 'New PC' и 'Acquired PC' клиник
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
df_new_PC_clinics = df_new_clinics_all[df_new_clinics_all.apply(is_pc_urgent, axis=1)].copy()
pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
def is_acq_pc(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
    if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_acq_pc, axis=1))].copy()

# Расчет Карт Лечения
TREATMENT_RADIUS_KM = 10; treatment_dates = {}
patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
# Упрощенный цикл для скорости, т.к. сам расчет не нужен для info()
for zip_code in patient_zips_coords_df['Filtered_Patient_ZipCode']:
     treatment_dates[zip_code] = {'treatment_date_NEW_PC': pd.NaT, 'treatment_date_ACQUIRED_PC': pd.NaT} # Просто создаем структуру
df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')

# Агрегация данных и создание Y
df_panel_info = final_data.copy()
if 'Department' not in df_panel_info.columns: df_panel_info['Department'] = 'Unknown'
else: df_panel_info['Department'] = df_panel_info['Department'].fillna('Unknown').astype(str)
df_panel_info['Is_Emergency'] = (df_panel_info['VisitType'] == 'Emergency').astype(int); df_panel_info['Is_Regular'] = (df_panel_info['VisitType'] != 'Emergency').astype(int)
df_panel_info['Year_Month'] = df_panel_info['Date'].dt.strftime('%Y-%m')
grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
df_agg_info = df_panel_info.groupby(grouping_cols).agg(
    Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_info.loc[x.index, 'Is_Emergency'] == 1].sum()),
    Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_info.loc[x.index, 'Is_Regular'] == 1].sum()),
    Regular_PC_Hosp_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[(df_panel_info.loc[x.index, 'Is_Regular'] == 1) & (df_panel_info.loc[x.index, 'Department'].isin(['Family Practice', 'Internal Medicine']))].sum())
).reset_index()
df_agg_info['Y_ER_Total'] = np.log1p(df_agg_info['Total_Emergency_Visits'])
df_agg_info['Y_Regular_PC_at_HOSPITAL'] = np.log1p(df_agg_info['Regular_PC_Hosp_Visits'])
df_analysis = df_agg_info.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
print("✅ Финальный агрегированный DataFrame (df_analysis) подготовлен.")

# --- ЗАДАЧА 1: Вывод final_data.info() ---
print("\n\n" + "="*70); print("--- ЗАДАЧА 1: Структура Финального Датасета (df_analysis.info()) ---"); print("="*70)

# Перехватываем вывод info() в строку
buffer = io.StringIO()
df_analysis.info(buf=buffer, verbose=True, show_counts=True) # verbose=True для полного вывода
info_output = buffer.getvalue()

# Сохраняем в файл
info_filename = "slide6_final_data_info.txt"
with open(info_filename, "w", encoding="utf-8") as f:
    f.write(info_output)
print(f"✅ Вывод .info() сохранен в файл: {info_filename}")
# Печатаем также на экран для просмотра
print(info_output)


# --- ЗАДАЧА 2: Визуализация Географии "Воздействия" (Карта Клиник) ---
print("\n\n" + "="*70); print("--- ЗАДАЧА 2: Карта Расположения Клиник ---"); print("="*70)

# 2.1 Создание "чистых" списков клиник
confounder_zip = '61364'; confounder_date = pd.to_datetime('2021-09-01')
date_cutoff = pd.to_datetime('2023-01-01')

# Чистый список New PC
df_new_PC_clinics_CLEAN = df_new_PC_clinics[
    ~((df_new_PC_clinics['clinic_zip'] == confounder_zip) & (df_new_PC_clinics['event_date'] == confounder_date)) &
    (df_new_PC_clinics['event_date'] < date_cutoff)
].copy()

# Чистый список Acquired PC
df_acquired_pc_CLEAN = df_acquired_pc_clinics[
    df_acquired_pc_clinics['event_date'] < date_cutoff
].copy()

print(f"Для карты отобрано:")
print(f"  - {len(df_new_PC_clinics_CLEAN)} 'чистых' New PC клиник.")
print(f"  - {len(df_acquired_pc_CLEAN)} 'чистых' Acquired PC клиник.")

# 2.2 Создание карты с matplotlib
fig_map, ax_map = plt.subplots(figsize=(10, 12)) # Соотношение сторон ближе к Иллинойсу

# Можно добавить все ZIP-коды серым фоном для контекста (если нужно)
# ax_map.scatter(zip_coords['IntPtLon'], zip_coords['IntPtLat'], color='lightgrey', s=1, alpha=0.3, label='Все ZIP Иллинойса')

# Наносим Acquired PC (Красные Квадраты)
if not df_acquired_pc_CLEAN.empty:
    ax_map.scatter(df_acquired_pc_CLEAN['clinic_lon'], df_acquired_pc_CLEAN['clinic_lat'],
                   color='red', marker='s', s=50, alpha=0.8, label=f'Acquired PC ({len(df_acquired_pc_CLEAN)})')

# Наносим New PC (Синие Кружки)
if not df_new_PC_clinics_CLEAN.empty:
    ax_map.scatter(df_new_PC_clinics_CLEAN['clinic_lon'], df_new_PC_clinics_CLEAN['clinic_lat'],
                   color='blue', marker='o', s=50, alpha=0.8, label=f'New PC/Urgent ({len(df_new_PC_clinics_CLEAN)})')

# Оформление
ax_map.set_title("Расположение 'Чистых' Клиник для Анализа Event Study", fontsize=16)
ax_map.set_xlabel("Долгота")
ax_map.set_ylabel("Широта")
ax_map.legend(title="Тип Клиники")
ax_map.grid(True)
ax_map.set_aspect('equal', adjustable='box') # Сохраняем пропорции
plt.tight_layout()

# Сохранение карты
map_filename = "slide6_clinic_map.png"
try:
    plt.savefig(map_filename, dpi=150) # DPI для лучшего качества
    print(f"✅ Карта сохранена в файл: {map_filename}")
except Exception as e:
    print(f"❌ ОШИБКА при сохранении карты: {e}")
plt.close(fig_map) # Закрываем, чтобы не отображать в ноутбуке сразу


# --- ЗАДАЧА 3: Описательная Статистика Ключевых Переменных ---
print("\n\n" + "="*70); print("--- ЗАДАЧА 3: Описательная Статистика (на сырых данных) ---"); print("="*70)

# Используем df_raw_data (загруженный и очищенный в шаге 2)
df_desc = df_raw_data.copy()

# 3.1 Количество Визитов (EncounterCount)
desc_encounter = df_desc['EncounterCount'].describe()
desc_encounter_er = df_desc[df_desc['VisitType'] == 'Emergency']['EncounterCount'].describe()
desc_encounter_reg = df_desc[df_desc['VisitType'] != 'Emergency']['EncounterCount'].describe()

# 3.2 Финансовый Класс (FinancialClassNM)
fin_class_dist = (df_desc['FinancialClassNM'].value_counts(normalize=True) * 100).round(1)
top5_fin_class = fin_class_dist.head(5)
other_fin_class_pct = fin_class_dist[5:].sum()

# 3.3 Тип Визита (VisitType)
visit_type_dist = (df_desc['VisitType'].value_counts(normalize=True) * 100).round(1)

# 3.4 Временной Охват
date_min = df_desc['Date'].min().strftime('%Y-%m-%d')
date_max = df_desc['Date'].max().strftime('%Y-%m-%d')

# 3.5 Количество Уникальных ZIP
n_unique_zip = df_desc['Filtered_Patient_ZipCode'].nunique()

# 3.6 Статистика для SVI колонки (Пример: X..Uninsured.Adults_zip)
# --- АДАПТИРУЙТЕ ИМЯ КОЛОНКИ ПРИ НЕОБХОДИМОСТИ ---
svi_col_for_desc = 'X..Uninsured.Adults_zip'
svi_desc = None
if svi_col_for_desc in df_desc.columns:
    # Берем уникальные значения SVI по ZIP-кодам
    df_svi_unique_desc = df_desc[['Filtered_Patient_ZipCode', svi_col_for_desc]].drop_duplicates('Filtered_Patient_ZipCode')
    # Преобразуем в число
    svi_numeric = pd.to_numeric(df_svi_unique_desc[svi_col_for_desc], errors='coerce')
    if svi_numeric.notna().any():
        svi_desc = svi_numeric.describe()
else:
    print(f"Предупреждение: Колонка SVI '{svi_col_for_desc}' не найдена для описательной статистики.")


# --- 3.7 Формирование Markdown Таблицы ---
md_table = f"""
## Описательная Статистика Данных (до агрегации)

**Период:** {date_min} - {date_max}
**Количество уникальных ZIP-кодов пациентов:** {n_unique_zip}
**Общее количество наблюдений (визитов):** {len(df_desc):,}

| Показатель                     | Статистика                     | Значение          |
| :----------------------------- | :----------------------------- | :---------------- |
| **Количество Визитов (Все)** | Среднее                        | {desc_encounter['mean']:,.2f}    |
|                                | Стандартное отклонение         | {desc_encounter['std']:,.2f}     |
|                                | Медиана                        | {desc_encounter['50%']:,.1f}     |
|                                | Мин                            | {desc_encounter['min']:,.0f}     |
|                                | Макс                           | {desc_encounter['max']:,.0f}     |
| **Количество Визитов (ER)** | Среднее                        | {desc_encounter_er['mean']:,.2f} |
|                                | Медиана                        | {desc_encounter_er['50%']:,.1f}  |
| **Количество Визитов (Reg)** | Среднее                        | {desc_encounter_reg['mean']:,.2f}|
|                                | Медиана                        | {desc_encounter_reg['50%']:,.1f} |
| **Тип Визита (%)** | Emergency                      | {visit_type_dist.get('Emergency', 0.0):.1f}% |
|                                | Regular (или другие)           | {100.0 - visit_type_dist.get('Emergency', 0.0):.1f}% |
| **Финансовый Класс (Топ-5 %)** | {top5_fin_class.index[0]} | {top5_fin_class.iloc[0]:.1f}%    |
|                                | {top5_fin_class.index[1]} | {top5_fin_class.iloc[1]:.1f}%    |
|                                | {top5_fin_class.index[2]} | {top5_fin_class.iloc[2]:.1f}%    |
|                                | {top5_fin_class.index[3]} | {top5_fin_class.iloc[3]:.1f}%    |
|                                | {top5_fin_class.index[4]} | {top5_fin_class.iloc[4]:.1f}%    |
|                                | Остальные                      | {other_fin_class_pct:.1f}%       |
"""

if svi_desc is not None:
    md_table += f"""
| **{svi_col_for_desc} (по ZIP)** | Среднее                    | {svi_desc['mean']:,.2f}           |
|                                | Стандартное отклонение     | {svi_desc['std']:,.2f}            |
|                                | Медиана                    | {svi_desc['50%']:,.2f}            |
|                                | Мин                        | {svi_desc['min']:,.2f}            |
|                                | Макс                       | {svi_desc['max']:,.2f}            |
"""

# Сохраняем Markdown в файл
desc_stat_filename = "slide6_descriptive_stats.md"
with open(desc_stat_filename, "w", encoding="utf-8") as f:
    f.write(md_table)
print(f"\n✅ Таблица описательной статистики сохранена в файл: {desc_stat_filename}")
# Печатаем таблицу для просмотра
print(md_table)


print("\n🎉 --- Генерация Данных для Слайда 6 завершена! ---")

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
import io # Для перехвата вывода info()
from tqdm import tqdm # Import tqdm

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")

print("--- 🚀 Генерация Данных для Слайда 6 ---")

# --- 1. Проверка и Загрузка Данных ---
# Загружаем агрегированные данные (final_data)
# Используем final_data, если он есть, иначе пытаемся загрузить
# **ВАЖНО**: Для Task 3 нам НУЖНЫ сырые данные, поэтому загружаем их в любом случае
raw_data_loaded = False
try:
    print("... Загрузка исходного файла для описательной статистики и SVI ...")
    # Определяем колонки SVI, которые могут понадобиться (пример)
    svi_cols_potential = ['X..Uninsured.Adults_zip'] # Фокусируемся на этой для примера
    cols_to_load_raw = ['Filtered_Patient_ZipCode', 'Date', 'EncounterCount',
                        'VisitType', 'FinancialClassNM', 'Department'] + svi_cols_potential
    # Убираем дубликаты и несуществующие колонки
    all_cols_in_raw = pd.read_csv("final_filtered_dataset_reordered.csv", nrows=0).columns
    cols_to_load_raw = [col for col in list(set(cols_to_load_raw)) if col in all_cols_in_raw]

    df_raw_data = pd.read_csv(
        "final_filtered_dataset_reordered.csv",
        usecols=cols_to_load_raw,
        dtype={'Filtered_Patient_ZipCode': str},
        parse_dates=['Date'],
        low_memory=False
    )
    raw_data_loaded = True
    print(f"  ✅ Исходный файл '{'final_filtered_dataset_reordered.csv'}' загружен ({len(df_raw_data)} строк).")

except FileNotFoundError:
    print(f"❌ ERROR: Cannot find file {'final_filtered_dataset_reordered.csv'}. Task 3 cannot be completed.")
    df_raw_data = None # Указываем, что данные не загружены
except ValueError as e:
    print(f"❌ ERROR: Проблема с колонками в исходном файле: {e}. Проверьте список 'svi_cols_potential'. Task 3 may fail.")
    df_raw_data = None
except Exception as e:
    print(f"❌ UNEXPECTED ERROR loading raw data: {e}. Task 3 may fail.")
    df_raw_data = None


# Загружаем final_data (агрегированные), если он не в памяти
if 'final_data' not in locals():
    print("Предупреждение: 'final_data' не найден. Пытаюсь загрузить 'aggregated_data.csv'")
    try:
        final_data = pd.read_csv('aggregated_data.csv', dtype={'Filtered_Patient_ZipCode': str, 'Zip': str})
        if 'Date' in final_data.columns and not pd.api.types.is_datetime64_any_dtype(final_data['Date']):
             final_data['Date'] = pd.to_datetime(final_data['Date'])
        if 'Zip' in final_data.columns and 'Filtered_Patient_ZipCode' not in final_data.columns:
            final_data = final_data.rename(columns={'Zip': 'Filtered_Patient_ZipCode'})
        print("  ✅ 'aggregated_data.csv' загружен.")
    except FileNotFoundError:
        print("❌ ОШИБКА: DataFrame 'final_data'/'aggregated_data.csv' не найден.")
        # Если не можем загрузить ни final_data, ни aggregated, останавливаемся
        if not raw_data_loaded:
             raise FileNotFoundError("Необходим 'final_data' или 'aggregated_data.csv' или сырые данные.")
        else:
             print("Продолжаем с сырыми данными для задач 2 и 3.")
             # Создаем пустой final_data, чтобы скрипт не упал дальше, Task 1 будет пропущен
             final_data = pd.DataFrame()


# Загружаем остальные файлы
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")

except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# Очистка координат и клиник
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True, how='left') # Use left join here
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type']) # Drop clinics without coords AFTER join
print("✅ Базовые данные клиник и координат очищены.")

# Очистка final_data (если он загружен)
if not final_data.empty:
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    print("✅ final_data очищен.")

# Очистка сырых данных для статистики (если они загружены)
if df_raw_data is not None:
    df_raw_data['Filtered_Patient_ZipCode'] = df_raw_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_raw_data = df_raw_data[~df_raw_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    key_cols_raw = ['Date', 'EncounterCount', 'VisitType', 'FinancialClassNM', 'Filtered_Patient_ZipCode']
    # Добавляем Department только если он есть
    if 'Department' in df_raw_data.columns:
        key_cols_raw.append('Department')
    df_raw_data = df_raw_data.dropna(subset=key_cols_raw) # Убираем пропуски в ключевых колонках
    print("✅ Сырые данные для статистики очищены.")


# --- 3. Подготовка финального агрегированного датасета (df_analysis) для info() ---
#    (Только если final_data существует)
df_analysis = pd.DataFrame() # Инициализируем
if not final_data.empty:
    print("\n--- Подготовка финального агрегированного датасета (df_analysis) для info() ---")
    # Идентификация 'New PC' и 'Acquired PC' клиник
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'New') & (df_clinics_info_full.apply(is_pc_urgent, axis=1))].copy()
    pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_acq_pc(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_acq_pc, axis=1))].copy()

    # Расчет Карт Лечения (НУЖНО РАССЧИТАТЬ ПРАВИЛЬНО для info())
    print("... Расчет карт лечения для Task 1 ...")
    TREATMENT_RADIUS_KM = 10; treatment_dates = {}
    patient_zips_for_map = final_data['Filtered_Patient_ZipCode'].unique() # Берем ZIPы из final_data
    patient_zips_coords_df_map = pd.DataFrame(patient_zips_for_map, columns=['Filtered_Patient_ZipCode'])
    patient_zips_coords_df_map = patient_zips_coords_df_map.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')

    for _, patient_row in tqdm(patient_zips_coords_df_map.iterrows(), total=len(patient_zips_coords_df_map), desc="Расчет Дат Лечения для info()"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows(): # Используем только PC
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        for _, clinic_row in df_acquired_pc_clinics.iterrows(): # Используем только PC
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
        if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
        treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
    df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
    print("... Карты лечения для Task 1 рассчитаны ...")


    # Агрегация данных и создание Y (используем существующий final_data)
    df_panel_info = final_data.copy()
    # Убедимся что Department есть
    if 'Department' not in df_panel_info.columns: df_panel_info['Department'] = 'Unknown'
    else: df_panel_info['Department'] = df_panel_info['Department'].fillna('Unknown').astype(str)

    df_panel_info['Is_Emergency'] = (df_panel_info['VisitType'] == 'Emergency').astype(int); df_panel_info['Is_Regular'] = (df_panel_info['VisitType'] != 'Emergency').astype(int)
    # Используем существующие Year/Month если есть, иначе создаем Year_Month
    if 'Year_Month' not in df_panel_info.columns:
         if 'Date' in df_panel_info.columns and pd.api.types.is_datetime64_any_dtype(df_panel_info['Date']):
             df_panel_info['Year_Month'] = df_panel_info['Date'].dt.strftime('%Y-%m')
             if 'Year' not in df_panel_info.columns: df_panel_info['Year'] = df_panel_info['Date'].dt.year
             if 'Month' not in df_panel_info.columns: df_panel_info['Month'] = df_panel_info['Date'].dt.month
         else:
             raise ValueError("Не найдены колонки 'Year_Month' или 'Date' в final_data.")

    # Проверяем наличие нужных колонок перед группировкой
    grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
    if not all(col in df_panel_info.columns for col in grouping_cols + ['EncounterCount', 'Department', 'Is_Emergency', 'Is_Regular']):
         raise ValueError(f"Не все колонки для агрегации ({grouping_cols}) или расчета Y найдены в final_data.")

    df_agg_info = df_panel_info.groupby(grouping_cols).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_info.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_info.loc[x.index, 'Is_Regular'] == 1].sum()),
        # Используем .loc для безопасного доступа по индексу x
        Regular_PC_Hosp_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: df_panel_info.loc[x.index][
            (df_panel_info.loc[x.index, 'Is_Regular'] == 1) &
            (df_panel_info.loc[x.index, 'Department'].isin(['Family Practice', 'Internal Medicine']))
        ]['EncounterCount'].sum())
    ).reset_index()

    df_agg_info['Y_ER_Total'] = np.log1p(df_agg_info['Total_Emergency_Visits'])
    df_agg_info['Y_Regular_PC_at_HOSPITAL'] = np.log1p(df_agg_info['Regular_PC_Hosp_Visits'])
    
    # --- ИСПРАВЛЕНИЕ: Используем merge, а не join для df_treatment_dates ---
    df_analysis = df_agg_info.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    
    # Создаем Date после merge
    df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
    df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    print("✅ Финальный агрегированный DataFrame (df_analysis) подготовлен.")


# --- ЗАДАЧА 1: Вывод df_analysis.info() ---
print("\n\n" + "="*70); print("--- TASK 1: Structure of Final Aggregated Dataset (df_analysis.info()) ---"); print("="*70)

info_filename = "slide6_final_data_info.txt"
if not df_analysis.empty:
    # Перехватываем вывод info() в строку
    buffer = io.StringIO()
    df_analysis.info(buf=buffer, verbose=True, show_counts=True) # verbose=True для полного вывода
    info_output = buffer.getvalue()

    # Сохраняем в файл
    with open(info_filename, "w", encoding="utf-8") as f:
        f.write(info_output)
    print(f"✅ .info() output saved to file: {info_filename}")
    # Печатаем также на экран для просмотра
    print("\n" + info_output)
    # --- Проверка Non-Null counts ---
    print("\nChecking Non-Null Counts for Treatment Dates:")
    print(df_analysis[['treatment_date_NEW_PC', 'treatment_date_ACQUIRED_PC']].notna().sum())

else:
    print("⚠️ df_analysis is empty or was not created. Skipping Task 1.")
    # Создаем пустой файл, если нужно
    with open(info_filename, "w", encoding="utf-8") as f:
        f.write("df_analysis was empty or could not be created.")


# --- ЗАДАЧА 2: Визуализация Географии "Воздействия" (Улучшенная Карта) ---
print("\n\n" + "="*70); print("--- TASK 2: Enhanced Map of Clinic Locations ---"); print("="*70)

# 2.1 Создание "чистых" списков клиник
confounder_zip = '61364'; confounder_date = pd.to_datetime('2021-09-01')
date_cutoff = pd.to_datetime('2023-01-01')
df_new_PC_clinics_CLEAN = df_new_PC_clinics[ (~((df_new_PC_clinics['clinic_zip'] == confounder_zip) & (df_new_PC_clinics['event_date'] == confounder_date))) & (df_new_PC_clinics['event_date'] < date_cutoff)].copy()
df_acquired_pc_CLEAN = df_acquired_pc_clinics[ df_acquired_pc_clinics['event_date'] < date_cutoff ].copy()

# 2.2 Определение координат госпиталей OSF (примерный список)
osf_hospital_zips = ['61637', '61108', '61801', '61701', '62002', '61401', '61350', '61764', '61443', '61462']
df_hospitals = zip_coords.loc[zip_coords.index.intersection(osf_hospital_zips)].reset_index() # Используем .loc и intersection

print(f"Plotting Map with:")
print(f"  - {len(df_new_PC_clinics_CLEAN)} 'Clean' New PC clinics.")
print(f"  - {len(df_acquired_pc_CLEAN)} 'Clean' Acquired PC clinics.")
print(f"  - {len(df_hospitals)} approximate OSF Hospital locations.")

# 2.3 Создание карты с matplotlib
fig_map, ax_map = plt.subplots(figsize=(10, 12))

# Фон: Все ZIP Иллинойса
ax_map.scatter(zip_coords['IntPtLon'], zip_coords['IntPtLat'], color='lightgrey', s=2, alpha=0.4, label='All IL ZIP Codes (approx)')

# Госпитали (Зеленые Звезды)
if not df_hospitals.empty:
    ax_map.scatter(df_hospitals['IntPtLon'], df_hospitals['IntPtLat'],
                   color='green', marker='*', s=150, alpha=0.9, label=f'OSF Hospitals (approx, {len(df_hospitals)})', edgecolors='black')

# Acquired PC (Красные Квадраты)
if not df_acquired_pc_CLEAN.empty:
    ax_map.scatter(df_acquired_pc_CLEAN['clinic_lon'], df_acquired_pc_CLEAN['clinic_lat'],
                   color='red', marker='s', s=60, alpha=0.8, label=f'Acquired PC Clinics ({len(df_acquired_pc_CLEAN)})')

# New PC (Синие Кружки)
if not df_new_PC_clinics_CLEAN.empty:
    ax_map.scatter(df_new_PC_clinics_CLEAN['clinic_lon'], df_new_PC_clinics_CLEAN['clinic_lat'],
                   color='blue', marker='o', s=60, alpha=0.8, label=f'New PC/Urgent Clinics ({len(df_new_PC_clinics_CLEAN)})')

# Оформление (на английском)
ax_map.set_title("Geographical Distribution of OSF HealthCare Facilities in Illinois", fontsize=16)
ax_map.set_xlabel("Longitude")
ax_map.set_ylabel("Latitude")
ax_map.legend(title="Facility Type")
ax_map.grid(True, linestyle='--', alpha=0.5)
ax_map.set_aspect('equal', adjustable='box') # Сохраняем пропорции

# Установка пределов для Иллинойса (примерно)
ax_map.set_xlim(-91.7, -87.4)
ax_map.set_ylim(36.8, 42.7)

plt.tight_layout()

# Сохранение карты
map_filename = "slide6_clinic_map_enhanced.png"
try:
    plt.savefig(map_filename, dpi=200) # Увеличили DPI
    print(f"✅ Enhanced map saved to file: {map_filename}")
except Exception as e:
    print(f"❌ ERROR saving map: {e}")
plt.close(fig_map) # Закрываем, чтобы не отображать в ноутбуке сразу


# --- ЗАДАЧА 3: Описательная Статистика Ключевых Переменных ---
print("\n\n" + "="*70); print("--- TASK 3: Descriptive Statistics (Raw Visit Data) ---"); print("="*70)

desc_stat_filename = "slide6_descriptive_stats.md"
md_table = "## Descriptive Statistics (Visit-Level Data)\n\n" # Начало таблицы

if df_raw_data is not None:
    df_desc = df_raw_data.copy()

    # 3.1 Overall Characteristics
    date_min = df_desc['Date'].min().strftime('%b %Y') # Формат Мес Год
    date_max = df_desc['Date'].max().strftime('%b %Y')
    n_visits = len(df_desc)
    n_unique_zip = df_desc['Filtered_Patient_ZipCode'].nunique()

    md_table += f"**Time Period:** {date_min} - {date_max}\n"
    md_table += f"**Total Patient Visits:** {n_visits:,}\n"
    md_table += f"**Unique Patient ZIP Codes:** {n_unique_zip:,}\n\n"

    # 3.2 Encounter Count
    desc_encounter = df_desc['EncounterCount'].describe()

    # 3.3 Visit Type Distribution
    visit_type_dist = (df_desc['VisitType'].value_counts(normalize=True) * 100)
    emergency_pct = visit_type_dist.get('Emergency', 0.0)
    regular_pct = 100.0 - emergency_pct # Все остальное считаем Regular

    # 3.4 Top 5 Financial Classes
    fin_class_dist = (df_desc['FinancialClassNM'].value_counts(normalize=True) * 100)
    top5_fin_class = fin_class_dist.head(5)
    other_fin_class_pct = fin_class_dist[5:].sum()
    fin_class_lines = ""
    for idx, val in top5_fin_class.items():
        fin_class_lines += f"|   - {idx:<25} | {val:>6.1f}% |\n"
    fin_class_lines += f"|   - {'Other':<25} | {other_fin_class_pct:>6.1f}% |\n"


    # --- Формирование Markdown Таблицы ---
    md_table += """
| Variable                      | Statistic        | Value         |
| :---------------------------- | :--------------- | :------------ |
| **Visit Count (Overall)** | Mean             | {:,.2f}       |
|                               | Std Dev          | {:,.2f}       |
|                               | Median           | {:,.1f}       |
|                               | Min              | {:,.0f}       |
|                               | Max              | {:,.0f}       |
| **Visit Type Distribution** | Emergency        | {:.1f}%       |
|                               | Regular          | {:.1f}%       |
""".format(
        desc_encounter['mean'], desc_encounter['std'], desc_encounter['50%'],
        desc_encounter['min'], desc_encounter['max'],
        emergency_pct, regular_pct
    )

    md_table += "| **Top 5 Financial Classes** |                  |               |\n"
    md_table += fin_class_lines

    # Сохраняем Markdown в файл
    with open(desc_stat_filename, "w", encoding="utf-8") as f:
        f.write(md_table)
    print(f"✅ Descriptive statistics table saved to file: {desc_stat_filename}")
    # Печатаем таблицу для просмотра
    print("\n" + md_table)

else:
    print("⚠️ Raw data (df_raw_data) not available. Skipping Task 3.")
    # Создаем пустой файл
    with open(desc_stat_filename, "w", encoding="utf-8") as f:
        f.write("Raw data was not available to generate descriptive statistics.")


print("\n🎉 --- Генерация Данных для Слайда 6 завершена! ---")

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
import io # Для перехвата вывода info()
from tqdm import tqdm # Import tqdm

# --- Установка/Проверка geopandas и contextily ---
try:
    import geopandas as gpd
    import contextily as ctx
    print("✅ 'geopandas' и 'contextily' уже установлены.")
except ImportError:
    print("Устанавливаю 'geopandas' и 'contextily' для построения карт...")
    import sys
    !{sys.executable} -m pip install geopandas contextily
    try:
        import geopandas as gpd
        import contextily as ctx
        print("✅ 'geopandas' и 'contextily' успешно установлены.")
    except ImportError:
        print("❌ ОШИБКА: Не удалось установить 'geopandas' или 'contextily'. Карта (Задача 2) не будет построена.")
        gpd = None
        ctx = None

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")

print("--- 🚀 Генерация Данных для Слайда 6 ---")

# --- 1. Проверка и Загрузка Данных ---
# ... (Этот блок остается без изменений) ...
# Загружаем агрегированные данные (final_data)
# Используем final_data, если он есть, иначе пытаемся загрузить
# **ВАЖНО**: Для Task 3 нам НУЖНЫ сырые данные, поэтому загружаем их в любом случае
raw_data_loaded = False
try:
    print("... Загрузка исходного файла для описательной статистики и SVI ...")
    # Определяем колонки SVI, которые могут понадобиться (пример)
    svi_cols_potential = ['X..Uninsured.Adults_zip'] # Фокусируемся на этой для примера
    cols_to_load_raw = ['Filtered_Patient_ZipCode', 'Date', 'EncounterCount',
                        'VisitType', 'FinancialClassNM', 'Department'] + svi_cols_potential
    # Убираем дубликаты и несуществующие колонки
    all_cols_in_raw = pd.read_csv("final_filtered_dataset_reordered.csv", nrows=0).columns
    cols_to_load_raw = [col for col in list(set(cols_to_load_raw)) if col in all_cols_in_raw]

    df_raw_data = pd.read_csv(
        "final_filtered_dataset_reordered.csv",
        usecols=cols_to_load_raw,
        dtype={'Filtered_Patient_ZipCode': str},
        parse_dates=['Date'],
        low_memory=False
    )
    raw_data_loaded = True
    print(f"  ✅ Исходный файл '{'final_filtered_dataset_reordered.csv'}' загружен ({len(df_raw_data)} строк).")

except FileNotFoundError:
    print(f"❌ ERROR: Cannot find file {'final_filtered_dataset_reordered.csv'}. Task 3 cannot be completed.")
    df_raw_data = None # Указываем, что данные не загружены
except ValueError as e:
    print(f"❌ ERROR: Проблема с колонками в исходном файле: {e}. Проверьте список 'svi_cols_potential'. Task 3 may fail.")
    df_raw_data = None
except Exception as e:
    print(f"❌ UNEXPECTED ERROR loading raw data: {e}. Task 3 may fail.")
    df_raw_data = None


# Загружаем final_data (агрегированные), если он не в памяти
if 'final_data' not in locals():
    print("Предупреждение: 'final_data' не найден. Пытаюсь загрузить 'aggregated_data.csv'")
    try:
        final_data = pd.read_csv('aggregated_data.csv', dtype={'Filtered_Patient_ZipCode': str, 'Zip': str})
        if 'Date' in final_data.columns and not pd.api.types.is_datetime64_any_dtype(final_data['Date']):
             final_data['Date'] = pd.to_datetime(final_data['Date'])
        if 'Zip' in final_data.columns and 'Filtered_Patient_ZipCode' not in final_data.columns:
            final_data = final_data.rename(columns={'Zip': 'Filtered_Patient_ZipCode'})
        print("  ✅ 'aggregated_data.csv' загружен.")
    except FileNotFoundError:
        print("❌ ОШИБКА: DataFrame 'final_data'/'aggregated_data.csv' не найден.")
        # Если не можем загрузить ни final_data, ни aggregated, останавливаемся
        if not raw_data_loaded:
             raise FileNotFoundError("Необходим 'final_data' или 'aggregated_data.csv' или сырые данные.")
        else:
             print("Продолжаем с сырыми данными для задач 2 и 3.")
             # Создаем пустой final_data, чтобы скрипт не упал дальше, Task 1 будет пропущен
             final_data = pd.DataFrame()


# Загружаем остальные файлы
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")

except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (код очистки без изменений) ...
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True, how='left') # Use left join here
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type']) # Drop clinics without coords AFTER join
print("✅ Базовые данные клиник и координат очищены.")
if not final_data.empty:
    final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
    final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    invalid_zips = ["99999", "Other", "0", "00000"]
    final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    print("✅ final_data очищен.")
if df_raw_data is not None:
    df_raw_data['Filtered_Patient_ZipCode'] = df_raw_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
    df_raw_data = df_raw_data[~df_raw_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
    key_cols_raw = ['Date', 'EncounterCount', 'VisitType', 'FinancialClassNM', 'Filtered_Patient_ZipCode']
    if 'Department' in df_raw_data.columns: key_cols_raw.append('Department')
    df_raw_data = df_raw_data.dropna(subset=key_cols_raw)
    print("✅ Сырые данные для статистики очищены.")


# --- 3. Подготовка финального агрегированного датасета (df_analysis) для info() ---
# ... (код без изменений) ...
df_analysis = pd.DataFrame() # Инициализируем
if not final_data.empty:
    print("\n--- Подготовка финального агрегированного датасета (df_analysis) для info() ---")
    # Идентификация 'New PC' и 'Acquired PC' клиник
    pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
    def is_pc_urgent(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
    df_new_PC_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'New') & (df_clinics_info_full.apply(is_pc_urgent, axis=1))].copy()
    pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
    def is_acq_pc(row):
        facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
        if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
        if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
        return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
    df_acquired_pc_clinics = df_clinics_info_full[(df_clinics_info_full['clinic_type'] == 'Acquired') & (df_clinics_info_full.apply(is_acq_pc, axis=1))].copy()

    # Расчет Карт Лечения (НУЖНО РАССЧИТАТЬ ПРАВИЛЬНО для info())
    print("... Расчет карт лечения для Task 1 ...")
    TREATMENT_RADIUS_KM = 10; treatment_dates = {}
    patient_zips_for_map = final_data['Filtered_Patient_ZipCode'].unique() # Берем ZIPы из final_data
    patient_zips_coords_df_map = pd.DataFrame(patient_zips_for_map, columns=['Filtered_Patient_ZipCode'])
    patient_zips_coords_df_map = patient_zips_coords_df_map.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
    
    for _, patient_row in tqdm(patient_zips_coords_df_map.iterrows(), total=len(patient_zips_coords_df_map), desc="Расчет Дат Лечения для info()"):
        patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
        earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
        for _, clinic_row in df_new_PC_clinics.iterrows(): # Используем только PC
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
        if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
        for _, clinic_row in df_acquired_pc_clinics.iterrows(): # Используем только PC
            clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
            if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
        if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
        treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
    df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
    print("... Карты лечения для Task 1 рассчитаны ...")
    
    # Агрегация данных и создание Y (используем существующий final_data)
    df_panel_info = final_data.copy()
    if 'Department' not in df_panel_info.columns: df_panel_info['Department'] = 'Unknown'
    else: df_panel_info['Department'] = df_panel_info['Department'].fillna('Unknown').astype(str)
    df_panel_info['Is_Emergency'] = (df_panel_info['VisitType'] == 'Emergency').astype(int); df_panel_info['Is_Regular'] = (df_panel_info['VisitType'] != 'Emergency').astype(int)
    if 'Year_Month' not in df_panel_info.columns:
         if 'Date' in df_panel_info.columns and pd.api.types.is_datetime64_any_dtype(df_panel_info['Date']):
             df_panel_info['Year_Month'] = df_panel_info['Date'].dt.strftime('%Y-%m')
             if 'Year' not in df_panel_info.columns: df_panel_info['Year'] = df_panel_info['Date'].dt.year
             if 'Month' not in df_panel_info.columns: df_panel_info['Month'] = df_panel_info['Date'].dt.month
         else: raise ValueError("Не найдены колонки 'Year_Month' или 'Date' в final_data.")
    grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
    if not all(col in df_panel_info.columns for col in grouping_cols + ['EncounterCount', 'Department', 'Is_Emergency', 'Is_Regular']):
         raise ValueError(f"Не все колонки для агрегации ({grouping_cols}) или расчета Y найдены в final_data.")
    df_agg_info = df_panel_info.groupby(grouping_cols).agg(
        Total_Emergency_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_info.loc[x.index, 'Is_Emergency'] == 1].sum()),
        Total_Regular_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: x[df_panel_info.loc[x.index, 'Is_Regular'] == 1].sum()),
        Regular_PC_Hosp_Visits = pd.NamedAgg(column='EncounterCount', aggfunc=lambda x: df_panel_info.loc[x.index][(df_panel_info.loc[x.index, 'Is_Regular'] == 1) & (df_panel_info.loc[x.index, 'Department'].isin(['Family Practice', 'Internal Medicine']))]['EncounterCount'].sum())
    ).reset_index()
    df_agg_info['Y_ER_Total'] = np.log1p(df_agg_info['Total_Emergency_Visits']); df_agg_info['Y_Regular_PC_at_HOSPITAL'] = np.log1p(df_agg_info['Regular_PC_Hosp_Visits'])
    df_analysis = df_agg_info.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
    df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
    df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})
    print("✅ Финальный агрегированный DataFrame (df_analysis) подготовлен.")


# --- ЗАДАЧА 1: Вывод df_analysis.info() ---
# ... (код без изменений) ...
print("\n\n" + "="*70); print("--- TASK 1: Structure of Final Aggregated Dataset (df_analysis.info()) ---"); print("="*70)
info_filename = "slide6_final_data_info.txt"
if not df_analysis.empty:
    buffer = io.StringIO(); df_analysis.info(buf=buffer, verbose=True, show_counts=True)
    info_output = buffer.getvalue()
    with open(info_filename, "w", encoding="utf-8") as f: f.write(info_output)
    print(f"✅ .info() output saved to file: {info_filename}")
    print("\n" + info_output)
    print("\nChecking Non-Null Counts for Treatment Dates:")
    print(df_analysis[['treatment_date_NEW_PC', 'treatment_date_ACQUIRED_PC']].notna().sum())
else:
    print("⚠️ df_analysis is empty or was not created. Skipping Task 1.")
    with open(info_filename, "w", encoding="utf-8") as f: f.write("df_analysis was empty or could not be created.")


# --- ЗАДАЧА 2: Визуализация Географии "Воздействия" (Улучшенная Карта) ---
print("\n\n" + "="*70); print("--- TASK 2: Enhanced Map of Clinic Locations ---"); print("="*70)

# --- ИЗМЕНЕНИЕ: Используем Geopandas для границ штата ---
if gpd is not None and ctx is not None:
    # 2.1 Создание "чистых" списков клиник
    confounder_zip = '61364'; confounder_date = pd.to_datetime('2021-09-01')
    date_cutoff = pd.to_datetime('2023-01-01')
    df_new_PC_clinics_CLEAN = df_new_PC_clinics[ (~((df_new_PC_clinics['clinic_zip'] == confounder_zip) & (df_new_PC_clinics['event_date'] == confounder_date))) & (df_new_PC_clinics['event_date'] < date_cutoff)].copy()
    df_acquired_pc_CLEAN = df_acquired_pc_clinics[ df_acquired_pc_clinics['event_date'] < date_cutoff ].copy()

    # 2.2 Определение координат госпиталей OSF
    osf_hospital_zips = ['61637', '61108', '61801', '61701', '62002', '61401', '61350', '61764', '61443', '61462']
    df_hospitals = zip_coords.loc[zip_coords.index.intersection(osf_hospital_zips)].reset_index()

    print(f"Plotting Map with:")
    print(f"  - {len(df_new_PC_clinics_CLEAN)} 'Clean' New PC clinics.")
    print(f"  - {len(df_acquired_pc_CLEAN)} 'Clean' Acquired PC clinics.")
    print(f"  - {len(df_hospitals)} approximate OSF Hospital locations.")

    # 2.3 Создание GeoDataFrames
    # Конвертируем все DataFrame'ы в GeoDataFrame'ы
    gdf_zip_coords = gpd.GeoDataFrame(
        zip_coords, geometry=gpd.points_from_xy(zip_coords.IntPtLon, zip_coords.IntPtLat), crs="EPSG:4326"
    )
    gdf_hospitals = gpd.GeoDataFrame(
        df_hospitals, geometry=gpd.points_from_xy(df_hospitals.IntPtLon, df_hospitals.IntPtLat), crs="EPSG:4326"
    )
    gdf_acquired_pc = gpd.GeoDataFrame(
        df_acquired_pc_CLEAN, geometry=gpd.points_from_xy(df_acquired_pc_CLEAN.clinic_lon, df_acquired_pc_CLEAN.clinic_lat), crs="EPSG:4326"
    )
    gdf_new_pc = gpd.GeoDataFrame(
        df_new_PC_clinics_CLEAN, geometry=gpd.points_from_xy(df_new_PC_clinics_CLEAN.clinic_lon, df_new_PC_clinics_CLEAN.clinic_lat), crs="EPSG:4326"
    )

    # 2.4 Загрузка границ штатов
    try:
        # URL к GeoJSON файлу границ штатов США
        states_geojson_url = "https://raw.githubusercontent.com/python-visualization/folium/main/examples/data/us-states.json"
        all_states = gpd.read_file(states_geojson_url)
        # Выбираем Иллинойс
        illinois = all_states[all_states['name'] == 'Illinois']
        
        # Переводим все в проекцию Web Mercator (EPSG:3857) для корректного отображения с basemap
        illinois = illinois.to_crs(epsg=3857)
        gdf_zip_coords = gdf_zip_coords.to_crs(epsg=3857)
        gdf_hospitals = gdf_hospitals.to_crs(epsg=3857)
        gdf_acquired_pc = gdf_acquired_pc.to_crs(epsg=3857)
        gdf_new_pc = gdf_new_pc.to_crs(epsg=3857)

        # 2.5 Создание карты с geopandas и contextily
        fig_map, ax_map = plt.subplots(figsize=(10, 12))

        # 1. Рисуем границу Иллинойса
        illinois.plot(ax=ax_map, color='whitesmoke', edgecolor='black', linewidth=1.5, label='Illinois Border', alpha=0.8)
        
        # 2. Фон: Все ZIP Иллинойса (можно убрать, если будет слишком грязно)
        # gdf_zip_coords.plot(ax=ax_map, color='lightgrey', markersize=1, alpha=0.3, label='All IL ZIP Codes')

        # 3. Госпитали (Зеленые Звезды)
        if not gdf_hospitals.empty:
            gdf_hospitals.plot(ax=ax_map, color='green', marker='*', markersize=250, alpha=0.9, label=f'OSF Hospitals (approx, {len(gdf_hospitals)})', edgecolors='black')

        # 4. Acquired PC (Красные Квадраты)
        if not gdf_acquired_pc.empty:
            gdf_acquired_pc.plot(ax=ax_map, color='red', marker='s', markersize=60, alpha=0.8, label=f'Acquired PC Clinics ({len(gdf_acquired_pc)})')

        # 5. New PC (Синие Кружки)
        if not gdf_new_pc.empty:
            gdf_new_pc.plot(ax=ax_map, color='blue', marker='o', markersize=60, alpha=0.8, label=f'New PC/Urgent Clinics ({len(gdf_new_pc)})')

        # 6. Добавляем фоновую карту (basemap)
        try:
            ctx.add_basemap(ax_map, crs=illinois.crs.to_string(), source=ctx.providers.CartoDB.PositronNoLabels, alpha=0.6)
        except Exception as e_ctx:
            print(f"Предупреждение: не удалось загрузить фоновую карту contextily: {e_ctx}. График будет без фона.")

        # Оформление (на английском)
        ax_map.set_title("Geographical Distribution of OSF HealthCare Facilities in Illinois", fontsize=16)
        ax_map.set_xlabel("Longitude")
        ax_map.set_ylabel("Latitude")
        ax_map.legend(title="Facility Type", loc='upper right')
        ax_map.grid(True, linestyle='--', alpha=0.5)
        # Убираем оси с lat/lon, т.к. есть карта
        ax_map.set_xticks([])
        ax_map.set_yticks([])

        plt.tight_layout()

        # Сохранение карты
        map_filename = "slide6_clinic_map_enhanced.png"
        plt.savefig(map_filename, dpi=200) # Увеличили DPI
        print(f"✅ Enhanced map saved to file: {map_filename}")
        plt.close(fig_map) # Закрываем

    except Exception as e_map:
        print(f"❌ ОШИБКА при создании карты geopandas: {e_map}")
        print("   Возможно, не удалось скачать файл границ штатов. Пропускаю Task 2.")

else:
    print("⚠️ 'geopandas' или 'contextily' не найдены. Пропускаю Task 2 (Карта).")
    print("   Пожалуйста, установите их (pip install geopandas contextily) и перезапустите.")


# --- ЗАДАЧА 3: Описательная Статистика Ключевых Переменных ---
# ... (код без изменений) ...
print("\n\n" + "="*70); print("--- TASK 3: Descriptive Statistics (Raw Visit Data) ---"); print("="*70)
desc_stat_filename = "slide6_descriptive_stats.md"
md_table = "## Descriptive Statistics (Visit-Level Data)\n\n" # Начало таблицы
if df_raw_data is not None:
    df_desc = df_raw_data.copy()
    # 3.1 Overall Characteristics
    date_min = df_desc['Date'].min().strftime('%b %Y'); date_max = df_desc['Date'].max().strftime('%b %Y')
    n_visits = len(df_desc); n_unique_zip = df_desc['Filtered_Patient_ZipCode'].nunique()
    md_table += f"**Time Period:** {date_min} - {date_max}\n"
    md_table += f"**Total Patient Visits:** {n_visits:,}\n"
    md_table += f"**Unique Patient ZIP Codes:** {n_unique_zip:,}\n\n"
    # 3.2 Encounter Count
    desc_encounter = df_desc['EncounterCount'].describe()
    # 3.3 Visit Type Distribution
    visit_type_dist = (df_desc['VisitType'].value_counts(normalize=True) * 100)
    emergency_pct = visit_type_dist.get('Emergency', 0.0); regular_pct = 100.0 - emergency_pct
    # 3.4 Top 5 Financial Classes
    fin_class_dist = (df_desc['FinancialClassNM'].value_counts(normalize=True) * 100)
    top5_fin_class = fin_class_dist.head(5); other_fin_class_pct = fin_class_dist[5:].sum()
    fin_class_lines = ""
    for idx, val in top5_fin_class.items(): fin_class_lines += f"|   - {idx:<25} | {val:>6.1f}% |\n"
    fin_class_lines += f"|   - {'Other':<25} | {other_fin_class_pct:>6.1f}% |\n"
    # --- Формирование Markdown Таблицы ---
    md_table += """
| Variable                      | Statistic        | Value         |
| :---------------------------- | :--------------- | :------------ |
| **Visit Count (Overall)** | Mean             | {:,.2f}       |
|                               | Std Dev          | {:,.2f}       |
|                               | Median           | {:,.1f}       |
|                               | Min              | {:,.0f}       |
|                               | Max              | {:,.0f}       |
| **Visit Type Distribution** | Emergency        | {:.1f}%       |
|                               | Regular          | {:.1f}%       |
""".format(desc_encounter['mean'], desc_encounter['std'], desc_encounter['50%'], desc_encounter['min'], desc_encounter['max'], emergency_pct, regular_pct)
    md_table += "| **Top 5 Financial Classes** |                  |               |\n"
    md_table += fin_class_lines
    # Сохраняем Markdown в файл
    with open(desc_stat_filename, "w", encoding="utf-8") as f: f.write(md_table)
    print(f"✅ Descriptive statistics table saved to file: {desc_stat_filename}")
    print("\n" + md_table)
else:
    print("⚠️ Raw data (df_raw_data) not available. Skipping Task 3.")
    with open(desc_stat_filename, "w", encoding="utf-8") as f: f.write("Raw data was not available to generate descriptive statistics.")

print("\n🎉 --- Генерация Данных для Слайда 6 завершена! ---")

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col # Для создания красивых таблиц
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО АНАЛИЗА ДЛЯ АКАДЕМИЧЕСКОЙ ПРЕЗЕНТАЦИИ ---")

# --- 1. Проверка и Загрузка Данных ---
# **ВАЖНО**: Этот скрипт предполагает, что 'final_data' — это DataFrame 
# УРОВНЯ ВИЗИТОВ (до агрегации), как в вашем исходном ноутбуке.
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' (на уровне визитов) не найден.")
    print("   Пожалуйста, убедитесь, что 'final_data' (filtered_top_icd_data) загружен.")
    # Пытаемся загрузить его, если он был сохранен
    try:
        print("   ... Пытаюсь загрузить 'filtered_data_no_other_icd.csv' как 'final_data' ...")
        final_data = pd.read_csv(
            'filtered_data_no_other_icd.csv', # Этот файл создается в вашем коде
            parse_dates=["Date"],
            dtype={'Filtered_Patient_ZipCode': str, 'Zip': str}
        )
        print("   ✅ Файл 'filtered_data_no_other_icd.csv' успешно загружен.")
    except FileNotFoundError:
         print("   ❌ ОШИБКА: 'filtered_data_no_other_icd.csv' не найден. Не могу продолжить.")
         raise
else:
    print("✅ 'final_data' (на уровне визитов) найден в памяти.")


# Загрузка остальных файлов
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (стандартный код очистки) ...
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True, how='left')
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
print("✅ Данные загружены и очищены.")

# --- 3. Идентификация "Чистых" Групп Клиник ---
date_cutoff = pd.to_datetime('2023-01-01')
confounder_zip = '61364'; confounder_date = pd.to_datetime('2021-09-01')

# New PC
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
df_new_PC_clinics_CLEAN = df_new_clinics_all[
    (df_new_clinics_all.apply(is_pc_urgent, axis=1)) &
    (~((df_new_clinics_all['clinic_zip'] == confounder_zip) & (df_new_clinics_all['event_date'] == confounder_date))) &
    (df_new_clinics_all['event_date'] < date_cutoff)
].copy()

# Acquired PC
df_acquired_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'Acquired'].copy()
pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
def is_acq_pc(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
    if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
df_acquired_pc_CLEAN = df_acquired_clinics_all[
    (df_acquired_clinics_all.apply(is_acq_pc, axis=1)) &
    (df_acquired_clinics_all['event_date'] < date_cutoff)
].copy()
print(f"Идентифицировано: {len(df_new_PC_clinics_CLEAN)} 'чистых' New PC, {len(df_acquired_pc_CLEAN)} 'чистых' Acquired PC.")

# --- 4. Расчет Карт Лечения (Метод Радиуса 10 км) ---
print("\n--- Расчет Карт Лечения (Метод Радиуса 10 км) ---")
TREATMENT_RADIUS_KM = 10; treatment_dates = {}
patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет Дат Лечения"):
    patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
    earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
    for _, clinic_row in df_new_PC_clinics_CLEAN.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
    if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
    for _, clinic_row in df_acquired_pc_CLEAN.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
    if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
    treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

# --- 5. Шаг 1: Агрегация данных и Создание Контролей ---
print("\n--- Шаг 1: Агрегация данных и создание Y-переменных и Контролей ---")
df_panel = final_data.copy()
df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')

# Определяем Топ-4 FinClass (исключаем NaN/Other)
fin_class_counts = df_panel['FinancialClassNM'].value_counts()
top_4_fin_classes = fin_class_counts.head(4).index.tolist()
# Убедимся, что Self-Pay в списке, если его нет в Топ-4
if 'Self-Pay' not in top_4_fin_classes and 'Self-Pay' in fin_class_counts.index:
    top_4_fin_classes = top_4_fin_classes[:3] + ['Self-Pay'] # Заменяем 4-й
print(f"Контроли FinClass (Топ-4): {top_4_fin_classes}")

# Создаем колонки ДО агрегации
df_panel['ER_Visits'] = df_panel['EncounterCount'] * df_panel['Is_Emergency']
df_panel['Total_Visits'] = df_panel['EncounterCount'] # Общие визиты
for fin_class in top_4_fin_classes:
    df_panel[f'Visits_{fin_class}'] = df_panel['EncounterCount'] * (df_panel['FinancialClassNM'] == fin_class)

# Агрегация
grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
agg_dict = {'ER_Visits': 'sum', 'Total_Visits': 'sum'}
agg_dict.update({f'Visits_{fc}': 'sum' for fc in top_4_fin_classes})
print("... Выполняю агрегацию ...")
df_agg = df_panel.groupby(grouping_cols).agg(agg_dict).reset_index()

# Создаем Y-переменную
df_agg['Y_ER_Total'] = np.log1p(df_agg['ER_Visits'])
# Создаем Time-Varying Controls (Shares)
df_agg['Total_Visits_Safe'] = df_agg['Total_Visits'] + 1e-6 # Для безопасного деления
for fin_class in top_4_fin_classes:
    df_agg[f'Share_{fin_class}'] = df_agg[f'Visits_{fin_class}'] / df_agg['Total_Visits_Safe']
print("✅ Y-переменная и Контроли (Shares) созданы.")

# Присоединяем Карты Лечения
df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

# --- 6. Шаг 2: Модель 1 - "Простой" DiD (для Таблицы) ---
print("\n" + "="*70); print("--- ШАГ 2: Модель 1 - 'Простой' DiD (Таблица) ---"); print("="*70)

# Создаем переменные Post_Treatment
df_analysis['Post_Treatment_NEW'] = ((df_analysis['Date'] >= df_analysis['treatment_date_NEW_PC'])).astype(int)
df_analysis['Post_Treatment_ACQUIRED'] = ((df_analysis['Date'] >= df_analysis['treatment_date_ACQUIRED_PC'])).astype(int)

OUTCOME = "Y_ER_Total"
# Составляем строку контролей
CONTROLS = ' + '.join([f'Share_{fc}' for fc in top_4_fin_classes])
FIXED_EFFECTS = "C(Zip) + C(Year_Month)" # Используем Year_Month для FE времени
models_did = {}

# Модель 1А (New PC)
formula_did_new = f"{OUTCOME} ~ Post_Treatment_NEW + {CONTROLS} + {FIXED_EFFECTS}"
print(f"\nЗапуск 'Простого' DiD для 'New PC' (N={len(df_analysis)})...")
try:
    # Используем smf.ols с FE. Это корректно для Staggered DiD с Post_Treatment, если FE поглощают все
    # Мы кластеризуем ошибки по Zip
    model_new = smf.ols(formula_did_new, data=df_analysis)
    models_did['New_PC'] = model_new.fit(cov_type='cluster', cov_kwds={'groups': df_analysis['Zip']})
    print("  ✅ Модель 'New PC' (Простой DiD) рассчитана.")
except Exception as e:
    print(f"  ❌ ОШИБКА Модели 'New PC': {e}")

# Модель 1Б (Acquired PC)
formula_did_acq = f"{OUTCOME} ~ Post_Treatment_ACQUIRED + {CONTROLS} + {FIXED_EFFECTS}"
print(f"\nЗапуск 'Простого' DiD для 'Acquired PC' (N={len(df_analysis)})...")
try:
    model_acq = smf.ols(formula_did_acq, data=df_analysis)
    models_did['Acquired_PC'] = model_acq.fit(cov_type='cluster', cov_kwds={'groups': df_analysis['Zip']})
    print("  ✅ Модель 'Acquired PC' (Простой DiD) рассчитана.")
except Exception as e:
    print(f"  ❌ ОШИБКА Модели 'Acquired PC': {e}")

# Вывод Таблицы
if models_did:
    table_filename = "slide6_simple_did_table.md"
    try:
        # Собираем только нужные модели
        model_list = [m for m in models_did.values() if m is not None]
        # Оставляем только интересующие нас коэффициенты
        regressors_to_show = ['Post_Treatment_NEW', 'Post_Treatment_ACQUIRED'] + [f'Share_{fc}' for fc in top_4_fin_classes]
        
        info_dict = {
            'R-squared': lambda x: f"{x.rsquared_adj:.3f}", # adj. R-squared
            'N': lambda x: f"{int(x.nobs):,}"
        }
        
        results_table = summary_col(
            model_list,
            model_names=['(1) New PC', '(2) Acquired PC'],
            stars=True,
            float_format="%.4f",
            regressor_order=regressors_to_show,
            drop_omitted=True, # Убираем FE из таблицы
            info_dict=info_dict
        )
        
        print("\n--- Таблица 2: Средний Эффект (Простой DiD) ---")
        print(results_table)
        
        # Сохраняем в Markdown
        with open(table_filename, "w", encoding="utf-8") as f:
            f.write(f"## Таблица 2: Средний Эффект 'Post_Treatment' на {OUTCOME}\n\n")
            f.write(results_table.as_markdown())
        print(f"\n✅ Таблица 2 сохранена в: {table_filename}")

    except Exception as e_table:
        print(f"❌ ОШИБКА при создании таблицы: {e_table}")


# =========================================================================
# --- Шаг 3: Модель 2 - Event Study (для Графика) ---
# =========================================================================
print("\n\n" + "="*70); print("--- ШАГ 3: Модель 2 - Event Study (Графики) ---"); print("="*70)

# (Включаем функции для запуска и отрисовки)
def run_event_study_model(formula, data, cluster_var='Zip', time_var='time_to_event'):
    model_result = None; outcome_col = formula.split('~')[0].strip()
    required_cols_run = [outcome_col, time_var, cluster_var, 'Year_Month'] + [f'Share_{fc}' for fc in top_4_fin_classes]
    if not all(col in data.columns for col in required_cols_run):
         print(f"  ⚠️ Пропуск: Отсутствуют необходимые колонки ({[col for col in required_cols_run if col not in data.columns]}) для '{outcome_col}'.")
         return None
    if not data.empty and data[outcome_col].notna().any() and data[time_var].nunique() >= 2:
        try:
            cols_exist = [col for col in required_cols_run if col in data.columns]
            data_clean = data[cols_exist].dropna().reset_index(drop=True)
            if data_clean.empty or data_clean[time_var].nunique() < 2: return None
            print(f"... Запуск Event Study с C({cluster_var}) для {outcome_col} (N={len(data_clean)})...")
            model = smf.ols(formula, data=data_clean)
            model_result = model.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]})
            print(f"  ✅ Модель с C({cluster_var}) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА с C({cluster_var}): {e}")
            # ... (Откат к модели без C(Zip)) ...
            try:
                 formula_no_zip = formula.replace(f"+ C({cluster_var})", "")
                 print(f"     ... Попытка без C({cluster_var}) ...")
                 cols_nozip = [col for col in cols_exist if col != cluster_var]
                 data_clean_nozip = data[cols_nozip].dropna().reset_index(drop=True)
                 cluster_groups_nozip = data.loc[data_clean_nozip.index, cluster_var]
                 if data_clean_nozip.empty or data_clean_nozip[time_var].nunique() < 2: return None
                 model_no_zip = smf.ols(formula_no_zip, data=data_clean_nozip)
                 model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': cluster_groups_nozip})
                 print(f"        ✅ Модель без C({cluster_var}) рассчитана.")
            except Exception as e2: print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}"); model_result = None
    else: print(f"  ⚠️ Нет данных или вариации для запуска модели '{outcome_col}'.")
    return model_result

def extract_event_study_effects_v3(model_result, base_period, time_var_name):
    # ... (код без изменений) ...
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\](?!:)"; # Добавил (?!:) чтобы исключить взаимодействия
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                month = int(match.group(1));
                if month == base_period: continue
                results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
    return df.sort_values('Relative_Month').reset_index(drop=True)

def plot_event_study(event_data, time_col, outcome_name, title, color, event_window, base_period, time_unit="Месяцы"):
     # ... (код без изменений) ...
     if not event_data.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax.errorbar(x=event_data[time_col], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Effect on {outcome_name}', color=color)
        else: ax.plot(event_data[time_col], event_data['Effect'], marker='o', linestyle='-', label=f'Effect on {outcome_name} (No CI)', color=color)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
        ax.set_title(title, fontsize=14); ax.set_xlabel(f"{time_unit} Relative to Event"); ax.set_ylabel(f"Effect on {outcome_name}")
        ax.set_xticks(range(-event_window, event_window + 1, 2)); ax.legend(); ax.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
     else: print(f"Не удалось извлечь/построить данные для: {title}")

# --- Модель 2А (Event Study New PC) ---
print("\n--- Запуск Модели 2А (Event Study 'New PC') ---")
EVENT_WINDOW = 12
BASE_PERIOD = -1
TIME_VAR_NEW = 'time_to_event_NEW_PC'
df_analysis['event_month_new'] = pd.to_datetime(df_analysis['treatment_date_NEW_PC']).dt.to_period('M')
df_analysis['current_month'] = pd.to_datetime(df_analysis['Date']).dt.to_period('M')
df_analysis[TIME_VAR_NEW] = (df_analysis['current_month'] - df_analysis['event_month_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
df_es_new_filtered = df_analysis[df_analysis[TIME_VAR_NEW].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
df_es_new_filtered[TIME_VAR_NEW] = df_es_new_filtered[TIME_VAR_NEW].astype(int)

formula_es_new = f"{OUTCOME} ~ C({TIME_VAR_NEW}, Treatment(reference={BASE_PERIOD})) + {CONTROLS} + C(Zip) + C(Year_Month)"
model_es_new_result = run_event_study_model(formula_es_new, df_es_new_filtered, time_var=TIME_VAR_NEW)
event_data_es_new = extract_event_study_effects_v3(model_es_new_result, BASE_PERIOD, TIME_VAR_NEW)
plot_event_study(event_data_es_new, 'Relative_Month', OUTCOME, "Graph 1: Event Study - Effect of 'New PC' (w/ Controls)", 'blue', EVENT_WINDOW, BASE_PERIOD)


# --- Модель 2Б (Event Study Acquired PC) ---
print("\n--- Запуск Модели 2Б (Event Study 'Acquired PC') ---")
TIME_VAR_ACQ = 'time_to_event_ACQUIRED_PC'
df_analysis['event_month_acq'] = pd.to_datetime(df_analysis['treatment_date_ACQUIRED_PC']).dt.to_period('M')
df_analysis['current_month'] = pd.to_datetime(df_analysis['Date']).dt.to_period('M')
df_analysis[TIME_VAR_ACQ] = (df_analysis['current_month'] - df_analysis['event_month_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)
df_es_acq_filtered = df_analysis[df_analysis[TIME_VAR_ACQ].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
df_es_acq_filtered[TIME_VAR_ACQ] = df_es_acq_filtered[TIME_VAR_ACQ].astype(int)

formula_es_acq = f"{OUTCOME} ~ C({TIME_VAR_ACQ}, Treatment(reference={BASE_PERIOD})) + {CONTROLS} + C(Zip) + C(Year_Month)"
model_es_acq_result = run_event_study_model(formula_es_acq, df_es_acq_filtered, time_var=TIME_VAR_ACQ)
event_data_es_acq = extract_event_study_effects_v3(model_es_acq_result, BASE_PERIOD, TIME_VAR_ACQ)
plot_event_study(event_data_es_acq, 'Relative_Month', OUTCOME, "Graph 2: Event Study - Effect of 'Acquired PC' (w/ Controls)", 'red', EVENT_WINDOW, BASE_PERIOD)


print("\n🎉 --- Анализ для Слайдов (Таблица + Графики) завершен! ---")



In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col # Для создания красивых таблиц
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО АНАЛИЗА ДЛЯ АКАДЕМИЧЕСКОЙ ПРЕЗЕНТАЦИИ ---")

# --- 1. Проверка и Загрузка Данных ---
# **ВАЖНО**: Этот скрипт предполагает, что 'final_data' — это DataFrame 
# УРОВНЯ ВИЗИТОВ (до агрегации), как в вашем исходном ноутбуке.
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' (на уровне визитов) не найден.")
    print("   Пожалуйста, убедитесь, что 'final_data' (filtered_top_icd_data) загружен.")
    # Пытаемся загрузить его, если он был сохранен
    try:
        print("   ... Пытаюсь загрузить 'filtered_data_no_other_icd.csv' как 'final_data' ...")
        final_data = pd.read_csv(
            'filtered_data_no_other_icd.csv', # Этот файл создается в вашем коде
            parse_dates=["Date"],
            dtype={'Filtered_Patient_ZipCode': str, 'Zip': str}
        )
        print("   ✅ Файл 'filtered_data_no_other_icd.csv' успешно загружен.")
    except FileNotFoundError:
         print("   ❌ ОШИБКА: 'filtered_data_no_other_icd.csv' не найден. Не могу продолжить.")
         raise
else:
    print("✅ 'final_data' (на уровне визитов) найден в памяти.")


# Загрузка остальных файлов
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (стандартный код очистки) ...
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True, how='left')
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
print("✅ Данные загружены и очищены.")

# --- 3. Идентификация "Чистых" Групп Клиник ---
date_cutoff = pd.to_datetime('2023-01-01')
confounder_zip = '61364'; confounder_date = pd.to_datetime('2021-09-01')

# New PC
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
df_new_PC_clinics_CLEAN = df_new_clinics_all[
    (df_new_clinics_all.apply(is_pc_urgent, axis=1)) &
    (~((df_new_clinics_all['clinic_zip'] == confounder_zip) & (df_new_clinics_all['event_date'] == confounder_date))) &
    (df_new_clinics_all['event_date'] < date_cutoff)
].copy()

# Acquired PC
df_acquired_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'Acquired'].copy()
pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
def is_acq_pc(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
    if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
df_acquired_pc_CLEAN = df_acquired_clinics_all[
    (df_acquired_clinics_all.apply(is_acq_pc, axis=1)) &
    (df_acquired_clinics_all['event_date'] < date_cutoff)
].copy()
print(f"Идентифицировано: {len(df_new_PC_clinics_CLEAN)} 'чистых' New PC, {len(df_acquired_pc_CLEAN)} 'чистых' Acquired PC.")

# --- 4. Расчет Карт Лечения (Метод Радиуса 10 км) ---
print("\n--- Расчет Карт Лечения (Метод Радиуса 10 км) ---")
TREATMENT_RADIUS_KM = 10; treatment_dates = {}
patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет Дат Лечения"):
    patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
    earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
    for _, clinic_row in df_new_PC_clinics_CLEAN.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
    if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
    for _, clinic_row in df_acquired_pc_CLEAN.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
    if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
    treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

# --- 5. Шаг 1: Агрегация данных и Создание Контролей ---
print("\n--- Шаг 1: Агрегация данных и создание Y-переменных и Контролей ---")
df_panel = final_data.copy()
df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')

# Определяем Топ-4 FinClass (исключаем NaN/Other)
fin_class_counts = df_panel['FinancialClassNM'].value_counts()
top_4_fin_classes = fin_class_counts.head(4).index.tolist()
if 'Self-Pay' not in top_4_fin_classes and 'Self-Pay' in fin_class_counts.index:
    top_4_fin_classes = top_4_fin_classes[:3] + ['Self-Pay'] # Заменяем 4-й

# --- ИСПРАВЛЕНИЕ 1: Создаем "чистые" имена для формул ---
# Создаем словарь: {'BC/BS': 'BC_BS', 'Medicare': 'Medicare', ...}
fin_class_map = {fc: re.sub(r'[^A-Za-z0-9_]+', '_', fc) for fc in top_4_fin_classes}
print(f"Контроли FinClass (Оригинал): {top_4_fin_classes}")
print(f"Контроли FinClass (Очищенные): {list(fin_class_map.values())}")

# Создаем колонки ДО агрегации, используя ОРИГИНАЛЬНЫЕ имена для фильтрации
# и ЧИСТЫЕ имена для новых колонок
df_panel['ER_Visits'] = df_panel['EncounterCount'] * df_panel['Is_Emergency']
df_panel['Total_Visits'] = df_panel['EncounterCount'] # Общие визиты
for fc_orig, fc_clean in fin_class_map.items():
    df_panel[f'Visits_{fc_clean}'] = df_panel['EncounterCount'] * (df_panel['FinancialClassNM'] == fc_orig)

# Агрегация, используя ЧИСТЫЕ имена
grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
agg_dict = {'ER_Visits': 'sum', 'Total_Visits': 'sum'}
agg_dict.update({f'Visits_{fc_clean}': 'sum' for fc_clean in fin_class_map.values()})
print("... Выполняю агрегацию ...")
df_agg = df_panel.groupby(grouping_cols).agg(agg_dict).reset_index()

# Создаем Y-переменную
df_agg['Y_ER_Total'] = np.log1p(df_agg['ER_Visits'])
# Создаем Time-Varying Controls (Shares), используя ЧИСТЫЕ имена
df_agg['Total_Visits_Safe'] = df_agg['Total_Visits'] + 1e-6
for fc_clean in fin_class_map.values():
    df_agg[f'Share_{fc_clean}'] = df_agg[f'Visits_{fc_clean}'] / df_agg['Total_Visits_Safe']
print("✅ Y-переменная и Контроли (Shares) созданы.")

# Присоединяем Карты Лечения
df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

# --- 6. Шаг 2: Модель 1 - "Простой" DiD (для Таблицы) ---
print("\n" + "="*70); print("--- ШАГ 2: Модель 1 - 'Простой' DiD (Таблица) ---"); print("="*70)

# Создаем переменные Post_Treatment
df_analysis['Post_Treatment_NEW'] = ((df_analysis['Date'] >= df_analysis['treatment_date_NEW_PC'])).astype(int)
df_analysis['Post_Treatment_ACQUIRED'] = ((df_analysis['Date'] >= df_analysis['treatment_date_ACQUIRED_PC'])).astype(int)

OUTCOME = "Y_ER_Total"
# --- ИСПРАВЛЕНИЕ 1: Используем ЧИСТЫЕ имена в строке контролей ---
CONTROLS = ' + '.join([f'Share_{fc_clean}' for fc_clean in fin_class_map.values()])
FIXED_EFFECTS = "C(Zip) + C(Year_Month)"
models_did = {}

# Модель 1А (New PC)
formula_did_new = f"{OUTCOME} ~ Post_Treatment_NEW + {CONTROLS} + {FIXED_EFFECTS}"
print(f"\nЗапуск 'Простого' DiD для 'New PC' (N={len(df_analysis)})...")
print(f"Формула: {formula_did_new}") # Печатаем формулу для проверки
try:
    model_new = smf.ols(formula_did_new, data=df_analysis.dropna(subset=['Post_Treatment_NEW'] + list(fin_class_map.values()))) # Добавили dropna
    models_did['New_PC'] = model_new.fit(cov_type='cluster', cov_kwds={'groups': df_analysis.dropna(subset=['Post_Treatment_NEW'] + list(fin_class_map.values()))['Zip']})
    print("  ✅ Модель 'New PC' (Простой DiD) рассчитана.")
except Exception as e:
    print(f"  ❌ ОШИБКА Модели 'New PC': {e}")

# Модель 1Б (Acquired PC)
formula_did_acq = f"{OUTCOME} ~ Post_Treatment_ACQUIRED + {CONTROLS} + {FIXED_EFFECTS}"
print(f"\nЗапуск 'Простого' DiD для 'Acquired PC' (N={len(df_analysis)})...")
print(f"Формула: {formula_did_acq}") # Печатаем формулу для проверки
try:
    model_acq = smf.ols(formula_did_acq, data=df_analysis.dropna(subset=['Post_Treatment_ACQUIRED'] + list(fin_class_map.values())))
    models_did['Acquired_PC'] = model_acq.fit(cov_type='cluster', cov_kwds={'groups': df_analysis.dropna(subset=['Post_Treatment_ACQUIRED'] + list(fin_class_map.values()))['Zip']})
    print("  ✅ Модель 'Acquired PC' (Простой DiD) рассчитана.")
except Exception as e:
    print(f"  ❌ ОШИБКА Модели 'Acquired PC': {e}")

# Вывод Таблицы
if models_did:
    table_filename = "slide6_simple_did_table.md"
    try:
        model_list = [m for m in models_did.values() if m is not None]
        # --- ИСПРАВЛЕНИЕ 1: Используем ЧИСТЫЕ имена ---
        regressors_to_show = ['Post_Treatment_NEW', 'Post_Treatment_ACQUIRED'] + [f'Share_{fc_clean}' for fc_clean in fin_class_map.values()]
        
        info_dict = {'R-squared': lambda x: f"{x.rsquared_adj:.3f}", 'N': lambda x: f"{int(x.nobs):,}"}
        
        results_table = summary_col(
            model_list, model_names=['(1) New PC', '(2) Acquired PC'],
            stars=True, float_format="%.4f",
            regressor_order=regressors_to_show,
            drop_omitted=True, info_dict=info_dict
        )
        
        print("\n--- Таблица 2: Средний Эффект (Простой DiD) ---")
        print(results_table)
        
        with open(table_filename, "w", encoding="utf-8") as f:
            f.write(f"## Таблица 2: Средний Эффект 'Post_Treatment' на {OUTCOME}\n\n"); f.write(results_table.as_markdown())
        print(f"\n✅ Таблица 2 сохранена в: {table_filename}")

    except Exception as e_table:
        print(f"❌ ОШИБКА при создании таблицы: {e_table}")


# =========================================================================
# --- Шаг 3: Модель 2 - Event Study (для Графика) ---
# =========================================================================
print("\n\n" + "="*70); print("--- ШАГ 3: Модель 2 - Event Study (Графики) ---"); print("="*70)

# (Включаем функции для запуска и отрисовки)
def run_event_study_model(formula, data, cluster_var='Zip', time_var='time_to_event'):
    model_result = None; outcome_col = formula.split('~')[0].strip()
    # --- ИСПРАВЛЕНИЕ 1: Используем ЧИСТЫЕ имена ---
    required_cols_run = [outcome_col, time_var, cluster_var, 'Year_Month'] + [f'Share_{fc_clean}' for fc_clean in fin_class_map.values()]
    if not all(col in data.columns for col in required_cols_run):
         print(f"  ⚠️ Пропуск: Отсутствуют необходимые колонки ({[col for col in required_cols_run if col not in data.columns]}) для '{outcome_col}'.")
         return None
    
    if not data.empty and data[outcome_col].notna().any() and data[time_var].nunique() >= 2:
        try:
            cols_exist = [col for col in required_cols_run if col in data.columns]
            data_clean = data[cols_exist].dropna().reset_index(drop=True)
            if data_clean.empty or data_clean[time_var].nunique() < 2: return None
            print(f"... Запуск Event Study с C({cluster_var}) для {outcome_col} (N={len(data_clean)})...")
            model = smf.ols(formula, data=data_clean)
            model_result = model.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]})
            print(f"  ✅ Модель с C({cluster_var}) рассчитана.")
        except Exception as e:
            print(f"  ❌ ОШИБКА с C({cluster_var}): {e}")
            # --- ИСПРАВЛЕНИЕ 2: Исправлена логика запасной модели ---
            try:
                 formula_no_zip = formula.replace(f" + C({cluster_var})", "") # Убираем FE Zip
                 print(f"     ... Попытка без C({cluster_var}) ...")
                 # data_clean УЖЕ создан и содержит Zip
                 if data_clean.empty or data_clean[time_var].nunique() < 2: return None
                 
                 model_no_zip = smf.ols(formula_no_zip, data=data_clean) # Используем data_clean
                 # Кластеризуем по Zip, даже если он не в FE
                 model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]}) 
                 print(f"        ✅ Модель без C({cluster_var}) рассчитана.")
            except Exception as e2: 
                 print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}"); model_result = None
    else: print(f"  ⚠️ Нет данных или вариации для запуска модели '{outcome_col}'.")
    return model_result

def extract_event_study_effects_v3(model_result, base_period, time_var_name):
    # ... (код без изменений) ...
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\](?!:)";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                month = int(match.group(1));
                if month == base_period: continue
                results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
    return df.sort_values('Relative_Month').reset_index(drop=True)

def plot_event_study(event_data, time_col, outcome_name, title, color, event_window, base_period, time_unit="Месяцы"):
     # ... (код без изменений) ...
     if not event_data.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax.errorbar(x=event_data[time_col], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Effect on {outcome_name}', color=color)
        else: ax.plot(event_data[time_col], event_data['Effect'], marker='o', linestyle='-', label=f'Effect on {outcome_name} (No CI)', color=color)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
        ax.set_title(title, fontsize=14); ax.set_xlabel(f"{time_unit} Relative to Event"); ax.set_ylabel(f"Effect on {outcome_name}")
        ax.set_xticks(range(-event_window, event_window + 1, 2)); ax.legend(); ax.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
     else: print(f"Не удалось извлечь/построить данные для: {title}")

# --- Модель 2А (Event Study New PC) ---
print("\n--- Запуск Модели 2А (Event Study 'New PC') ---")
EVENT_WINDOW = 12
BASE_PERIOD = -1
TIME_VAR_NEW = 'time_to_event_NEW_PC'
df_analysis['event_month_new'] = pd.to_datetime(df_analysis['treatment_date_NEW_PC']).dt.to_period('M')
df_analysis['current_month'] = pd.to_datetime(df_analysis['Date']).dt.to_period('M')
df_analysis[TIME_VAR_NEW] = (df_analysis['current_month'] - df_analysis['event_month_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
df_es_new_filtered = df_analysis[df_analysis[TIME_VAR_NEW].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
df_es_new_filtered[TIME_VAR_NEW] = df_es_new_filtered[TIME_VAR_NEW].astype(int)

# --- ИСПРАВЛЕНИЕ 1: Используем ЧИСТУЮ строку контролей ---
formula_es_new = f"{OUTCOME} ~ C({TIME_VAR_NEW}, Treatment(reference={BASE_PERIOD})) + {CONTROLS} + C(Zip) + C(Year_Month)"
model_es_new_result = run_event_study_model(formula_es_new, df_es_new_filtered, time_var=TIME_VAR_NEW)
event_data_es_new = extract_event_study_effects_v3(model_es_new_result, BASE_PERIOD, TIME_VAR_NEW)
plot_event_study(event_data_es_new, 'Relative_Month', OUTCOME, "Graph 1: Event Study - Effect of 'New PC' (w/ Controls)", 'blue', EVENT_WINDOW, BASE_PERIOD)


# --- Модель 2Б (Event Study Acquired PC) ---
print("\n--- Запуск Модели 2Б (Event Study 'Acquired PC') ---")
TIME_VAR_ACQ = 'time_to_event_ACQUIRED_PC'
df_analysis['event_month_acq'] = pd.to_datetime(df_analysis['treatment_date_ACQUIRED_PC']).dt.to_period('M')
df_analysis['current_month'] = pd.to_datetime(df_analysis['Date']).dt.to_period('M')
df_analysis[TIME_VAR_ACQ] = (df_analysis['current_month'] - df_analysis['event_month_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)
df_es_acq_filtered = df_analysis[df_analysis[TIME_VAR_ACQ].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
df_es_acq_filtered[TIME_VAR_ACQ] = df_es_acq_filtered[TIME_VAR_ACQ].astype(int)

# --- ИСПРАВЛЕНИЕ 1: Используем ЧИСТУЮ строку контролей ---
formula_es_acq = f"{OUTCOME} ~ C({TIME_VAR_ACQ}, Treatment(reference={BASE_PERIOD})) + {CONTROLS} + C(Zip) + C(Year_Month)"
model_es_acq_result = run_event_study_model(formula_es_acq, df_es_acq_filtered, time_var=TIME_VAR_ACQ)
event_data_es_acq = extract_event_study_effects_v3(model_es_acq_result, BASE_PERIOD, TIME_VAR_ACQ)
plot_event_study(event_data_es_acq, 'Relative_Month', OUTCOME, "Graph 2: Event Study - Effect of 'Acquired PC' (w/ Controls)", 'red', EVENT_WINDOW, BASE_PERIOD)


print("\n🎉 --- Анализ для Слайдов (Таблица + Графики) завершен! ---")

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col # Для создания красивых таблиц
import matplotlib.pyplot as plt
# ... (остальные импорты) ...
import warnings
import re
import os
from tqdm import tqdm # Индикатор прогресса

# --- 0. Настройка ---
# ... (код настройки без изменений) ...
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
pd.options.mode.chained_assignment = None
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7) # Размер графиков по умолчанию

print("--- 🚀 НАЧАЛО АНАЛИЗА ДЛЯ АКАДЕМИЧЕСКОЙ ПРЕЗЕНТАЦИИ (ИСПРАВЛЕНО) ---")

# --- 1. Проверка и Загрузка Данных ---
# ... (код без изменений) ...
if 'final_data' not in locals():
    print("❌ ОШИБКА: DataFrame 'final_data' (на уровне визитов) не найден.")
    print("   Пожалуйста, убедитесь, что 'final_data' (filtered_top_icd_data) загружен.")
    try:
        print("   ... Пытаюсь загрузить 'filtered_data_no_other_icd.csv' как 'final_data' ...")
        final_data = pd.read_csv(
            'filtered_data_no_other_icd.csv', # Этот файл создается в вашем коде
            parse_dates=["Date"],
            dtype={'Filtered_Patient_ZipCode': str, 'Zip': str}
        )
        print("   ✅ Файл 'filtered_data_no_other_icd.csv' успешно загружен.")
    except FileNotFoundError:
         print("   ❌ ОШИБКА: 'filtered_data_no_other_icd.csv' не найден. Не могу продолжить.")
         raise
else:
    print("✅ 'final_data' (на уровне визитов) найден в памяти.")
try:
    df_clinics_info_full = pd.read_csv("OSFResearch_Clinics_mastersheet_v2.csv")
    df_crosswalk = pd.read_csv("Crosswalk_IL.csv")
except FileNotFoundError as e:
    print(f"❌ ERROR: Cannot find file {e.filename}. Stopping.")
    raise

# --- 2. Очистка Данных ---
# ... (код без изменений) ...
final_data = final_data[final_data['Filtered_Patient_ZipCode'] != '49829']
final_data['Filtered_Patient_ZipCode'] = final_data['Filtered_Patient_ZipCode'].astype(str).str.replace(r"\.0$", "", regex=True)
invalid_zips = ["99999", "Other", "0", "00000"]
final_data = final_data[~final_data["Filtered_Patient_ZipCode"].isin(invalid_zips)]
df_crosswalk['zcta5'] = df_crosswalk['zcta5'].astype(str).str.replace(r"\.0$", "", regex=True)
zip_coords = df_crosswalk.set_index('zcta5')[['IntPtLat', 'IntPtLon']].dropna()
zip_coords['IntPtLat'] = pd.to_numeric(zip_coords['IntPtLat'], errors='coerce'); zip_coords['IntPtLon'] = pd.to_numeric(zip_coords['IntPtLon'], errors='coerce')
zip_coords = zip_coords.dropna()
df_clinics_info_full['Zip'] = df_clinics_info_full['Zip'].astype(str).str.replace(r"\.0$", "", regex=True)
df_clinics_info_full = df_clinics_info_full.rename(columns={'New or Acquired': 'clinic_type', 'First Financial Date': 'event_date', 'Zip': 'clinic_zip'})
df_clinics_info_full['event_date'] = pd.to_datetime(df_clinics_info_full['event_date'])
df_clinics_info_full = df_clinics_info_full.merge(zip_coords.rename(columns={'IntPtLat':'clinic_lat', 'IntPtLon':'clinic_lon'}), left_on='clinic_zip', right_index=True, how='left')
df_clinics_info_full = df_clinics_info_full.dropna(subset=['clinic_lat', 'clinic_lon', 'event_date', 'clinic_type'])
print("✅ Данные загружены и очищены.")

# --- 3. Идентификация "Чистых" Групп Клиник ---
# ... (код без изменений) ...
date_cutoff = pd.to_datetime('2023-01-01')
confounder_zip = '61364'; confounder_date = pd.to_datetime('2021-09-01')
df_new_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'New'].copy()
pc_urgent_keywords = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Urgent Care', 'URGO', 'RHC', 'Family Medicine']
def is_pc_urgent(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_urgent_keywords)
df_new_PC_clinics_CLEAN = df_new_clinics_all[
    (df_new_clinics_all.apply(is_pc_urgent, axis=1)) &
    (~((df_new_clinics_all['clinic_zip'] == confounder_zip) & (df_new_clinics_all['event_date'] == confounder_date))) &
    (df_new_clinics_all['event_date'] < date_cutoff)
].copy()
df_acquired_clinics_all = df_clinics_info_full[df_clinics_info_full['clinic_type'] == 'Acquired'].copy()
pc_keywords_acq = ['Primary Care', 'Family Practice', 'FP', 'PromptCare', 'PC', 'Internal Medicine', 'IM', 'RHC']
def is_acq_pc(row):
    facility_name = str(row.get('Facility', '')).lower(); specialty_name = str(row.get('Specialty', '')).lower()
    if 'internal medicine' in facility_name or 'internal medicine' in specialty_name: return True
    if re.search(r'\bim\b', facility_name) or re.search(r'\bim\b', specialty_name): return True
    return any(keyword.lower() in facility_name or keyword.lower() in specialty_name for keyword in pc_keywords_acq if keyword != 'IM')
df_acquired_pc_CLEAN = df_acquired_clinics_all[
    (df_acquired_clinics_all.apply(is_acq_pc, axis=1)) &
    (df_acquired_clinics_all['event_date'] < date_cutoff)
].copy()
print(f"Идентифицировано: {len(df_new_PC_clinics_CLEAN)} 'чистых' New PC, {len(df_acquired_pc_CLEAN)} 'чистых' Acquired PC.")

# --- 4. Расчет Карт Лечения (Метод Радиуса 10 км) ---
# ... (код без изменений) ...
print("\n--- Расчет Карт Лечения (Метод Радиуса 10 км) ---")
TREATMENT_RADIUS_KM = 10; treatment_dates = {}
patient_zips_coords_df = final_data[['Filtered_Patient_ZipCode']].drop_duplicates()
patient_zips_coords_df = patient_zips_coords_df.merge(zip_coords.rename(columns={'IntPtLat':'patient_lat', 'IntPtLon':'patient_lon'}),left_on='Filtered_Patient_ZipCode', right_index=True, how='inner')
for _, patient_row in tqdm(patient_zips_coords_df.iterrows(), total=len(patient_zips_coords_df), desc="Расчет Дат Лечения"):
    patient_zip = patient_row['Filtered_Patient_ZipCode']; patient_coords = (patient_row['patient_lat'], patient_row['patient_lon'])
    earliest_date_new_pc = pd.NaT; earliest_date_acquired_pc = pd.NaT; nearby_new_pc_dates = []; nearby_acquired_pc_dates = []
    for _, clinic_row in df_new_PC_clinics_CLEAN.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_new_pc_dates.append(clinic_row['event_date'])
    if nearby_new_pc_dates: earliest_date_new_pc = min(nearby_new_pc_dates)
    for _, clinic_row in df_acquired_pc_CLEAN.iterrows():
        clinic_coords = (clinic_row['clinic_lat'], clinic_row['clinic_lon'])
        if geodesic(patient_coords, clinic_coords).kilometers <= TREATMENT_RADIUS_KM: nearby_acquired_pc_dates.append(clinic_row['event_date'])
    if nearby_acquired_pc_dates: earliest_date_acquired_pc = min(nearby_acquired_pc_dates)
    treatment_dates[patient_zip] = {'treatment_date_NEW_PC': earliest_date_new_pc, 'treatment_date_ACQUIRED_PC': earliest_date_acquired_pc}
df_treatment_dates = pd.DataFrame.from_dict(treatment_dates, orient='index')
print(f"\n✅ Даты первого события (New PC / Acquired PC) рассчитаны.")

# --- 5. Шаг 1: Агрегация данных и Создание Контролей ---
print("\n--- Шаг 1: Агрегация данных и создание Y-переменных и Контролей ---")
df_panel = final_data.copy()
df_panel['Is_Emergency'] = (df_panel['VisitType'] == 'Emergency').astype(int)
df_panel['Year_Month'] = df_panel['Date'].dt.strftime('%Y-%m')
fin_class_counts = df_panel['FinancialClassNM'].value_counts()
top_4_fin_classes = fin_class_counts.head(4).index.tolist()
if 'Self-Pay' not in top_4_fin_classes and 'Self-Pay' in fin_class_counts.index:
    top_4_fin_classes = top_4_fin_classes[:3] + ['Self-Pay']
fin_class_map = {fc: re.sub(r'[^A-Za-z0-9_]+', '_', fc) for fc in top_4_fin_classes}
print(f"Контроли FinClass (Оригинал): {top_4_fin_classes}")
print(f"Контроли FinClass (Очищенные): {list(fin_class_map.values())}")
df_panel['ER_Visits'] = df_panel['EncounterCount'] * df_panel['Is_Emergency']
df_panel['Total_Visits'] = df_panel['EncounterCount']
for fc_orig, fc_clean in fin_class_map.items():
    df_panel[f'Visits_{fc_clean}'] = df_panel['EncounterCount'] * (df_panel['FinancialClassNM'] == fc_orig)
grouping_cols = ['Filtered_Patient_ZipCode', 'Year_Month', 'Year', 'Month']
agg_dict = {'ER_Visits': 'sum', 'Total_Visits': 'sum'}
agg_dict.update({f'Visits_{fc_clean}': 'sum' for fc_clean in fin_class_map.values()})
print("... Выполняю агрегацию ...")
df_agg = df_panel.groupby(grouping_cols).agg(agg_dict).reset_index()
df_agg['Y_ER_Total'] = np.log1p(df_agg['ER_Visits'])
df_agg['Total_Visits_Safe'] = df_agg['Total_Visits'] + 1e-6
for fc_clean in fin_class_map.values():
    df_agg[f'Share_{fc_clean}'] = df_agg[f'Visits_{fc_clean}'] / df_agg['Total_Visits_Safe']
print("✅ Y-переменная и Контроли (Shares) созданы.")
df_analysis = df_agg.merge(df_treatment_dates, left_on='Filtered_Patient_ZipCode', right_index=True, how='left')
df_analysis['Date'] = pd.to_datetime(df_analysis['Year_Month'] + '-01')
df_analysis = df_analysis.rename(columns={'Filtered_Patient_ZipCode': 'Zip'})

# --- 6. Шаг 2: Модель 1 - "Простой" DiD (для Таблицы) ---
print("\n" + "="*70); print("--- ШАГ 2: Модель 1 - 'Простой' DiD (Таблица) ---"); print("="*70)
df_analysis['Post_Treatment_NEW'] = ((df_analysis['Date'] >= df_analysis['treatment_date_NEW_PC'])).astype(int)
df_analysis['Post_Treatment_ACQUIRED'] = ((df_analysis['Date'] >= df_analysis['treatment_date_ACQUIRED_PC'])).astype(int)
OUTCOME = "Y_ER_Total"
CONTROLS = ' + '.join([f'Share_{fc_clean}' for fc_clean in fin_class_map.values()])
FIXED_EFFECTS = "C(Zip) + C(Year_Month)"
models_did = {}

# --- ИСПРАВЛЕНИЕ ОШИБКИ 1: 'dropna' subset теперь использует `Share_...` ---
cols_to_check_new = ['Post_Treatment_NEW'] + [f'Share_{fc}' for fc in fin_class_map.values()]
cols_to_check_acq = ['Post_Treatment_ACQUIRED'] + [f'Share_{fc}' for fc in fin_class_map.values()]
df_did_new_clean = df_analysis.dropna(subset=cols_to_check_new)
df_did_acq_clean = df_analysis.dropna(subset=cols_to_check_acq)
# -----------------------------------------------------------

# Модель 1А (New PC)
formula_did_new = f"{OUTCOME} ~ Post_Treatment_NEW + {CONTROLS} + {FIXED_EFFECTS}"
print(f"\nЗапуск 'Простого' DiD для 'New PC' (N={len(df_did_new_clean)})...")
print(f"Формула: {formula_did_new}")
try:
    model_new = smf.ols(formula_did_new, data=df_did_new_clean)
    models_did['New_PC'] = model_new.fit(cov_type='cluster', cov_kwds={'groups': df_did_new_clean['Zip']})
    print("  ✅ Модель 'New PC' (Простой DiD) рассчитана.")
except Exception as e:
    print(f"  ❌ ОШИБКА Модели 'New PC': {e}")

# Модель 1Б (Acquired PC)
formula_did_acq = f"{OUTCOME} ~ Post_Treatment_ACQUIRED + {CONTROLS} + {FIXED_EFFECTS}"
print(f"\nЗапуск 'Простого' DiD для 'Acquired PC' (N={len(df_did_acq_clean)})...")
print(f"Формула: {formula_did_acq}")
try:
    model_acq = smf.ols(formula_did_acq, data=df_did_acq_clean)
    models_did['Acquired_PC'] = model_acq.fit(cov_type='cluster', cov_kwds={'groups': df_did_acq_clean['Zip']})
    print("  ✅ Модель 'Acquired PC' (Простой DiD) рассчитана.")
except Exception as e:
    print(f"  ❌ ОШИБКА Модели 'Acquired PC': {e}")

# Вывод Таблицы
# ... (код вывода таблицы без изменений) ...
if models_did:
    table_filename = "slide6_simple_did_table.md"
    try:
        model_list = [m for m in models_did.values() if m is not None]
        if model_list: # Только если есть хотя бы одна успешная модель
            regressors_to_show = ['Post_Treatment_NEW', 'Post_Treatment_ACQUIRED'] + [f'Share_{fc_clean}' for fc_clean in fin_class_map.values()]
            info_dict = {'R-squared': lambda x: f"{x.rsquared_adj:.3f}", 'N': lambda x: f"{int(x.nobs):,}"}
            results_table = summary_col(
                model_list, model_names=['(1) New PC', '(2) Acquired PC'],
                stars=True, float_format="%.4f",
                regressor_order=regressors_to_show,
                drop_omitted=True, info_dict=info_dict
            )
            print("\n--- Таблица 2: Средний Эффект (Простой DiD) ---")
            print(results_table)
            with open(table_filename, "w", encoding="utf-8") as f:
                f.write(f"## Таблица 2: Средний Эффект 'Post_Treatment' на {OUTCOME}\n\n"); f.write(results_table.as_markdown())
            print(f"\n✅ Таблица 2 сохранена в: {table_filename}")
        else:
            print("❌ Не удалось создать таблицу: ни одна модель DiD не была успешно рассчитана.")
    except Exception as e_table:
        print(f"❌ ОШИБКА при создании таблицы: {e_table}")


# =========================================================================
# --- Шаг 3: Модель 2 - Event Study (для Графика) ---
# =========================================================================
print("\n\n" + "="*70); print("--- ШАГ 3: Модель 2 - Event Study (Графики) ---"); print("="*70)

# --- ИСПРАВЛЕНИЕ ОШИБКИ 2: Модифицируем run_event_study_model ---
# Убираем C(Zip) из-за нехватки данных (N=204) и исправляем запасной вариант
def run_event_study_model(formula, data, cluster_var='Zip', time_var='time_to_event'):
    model_result = None; outcome_col = formula.split('~')[0].strip()
    # Используем ЧИСТЫЕ имена
    required_cols_run = [outcome_col, time_var, cluster_var, 'Year_Month'] + [f'Share_{fc_clean}' for fc_clean in fin_class_map.values()]
    if not all(col in data.columns for col in required_cols_run):
         print(f"  ⚠️ Пропуск: Отсутствуют необходимые колонки ({[col for col in required_cols_run if col not in data.columns]}) для '{outcome_col}'.")
         return None
    
    if not data.empty and data[outcome_col].notna().any() and data[time_var].nunique() >= 2:
        # --- ИСПРАВЛЕНИЕ: ПРОВЕРКА N ПЕРЕД C(Zip) ---
        if len(data) < 1000: # Эвристика: если данных < 1000, C(Zip), вероятно, не сработает
            print(f"  ⚠️ N={len(data)} слишком мало для C(Zip). Запускаем без C(Zip)...")
            try:
                 formula_no_zip = formula.replace(f" + C({cluster_var})", "") # Убираем FE Zip
                 data_clean = data[required_cols_run].dropna().reset_index(drop=True) # Очищаем
                 if data_clean.empty or data_clean[time_var].nunique() < 2: return None
                 
                 model_no_zip = smf.ols(formula_no_zip, data=data_clean) # Используем data_clean
                 # Кластеризуем по Zip
                 model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]}) 
                 print(f"        ✅ Модель без C({cluster_var}) рассчитана (N={len(data_clean)}).")
            except Exception as e2: 
                 print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}"); model_result = None
        else: # Данных достаточно, пробуем с C(Zip)
            try:
                cols_exist = [col for col in required_cols_run if col in data.columns]
                data_clean = data[cols_exist].dropna().reset_index(drop=True)
                if data_clean.empty or data_clean[time_var].nunique() < 2: return None
                print(f"... Запуск Event Study с C({cluster_var}) для {outcome_col} (N={len(data_clean)})...")
                model = smf.ols(formula, data=data_clean)
                model_result = model.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]})
                print(f"  ✅ Модель с C({cluster_var}) рассчитана.")
            except Exception as e:
                print(f"  ❌ ОШИБКА с C({cluster_var}): {e}")
                # Попытка отката (как в первом случае)
                try:
                     formula_no_zip = formula.replace(f" + C({cluster_var})", "")
                     print(f"     ... Попытка без C({cluster_var}) ...")
                     if data_clean.empty or data_clean[time_var].nunique() < 2: return None
                     model_no_zip = smf.ols(formula_no_zip, data=data_clean)
                     model_result = model_no_zip.fit(cov_type='cluster', cov_kwds={'groups': data_clean[cluster_var]}) 
                     print(f"        ✅ Модель без C({cluster_var}) рассчитана.")
                except Exception as e2: 
                     print(f"        ❌ ОШИБКА без C({cluster_var}): {e2}"); model_result = None
    else: print(f"  ⚠️ Нет данных или вариации для запуска модели '{outcome_col}'.")
    return model_result

# ... (Функции extract и plot без изменений) ...
def extract_event_study_effects_v3(model_result, base_period, time_var_name):
    # ... (код без изменений) ...
    pattern = rf"C\({time_var_name}.*?Treatment\(reference={base_period}\)\)\[T\.(-?\d+)\](?!:)";
    if model_result is None: return pd.DataFrame()
    params = model_result.params.filter(regex=pattern); conf = model_result.conf_int().filter(regex=pattern, axis=0)
    results = {'Relative_Month': [], 'Effect': [], 'Conf_Low': [], 'Conf_High': []}
    results['Relative_Month'].append(base_period); results['Effect'].append(0); results['Conf_Low'].append(0); results['Conf_High'].append(0)
    for idx in params.index:
        try:
            match = re.search(pattern, idx);
            if match:
                month = int(match.group(1));
                if month == base_period: continue
                results['Relative_Month'].append(month); results['Effect'].append(params[idx])
                results['Conf_Low'].append(conf.loc[idx, 0]); results['Conf_High'].append(conf.loc[idx, 1])
        except Exception as e: print(f"  Warning: Could not parse month from '{idx}': {e}")
    df = pd.DataFrame(results).drop_duplicates(subset=['Relative_Month'])
    return df.sort_values('Relative_Month').reset_index(drop=True)

def plot_event_study(event_data, time_col, outcome_name, title, color, event_window, base_period, time_unit="Месяцы"):
     # ... (код без изменений) ...
     if not event_data.empty:
        fig, ax = plt.subplots(figsize=(12, 7))
        errors = [event_data['Effect'] - event_data['Conf_Low'], event_data['Conf_High'] - event_data['Effect']]
        valid_error = ~np.isnan(errors[0]) & ~np.isnan(errors[1])
        if valid_error.all(): ax.errorbar(x=event_data[time_col], y=event_data['Effect'], yerr=errors, fmt='-o', capsize=3, label=f'Effect on {outcome_name}', color=color)
        else: ax.plot(event_data[time_col], event_data['Effect'], marker='o', linestyle='-', label=f'Effect on {outcome_name} (No CI)', color=color)
        ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axhline(0, color='black', linestyle='-', linewidth=0.8); ax.axvline(x=-0.5, color='grey', linestyle=':', label=f'Event (Month 0)')
        ax.set_title(title, fontsize=14); ax.set_xlabel(f"{time_unit} Relative to Event"); ax.set_ylabel(f"Effect on {outcome_name}")
        ax.set_xticks(range(-event_window, event_window + 1, 2)); ax.legend(); ax.grid(True, axis='y', linestyle=':')
        plt.tight_layout(); plt.show()
     else: print(f"Не удалось извлечь/построить данные для: {title}")

# --- Модель 2А (Event Study New PC) ---
print("\n--- Запуск Модели 2А (Event Study 'New PC') ---")
EVENT_WINDOW = 12
BASE_PERIOD = -1
TIME_VAR_NEW = 'time_to_event_NEW_PC'
df_analysis['event_month_new'] = pd.to_datetime(df_analysis['treatment_date_NEW_PC']).dt.to_period('M')
df_analysis['current_month'] = pd.to_datetime(df_analysis['Date']).dt.to_period('M')
df_analysis[TIME_VAR_NEW] = (df_analysis['current_month'] - df_analysis['event_month_new']).apply(lambda x: x.n if pd.notna(x) else np.nan)
df_es_new_filtered = df_analysis[df_analysis[TIME_VAR_NEW].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
df_es_new_filtered[TIME_VAR_NEW] = df_es_new_filtered[TIME_VAR_NEW].astype(int)

# Используем ЧИСТУЮ строку контролей
formula_es_new = f"{OUTCOME} ~ C({TIME_VAR_NEW}, Treatment(reference={BASE_PERIOD})) + {CONTROLS} + C(Zip) + C(Year_Month)"
# --- N=204, поэтому C(Zip) будет удален автоматически ---
model_es_new_result = run_event_study_model(formula_es_new, df_es_new_filtered, time_var=TIME_VAR_NEW)
event_data_es_new = extract_event_study_effects_v3(model_es_new_result, BASE_PERIOD, TIME_VAR_NEW)
plot_event_study(event_data_es_new, 'Relative_Month', OUTCOME, "Graph 1: Event Study - Effect of 'New PC' (w/ Controls, No Zip FE)", 'blue', EVENT_WINDOW, BASE_PERIOD) # Добавил в заголовок


# --- Модель 2Б (Event Study Acquired PC) ---
print("\n--- Запуск Модели 2Б (Event Study 'Acquired PC') ---")
TIME_VAR_ACQ = 'time_to_event_ACQUIRED_PC'
df_analysis['event_month_acq'] = pd.to_datetime(df_analysis['treatment_date_ACQUIRED_PC']).dt.to_period('M')
df_analysis['current_month'] = pd.to_datetime(df_analysis['Date']).dt.to_period('M')
df_analysis[TIME_VAR_ACQ] = (df_analysis['current_month'] - df_analysis['event_month_acq']).apply(lambda x: x.n if pd.notna(x) else np.nan)
df_es_acq_filtered = df_analysis[df_analysis[TIME_VAR_ACQ].between(-EVENT_WINDOW, EVENT_WINDOW)].copy()
df_es_acq_filtered[TIME_VAR_ACQ] = df_es_acq_filtered[TIME_VAR_ACQ].astype(int)

# Используем ЧИСТУЮ строку контролей
formula_es_acq = f"{OUTCOME} ~ C({TIME_VAR_ACQ}, Treatment(reference={BASE_PERIOD})) + {CONTROLS} + C(Zip) + C(Year_Month)"
# --- N=388, поэтому C(Zip) также будет удален ---
model_es_acq_result = run_event_study_model(formula_es_acq, df_es_acq_filtered, time_var=TIME_VAR_ACQ)
event_data_es_acq = extract_event_study_effects_v3(model_es_acq_result, BASE_PERIOD, TIME_VAR_ACQ)
plot_event_study(event_data_es_acq, 'Relative_Month', OUTCOME, "Graph 2: Event Study - Effect of 'Acquired PC' (w/ Controls, No Zip FE)", 'red', EVENT_WINDOW, BASE_PERIOD) # Добавил в заголовок


print("\n🎉 --- Анализ для Слайдов (Таблица + Графики) завершен! ---")

